# MedXIR hybrid retrieval: query variants + dense + PRF + BM25 + rerank

Generated by `scripts/build_hybrid_notebook.py`. Edit the generator, not this file.

RTX Pro 6000, Internet OFF. Inputs (all found by glob under `/kaggle/input`):
kernel outputs `medxir-package-models`, `medxir-embed-corpus-4b-s0/s1/s2`,
`medxir-bm25-vi`, `medxir-package-rerankers`; datasets `medxir-retrieval-inputs`,
`ntklinhfitus/medxir-chunk-corpus-v2`, `ntklinhfitus/medxir-vectors-qwen3-4b-additions`.
Attach one full chunk corpus and one copy of each vector part. Share the private ntklinhfitus
datasets with the account running this notebook before starting it.

Model selection: `EMBEDDING_MODEL = 'qwen4b' | 'qwen8b' | 'octen8b'` for one model.
For weighted RRF across models, set `USE_MODEL_FUSION=True` and edit `MODEL_WEIGHTS`.
Weights are normalized to sum to one; zero disables that model and its inputs. BM25 is added once.
With reranking on, model weights determine which candidates reach the reranker.
Each model is encoded, searched in its own space (2560/4096/4096 dimensions), then fused by chunk ID.
Vector matrices are loaded/searched/released sequentially, with the largest full matrix about 23.7 GB.

Additional inputs for Qwen8B: `boinhbo/medxir-package-qwen8b`,
`dinhbaobao/medxir-embed-corpus-qwen8b-s0/s1/s2`,
`ntklinhfitus/medxir-vectors-qwen3-8b-additions`.
Additional inputs for Octen8B: `boinhbo/medxir-package-octen8b`,
`boinhbo/medxir-embed-corpus-octen8b-s0/s1/s2`,
`ntklinhfitus/medxir-vectors-octen-8b-additions`.
Only active models need these inputs. `ENCODE_HERE=False` requires model-labelled variant caches
written by this notebook; old 1200-query outputs do not include custom instructions or sub-queries.

Steps: input/ID preflight -> query variants -> encode each active embedding model -> exact search over its 2.89M chunk vectors held on
the GPU -> PRF search -> BM25 candidates -> weighted RRF pool -> rerank -> document scores (max/top3) ->
submission ZIP (kept under 100 MB) + `pool.parquet` to rebuild variants offline without a GPU ->
extra ZIPs `medxir-expand-nb60-d200[-drr-bge|-drr-bge-only]-submission.zip`: neighbour spans of the top
EXPAND_NB chunks, documents optionally re-ordered by bge-reranker-v2-m3 (DOC_RERANK, +~25 min for 480k pairs).

Historical single-4B time with every switch on (bge reranker, POOL_MAX 600): about 45-75 min. Rerank ~720k pairs at ~530 pairs/s
(benchmark 26/10) is 20-30 min; the 4B model load (4-9 min) and the 14.8 GB vector read are most of the rest.
Without the reranker: about 20-45 min for 4B. More models add encoding and vector loading/search time;
these are historical estimates, not multi-model GPU benchmarks. Progress and rerank ETA go to progress.txt.
BM25 candidates: rerun `kaggle_bm25_vi` on corpus v2 (its bm25_manifest.json states the corpus, checked here); with
the v1 run, chunks added in v2 enter the pool through dense search only.

In [ ]:
# ===================== Config: the only cell to edit =====================
EMBEDDING_MODEL = 'qwen4b'   # single model: 'qwen4b', 'qwen8b', 'octen8b'
USE_MODEL_FUSION = True     # True: use MODEL_WEIGHTS below; EMBEDDING_MODEL is then ignored
MODEL_WEIGHTS = {'qwen4b': 0.35, 'qwen8b': 0.35, 'octen8b': 0.30}   # 30/10 fusion run: 0.2539 (nb60-d200-drr-bge-only)
USE_TRANSLATION = False     # 31/10: 0.2383 vs 0.2539 (ZH documents 15% -> 35%). EN + ZH translations of each query (scripts/translate_queries.py, embedded below) are
                            # searched as extra dense queries (TRANS_K candidates, RRF weight W_TRANS per model)
TRANS_K, W_TRANS = 300, 0.5 # W_TRANS as the sub-query / PRF lists
RERANK_SAME_LANG = False    # EN/ZH chunks and documents: reranker score = max(VI query, query in the chunk's language)
# Weighted RRF, normalized to sum=1. Zero skips a model and its inputs.
# With USE_RERANK=True these weights select the candidate pool; the reranker then orders that pool.
USE_QUERY_SPLIT = True      # rule-based cleanup + split of long queries; each sub-query is searched on its own
USE_CUSTOM_INSTRUCT = True  # task-specific Qwen3-Embedding instruction instead of the model-card default
USE_PRF = True              # pseudo-relevance feedback: query moved toward its top PRF_K chunks, searched again
USE_BM25 = True             # add BM25 (Vietnamese chunks) candidates to the pool
USE_RERANK = True           # cross-encoder scores the fused pool; chunk and document scores come from it
DOC_AGG = 'top3'            # document score: 'max' (old: best chunk) or 'top3' (sum of the 3 best chunks)
LANGS = 'vi,en,zh'          # corpus languages searched, e.g. 'vi' or 'vi,zh' (BM25 covers 'vi' only)
EXPECTED_CORPUS_VERSION = 'v2'  # rejects accidentally attaching the older full corpus
BM25_CORPUS_VERSION = 'v1'  # BM25 run without bm25_manifest.json (the v1 run); newer runs state their corpus

RERANKER = 'bge-reranker-v2-m3'  # 'Qwen3-Reranker-0.6B' is ~3.4x slower, 'Qwen3-Reranker-4B' ~14x (too slow here)
N_DOCS = 200                # 200 or 300: d300 raised Doc R 0.473 -> 0.519 but Doc F2 fell (FINAL 0.2235 -> 0.2125)
N_CHUNKS = 150              # lowered by 10 until the ZIP fits MAX_ZIP_MB (dashboard limit 100 MB)
MAX_ZIP_MB = 99.0

# Extra ZIPs from the same pool after the hybrid ZIP (scripts/expand_chunks.py), named medxir-expand-nb60-d200-*.
EXPAND_NB = 60              # neighbours (order +-1) of the top EXPAND_NB chunks merged into spans (nb60 0.2235); 0 = off
DOC_RERANK = 'only'         # bge-reranker-v2-m3 on candidate documents: 'off', 'fusion' (RRF with the chunk-based
                            # order, 0.2253), 'only' (bge order, 0.2253) or 'both' (two ZIPs, scored once)
DOC_RERANK_DEPTH = N_DOCS + 200  # candidate documents scored per query (the cut is at N_DOCS); 29/10 run: 400
DOC_LEAD_CHARS = 400        # document start (title / lead) shown before its best pool chunk

DENSE_K = 500               # candidates of the full query
SUB_K = 100                 # candidates of each sub-query
PRF_K, PRF_ALPHA, PRF_KEEP = 5, 0.3, 300  # feedback chunks, weight of their mean, candidates of the PRF query
BM25_K = 300                # BM25 candidates per query (the attached run kept 500)
POOL_MAX = 600              # fused candidates sent to the reranker per query (pairs = 1,200 x POOL_MAX)
RRF_K = 60
W_SUB, W_PRF, W_BM25 = 0.5, 0.5, 0.3  # RRF weights; the full-query dense list has weight 1
SPLIT_MIN_WORDS, MIN_SUB_WORDS, MAX_SUBQ = 20, 4, 4

INSTRUCT_DEFAULT = 'Given a web search query, retrieve relevant passages that answer the query'
INSTRUCT_CUSTOM = ('Given a Vietnamese patient question about a health or medical problem, retrieve medical '
                   'passages in Vietnamese, English or Chinese that answer the question')
SEARCH_TF32 = True          # TF32 matmul (fp16 vectors are exact in TF32); False = full float32 like the baseline
ENCODE_HERE = True          # False: read query_variant_vectors.npy + query_variants.json from an attached earlier run
QUERY_LIMIT = 0             # >0: first N queries only (speed check; no submission is written)

In [ ]:
import faulthandler
import json
import os
import resource
import time
import traceback
from contextlib import contextmanager
from pathlib import Path

OUT = Path('/kaggle/working')
OUT.mkdir(parents=True, exist_ok=True)
T0 = time.perf_counter()
PROGRESS = OUT / 'progress.txt'
FAULT_FILE = open(OUT / 'faulthandler.txt', 'w')  # native crashes dump the Python stack here
faulthandler.enable(file=FAULT_FILE, all_threads=True)
FAILED = None


def note(message):
    rss_gib = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2 ** 20  # Linux reports KiB
    line = f'{time.perf_counter() - T0:8.1f}s  peak_rss={rss_gib:.1f}GiB  {message}'
    print(line, flush=True)
    with open(PROGRESS, 'a', encoding='utf-8') as file:
        file.write(line + '\n')


@contextmanager
def stage(name):
    # A Python exception is written to retrieval_error.txt and does not stop the kernel, so the
    # session ends COMPLETE and the files can be fetched; later steps are skipped.
    global FAILED
    note(f'start {name}')
    try:
        yield
        note(f'done {name}')
    except Exception:
        FAILED = name
        (OUT / 'retrieval_error.txt').write_text(f'stage: {name}\n\n{traceback.format_exc()}', encoding='utf-8')
        note(f'FAILED {name}; traceback in retrieval_error.txt')

In [ ]:
# Copied from scripts/hybrid_retrieval.py by scripts/build_hybrid_notebook.py. Edit the script and rebuild.
"""CPU-side logic of the hybrid retrieval notebook (scripts/build_hybrid_notebook.py copies this file into it).

Query variants: rule-based cleanup of patient chit-chat and a split of long queries into sentences and
comma clauses; no LLM. Every variant is searched on its own and the lists are fused with weighted RRF.
Document ranking: a document scores the max, or the sum of its 3 best chunk scores (DOC_AGG). Scores
only depend on the submitted sets, so the ranking matters through what falls inside N_DOCS / N_CHUNKS.

With one dense list, DOC_AGG='max' and no reranker, assemble() gives exactly the old d200-c150 lists
(scripts/retrieve_local_4b.py): docs in order of their best chunk, chunk docs appended when missing.
"""
import re
import unicodedata

CHITCHAT = re.compile(
    r'(?i)\b(?:'
    r'(?:xin )?chào (?:bác s[ĩỹi]|bs)[^,.?!]*[,.!]?'
    r'|(?:cho|xin cho) (?:em|tôi|mình|cháu|con|chị|anh) (?:được )?hỏi(?: (?:bác s[ĩỹi]|bs))?(?: là)?'
    r'|(?:em |tôi |mình )?xin hỏi'
    r'|bác s[ĩỹi] ơi'
    r'|(?:em |tôi |mình |xin )?(?:chân thành )?cảm ơn(?: (?:bác s[ĩỹi]|bs))?(?: nhiều)?(?: ạ)?'
    r'|mong (?:bác s[ĩỹi]|bs)[^.?!]*(?:tư vấn|giải đáp|trả lời|giúp)[^.?!]*'
    r'|nhờ (?:bác s[ĩỹi]|bs)[^.?!]*(?:tư vấn|giải đáp)[^.?!]*'
    r')')
SENTENCE = re.compile(r'(?<=[.?!;…])\s+|\n+')


def clean_query(text):
    """Drop greetings, thanks and 'cho em hỏi'-style openers; keep every medical word."""
    text = CHITCHAT.sub(' ', unicodedata.normalize('NFC', text))
    return re.sub(r'\s+', ' ', text).strip(' ,.')


def sub_queries(query, split_min_words=20, min_sub_words=4, max_subq=4):
    """Sentences of a long query, plus the comma clauses of a sentence when every clause has 3+ words.

    Short queries (most are one sentence of ~18 words) return []: they are searched whole only.
    """
    if len(query.split()) < split_min_words:
        return []
    found = []
    for sentence in SENTENCE.split(clean_query(query)):
        sentence = sentence.strip(' ,.;')
        if len(sentence.split()) < min_sub_words:
            continue
        found.append(sentence)
        clauses = [part.strip() for part in sentence.split(',')]
        if len(clauses) >= 2 and all(len(part.split()) >= 3 for part in clauses):
            found.extend(clauses)
    seen, out = {query.lower().strip()}, []
    for text in found:
        if text.lower() not in seen:
            seen.add(text.lower())
            out.append(text)
    return out[:max_subq]


def rrf(lists, k=60):
    """Weighted reciprocal-rank fusion. lists: [(ranked ids, weight)]. Returns [(id, score)], best first.

    Ties keep the order of first appearance, so a single list comes back in its own order.
    """
    score, first = {}, {}
    for ids, weight in lists:
        for rank, cid in enumerate(ids):
            score[cid] = score.get(cid, 0.0) + weight / (k + rank + 1)
            first.setdefault(cid, len(first))
    return sorted(score.items(), key=lambda item: (-item[1], first[item[0]]))


def model_candidate_lists(results, weights, qi, sub_weight, prf_weight, trans_weight=0.0):
    """One RRF stage: model weights multiply full/sub/PRF/translation lists; BM25 is added once by caller."""
    lists = []
    for key, weight in weights.items():
        result = results[key]
        lists.append((list(result['full_ids'][qi]), weight))
        lists.extend((list(ids), weight * sub_weight) for ids in result['sub_lists'][qi])
        if trans_weight and result.get('trans_lists'):
            lists.extend((list(ids), weight * trans_weight) for ids in result['trans_lists'][qi])
        if result['prf_ids'] is not None:
            lists.append((list(result['prf_ids'][qi]), weight * prf_weight))
    return lists


def doc_ranking(scored, docs_of, agg='max'):
    """Documents of the scored chunks ([(chunk id, score)], best first), best document first.

    agg='max': a document scores its best chunk (the old ranking). agg='top3': the sum of its 3 best chunks,
    so a document with several matching chunks beats one with a single lucky match.
    """
    per_doc, first = {}, {}
    for cid, score in scored:
        for doc_id in docs_of(cid):
            per_doc.setdefault(doc_id, []).append(score)
            first.setdefault(doc_id, len(first))
    if agg == 'max':
        value = {doc: max(scores) for doc, scores in per_doc.items()}
    elif agg == 'top3':
        value = {doc: sum(sorted(scores, reverse=True)[:3]) for doc, scores in per_doc.items()}
    else:
        raise ValueError(f'unknown DOC_AGG {agg!r}')
    return sorted(per_doc, key=lambda doc: (-value[doc], first[doc]))


def assemble(scored, fill_ids, docs_of, n_docs, n_chunks, agg='max'):
    """One query's (relevant_docs, [(doc_id, chunk id)]).

    scored: [(chunk id, score)] best first, the chunks with a final score (the reranked pool, or all fused
    candidates). fill_ids: further candidate chunk ids in fused order, used only to top the document list
    up to n_docs. Chunk documents missing from the list are appended, as in the earlier submissions.
    """
    docs = doc_ranking(scored, docs_of, agg)[:n_docs]
    listed = set(docs)
    for cid in fill_ids:
        if len(docs) >= n_docs:
            break
        for doc_id in docs_of(cid):
            if doc_id not in listed and len(docs) < n_docs:
                listed.add(doc_id)
                docs.append(doc_id)
    chunks = []
    for cid, _ in scored[:n_chunks]:
        for doc_id in docs_of(cid):
            chunks.append((doc_id, cid))
            if doc_id not in listed:
                listed.add(doc_id)
                docs.append(doc_id)
    return docs, chunks

# Input validation copied from scripts/hybrid_inputs.py.
"""CPU preflight for hybrid retrieval: part discovery, coverage and row alignment."""
import hashlib
import json
import math
from collections import Counter
from pathlib import Path

EMBEDDING_SPECS = {
    'qwen4b': dict(model='Qwen/Qwen3-Embedding-4B', dim=2560, doc_prefix='', directory='Qwen3-Embedding-4B'),
    'qwen8b': dict(model='Qwen/Qwen3-Embedding-8B', dim=4096, doc_prefix='', directory='Qwen3-Embedding-8B'),
    'octen8b': dict(model='Octen/Octen-Embedding-8B', dim=4096, doc_prefix=' ', directory='Octen-Embedding-8B'),
}


def embedding_weights(selected, fusion, weights):
    """Normalized model weights: zero-weight models need no attached assets."""
    if not fusion:
        assert selected in EMBEDDING_SPECS, f'Unknown EMBEDDING_MODEL: {selected}'
        return {selected: 1.0}
    assert set(weights) == set(EMBEDDING_SPECS), 'MODEL_WEIGHTS must specify qwen4b, qwen8b, octen8b'
    assert all(isinstance(w, (int, float)) and not isinstance(w, bool) and math.isfinite(w) and w >= 0
               for w in weights.values()), 'Model weights must be finite and nonnegative'
    total = sum(weights.values())
    assert math.isfinite(total) and total > 0, 'At least one model weight must be positive'
    return {k: weights[k] / total for k in EMBEDDING_SPECS if weights[k] > 0}


def group_embedding_vectors(paths, active):
    """Resolve model identity from its manifest, never from dimension or part filename."""
    import re
    by_repo = {EMBEDDING_SPECS[k]['model']: k for k in active}
    found = {k: [] for k in active}
    manifests = {}
    for raw in paths:
        path = Path(raw)
        if not re.fullmatch(r'(vi|en|zh)-part-\d+', path.stem):
            continue
        manifest_path = path.parent.parent / 'embed_manifest.json'
        if manifest_path not in manifests:
            manifests[manifest_path] = json.loads(manifest_path.read_text(encoding='utf-8'))
        key = by_repo.get(manifests[manifest_path].get('model'))
        if key is not None:
            found[key].append(str(path))
    for key in active:
        assert found[key], f'Missing corpus vectors for {key}; attach its three baseline sources and additions dataset'
    return found


def validate_query_cache(saved, vectors, key, variants, task, query_ids, legacy=False, revision=None):
    import numpy as np
    assert saved['variants'] == variants and saved['task'] == task, 'Cached query variants/instruction differ'
    if legacy:
        assert key == 'qwen4b', 'Unlabelled legacy query vectors are supported only for qwen4b'
        assert saved.get('model', EMBEDDING_SPECS[key]['model']) == EMBEDDING_SPECS[key]['model']
    else:
        assert saved['model'] == EMBEDDING_SPECS[key]['model'], 'Cached query model differs'
        assert saved['query_ids'] == list(query_ids), 'Cached query ID order differs'
        assert saved['query_format'] == 'Instruct: {task}\nQuery:{text}' and saved['max_length'] == 1024, 'Cached query encoding contract differs'
        assert revision is None or saved.get('model_revision') == revision, 'Cached model revision differs'
    assert vectors.shape == (len(variants), EMBEDDING_SPECS[key]['dim']) and np.isfinite(vectors).all(), 'Invalid cached query vectors'
    assert np.max(abs(np.linalg.norm(vectors.astype(np.float32), axis=1) - 1)) < .01, 'Cached queries must have unit norm'


def unique_parts(paths, chunk=False):
    result = {}
    for raw in paths:
        path = Path(raw)
        key = f'{path.parent.name}-{path.stem}' if chunk else path.stem
        assert key.split('-')[0] in {'vi', 'en', 'zh'}, f'Unknown language in part: {path}'
        assert key not in result, f'Duplicate part {key}: {result.get(key)} and {path}'
        result[key] = str(path)
    assert result, 'No chunk parts found' if chunk else 'No vector parts found'
    return result


def select_input_parts(chunk_files, vector_files, langs):
    langs = list(langs)
    assert langs and len(set(langs)) == len(langs) and set(langs) <= {'vi', 'en', 'zh'}, f'Invalid LANGS: {langs}'
    chunks = unique_parts(chunk_files, chunk=True)
    vectors = unique_parts(vector_files)
    chunks = {k: p for k, p in chunks.items() if k.split('-')[0] in langs}
    vectors = {k: p for k, p in vectors.items() if k.split('-')[0] in langs}
    assert {k.split('-')[0] for k in chunks} == set(langs), f'Missing chunk languages: {langs}'
    assert set(chunks) == set(vectors), (
        f'Chunk/vector parts differ: missing vectors={sorted(set(chunks)-set(vectors))}; '
        f'extra vectors={sorted(set(vectors)-set(chunks))}')
    return chunks, vectors


def json_fingerprint(path):
    path = Path(path)
    return dict(path=str(path), sha256=hashlib.sha256(path.read_bytes()).hexdigest())


def validate_hybrid_inputs(chunk_files, vector_files, langs, manifest_path, expected_version='v2', progress=None, model_key='qwen4b'):
    import numpy as np
    import pyarrow.parquet as pq

    spec = EMBEDDING_SPECS[model_key]
    manifest_path = Path(manifest_path)
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    assert not expected_version or manifest['dataset_version'] == expected_version, (
        f'Expected corpus {expected_version}, found {manifest.get("dataset_version")}')
    chunks, vectors = select_input_parts(chunk_files, vector_files, langs)
    schemas = {}; rows = Counter(); seen = set(); vector_manifests = {}
    for key in sorted(vectors):
        path = Path(vectors[key]); chunk_path = Path(chunks[key]); lang = key.split('-')[0]
        assert chunk_path.resolve().is_relative_to(manifest_path.parent.resolve()), f'Chunk part outside corpus root: {chunk_path}'
        embed_manifest_path = path.parent.parent / 'embed_manifest.json'
        if str(embed_manifest_path) not in vector_manifests:
            m = json.loads(embed_manifest_path.read_text(encoding='utf-8'))
            assert m['complete'] and m['model'] == spec['model'] and m['dim'] == spec['dim'], f'Incompatible vector manifest: {embed_manifest_path}'
            assert m.get('doc_prefix', '') == spec['doc_prefix'], f'Incompatible document prefix: {embed_manifest_path}'
            assert m['max_length'] == 1024 and m['dtype'] == 'torch.bfloat16', f'Incompatible embedding contract: {embed_manifest_path}'
            for field, expected in [('instruction', ''), ('pooling', 'last_token'), ('padding_side', 'left'), ('normalize', True)]:
                assert m.get(field, expected) == expected, f'Incompatible {field}: {embed_manifest_path}'
            vector_manifests[str(embed_manifest_path)] = dict(**json_fingerprint(embed_manifest_path), model=m['model'], dim=m['dim'],
                model_revision=m.get('model_revision'), output_dtype=m.get('output_dtype', 'float16'))
        array = np.load(path, mmap_mode='r', allow_pickle=False)
        assert array.ndim == 2 and array.dtype == np.float16 and array.shape[1] == spec['dim'], (key, array.dtype, array.shape)
        ids_path = path.with_name(f'{key}-ids.parquet')
        ids_table = pq.read_table(ids_path, columns=['chunk_id'])
        chunk_table = pq.read_table(chunk_path, columns=['chunk_id', 'lang'])
        assert ids_table['chunk_id'].null_count == chunk_table['chunk_id'].null_count == 0, f'Null ID in {key}'
        ids = ids_table['chunk_id'].to_numpy(); expected_ids = chunk_table['chunk_id'].to_numpy()
        assert array.shape[0] == len(ids) == len(expected_ids) and len(ids) > 0, f'Row count mismatch: {key}'
        assert np.array_equal(ids, expected_ids), f'Chunk ID order mismatch: {key}'
        assert set(chunk_table['lang'].to_pylist()) == {lang}, f'Wrong language rows: {key}'
        part_ids = set(ids.tolist())
        assert len(part_ids) == len(ids) and not seen.intersection(part_ids), f'Duplicate chunk IDs: {key}'
        seen.update(part_ids); rows[lang] += len(ids)
        schemas[key] = dict(chunk_path=str(chunk_path), vector_path=str(path), ids_path=str(ids_path), rows=len(ids), language=lang)
        del array, ids, expected_ids, ids_table, chunk_table, part_ids
        if progress: progress(f'preflight {key}: {schemas[key]["rows"]:,} aligned IDs')
    for lang in langs:
        assert rows[lang] == manifest['languages'][lang]['chunks'], f'Corpus manifest count mismatch for {lang}: {rows[lang]}'
    provenance = dict(model_key=model_key, model=spec['model'], dim=spec['dim'], corpus_version=manifest['dataset_version'], corpus_manifest=json_fingerprint(manifest_path),
        selected_languages=list(langs), chunk_count=sum(rows.values()), vector_count=sum(rows.values()),
        rows_by_language=dict(rows), part_count=len(schemas), parts=schemas,
        vector_manifests=list(vector_manifests.values()), all_part_ids_aligned=True)
    return chunks, vectors, provenance

"""Qwen reranker inputs: preserve the classification suffix when truncating.

Matches the official Qwen3-Reranker Transformers recipe: reserve prompt tokens,
truncate the body, append the untouched prefix/suffix, and left-pad the batch.
"""

QWEN_RERANK_PREFIX = ('<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the '
                     'Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n')
QWEN_RERANK_SUFFIX = '<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'


def qwen_reranker_inputs(tokenizer, queries, documents, instruction, max_length, return_tensors='pt'):
    if len(queries) != len(documents) or not queries:
        raise ValueError('Expected equal nonempty query/document batches')
    if tokenizer.padding_side != 'left':
        raise ValueError('Qwen last-position scoring requires left padding')
    prefix = tokenizer.encode(QWEN_RERANK_PREFIX, add_special_tokens=False)
    suffix = tokenizer.encode(QWEN_RERANK_SUFFIX, add_special_tokens=False)
    available = max_length - len(prefix) - len(suffix)
    if available < 1:
        raise ValueError('Token limit leaves no room for query/document body')
    bodies = [f'<Instruct>: {instruction}\n<Query>: {q}\n<Document>: {d}' for q,d in zip(queries,documents)]
    encoded = tokenizer(bodies, padding=False, truncation=True, add_special_tokens=False,
                        return_attention_mask=False, max_length=available)
    ids = [prefix + body + suffix for body in encoded['input_ids']]
    assert all(len(row) <= max_length and row[-len(suffix):] == suffix for row in ids)
    return tokenizer.pad({'input_ids': ids}, padding=True, return_tensors=return_tensors)

## 1. Inputs and GPU check

In [ ]:
if FAILED is None:
    with stage('setup'):
        import glob
        import subprocess
        import zipfile

        import numpy as np
        import pandas as pd
        import pyarrow as pa
        import pyarrow.compute as pc
        import pyarrow.parquet as pq
        import torch

        CONFIG = {name: value for name, value in globals().items()
                  if name.isupper() and name not in {'OUT', 'T0', 'PROGRESS', 'FAULT_FILE', 'FAILED', 'CONFIG', 'EMBEDDING_SPECS'}
                  and isinstance(value, (bool, int, float, str, dict))}
        TIMINGS = {}


        def one_match(pattern, description):
            hits = sorted(glob.glob(pattern, recursive=True))
            assert len(hits) == 1, f'{description}: expected one match, found {hits[:10]}'
            return Path(hits[0])


        # The attached inputs, so a missing or doubled source shows in the log before any check fails.
        for root in sorted(glob.glob('/kaggle/input/*/*/*')) or sorted(glob.glob('/kaggle/input/*')):
            print('input:', root, sorted(os.listdir(root))[:12] if os.path.isdir(root) else '')

        QUERY_PATH = one_match('/kaggle/input/**/query.parquet', 'query.parquet')
        CORPUS_PATH = one_match('/kaggle/input/**/links_corpus_vi_en_zh.parquet', 'official corpus ids')
        HASH_PATH = one_match('/kaggle/input/**/hash_groups.parquet', 'hash_groups.parquet (medxir-chunk-corpus)')
        CHUNK_MANIFEST_PATH = HASH_PATH.parent / 'chunk_manifest.json'
        assert CHUNK_MANIFEST_PATH.is_file(), f'Missing corpus manifest: {CHUNK_MANIFEST_PATH}'
        CHUNK_FILES = sorted(glob.glob('/kaggle/input/**/chunks/*/part-*.parquet', recursive=True))
        VECTOR_FILES = sorted(p for p in glob.glob('/kaggle/input/**/vectors/*.npy', recursive=True) if not p.endswith('.tmp.npy'))
        SEARCH_LANGS = [lang.strip() for lang in LANGS.split(',') if lang.strip()]
        ACTIVE_MODEL_WEIGHTS = embedding_weights(EMBEDDING_MODEL, USE_MODEL_FUSION, MODEL_WEIGHTS)
        MODEL_VECTOR_FILES = group_embedding_vectors(VECTOR_FILES, ACTIVE_MODEL_WEIGHTS)
        MODEL_VECTOR_PATHS, MODEL_INPUTS, MODEL_REVISIONS = {}, {}, {}
        for model_key in ACTIVE_MODEL_WEIGHTS:
            CHUNK_PATH, paths, info = validate_hybrid_inputs(
                CHUNK_FILES, MODEL_VECTOR_FILES[model_key], SEARCH_LANGS, CHUNK_MANIFEST_PATH,
                EXPECTED_CORPUS_VERSION, progress=note, model_key=model_key)
            MODEL_VECTOR_PATHS[model_key], MODEL_INPUTS[model_key] = paths, info
            revisions = {m['model_revision'] for m in info['vector_manifests'] if m['model_revision']}
            assert len(revisions) <= 1, f'{model_key}: vector sources use different revisions'
            MODEL_REVISIONS[model_key] = next(iter(revisions), None)
            note(f'{model_key}: {info["part_count"]} parts, {info["vector_count"]:,} vectors, '
                 f'{info["vector_count"] * info["dim"] * 2 / 1e9:.2f} GB fp16')
        INPUT_PROVENANCE = dict(next(iter(MODEL_INPUTS.values())), models=MODEL_INPUTS,
                                fusion='weighted_rrf' if len(ACTIVE_MODEL_WEIGHTS) > 1 else 'single_model',
                                normalized_model_weights=ACTIVE_MODEL_WEIGHTS)
        CHUNK_FILES = [CHUNK_PATH[key] for key in sorted(CHUNK_PATH)]
        note(f'active embedding weights: {ACTIVE_MODEL_WEIGHTS}; languages={SEARCH_LANGS}')
        BM25_ACTIVE = USE_BM25 and 'vi' in SEARCH_LANGS
        BM25_PROVENANCE = dict(requested=USE_BM25, active=BM25_ACTIVE, corpus_version=BM25_CORPUS_VERSION,
                               coverage='not_used', candidates_path=None)
        MODEL_PATHS = {}
        if ENCODE_HERE:
            for model_key in ACTIVE_MODEL_WEIGHTS:
                spec = EMBEDDING_SPECS[model_key]
                path = one_match(f'/kaggle/input/**/{spec["directory"]}/config.json', f'packaged {model_key}').parent
                package = json.loads((path.parent / 'MODELS.json').read_text(encoding='utf-8'))['models'][spec['model']]
                assert MODEL_REVISIONS[model_key] is None or package['sha'] == MODEL_REVISIONS[model_key], f'{model_key}: query model / document revision mismatch'
                MODEL_REVISIONS[model_key] = package['sha']
                MODEL_PATHS[model_key] = path
        if BM25_ACTIVE:
            BM25_PATH = one_match('/kaggle/input/**/vi-candidates-k500.npz', 'BM25 candidates (medxir-bm25-vi)')
            bm25_manifest = BM25_PATH.with_name('bm25_manifest.json')
            if bm25_manifest.is_file():   # written by the BM25 runs since corpus v2; older runs only have the config label
                bm25_info = json.loads(bm25_manifest.read_text(encoding='utf-8'))
                BM25_CORPUS_VERSION = bm25_info['corpus_version']
                if BM25_CORPUS_VERSION == INPUT_PROVENANCE['corpus_version'] and 'vi' in SEARCH_LANGS:
                    assert bm25_info['vi_chunks'] == INPUT_PROVENANCE['rows_by_language']['vi'], ('BM25 / corpus VI chunks differ', bm25_info)
            BM25_PROVENANCE.update(corpus_version=BM25_CORPUS_VERSION,candidates_path=str(BM25_PATH), coverage='same_corpus' if BM25_CORPUS_VERSION == INPUT_PROVENANCE['corpus_version'] else 'older_or_different_corpus')
            if BM25_PROVENANCE['coverage'] != 'same_corpus':
                note(f'BM25 candidates cover {BM25_CORPUS_VERSION}; current corpus is {INPUT_PROVENANCE["corpus_version"]}. '
                     'Added chunks enter through dense search until BM25 is rebuilt.')
        if USE_RERANK:
            RERANK_PATH = one_match(f'/kaggle/input/**/{RERANKER}/config.json', f'packaged {RERANKER}').parent
        assert DOC_RERANK in {'off', 'fusion', 'only', 'both'}, f'unknown DOC_RERANK {DOC_RERANK!r}'
        assert DOC_RERANK == 'off' or DOC_RERANK_DEPTH > N_DOCS, 'DOC_RERANK_DEPTH must reach past the N_DOCS cut'
        if DOC_RERANK != 'off':
            DOC_RERANK_PATH = one_match('/kaggle/input/**/bge-reranker-v2-m3/config.json', 'packaged bge-reranker-v2-m3').parent

        queries = pd.read_parquet(QUERY_PATH, columns=['id', 'query'])
        assert len(queries) == 1200 and queries['id'].is_unique
        if QUERY_LIMIT:
            queries = queries.head(QUERY_LIMIT).reset_index(drop=True)
        QUERY_IDS = [int(q) for q in queries['id']]
        QUERY_TEXT = queries['query'].astype(str).tolist()

        print(subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total,driver_version', '--format=csv'],
                             capture_output=True, text=True).stdout)
        GPU_NAME = torch.cuda.get_device_name(0)
        assert '6000' in GPU_NAME, f'Expected RTX Pro 6000, got {GPU_NAME!r}'
        print('GPU:', GPU_NAME, '| queries:', len(queries), '| model vector parts:', {k: list(v) for k, v in MODEL_VECTOR_PATHS.items()})
        print(json.dumps(CONFIG, ensure_ascii=False, indent=1))

## 2. Query variants (USE_QUERY_SPLIT)

In [ ]:
# Query translations (Qwen/Qwen3-4B-Instruct-2507, scripts/translate_queries.py), embedded by the generator.
TRANSLATIONS_JSON = r'''{"model": "Qwen/Qwen3-4B-Instruct-2507", "revision": "cdbee75f17c01a7cc42f958dc650907174af0554", "items": [{"id": 1, "vi": "Em bị viêm xung huyết 1/3 dưới thực quản, viêm dạ dày trợt xung huyết mạn tính, sẹo loét hành tá tràng. Cho em hỏi các bệnh này của em có nặng và có thể khỏi được không. Em vừa uống xong đợt thuốc được 1 2 ngày thì giờ em lại thấy triệu chứng bị nghẹn ở cổ và hơi buồn nôn. không biết có phải bệnh nặng lên hay do sao.", "en": "I have inflammation of the lower third of the esophagus, chronic erosive hemorrhagic gastritis, and peptic ulcer scar in the duodenal bulb. I would like to know whether these conditions are serious and whether they can be cured. I just finished a course of medication two days ago, and now I am experiencing symptoms of choking in the neck and a feeling of nausea. I don't know if this means my condition has worsened or if it's due to something else.", "zh": "我患有食管下1/3的血管炎、慢性胃黏膜糜烂出血性胃炎、十二指肠溃疡瘢痕。我想问一下，这些病严重吗？能否痊愈？我刚吃完一两天的药，现在又出现喉咙发堵和恶心的症状，不知道是病情加重还是其他原因。"}, {"id": 2, "vi": "Răng có lỗ chấm đen, chảy máu khi thức dậy là sao?", "en": "Black hole in the tooth, bleeding when waking up, what is this?", "zh": "牙齿有黑色孔洞，早上起床时出血是怎么回事？"}, {"id": 3, "vi": "Kết quả của Beta-HCG là dưới 0.10 U/L thì là như thế nào ạ?", "en": "What does a Beta-HCG result below 0.10 U/L indicate?", "zh": "β-HCG 检测结果低于 0.10 U/L 表示什么？"}, {"id": 4, "vi": "Chỉ số SCC của em sau khi xét nghiệm là 1.52 vẫn nằm trong ngưỡng bình thường nhưng có vẻ hơi cao. Em có cần để ý, chú ý thêm hay sau mấy tháng phải xét nghiệm lại không. Bác sĩ có thể giải thích thêm về chỉ số của em không.", "en": "My SCC value after the test is 1.52, which still falls within the normal range but seems slightly high. Should I pay more attention, be more cautious, or should I get retested after a few months? Could the doctor please explain my value further?", "zh": "检测后您的SCC值为1.52，虽然仍在正常范围内，但似乎略高。您是否需要特别注意，或者需要几个月后再次检测？医生能否进一步解释一下您的这个指标？"}, {"id": 5, "vi": "Cách phòng tránh nguy cơ lây nhiễm bệnh lao đa kháng là gì?", "en": "How to prevent the risk of infection with multidrug-resistant tuberculosis?", "zh": "如何预防多重耐药结核病的感染？"}, {"id": 6, "vi": "Nhân tuyến giáp có nên điều trị hay không và can thiệp những gì?", "en": "Should thyroid nodules be treated and what interventions are required?", "zh": "甲状腺腺体是否需要治疗以及如何干预？"}, {"id": 7, "vi": "Thường xuyên nổi nóng không kiềm chế được, phải làm sao?", "en": "Frequent hot flashes that cannot be controlled, what should I do?", "zh": "经常无法控制地出现发热，该怎么办？"}, {"id": 8, "vi": "Đau họng, khó ngủ có nên ngậm viên ho lotus chiết xuất cao thường xuân không?", "en": "Sore throat, difficulty sleeping, should I suck on Lotus extract high-concentration Changxun lozenges?", "zh": "喉咙痛、失眠是否可以含服高浓度的莲子提取的常春片？"}, {"id": 9, "vi": "Em bị dị ứng một loại thuốc giảm đau, khi uống thuốc có thành phần này chỉ bị sưng mắt, nếu ngưng uống triệu chứng tự hết. Vậy em có thể tiêm vắc xin Covid-19 được không?", "en": "I have an allergy to a pain reliever; when taking a medication containing this ingredient, I only experience swelling of the eyes, and the symptoms resolve spontaneously upon discontinuation. Therefore, can I receive the Covid-19 vaccine?", "zh": "我对一种止痛药产生过敏，服用含有该成分的药物时只会出现眼肿，停药后症状自行消失。那么我能否接种新冠疫苗？"}, {"id": 10, "vi": "Em có dùng một loại thuốc tăng cân nhanh lúc dùng thì ăn cơm ngon thật như mà khi em dùng hết em cảm thấy rất chán ăn không ăn nổi một bát cố ăn nhưng lại buồn nôn vậy cho em hỏi em có bị làm sao không", "en": "I have been using a fast-weight-gain medication; when I was using it, I really enjoyed eating rice. However, after I finished using it, I felt very loss of appetite, unable to eat a bowl, and only managed to eat a little, but I felt nauseous. I would like to know if I have any problem.", "zh": "我曾经用过一种快速增重的药物，用的时候吃米饭特别香，但一旦用完，就感觉非常没胃口，吃不了半碗，只能勉强吃一点，还伴有恶心。请问我会有什么问题吗？"}, {"id": 11, "vi": "Tư vấn về ghép thận đối với người bị suy thận mãn giai đoạn cuối", "en": "Consultation on kidney transplantation for patients with end-stage chronic kidney disease", "zh": "关于终末期慢性肾衰竭患者的肾移植咨询"}, {"id": 12, "vi": "Con em đẻ được tháng rốn vẫn ướt. Vậy có làm sao không?", "en": "The baby's navel is still wet one month after birth. Is there anything wrong with this?", "zh": "宝宝出生一个月脐部仍然湿润，这种情况有问题吗？"}, {"id": 13, "vi": "Uống thuốc điều trị lao phổi nhưng thấy người mệt mỏi, ăn uống khó tiêu, chướng bụng có phải bị viêm gan không?", "en": "Taking anti-tuberculosis medication but feeling fatigue, poor appetite, and abdominal distension—could this be due to liver inflammation?", "zh": "服用肺结核治疗药物后出现乏力、食欲差、腹胀，是否可能是肝炎？"}, {"id": 14, "vi": "Hôm nay tôi đi khám bệnh ở Đức Giang, bác sĩ siêu âm ổ bụng kết luận là nhu mô gan thô. Bác sĩ cho hỏi nhu mô gan thô là gì?", "en": "Today I went to see a doctor at Duoc Giang. The abdominal ultrasound doctor concluded that the liver parenchyma is coarse. The doctor asks, what is coarse liver parenchyma?", "zh": "今天我到德江医院就诊，医生做腹部超声检查后指出是肝实质粗。医生请问肝实质粗是什么意思？"}, {"id": 15, "vi": "Em là nam , sức khỏe bình thường không bị HIV . Hôm rồi bạn gái em có làm cho em xuất tinh ra tay cô ấy , vô tình dính tinh dịch tinh trùng vào 1 vết thương hở do đứt tay cách đó 1 ngày , chưa lành hẳn . Vậy liệu bạn gái em có bị bệnh liên quan đến lây nhiễm như HIV không.", "en": "I am male, in good health and not infected with HIV. Recently, my girlfriend had intercourse with me, and accidentally came into contact with semen during a cut on my hand, which occurred one day ago and has not yet healed. Therefore, is my girlfriend at risk of contracting diseases related to infection such as HIV?", "zh": "我是男性，身体健康，没有感染HIV。前几天女友曾让我射精到她的手上，不小心精液精液进入了她前一天割伤的手部伤口，伤口尚未痊愈。请问女友是否有感染与传播相关的疾病，比如HIV？"}, {"id": 16, "vi": "Tôi năm nay 45 tuổi, thường hay bị cay mắt và chảy nước mắt sống. Mắt nhìn mờ, chữ nhỏ không đọc được. Tôi bị bệnh gì?", "en": "I am 45 years old, often experiencing eye burning and excessive tearing. My vision is blurry, and I cannot read small print. What disease do I have?", "zh": "我今年45岁，经常出现眼灼热和流泪症状，视力模糊，小字看不清。我得的是什么病？"}, {"id": 17, "vi": "Thuốc kích thích HGH có làm tăng tế bào ung thư không?", "en": "Do growth hormone stimulants increase cancer cells?", "zh": "刺激性HGH药物会促进癌细胞增长吗？"}, {"id": 18, "vi": "Sử dụng rượu bia có phải là nguyên nhân chính khiến tinh trùng không di động và tiến tới?", "en": "Is the use of alcohol and beer a main cause of non-motile and non-progressing sperm?", "zh": "饮酒是否是导致精子不动和无法前进的主要原因？"}, {"id": 19, "vi": "Người tiểu đường bị đục thủy tinh thể, có thay/mổ được không ạ?", "en": "Does a diabetic patient with cataract have a possibility of replacement/operation?", "zh": "糖尿病患者出现晶状体浑浊，可以更换或手术吗？"}, {"id": 20, "vi": "Trẻ 4 tuổi tiểu 2 tiếng 1 lần 50ml liệu có sao không?", "en": "Is it problematic for a 4-year-old child to urinate every 2 hours, 50ml each time?", "zh": "4岁孩子每2小时小便50毫升，是否正常？"}, {"id": 21, "vi": "Bé nhà em sốt cao tới 39 độ nay mới hết 1 bữa thì người bé nổi ban đầy người . Vậy cho em hỏi có nguy hiểm gì không hả bác", "en": "My child has had a high fever up to 39 degrees, which has now subsided, but then the child developed a rash all over the body. So I would like to ask, is there any danger?", "zh": "我孩子刚才高烧到39度，现在才退烧，但孩子全身突然出现皮疹。请问医生这种情况有危险吗？"}, {"id": 22, "vi": "Khi nào công dân thông thường và doanh nghiệp không thuộc nhóm tuyến đầu như chúng tôi được tiêm chủng (miễn phí hoặc có phí) vậy?", "en": "When are ordinary citizens and enterprises not belonging to our frontline groups vaccinated (free or paid) instead?", "zh": "普通公民和不属于我们这类一线单位的企业员工在什么情况下会接种疫苗（免费或收费）呢？"}, {"id": 23, "vi": "Thưa bác sĩ, em sinh em bé được 22 tháng, em vẫn đang cho bé bú. Nhưng gần đây tóc em rụng rất nhiều, càng ngày càng nhiều, thêm vào đó thỉnh thoảng em có cảm giác ớn lạnh, bốc hỏa, người rất dễ mệt mỏi, xương khớp rất dễ nhức mỏi. Có phải em bị mất cân bằng nội tiết sau sinh không ạ? Nếu phải thì em nên làm thế nào để cải thiện sức khỏe ạ?", "en": "Dear doctor, I have been breastfeeding my baby for 22 months. However, recently I have been experiencing a lot of hair loss, and it's getting worse over time. Additionally, occasionally I feel cold chills, hot flashes, and I feel very easily fatigued, with frequent muscle and joint pain. Could this be due to postpartum hormonal imbalance? If so, what should I do to improve my health?", "zh": "医生您好，我产后22个月，目前仍在哺乳。但最近头发掉得非常多，而且越来越严重，此外偶尔会感到发冷、潮热，容易疲劳，关节和骨骼也经常酸痛。请问这是产后内分泌失调吗？如果是，我应该怎么做才能改善健康状况呢？"}, {"id": 24, "vi": "Hôm qua em có đi khám phụ khoa, bác sĩ khám mỏ vịt nói em có nhiều dịch. Bác sĩ có xét nghiệm dịch đó của em thì nói là em không sao. Nhưng sau khi đi tiểu thì trong nước tiểu em có nhiều sợi trắng giống đờm. Vậy có sao không ạ?", "en": "Yesterday I went to see a gynecologist, and the doctor examined my vulva and said I have a lot of discharge. The doctor tested the discharge and said I am fine. However, after urination, I notice many white threads in my urine, similar to phlegm. So, is this a problem?", "zh": "昨天我去妇科检查，医生检查外阴说我的分泌物很多。医生对我的分泌物做了检查，说我没有问题。但小便后，我的尿液里有很多白色丝状物，像痰一样。这样有问题吗？"}, {"id": 25, "vi": "Bác sĩ ơi, cho em hỏi con nhà em mới sinh được 23 ngày tuổi, mấy ngày gần đây thường xuyên nôn trớ, còn đi ngoài ra bọt và lúc nào cũng ướt, đít đỏ rát và đau. Em có bôi thuốc hăm nhưng đít lúc nào cũng ướt và dính chút phân nên không đỡ. Cho em hỏi đi ngoài và nôn trớ như vậy có phải là bệnh lý không?", "en": "Doctor, I would like to ask you about my newborn child who is 23 days old. Recently, my child has been vomiting frequently, with diarrhea that contains foam, and is always wet. The bottom is red, painful, and irritated. I have been applying diaper cream, but the bottom remains wet and sticky with stool, so it hasn't improved. I would like to know if vomiting and diarrhea like this indicate a pathological condition?", "zh": "医生您好，我孩子刚出生23天，最近几天经常呕吐，大便带有泡沫，总是湿漉漉的，屁股发红、疼痛。我给孩子用了护臀药，但屁股总是湿的，还粘着一点大便，效果不好。请问这种情况大便和呕吐是否属于病理状态？"}, {"id": 26, "vi": "Bị đau dây thần kinh số V có phẫu thuật cắt chọn lọc rễ thần kinh sau hạch Gasser được không?", "en": "Can patients with trigeminal nerve pain undergo selective nerve root section surgery after Gasser's ganglion?", "zh": "是否有痛觉神经根V的病例可以进行加斯珀淋巴结后选择性神经根切断手术？"}, {"id": 27, "vi": "Đau khớp cổ tay, cổ chân và khớp ngón tay, ngón chân. Con khoai và được chuẩn đoán viêm túi dịch khớp gối đã được bác sĩ tại viện cho uống thuốc meloxicam nhưng đến nay vẫn đau và đi lại rất khó khăn vậy bác sĩ cho em thêm tư vấn.", "en": "Pain in the elbow, knee, and finger and toe joints. The patient has been diagnosed with knee joint effusion and has been prescribed meloxicam by the doctor at the hospital, but pain persists and mobility remains very difficult. Thus, the doctor kindly requests further advice.", "zh": "手腕、脚踝和手指、脚趾疼痛。医生在医院诊断为膝关节滑膜炎，已开具美洛昔康药物治疗，但至今仍疼痛，行走非常困难，因此恳请医生给予进一步指导。"}, {"id": 28, "vi": "Đau 2 bên răng nhai trong quá trình nới răng khi niềng răng có sao không?", "en": "Is it problematic to have pain on both sides of the teeth during the process of loosening teeth when wearing braces?", "zh": "戴牙套过程中，松牙时两侧咀嚼牙疼痛吗？"}, {"id": 29, "vi": "Bà cháu năm nay hơn 80 tuổi, bà em tự dựng bị đau chân và tay cùng một bên trái. Không nhấc tay và chân lên được. Đau nhất là phần khuỷu tay, chân. Mỗi lần tay nhấc lên sẽ bị giật giật ở khuỷu tay. Bác sĩ chẩn đoán xem bà cháu bị sao vậy ạ?", "en": "The patient is over 80 years old. The elder sister has been experiencing pain in the left arm and leg. She is unable to lift her arm or leg. The pain is most severe at the elbow and knee. Each time she tries to lift her arm, she experiences a jerking sensation at the elbow. Doctor, what diagnosis do you think the patient has?", "zh": "这位老人家今年超过80岁，她妹妹自己出现一侧左侧手脚疼痛，无法抬起手和脚。疼痛最严重的是肘部和脚踝部位。每次抬手时肘部都会出现抽搐。医生请问一下，老人家这是什么病？"}, {"id": 30, "vi": "Bị kim tiêm lấy máu đâm vào tay có nguy hiểm không?", "en": "Is it dangerous to have a blood draw needle prick on the arm?", "zh": "打血样针扎到手臂有危险吗？"}, {"id": 31, "vi": "Cho em hỏi kỹ thuật soi tươi khác kỹ thuật soi trực tiếp vi khuẩn nhuộm soi (nhuộm gram, nhuộm xanh methylen) như thế nào? Tại sao em làm kỹ thuật soi tươi dịch niệu đạo tất cả các kết quả âm tính mà nhuộm gram, nhuộm xanh kết quả bạch cầu 3+, cầu khuẩn gram dương +, trực khuẩn gram dương +, cầu khuẩn gram âm -, trực khuẩn gram âm -?", "en": "May I ask how fresh wet mount differs from direct bacterial staining (Gram staining, methylene blue staining)? Why do I get all negative results with fresh wet mount of urethral secretions, while Gram and methylene blue staining show 3+ white blood cells, Gram-positive cocci, Gram-positive rods, Gram-negative cocci, and Gram-negative rods?", "zh": "我想问一下新鲜涂片检查和染色细菌涂片检查（革兰染色、美蓝染色）之间的区别是什么？为什么我做尿道分泌物新鲜涂片检查所有结果均为阴性，而革兰染色和美蓝染色结果却显示白细胞3+，革兰阳性球菌+，革兰阳性杆菌+，革兰阴性球菌-，革兰阴性杆菌-？"}, {"id": 32, "vi": "Nghẹt, chảy nước mũi, hắt hơi liên tục khi thay đổi thời tiết hay ra chỗ đông người, phải làm sao?", "en": "Nasal congestion, runny nose, and continuous sneezing when weather changes or entering crowded places—what should I do?", "zh": "天气变化或进入人群密集场所时出现鼻塞、流鼻涕、打喷嚏，该怎么办？"}, {"id": 33, "vi": "Con em được 8 tháng ạ, em bị ít sữa, tối thì bé ti mẹ, mẹ không đủ sữa, bỏ ti nhưng ti bình bé lại không ạ, nhưng nước thì bé ti ạ. Vậy để làm sao bé ti bình ạ, bé bị chậm tăng cân, mỗi lần ăn hay uống sữa đều khóc ạ.", "en": "The child is 8 months old. The baby has poor milk supply, and at night the baby feeds from mother, but the mother does not have enough milk, so the baby stops breastfeeding but starts feeding from bottle, yet the baby drinks milk. So, how can the baby feed from bottle? The baby has slow weight gain, and cries every time feeding or drinking milk.", "zh": "宝宝8个月，奶量少，晚上宝宝吃母乳，妈妈奶水不够，停止喂母乳后，改用奶瓶喂养，宝宝反而不喝奶瓶，但会喝奶。请问该如何让宝宝喝奶瓶？宝宝生长发育迟缓，每次吃奶或喝奶都会哭闹。"}, {"id": 34, "vi": "Điều trị viêm nhiễm tuyến lệ khi cho con bú thế nào?", "en": "How to treat blepharitis during breastfeeding?", "zh": "哺乳期间如何治疗泪腺感染？"}, {"id": 35, "vi": "Con gái tôi năm nay 5 tuổi, cháu kém ăn và gầy, cháu có cần phải xét nghiệm máu nhũng chỉ số nào để điều chỉnh chế độ ăn cho cháu không.", "en": "My daughter is now 5 years old. She has poor appetite and is underweight. Do we need to perform any blood tests to check for any indices in order to adjust her diet?", "zh": "我女儿今年5岁，食欲差、消瘦，需要做哪些血液检查来调整她的饮食方案吗？"}, {"id": 36, "vi": "Con tôi bị sốt 1 ngày đã hết, sau khi hết sốt nổi mẩn đỏ li ti khắp cơ thể, xin hỏi bác sĩ là con tôi bị bệnh gì ạ.", "en": "My child had a fever for one day then it resolved, after the fever subsided, a mild red rash appeared all over the body. I would like to ask the doctor what illness my child has.", "zh": "我孩子一天前发烧，现在已经退烧，退烧后全身出现轻微的红疹，想请教医生孩子得的是什么病？"}, {"id": 37, "vi": "Rối loạn sinh tủy sau khi hóa trị bạch cầu cấp dòng tủy nghĩa là gì?", "en": "What does bone marrow dysfunction after acute myeloid leukemia chemotherapy mean?", "zh": "急性髓系白血病化疗后出现造血功能障碍是什么意思？"}, {"id": 38, "vi": "Điều trị trào ngược dạ dày thực quản như thế nào?", "en": "How to treat gastroesophageal reflux disease?", "zh": "如何治疗胃食管反流病？"}, {"id": 39, "vi": "Khối u dưới niêm mạc hang vị dạ dày có nên sinh thiết không", "en": "Should the submucosal mass in the gastric antrum be biopsied?", "zh": "胃窦黏膜下肿块是否需要活检"}, {"id": 40, "vi": "Đau dương vật theo từng cơn, xuất hiện nhiều về chiều và tối phải làm sao?", "en": "Pain in the penis occurring in episodes, appearing more in the afternoon and evening, what should I do?", "zh": "阴茎间歇性疼痛，尤其在下午和晚上加重，该怎么办？"}, {"id": 41, "vi": "Em năm nay 25 tuổi. Em nằm ngủ hay bị tinh trùng ra. Em bị sao và chữa như thế nào?", "en": "I am 25 years old. I often have semen leakage while sleeping. What is the cause and how should it be treated?", "zh": "我今年25岁。我睡觉时经常有精液流出。我是什么原因，该怎么治疗？"}, {"id": 42, "vi": "Mẹ bầu tuần 35 bị đau nhức vùng háng có ảnh hưởng đến thai nhi không?", "en": "Does a pregnant mother at 35 weeks experiencing pain and discomfort in the hip area affect the fetus?", "zh": "怀孕35周的母亲出现髋部疼痛，是否会影响胎儿？"}, {"id": 43, "vi": "Tôi bị đau lưng khi cúi xuống và bị tức thăn hai bên thận mà tiền sử tôi bị thoát vị đĩa đệm, tôi tập thể dục thường xuyên như thế tôi có vấn đề gì về thận không bác sĩ nay tôi 48 tuổi.", "en": "I have lower back pain when bending down and have flank pain on both sides of the kidneys, and I have a history of disc herniation. I exercise regularly. Doctor, I am 48 years old. Do I have any kidney problems?", "zh": "我弯腰时会感到腰痛，同时两侧肾脏有压迫感，我有椎间盘突出病史，平时经常锻炼，医生，我现在48岁，我是否有肾脏问题？"}, {"id": 44, "vi": "Môi sưng, tai ngứa kèm da phát ban là dấu hiệu của bệnh gì?", "en": "Swollen mouth, itchy ear accompanied by skin rash – what disease is this?", "zh": "口肿、耳痒伴皮疹是哪种疾病的症状？"}, {"id": 45, "vi": "Bé 5 tháng tuổi bộ phận sinh dục nữ ngày nào cũng ra ít dịch trong có sao không", "en": "5-month-old girl, female genital area secretes a small amount of discharge daily, is this normal?", "zh": "5个月大的女孩，外阴每天都会少量分泌物，是否正常"}, {"id": 46, "vi": "Em có kinh ngày mùng 9 và kỳ kinh của em tháng này 32 ngày.Kỳ kinh của em thường là 30.32 ngày.Em quan hệ ngày 20.21.23 thì có mang thai được không và nếu có thì ngày bao nhiêu em có thể xét nghiệm máu để bit mình đã có thai.Em cũng muốn có thai tháng này.", "en": "I have my period on the 9th and this month my cycle is 32 days. My usual cycle is 30-32 days. I had intercourse on days 20, 21, and 23. Could I be pregnant? And if so, on which day can I take a blood test to confirm pregnancy? I also want to get pregnant this month.", "zh": "我的月经是每月9号来，这个月的周期是32天，平时周期是30-32天。我分别在20、21、23天有性生活，这样会怀孕吗？如果怀孕了，我可以在哪一天做血液检查来确认是否怀孕？我也希望这个月能怀孕。"}, {"id": 47, "vi": "Kiểm tra vết va đập xương ống chân bằng cách nào?", "en": "How to examine the femoral shaft fracture?", "zh": "如何检查小腿骨挫伤？"}, {"id": 48, "vi": "Người nhà tôi vừa đi truyền hóa chất 1 ngày, hiện tại thấy buồn nôn nhiều. Người nhà tôi có thể uống thuốc gì được không?", "en": "My family member just finished a chemotherapy infusion one day ago, and is now experiencing significant nausea. Can my family member take any medication?", "zh": "我家人刚接受了1天的化疗，现在出现明显恶心。家人可以吃些什么药呢？"}, {"id": 49, "vi": "Tôi có huyết áp trung bình khoảng 140/90 mmHg, trước tiêm có ngậm viên thuốc hạ huyết áp, sau đó đo lại thì huyết áp trong mức cho phép và được tiêm vaccine. Sau tiêm 12 giờ, tôi có phản ứng ớn lạnh, sốt 37,4 độ C, khó ngủ đêm đầu tiên, chỗ tiêm hơi đau, sáng hôm sau thì hết hẳn. Huyết áp theo dõi các ngày sau đó vẫn ở mức trung bình 140/90 mmHg. Xin hỏi tôi có cần phải uống thuốc huyết áp không? Ngoài ra, tôi có thể sử dụng thực phẩm chức năng để ổn định huyết áp, ngừa cục máu đông không?", "en": "My blood pressure is around 140/90 mmHg, before injection I took a tablet to lower blood pressure, and after that my blood pressure was within the permissible range and I received the vaccine. Twelve hours after injection, I experienced chills, fever of 37.4°C, difficulty sleeping during the first night, mild pain at the injection site, and by the next morning these symptoms completely resolved. My blood pressure has remained around 140/90 mmHg in the following days. I would like to ask whether I need to take blood pressure medication? Additionally, can I use dietary supplements to stabilize my blood pressure and prevent blood clots?", "zh": "我的血压大约为140/90 mmHg，注射前服用了降压药，注射后测量血压在允许范围内，随后进行了疫苗接种。注射12小时后，我出现了发冷、体温37.4℃、首夜睡眠困难、注射部位轻微疼痛，第二天早上症状完全消失。此后几天的血压监测仍维持在140/90 mmHg。请问，我是否需要长期服用降压药？此外，我能否使用保健品来稳定血压、预防血栓形成？"}, {"id": 50, "vi": "Bố em bị bệnh Alzheimer do ty thể thì bố có truyền bệnh cho người con nào trong gia đình không?", "en": "Does the father have Alzheimer's disease due to mitochondrial issue, and can he pass on the disease to any of his family children?", "zh": "我哥哥因线粒体疾病患有阿尔茨海默病，他会不会把这种病传给家庭中的任何一个子女？"}, {"id": 51, "vi": "Cách điều trị đau đầu, tắc mạch máu kèm rối loạn tiền đình?", "en": "How to treat headache, cerebral vasospasm accompanied by vestibular disorder?", "zh": "偏头痛、脑血管阻塞伴前庭功能障碍的治疗方法？"}, {"id": 52, "vi": "Thiểu sản động mạch thông sau hai bên là gì?", "en": "What is bilateral posterior cerebral artery insufficiency?", "zh": "双侧后动脉发育不良是什么？"}, {"id": 53, "vi": "Khó thở, đau tức ngực là biểu hiện của bệnh gì?", "en": "Difficulty breathing, chest pain is a sign of what disease?", "zh": "呼吸困难、胸痛是何种疾病的症状？"}, {"id": 54, "vi": "Đã mổ niệu đạo nhưng có triệu chứng của rối loạn cương dương phải điều trị thế nào?", "en": "Has undergone urethral surgery but has symptoms of erectile dysfunction—how should it be treated?", "zh": "已进行尿道手术，但出现勃起功能障碍，应如何治疗？"}, {"id": 55, "vi": "Sụt cân nhiều, mỏi ở đầu gối là dấu hiệu bệnh gì?", "en": "Rapid weight loss, joint pain in the knees - what disease is this a sign of?", "zh": "体重明显下降，膝盖乏力是何种疾病的症状？"}, {"id": 56, "vi": "Mũi có vết bớt đỏ dài lan xung quanh mũi điều trị thế nào?", "en": "What is the treatment for a red, long scar around the nose?", "zh": "鼻子周围有一条红色的疤痕，该如何治疗？"}, {"id": 57, "vi": "Cháu mới khám ở bên mình được khoảng 3 tháng, có Bác sĩ gọi tư vấn kết quả có nói là lượng bạch cầu của cháu thấp tuy nhiêu không có vấn đề gì. Gần đây thỉnh thoảng cháu có bị đánh trống ngực, hồi hộp vậy cho cháu hỏi tình trạng này có vấn đề gì không", "en": "My child has been visiting our clinic for about 3 months. The doctor called for consultation and mentioned that the child's white blood cell count is low, but there is no problem. Recently, the child occasionally experiences palpitations and restlessness. I would like to ask whether this condition is problematic.", "zh": "我大约三个月前在你们这里就诊，医生看过结果后告诉我白细胞数量偏低，但并无问题。最近偶尔会感到心悸、胸闷，想问问这种情况是否有问题？"}, {"id": 58, "vi": "Làm cách nào để hết mụn vĩnh viễn.", "en": "How to get rid of persistent acne.", "zh": "如何才能消除顽固性痘痘。"}, {"id": 59, "vi": "Làm thế nào để giảm đau khi bị chàm khô trong lòng bàn chân?", "en": "How to reduce pain when suffering from dry eczema on the sole of the foot?", "zh": "如何缓解足底干性湿疹的疼痛？"}, {"id": 60, "vi": "Cháu đang là học sinh lớp 10. Gần đây mắt cháu có cảm giác khô liên tục, da mặt cũng xấu đi có thể do tình trạng thiếu hụt vitamin A trong thực đơn hằng ngày. Bác sĩ có thể hướng dẫn cháu cách để cung cấp các loại vitamin cũng như các lượng chất cần thiết để cải thiện tình trạng trên một cách khoa học nhất không.", "en": "I am a 10th-grade student. Recently, I have been experiencing continuous dry eyes, and my facial skin has worsened, possibly due to a deficiency of vitamin A in my daily diet. Could the doctor please guide me on how to provide various vitamins and the necessary amounts of nutrients in a scientific way to improve my condition?", "zh": "我目前是高一学生。最近眼睛有持续干涩的感觉，面部皮肤也变差了，可能是因为日常饮食中缺乏维生素A。医生能否指导我如何科学地补充各类维生素以及所需剂量，以改善上述情况？"}, {"id": 61, "vi": "Có 2 nang thận phải có cần uống thuốc không", "en": "Are 2 right kidney cysts requiring medication?", "zh": "右侧有2个肾囊，是否需要服药？"}, {"id": 62, "vi": "Có nhiều ý kiến cho rằng tiêm vaccine Covid-19 của AstraZeneca có thể gây ra tình trạng đông máu, xuất hiện cục máu đông. Vậy chúng ta có cần đi xét nghiệm tình trạng đông máu của cơ thể hay không trước khi tiêm vaccine Covid-19?", "en": "There are many opinions suggesting that receiving the AstraZeneca COVID-19 vaccine may cause blood clotting conditions, including the formation of blood clots. Therefore, should we undergo a blood coagulation test before receiving the COVID-19 AstraZeneca vaccine?", "zh": "有许多观点认为接种阿斯利康新冠疫苗可能导致血液凝固，出现血栓。那么，在接种新冠疫苗前，我们是否需要检查身体的凝血状况呢？"}, {"id": 63, "vi": "Em bầu được 36 tuần, đêm qua em có bị són tiểu liên tục, sáng nay dậy em thấy có dịch nhầy trắng trong và một ít máu. Bây giờ vẫn còn hiện tượng són tiểu có chút nhầy nhưng không có máu. Em như vậy là bị sao ạ", "en": "I am 36 weeks pregnant. Last night I experienced continuous urinary incontinence. This morning, upon waking, I noticed a white, mucous discharge and a little blood. Now I still have a slight urinary incontinence with a bit of mucus, but no blood. What is this condition?", "zh": "我怀孕36周，昨晚开始反复出现尿失禁，今天早上醒来发现有白色黏液和少量血液。现在仍有少量黏液性尿失禁，但没有血液。我这种情况是怎样的？"}, {"id": 64, "vi": "Bác sĩ ơi cho em hỏi bố em bị tiểu đường + tăng huyết áp + gút + mỡ máu thì bố em nên ăn uống như thế nào ạ?", "en": "Doctor, I would like to ask you: my father has diabetes + hypertension + gout + high cholesterol. What should my father eat?", "zh": "医生您好，我父亲患有糖尿病、高血压、痛风和高血脂，我父亲应该怎样饮食呢？"}, {"id": 65, "vi": "Thân nhiệt 37.4 độ C trong chu kỳ kinh có phải là sốt không?", "en": "Is a body temperature of 37.4°C during the menstrual cycle considered fever?", "zh": "体温37.4℃在月经周期内算不算发热？"}, {"id": 66, "vi": "Cách chăm sóc trẻ sốt phát ban tại nhà", "en": "How to care for a child with fever and rash at home", "zh": "如何在家中护理发热出疹的儿童"}, {"id": 67, "vi": "Kết quả men gan GPT 69,76 liệu có nguy hiểm không?", "en": "Is a liver enzyme GPT result of 69.76 dangerous?", "zh": "转氨酶GPT 69.76是否危险？"}, {"id": 68, "vi": "Phương pháp khắc phục tình trạng trẻ sơ sinh bẹp đầu", "en": "Methods to correct the condition of newborns with flat head", "zh": "纠正新生儿头颅扁平的治疗方法"}, {"id": 69, "vi": "Kích thước tiền liệt tuyến 33x24x24mm, khối echo trống 6.4mm có nguy hiểm gì không?", "en": "Is there any danger with a prostate gland size of 33x24x24mm and a 6.4mm anechoic area?", "zh": "前列腺大小 33×24×24 毫米，回声空缺 6.4 毫米，是否有危险？"}, {"id": 70, "vi": "Nghẹn họng, ợ nóng sau khi uống thuốc điều trị viêm họng là do đâu?", "en": "Throat obstruction, acid reflux after taking medication for pharyngitis - what is the cause?", "zh": "服咽喉炎药物后出现咽喉堵塞、反酸的原因是什么？"}, {"id": 71, "vi": "Dạo gần một tuần gần đây em đi táo bón và lỏng 2 tuần trước thì táo bón,phân gần tuần nay mỗi ngày một hình dạng khác nhau,hôm nay buồn đi vệ sinh đến 3 lần cách dây chín tháng em bị nứt kẽ hậu môn và em bị trĩ nữa ,hôm nay em chùi đít có một tí máu ,máu không bắn giọt ,không lẫn trong phân không có nhầy ,mà em thấy biểu hiện của em rất giống bệnh hội chứng kích thích ở ruột như đau lưng ,đau ngực ,đau quặn bụng từng cơn dưới rốn ,đi vệ sinh với hết đau ,cảm giác không hết phân ,hay bị đầy hơi chướng bụng ,bụng không đói mà vẫn sôi ùng ục ,cho em hỏi bệnh trĩ hoặc viêm nứt kẽ hậu môn và hội chứng kích thích ở ruột có tồn tại một lúc không", "en": "Recently, I have had constipation for nearly a week, and two weeks ago I also had constipation. For the past week, my stool has been of different forms each day. Today, I have felt the urge to defecate three times. Nine months ago, I developed anal fissures and I also have hemorrhoids. Today, when I clean my anus, I noticed a small amount of blood, which is not spraying, not mixed with the stool, and there is no mucus. I feel my symptoms are very similar to irritable bowel syndrome, such as back pain, chest pain, cramping pain under the navel in episodes, relief after defecation, feeling of incomplete evacuation, frequent bloating and gas, and abdominal distension without hunger. I would like to know whether hemorrhoids or anal fissure and irritable bowel syndrome can coexist at the same time.", "zh": "最近一周我有便秘，两周前也有便秘，最近一周每天大便形态都不同，今天排便次数达到三次，距离三个月前我出现肛裂和痔疮已有九个月了，今天擦拭肛门时发现有少量血液，血液没有喷溅，不混在大便中，也没有黏液。我感觉自己的症状很像肠易激综合征，如背痛、胸痛、脐部以下间歇性腹痛、排便后疼痛缓解、排便不尽感、经常胀气、腹部不饿却有咕噜声。请问，痔疮或肛裂和肠易激综合征可以同时存在吗？"}, {"id": 72, "vi": "Trẻ sơ sinh viêm đường hô hấp có thể ngủ phòng máy lạnh không?", "en": "Can newborns with respiratory infection sleep in a room with air conditioning?", "zh": "新生儿呼吸道感染可以放在空调房里睡觉吗？"}, {"id": 73, "vi": "Em bầu 3 tháng mà hay cúm và giờ có hiện tượng đau họng và ho nữa ạ. Giờ em phải làm thế nào ạ?", "en": "I am 3 months pregnant and often have flu, and now I have throat pain and cough as well. What should I do now?", "zh": "我怀孕三个月，经常感冒，现在又出现喉咙痛和咳嗽的情况，现在该怎么办？"}, {"id": 74, "vi": "Nên uống thuốc và kết hợp vận động thế nào để giúp phục hồi chức năng vận động, ngôn ngữ sau đột quỵ não?", "en": "What combination of medication and exercise should be taken to help restore motor and language function after stroke?", "zh": "应该如何用药并结合运动以帮助脑卒中后运动和语言功能的恢复？"}, {"id": 75, "vi": "Vì sao trẻ sơ sinh bị phồng bao quy đầu khi đi tiểu?", "en": "Why do newborns have penile ballooning when urinating?", "zh": "为什么新生儿小便时包皮 ballooning？"}, {"id": 76, "vi": "Sau cắt hạch giao cảm tỷ lệ tái phát lại có cao không?", "en": "After sympathetic ganglionectomy, is the recurrence rate high?", "zh": "交感神经切除后复发率高吗？"}, {"id": 77, "vi": "Cách đây 2 tháng em cố rặn nhổ nước bọt ra ít máu, đi CT phổi và siêu âm ổ bụng bác sĩ chỉ kết luận bị viêm phế quản và mới đây em nội soi họng cũng không việc gì mà thi thoảng em cứ cố rặn ra nước bọt mà thi thoảng nó cũng ra máu là bệnh gì, máu nó cũng không đậm nó hơi nhạt thôi, kiểu là em nhổ nước bọt bình thường nó không ra máu mà em cứ nhổ mạnh mà không có nước bọt thì nó lại chảy máu.", "en": "Two months ago, I forcibly tried to expel saliva and noticed a small amount of blood. I underwent a chest CT and abdominal ultrasound, and the doctor concluded I had bronchitis. Recently, I had a laryngoscopy which showed no abnormalities. However, whenever I forcefully try to expel saliva, occasionally it comes out with blood. The blood is not dark; it's quite light. Normally, when I expel saliva, there is no blood, but when I forcefully try to expel saliva without any saliva, it starts to bleed.", "zh": "两个月前我用力挤出唾液时有少量出血，做了胸部CT和腹部超声，医生诊断为支气管炎。最近我做了喉镜检查，没有发现问题。但最近我总是用力挤唾液，有时会出血，血色不深，比较淡。平时正常挤唾液不会出血，只有当我用力挤而没有唾液时才会出血。"}, {"id": 78, "vi": "Có biện pháp nào để phòng ngừa viêm cầu khuẩn nhóm B lây từ mẹ sang con không ạ? Làm sao để biết mẹ đã mắc liên cầu khuẩn nhóm B?", "en": "Is there any measure to prevent group B streptococcus maternal-to-fetal transmission? How to know if the mother has group B streptococcus?", "zh": "有没有预防母婴传播B族链球菌感染的措施？如何判断母亲是否感染了B族链球菌？"}, {"id": 79, "vi": "Uống thuốc tránh thai khi đang cho con bú có được không?", "en": "Can I take oral contraceptive medication while breastfeeding?", "zh": "哺乳期间服用避孕药可以吗？"}, {"id": 80, "vi": "Em là nữ 29 tuổi. Khoảng 3 hôm gần đây em để ý cổ có nốt ngứa. Vì nghĩ do muỗi đốt nên em chỉ bôi dầu và cũng có gãi 1 chút. Tình trạng các nốt không hết mà còn lan thêm những nốt khác ra xung quanh như ảnh, trông gần như muỗi đốt nhưng đỏ hơn và nhỏ. Gãi xước da thì hết ngứa nhưng lại lên nốt khác. Bác sĩ cho em hỏi đây có thể là bệnh gì và em nên làm thế nào?", "en": "I am a 29-year-old female. About 3 days ago, I have noticed itching bumps on my neck. Since I thought it was due to mosquito bites, I only applied oil and also scratched a little. The bumps have not gone away and have instead spread to additional bumps around the area, as shown in the photo; they look similar to mosquito bites but are redder and smaller. Scratching causes the itching to subside but new bumps appear. Doctor, I would like to ask what disease this could be and what I should do?", "zh": "我是一名29岁的女性。最近大约三天，我注意到脖子上出现了瘙痒的丘疹。因为怀疑是蚊虫叮咬，所以我只涂了油，并且轻轻抓了几次。但这些丘疹并没有消失，反而在周围又出现了新的丘疹，如图片所示，看起来像蚊虫叮咬，但颜色更红，且更小。抓挠后皮肤发红，瘙痒会缓解，但又会冒出新的丘疹。医生请问，这可能是哪种疾病？我应该怎么办？"}, {"id": 81, "vi": "Kết quả chụp X - quang đậm 2 bên rốn phổi là sao?", "en": "What is the result of the bilateral thoracic spine X-ray?", "zh": "双侧肺门X光片结果如何？"}, {"id": 82, "vi": "Đau trong miệng kèm theo sưng má có nguy hiểm không?", "en": "Is pain in the mouth accompanied by swelling of the cheek dangerous?", "zh": "口腔疼痛伴面部肿胀是否危险？"}, {"id": 83, "vi": "Khi nhỏ lọ thuốc natri clorid vào thì đỡ được khoảng tầm 4 đến 5 phút sau đó là lại mỏi và khó chịu.", "en": "When administering the sodium chloride solution, relief is achieved for about 4 to 5 minutes, then muscle fatigue and discomfort return.", "zh": "滴注氯化钠溶液后，约4到5分钟会感觉好转，之后又出现疲劳和不适。"}, {"id": 84, "vi": "Bé nhà em 5 tháng 22 ngày, nặng 10kg, bé nhà em ăn cả sữa ngoài đó là loại sữa meiji của nhật. Em đã cho bé ăn dặm lúc 5 tháng 7 ngày.Mỗi lần bé đi ngoài thì thường 3 ngày đi 1 lân mà mỗi lần đi phân 3 ngày 1 lần thì phân màu xanh còn phân2 ngày thì phân màu vàng.Hôm nay bé tự nhiên phân màu vàng bình thường mà có ra hạt đen đen dây dài khoảng 2mm( như đàn kiến đen vây).", "en": "My child is 5 months and 22 days old, weighing 10 kg. My child drinks formula milk, specifically Meiji Japanese milk. I started giving my child solid food at 5 months and 7 days. Every time my child has a bowel movement, it usually occurs every 3 days, and when the bowel movement happens every 3 days, the stool is blue; when it happens every 2 days, the stool is yellow. Today, my child's stool is normally yellow, but there are black, long, thread-like particles about 2 mm in length (resembling black ants' trails).", "zh": "我女儿5个月22天，体重10kg，她吃母乳外还吃日本的美津浓奶粉。我从5个月7天开始给她添加辅食。每次她排便通常3天一次，如果3天排一次大便，大便呈绿色；如果2天排一次，大便呈黄色。今天她突然大便呈正常黄色，但出现了黑色细长的颗粒，长约2mm（像黑色蚂蚁的触角）。"}, {"id": 85, "vi": "Tôi thuộc nhóm nào đối với tiêm vắc xin Covid-19 vì có một số bệnh sau: 1. Hở van tim 2 lá nhẹ, thỉnh thoảng tức ngực, khó thở, mệt mỏi, đang uống thuốc mỗi ngày một viên. 2. Thỉnh thoảng dị ứng, mề đay, mẩn ngứa do cơ địa dị ứng. 3. Ung thư vú đã điều trị ổn định 14 năm. 4. Huyết áp thường thấp 100/60, mạch đập không ổn định, 60-85. 5. Thỉnh thoảng đau đầu, tiền đình, viêm xoang. 6. Viêm đại tràng, viêm dạ dày.", "en": "Which group do I belong to for receiving the Covid-19 vaccine because I have the following conditions: 1. Mitral valve defect with two leaflets, mild, occasionally chest pain, shortness of breath, fatigue, currently taking one tablet per day. 2. Occasionally allergic, hives, itching due to allergic constitution. 3. Breast cancer already treated and stable for 14 years. 4. Blood pressure usually low at 100/60, irregular pulse, 60-85. 5. Occasionally headache, vertigo, sinusitis. 6. Colitis, gastritis.", "zh": "我属于哪一类人群可以接种新冠疫苗，因为我有以下一些疾病：1. 二尖瓣轻度关闭不全，偶尔有胸痛、呼吸困难、乏力，目前正在每日服用一片药物。2. 偶尔出现过敏、荨麻疹、瘙痒，因体质过敏所致。3. 乳腺癌已稳定治疗14年。4. 血压通常偏低，为100/60，心率不稳定，为60-85次/分钟。5. 偶尔有头痛、眩晕、鼻窦炎。6. 肠炎、胃炎。"}, {"id": 86, "vi": "Chỉ số hồng cầu 5,93 nhưng thể tích hồng cầu nhỏ hơn bình thường có ảnh hưởng gì không?", "en": "What impact does a hemoglobin index of 5.93 but a lower-than-normal hematocrit have?", "zh": "红细胞计数为5.93，但红细胞体积低于正常值，这会有何影响？"}, {"id": 87, "vi": "Điều trị trầm cảm nhẹ bằng thuốc gì tốt nhất?", "en": "What is the best medication for treating mild depression?", "zh": "治疗轻度抑郁症最好的药物是什么？"}, {"id": 88, "vi": "Kết quả người nam có tăng kích thước vệ tinh nhiễm sắc thể số 21 là như thế nào?", "en": "What are the results for a male patient with increased satellite size of chromosome 21?", "zh": "男性患者染色体21号卫星结构增大检查结果如何？"}, {"id": 89, "vi": "Run tay, khó thở, kèm theo rối loạn kinh nguyệt là triệu chứng của bệnh tuyến giáp?", "en": "Hand tremor, difficulty breathing, accompanied by menstrual disorder, are symptoms of thyroid disease?", "zh": "手抖、呼吸困难，伴有月经紊乱是甲状腺疾病的症状吗？"}, {"id": 90, "vi": "Trẻ trên 3 tuổi uống sữa công thức có khả năng dậy thì sớm không?", "en": "Can children over 3 years of age who drink formula milk develop precocious puberty?", "zh": "3岁以上儿童喝配方奶会引发早熟吗？"}, {"id": 91, "vi": "Bạch cầu mono giảm 2.1% có nguy hiểm hay có vấn đề gì không?", "en": "Is a monocytosis reduction of 2.1% dangerous or indicative of any problem?", "zh": "单核细胞白细胞减少2.1%是否有危险或存在什么问题？"}, {"id": 92, "vi": "Bệnh dại có lây qua máu từ động vật sang người không ạ?", "en": "Can rabies be transmitted from animals to humans through blood?", "zh": "狂犬病能通过动物的血液传播给人类吗？"}, {"id": 93, "vi": "Vào ngày 15 em bị ra máu bất thường, sau khi nhập viện kiểm tra thì kết luận em bị sảy thai sớm (beta kiểm tra khoảng 355). Đến ngày 18 tháng 12 thì em được xuất viện và máu ngừng chảy vào sáng 19 tháng 12. Em có đi kiểm tra lại và siêu âm đầu dò thì kết quả hoàn toàn bình thường. Nhưng đến ngày 26 em lại bị ra máu ồ ạt và có cục máu đông chảy ra. Em có gọi điện cho người phụ trách thì họ bảo theo dõi ở nhà vì có thể nhau đợt trước chưa bong hết ra và không có vấn đề gì. Vậy trường hợp của em có bị sao không? Nếu khám thì nên khám chuyên sâu như thế nào để đảm bảo cho đợt mang thai tiếp theo ạ?", "en": "On day 15, I experienced abnormal bleeding. After admission and examination, I was diagnosed with early miscarriage (beta test result around 355). On December 18, I was discharged from the hospital and bleeding stopped on the morning of December 19. I had a follow-up check-up and transvaginal ultrasound, which showed completely normal results. However, on day 26, I experienced heavy bleeding with clots passing out. I called the responsible person, who advised me to stay at home and monitor, as it might be that the previous miscarriage tissue had not completely expelled and there was no serious issue. So, is my case problematic? If I go for a check-up, what specialized examination should I undergo to ensure a safe subsequent pregnancy?", "zh": "15号出现异常出血，入院检查后诊断为早期流产（β值约为355）。12月18日出院，12月19日上午出血停止。之后我进行了复查和阴道超声检查，结果完全正常。但26号又出现大量出血，并有血块排出。我致电负责人员，他们建议在家观察，因为可能之前流产的组织未完全排出，目前没有问题。请问我的情况是否存在问题？如果需要检查，应该进行哪些专项检查以确保下次怀孕的安全？"}, {"id": 94, "vi": "Cổ tử cung ngắn có hay xảy ra hiện tượng đau cứng bụng không ạ? Vài ngày trở lại đây em hay bị tức bụng phía dưới cửa mình. Có phải bị tụt cổ tử cung xuống thấp không ạ? Nguyên nhân của hiện tượng này là sao ạ?", "en": "Does a short cervix often cause abdominal pain and stiffness? Recently, I have been experiencing lower abdominal discomfort near my vaginal opening. Could this be due to cervical descent? What is the cause of this phenomenon?", "zh": "宫颈短是否有腹痛僵硬的情况？最近几天我经常感到下腹靠近阴道处有不适感。这是宫颈脱垂吗？这种情况的原因是什么？"}, {"id": 95, "vi": "Em hỏi là em bị chậm kinh 9 ngày thì em đi siêu âm đầu dò thì thấy ra máu sau khi siêu âm và từ lúc siêu âm song là em bắt đầu cảm thấy bụng dưới đau âm ỉ và đến ngày hôm nay là em thấy máu màu nâu đen nó ra cho em hỏi là có phải em bị sảy thai rồi không?", "en": "I want to ask: I have missed my period by 9 days, then I did a transabdominal ultrasound and noticed bleeding after the ultrasound. Since the ultrasound, I have started feeling a dull pain in my lower abdomen, and today I notice brown-black bleeding. I want to know if this means I have had a miscarriage?", "zh": "我问一下，我月经推迟了9天，去做经阴道超声检查时发现检查后有出血，从做超声检查开始我就开始感到下腹隐隐作痛，到今天我看到的是褐色的血，我想问一下，我是不是流产了？"}, {"id": 96, "vi": "Bại liệt có thể điều trị bằng ghép tế bào gốc không?", "en": "Can stem cell transplantation be used to treat paralysis?", "zh": "再生细胞移植能否治疗偏瘫？"}, {"id": 97, "vi": "Xét nghiệm máu chỉ số HGB là 99.0, chỉ số HTC là 0.307 có nguy hiểm không?", "en": "Blood test values: HGB is 99.0, HTC is 0.307. Is this dangerous?", "zh": "血常规检查血红蛋白值为99.0，血细胞比容为0.307，是否有危险？"}, {"id": 98, "vi": "Đường ruột tôi bị yếu, mỗi khi ăn đồ lạ là bị đau, còn quặn bụng, đi cầu hoài. Bây giờ tôi đang uống thuốc Biolac viên nang 500mg thì thấy đỡ hơn nhiều. Tôi uống hoài có sao không?", "en": "My intestines are weak; every time I eat something unfamiliar, I get pain, cramps, and a constant urge to have a bowel movement. Currently, I am taking Biolac capsule 500mg, and I feel much better. Is it okay to keep taking this medication?", "zh": "我的肠道功能虚弱，一吃陌生食物就会腹痛、腹绞痛，还总是想排便。现在我一直在吃500mg的Biolac胶囊，感觉明显好转。我这样一直吃可以吗？"}, {"id": 99, "vi": "Em có xét nghiệm chỉ số AMH và được kết quả 1.555. Chỉ số này có thấp quá không? Và có ảnh hưởng nhiều đến số lượng trứng không?", "en": "My AMH test result is 1.555. Is this value too low? And does it significantly affect the number of eggs?", "zh": "我做了AMH检测，结果是1.555。这个数值偏低吗？会对卵子数量产生很大影响吗？"}, {"id": 100, "vi": "Con tôi 10 tháng bị sốt hết rồi giờ phát ban khắp người có sao không", "en": "My child 10 months old had fever for a while, then now has rash all over the body, is there anything wrong?", "zh": "我孩子10个月，发烧好了之后现在全身出疹，有没有问题？"}, {"id": 101, "vi": "Tôi hay bị đỏ mặt, tôi tìm hiểu thì được biết là 1 loại bệnh. Ở viện mình có khám về bệnh này không ạ, khi đi khám thì có phải làm xét nghiệm gì không? Bệnh này có phải uống thuốc nhiều không và uống trong bao lâu thì sẽ khỏi ạ?", "en": "I often experience facial redness. I have learned that this is a type of disease. Does your hospital offer consultations for this condition? When visiting for consultation, do I need to undergo any tests? Is medication required for this condition, and if so, how long will I need to take it to recover?", "zh": "我经常脸红，了解到这是一种疾病。请问贵院是否有针对这种疾病的诊疗服务？就诊时需要做哪些检查？这种病需要长期服药吗？需要服药多久才能痊愈？"}, {"id": 102, "vi": "Em hiện đang mang thai 32 tuần. Tuần trước em vừa siêu âm Doppler, tuần này em đi siêu âm 4D có sao không ạ? Siêu âm nhiều có ảnh hưởng đến thai nhi không?", "en": "I am currently 32 weeks pregnant. Last week I had a Doppler ultrasound, and this week I want to have a 4D ultrasound—will that be okay? Could having multiple ultrasounds affect the fetus?", "zh": "我目前怀孕32周。上一周我做了多普勒超声，这一周我能做4D超声吗？做太多超声对胎儿有影响吗？"}, {"id": 103, "vi": "Em bị hành kinh hôm nay là ngày thứ 9 nhưng vẫn ra nhiều, ra chủ yếu máu cục. Em dùng thuốc chống đông và đã xét nghiệm Irn chỉ số 3,47 thì có ảnh hưởng đến hành kinh không ?", "en": "My period today is on day 9 but still heavy, mainly with clots. I am taking anticoagulant medication and my INR value is 3.47—will this affect my period?", "zh": "我今天是经期第9天，但仍然出血较多，主要为血块。我服用了抗凝药物，并且检查了INR，数值为3.47，这会影响月经吗？"}, {"id": 104, "vi": "Bổ sung vitamin E với thời gian như thế nào là hợp lý?", "en": "What is the appropriate duration for vitamin E supplementation?", "zh": "补充维生素E的时间应如何？"}, {"id": 105, "vi": "Tôi sinh năm 1979 đang có bệnh huyết áp cao và hiện đang uống thuốc viên Amlodipin, bây giờ tôi lại có thêm triệu chứng thiếu hụt canxi, rất mong bác sĩ tư vấn dùm là tôi có thể uống loại canxi nào và uống khoảng bao lâu cho phù hợp với bệnh của tôi.", "en": "I was born in 1979 and currently have high blood pressure, and I am currently taking Amlodipin tablets. Now I have developed a new symptom of calcium deficiency. I kindly request the doctor's advice on which type of calcium I can take and for how long it is appropriate for my condition.", "zh": "我出生于1979年，目前患有高血压，正在服用氨氯地平片。现在又出现了缺钙的症状，希望医生能给我建议，我应该服用哪种钙剂，以及需要服用多久才合适。"}, {"id": 106, "vi": "Đau rát miệng, khắp người như có kim chích có nên đi khám không?", "en": "Should I go for a check-up due to mouth pain and all over body sensation like needle pricks?", "zh": "口腔疼痛，全身像有针刺感，是否需要就诊？"}, {"id": 107, "vi": "Ăn bình bát kèm thanh long có sao không?", "en": "Is it okay to eat plain rice with papaya?", "zh": "平贝配龙眼吃有问题吗？"}, {"id": 108, "vi": "Tư vấn về phương pháp điều trị bệnh u mạch gan trái", "en": "Consultation on the treatment method for left hepatic vascular tumor", "zh": "关于左肝血管瘤的治疗方法咨询"}, {"id": 109, "vi": "Em 22 tuổi, chưa lập gia đình. Em mới đi kiểm tra sức khỏe tổng quát và khá lo lắng khi xét nghiệm AMH 2.0. Chỉ số như vậy có thấp không? Mong được bác sĩ tư vấn.", "en": "I am 22 years old and have not married. I just underwent a general health check-up and am quite concerned about my AMH level of 2.0. Is this level considered low? I hope to get advice from a doctor.", "zh": "我22岁，尚未结婚。我最近做了全身健康检查，对AMH 2.0的检测结果感到非常担忧。这样的数值偏低吗？希望医生能给予指导。"}, {"id": 110, "vi": "Bị Basedow thì có tập thể hình được không?", "en": "Can people with Basedow's disease do exercise?", "zh": "患有格雷夫斯病是否可以进行健身锻炼？"}, {"id": 111, "vi": "Xin cho em hỏi về chỉ số Estradiol (E2) mức trung bình khi mang thai ạ?", "en": "May I ask about the average Estradiol (E2) level during pregnancy?", "zh": "请问怀孕期间雌二醇（E2）的正常平均值是多少？"}, {"id": 112, "vi": "Mình thai ivf bị lưu 8 tuần, hiện mình vừa hút thai xong bác sĩ có dặn 2 ngày sau kiểm tra lại chỉ số beta. Mình xét nghiệm chỉ số beta là1877.47...Liệu có phải minhf có thai ngoài tử cung không. Mình chuyển 3 phôi thì được 2 phôi.", "en": "I underwent IVF and had a pregnancy maintained until 8 weeks. I just had a suction abortion, and the doctor advised me to check the beta level again two days later. I tested my beta level and it is 1877.47. Is it possible that I have an ectopic pregnancy? I transferred three embryos and only two implanted.", "zh": "我做试管婴儿，胚胎移植8周，刚清宫完，医生嘱咐2天后复查beta值。我查了beta值是1877.47……我是否患有宫外孕？我移植了3个胚胎，只着床了2个。"}, {"id": 113, "vi": "Có thai ngay sau khi ngừng thuốc ngừa thai hàng ngày có ảnh hưởng đến thai nhi không", "en": "Does having a pregnancy immediately after stopping daily contraceptive medication affect the fetus?", "zh": "停用每日避孕药后立即怀孕，是否会影响胎儿？"}, {"id": 114, "vi": "Hôm trước mình có đi xét nghiệm máu thì có chỉ số CRP-hs 17,8; mình có hiện tượng như sau: khi ngồi ghế cứng lâu hoặc nằm sấp, dưới vùng bẹn phải có phát tiếng kêu ùng ục.", "en": "Last day I had a blood test and my CRP-hs value was 17.8; I have the following symptoms: when sitting on a hard chair for a long time or lying on my stomach, there is a popping sound under the right iliac region.", "zh": "昨天我做了血检，发现CRP-hs为17.8；我有如下症状：当长时间坐在硬椅子上或仰卧时，右侧腹股沟区域会发出“咕噜”声。"}, {"id": 115, "vi": "Bà bầu 23 tuần ăn rau muống có ảnh hưởng thuốc procare diamond và canxi không", "en": "Does a 23-week pregnant woman eating water spinach affect Procare Diamond and calcium medication?", "zh": "23周孕妇吃空心菜会影响Procare Diamond和钙片吗"}, {"id": 116, "vi": "Sốt kéo dài có triệu chứng ho sặc, rét run và đau nhức tay chân phải làm thế nào?", "en": "What should I do for prolonged fever with cough, chills and runny limbs and body pain?", "zh": "持续发热，伴有阵发性咳嗽、寒战和四肢疼痛，该如何处理？"}, {"id": 117, "vi": "Viêm đáy mắt điều trị thế nào hiệu quả nhất?", "en": "How should uveitis be treated most effectively?", "zh": "如何最有效地治疗眼底炎？"}, {"id": 118, "vi": "Trẻ sơ sinh nhiễm trùng máu: Nguyên nhân và hướng điều trị", "en": "Neonatal sepsis: Causes and treatment approach", "zh": "新生儿败血症：病因及治疗方向"}, {"id": 119, "vi": "Tôi 65 tuổi, từng điều trị tiểu đường tuýp 2 bằng thuốc viên, đã ổn định và ngưng dùng thuốc đã 15 tháng. Tôi có bệnh suy giãn tĩnh mạch chân vẫn đang phải uống thuốc từ 10 năm nay. Vậy xin được hỏi tôi có thể tham gia tiêm vaccine thử nghiệm Nanocovac được không?", "en": "I am 65 years old, previously treated for type 2 diabetes with oral medication, and have been stable for 15 months without medication. I have chronic venous insufficiency in my legs, which I have been taking medication for for 10 years. Therefore, I would like to ask whether I can participate in the Nanocovac vaccine trial?", "zh": "我65岁，曾用口服药治疗2型糖尿病，病情已稳定，并停药15个月。我有下肢静脉曲张病史，已持续服药10年。请问，我可以参加纳诺科瓦疫苗的临床试验吗？"}, {"id": 120, "vi": "2 tuần nay em cảm giác mệt mỏi, tay chân mệt mỏi, nhức. Nặng ở lồng ngực. Ăn vào có cảm giác muốn nôn. Em đi siêu âm, người ta bảo bị dạ dày rồi cho thuốc uống. Nhưng tình trạng vẫn không giảm. Bác sĩ tư vấn giúp. Em không có cảm giác đau ở bụng.", "en": "For the past 2 weeks, I have felt fatigue, tiredness in hands and feet, pain. The pain is more severe in the chest. I feel nauseous after eating. I went for an ultrasound, and they told me I have gastritis and prescribed oral medication. However, my condition has not improved. Doctor, please advise. I do not have any abdominal pain.", "zh": "最近两周我感到乏力，四肢无力，疼痛，主要在胸部。进食时有恶心感。我做了超声检查，医生说我患有胃病，并开了药物服用。但症状仍未缓解。医生请给予建议。我腹部没有疼痛感。"}, {"id": 121, "vi": "Ngày thứ 2 em xét nghiệm beta được 149.53 đến ngày chậm kinh thứ 9 em có đi siêu âm nhưng chưa thấy túi ối. Bsi chẩn đoán thai chậm phát triển hoặc TNTC, cho em làm xét nghiệm beta được 1285, bác sĩ cho em thuốc tăng nội tiết để bổ sung giúp thai phát triển. Đến ngày 10 sau chậm kinh em xét nghiệm beta 1319. Thực sự em đang rất lo lắng, mong bác sĩ tư vấn tình trạng của em và cách giải quyết.", "en": "On day 2, my beta level was 149.53. On day 9 of missed period, I underwent ultrasound but no amniotic sac was seen. The nurse diagnosed fetal growth retardation or TNC. I was given a beta test result of 1285; the doctor prescribed hormone supplements to support fetal development. On day 10 of missed period, my beta level was 1319. I am truly very worried and hope the doctor can advise on my condition and how to proceed.", "zh": "第二天检查hCG值为149.53，到迟经第9天进行了B超检查，但未见到胎囊。护士诊断为胚胎发育迟缓或待观察，建议我再次检查hCG值为1285，医生给予激素类药物以促进胚胎发育。迟经10天时hCG值为1319。我确实非常担心，希望医生能咨询我的情况并提供解决方案。"}, {"id": 122, "vi": "Trường hợp nào trẻ giãn não thất 2 bên bẩm sinh không có biểu hiện bệnh?", "en": "In which case does a child have bilateral congenital enlarged ventricles without clinical manifestations?", "zh": "哪些先天性双侧脑室扩张的儿童没有临床表现？"}, {"id": 123, "vi": "Viêm họng tái đi tái lại, cổ nổi hạch nguyên nhân do đâu?", "en": "Recurrent pharyngitis, enlarged cervical lymph nodes, what is the cause?", "zh": "反复咽炎，颈部淋巴结肿大，原因是什么？"}, {"id": 124, "vi": "Bé đau đầu là do đâu?", "en": "What is the cause of the child's headache?", "zh": "孩子头痛的原因是什么？"}, {"id": 125, "vi": "Trẻ 3 tháng rưỡi nặng 5,5kg có nhẹ cân không, mẹ đang cho con bú có uống được omega-3 không và nên uống loại nào?", "en": "A 3.5-month-old child weighing 5.5 kg is underweight, is the mother breastfeeding, can she take omega-3, and which type should she take?", "zh": "3个月半、重5.5公斤的婴儿是否偏轻？母亲正在哺乳，可以服用Omega-3吗？应该服用哪种？"}, {"id": 126, "vi": "Đầu tháng 7 này tôi tiêm mũi thứ 3 ngừa viêm gan siêu vi B nhưng hôm 24/6/2021 tôi có chích ngừa Covid-19 mũi một. Vậy cho hỏi khoảng bao lâu tôi mới có thể chích ngừa mũi 3 viêm gan siêu vi B được?", "en": "This month, I received the third dose of hepatitis B vaccine, but on June 24, 2021, I received the first dose of COVID-19 vaccine. Thus, I would like to know how long I need to wait before I can receive the third dose of hepatitis B vaccine?", "zh": "今年7月初我接种了乙肝疫苗第三针，但2021年6月24日我接种了新冠疫苗第一针。请问，我需要等待多久才能接种乙肝疫苗第三针？"}, {"id": 127, "vi": "Thuốc mê mất tác dụng nguyên nhân là gì?", "en": "Why does the anesthetic drug lose its effect?", "zh": "麻醉药失效的原因是什么？"}, {"id": 128, "vi": "Tại sao gần sáng thường bị tỉnh giấc, liên tục bị hắt xì?", "en": "Why do I usually wake up in the early morning and continuously have nosebleeds?", "zh": "为什么早上经常醒来，持续打喷嚏？"}, {"id": 129, "vi": "Hiện e đang bầu 18 tuần và bị sốt 38 độ do cảm lạnh kèm theo ho, đau đầu, đau họng, liệu e có thể uống thuốc ceflixim được không ạ?", "en": "I am currently 18 weeks pregnant and have a fever of 38 degrees due to a cold accompanied by cough, headache, and sore throat. Can I take cefixime?", "zh": "我目前怀孕18周，因感冒出现发烧38度，伴有咳嗽、头痛、喉咙痛，我可以服用头孢克肟吗？"}, {"id": 130, "vi": "Con tôi được 1 tháng rồi. Từ lúc mới sinh cháu bị ra ghèn vàng ở mắt. Các bác sĩ ở khoa sản có hướng dẫn tôi về rửa nước muối sinh lý cho cháu nhưng vẫn không khỏi. Tôi cho cháu đi khám thì bác sĩ bảo bị viêm kết mạc và cho nước muối sinh lý, thuốc Bralcib về trả mắt cho cháu. Tôi về trả thuốc như hướng dẫn của bác sĩ được 1 ngày thì thấy mắt cháu hơi đỏ khi trả thuốc, mắt lúc nào cũng ướt như vừa khóc xong, và vẫn bị ra ghèn vàng. Bác sĩ cho tôi hỏi có phải con tôi bị tắc tuyến lệ không ạ. Nếu đúng thì cháu chữa trị như thế nào và bệnh này có ảnh hưởng đến khả năng nhìn của mắt cháu sau này không", "en": "My child is one month old. Since birth, my baby has had yellow discharge from the eyes. The gynecologists advised me to wash the baby's eyes with normal saline, but it hasn't improved. I took my child to see a doctor, who diagnosed conjunctivitis and prescribed normal saline and Bralcib to apply to the eyes. After returning home and applying the medication as instructed for one day, I noticed the baby's eyes became slightly red after application, the eyes were always wet as if just crying, and there is still yellow discharge. The doctor asked me if my child has blocked lacrimal ducts. If so, how should my child be treated, and will this condition affect my child's vision in the future?", "zh": "我孩子一个月了。从出生开始，孩子眼睛就出现黄色分泌物。产科医生建议我用生理盐水给孩子清洗，但效果不好。我带孩子去医院检查，医生诊断为结膜炎，开了生理盐水和布拉西比（Bralcib）滴眼液。我按照医生的指导用药一天后，发现孩子眼睛有点发红，眼睛一直湿润，像刚哭过一样，而且仍然有黄色分泌物。医生请问一下，是不是我孩子得了泪道阻塞？如果真是这样，该怎么治疗？这种病会不会影响孩子以后的视力？"}, {"id": 131, "vi": "Em hay dị ứng các món ăn, em đi tắm biển cũng hay bị dị ứng các loại sứa biển. Cách đây khoảng 1 năm, em ăn mật ong chung với đậu hũ gây nên khó thở, nổi ban đỏ khắp người phải đi bệnh viện, khi vào được tiêm 2 mũi thuốc xong là hết ngay. Vậy xin hỏi em có nên chích vắc xin không, và có thuốc gì để điều trị không?", "en": "I have allergies to food items, and I also get allergic reactions to sea creatures when I go swimming at the beach. About one year ago, I ate honey together with tofu, which caused shortness of breath and widespread red rash all over my body, requiring hospitalization. After receiving two injections, I recovered immediately. Therefore, I would like to ask whether I should receive an allergy shot, and if there are any medications for treatment?", "zh": "我对食物过敏，去海边洗澡也常对海蜇类物质过敏。大约一年前，我将蜂蜜和豆腐一起食用，导致呼吸困难，全身出现红斑，需去医院就诊，注射两针后症状立即消失。请问是否需要接种疫苗？是否有药物可以治疗？"}, {"id": 132, "vi": "Em bị nổi cái này cả tuần nay, bôi thuốc hoài không hết. Lâu lâu cũng ngứa. Nó nổi vòng vòng ở chân thôi. Em bị gì vậy ạ và dùng thuốc gì bôi hết ạ?", "en": "I have had this rash for the past week, applying medication without any relief. It itches occasionally. It only appears in circles on my feet. What condition do I have, and which topical medication should I use to treat it?", "zh": "我这周一直在脚上出现这种疹子，一直涂药都不好，偶尔还会痒。只在脚上出现环状的疹子。我这是什么病啊，用什么药涂可以好呢？"}, {"id": 133, "vi": "Viêm xoang cấp mủ, nghẹt mũi dịch chảy nhiều và có mùi hôi điều trị bằng phương pháp nào?", "en": "Acute purulent sinusitis, nasal congestion, abundant purulent discharge with foul odor, what treatment method is used?", "zh": "脓性急性鼻窦炎，大量黏性鼻塞并有臭味，应如何治疗？"}, {"id": 134, "vi": "Hiện tại tôi bị dịch âm đạo màu trắng đục ra như nước và đọng thành bột, hỏi điều trị làm sao?", "en": "Currently I have a vaginal infection with white cloudy discharge resembling water and forming a powder-like consistency. What is the treatment?", "zh": "目前我阴道有白色浑浊分泌物，类似水状，凝结成粉末状，该如何治疗？"}, {"id": 135, "vi": "Tôi bị mạch vành đã đặt stent, sau đó bác sĩ khám cho biết thêm bị tăng hồng cầu vô căn đã điều trị 12 tháng, nay thấy khỏe có cần tái khám và uống thuốc tiếp không ạ, uống thuốc nhiều có ảnh hưởng gì không ạ?", "en": "I have coronary artery disease and have had a stent placed. Afterwards, my doctor examined me and found I have primary polycythemia, which I have been treated for 12 months. Now I feel better, do I need to see the doctor again and continue taking medication? Is there any impact from taking medication for a long time?", "zh": "我患有冠状动脉疾病，已植入支架，之后医生检查发现还伴有原发性红细胞增多症，已治疗12个月，现在感觉身体状况良好，是否需要再次就诊并继续服药？长期服药会有影响吗？"}, {"id": 136, "vi": "Vệt đen lơ lửng trong mắt sau phẫu thuật đục thủy tinh thể có phải do đục dịch kính không?", "en": "Are floating black spots in the eye after cataract surgery due to vitreous opacity?", "zh": "术后眼睛出现漂浮的黑点是否由玻璃体混浊引起？"}, {"id": 137, "vi": "Dương vật không cương cứng có chữa khỏi được không?", "en": "Can a hard penis be cured?", "zh": "阴茎不能勃起能否治愈？"}, {"id": 138, "vi": "Cháu có vết xước ở tay, vô tình cầm phải con dao mổ chó bị ốm, chuôi con dao có vết máu và cháu đã rửa con dao bằng xà phòng trước khi cầm, con dao mổ chó ốm đã được 13giờ, tư vấn giúp chị", "en": "I have a cut on my hand; accidentally I picked up a dog surgery knife that was from a sick dog, the handle of the knife has blood on it, and I washed the knife with soap before handling it; the dog surgery knife has been for 13 hours; advise me please", "zh": "我手上有一处擦伤，不小心碰到了一只生病的狗的手术刀，刀柄上有血迹，我在使用前用肥皂清洗了刀具，这只狗的手术刀已经放置了13小时，麻烦咨询一下。"}, {"id": 139, "vi": "Em có 1 lần bị tai nạn và mắt bị đập xuống xe. Lúc đó mắt bên trái em bầm tím không mở lên được và có 1 vết thẹo may ngay mặt. Tới bệnh viện 115 chụp hình các thứ bên trong thì báo là không có gãy xương hay có gì nghiêm trọng chỉ có ngoài da rồi cho thuốc về uống tan máu bầm mắt. Nhưng sau 2 tháng kể từ ngày em bị tai nạn mắt em hơi lõm vào trong . Thưa bác sĩ là như vậy có nguy hiểm gì không ạ", "en": "I had an accident and my eye hit the car. At that time, my left eye was bruised and could not open, and there was a scar right on my face. I went to hospital 115 for imaging of the internal structures, and they reported no bone fractures or serious injuries, only external bruising, and prescribed medication to take orally to dissolve the bruising in the eye. However, two months after the accident, my eye slightly sinks inward. Dear doctor, is this dangerous?", "zh": "我曾遭遇一次意外，眼睛撞到了汽车上。当时左眼出现淤青，无法睁开，面部还有一处疤痕。去医院115检查，拍了内部影像，结果显示没有骨折或其他严重问题，只是皮肤损伤，开了口服药物来消散眼内淤血。但事故发生两个月后，我发现左眼略微向内凹陷。请问医生，这种情况是否有危险？"}, {"id": 140, "vi": "Điều trị nang thùy phải tuyến giáp 3mm như thế nào?", "en": "How to treat a 3mm right thyroid nodule?", "zh": "右侧甲状腺结节3毫米如何治疗？"}, {"id": 141, "vi": "Nay sáng em có xét nghiệm máu về nồng độ axit uric và của em lên 510 thì giờ em phải làm sao, uống loại thuốc gì ạ?", "en": "This morning I had a blood test for uric acid level and my result came up 510, then what should I do, which medication should I take?", "zh": "今天早上我做了血液检查，尿酸浓度达到510，现在该怎么办，该吃什么药呢？"}, {"id": 142, "vi": "Em bị sốt xuất huyết nên uống thuốc gì ạ?", "en": "What medication should my child take for hemorrhagic fever?", "zh": "妹妹得了出血热该吃什么药？"}, {"id": 143, "vi": "Cho em hỏi cần tiêm phòng những gì trước khi lập gia đình ạ?", "en": "May I ask what vaccinations are needed before getting married?", "zh": "请问结婚前需要接种哪些疫苗？"}, {"id": 144, "vi": "Em có thai được 7 tuần 2 ngày bị ra dịch nhầy màu nâu em có đi siêu âm bác sỹ kết luận bị bóc tách màng nuôi 9 mm và dịch màng nuôi 2 mm. Bác sĩ kê đơn thuốc là thuốc duphaston và thuốc nospa. Em muốn hỏi uống 2 loại thuốc này có ảnh hưởng đến thai không.", "en": "I am 7 weeks and 2 days pregnant and have been experiencing brownish clear discharge. I went for an ultrasound, and the doctor concluded I have a detached amniotic membrane measuring 9 mm and amniotic fluid measuring 2 mm. The doctor prescribed Duphaston and Nospa. I would like to ask whether taking these two medications affects the pregnancy.", "zh": "我怀孕7周2天，出现棕色黏液，去医院做超声检查，医生诊断为蜕膜剥离，蜕膜厚度9毫米，羊膜液2毫米。医生开了杜芬松和诺斯帕两种药。我想问一下，服用这两种药对胎儿有没有影响？"}, {"id": 145, "vi": "Em là nam 21 tuổi. Ngày uống 1 viên vitamin E 400 IU có tốt không ạ? Ngày nào cũng thế ạ?", "en": "I am a 21-year-old male. Is it okay to take 1 vitamin E capsule of 400 IU daily? Every day?", "zh": "我是一名21岁的男性。每天服用1粒400 IU的维生素E可以吗？每天都这样吃可以吗？"}, {"id": 146, "vi": "Vừa tiểu xong lại mắc tiểu là bị làm sao?", "en": "After urinating, I have to urinate again, what's wrong?", "zh": "刚尿完又尿出来是怎么了？"}, {"id": 147, "vi": "Bé nhà em bị còi xương hiện cháu đang dùng d3 cộng k2 và hartus thì cháu dùng đồng thời hai loại này trong thời gian bao lâu ạ?", "en": "My child has rickets and is currently taking D3 plus K2 and Hartus. How long should my child take both of these medications together?", "zh": "我孩子患有佝偻病，目前孩子在服用D3加K2和Hartus，那么孩子同时服用这两种药物需要持续多久？"}, {"id": 148, "vi": "Người yêu em năm nay 21 tuổi. Từ khi học lớp 9, đêm ngủ cứ giật mình và tới tháng đau kinh dữ dội. Nguyên nhân bị sao vậy?", "en": "My boyfriend is 21 years old this year. Since 9th grade, he has had nightmares at night and severe menstrual pain every month. What is the cause?", "zh": "我男朋友今年21岁。从九年级开始，每晚睡觉都会突然惊醒，而且每个月经期都会非常剧烈疼痛。这是什么原因？"}, {"id": 149, "vi": "Bị vảy nến lâu năm nên điều trị thế nào để không ảnh hưởng quá trình mang thai?", "en": "How should psoriasis, which has been present for a long time, be treated so as not to affect the pregnancy process?", "zh": "长期患有银屑病应该如何治疗，以免影响怀孕过程？"}, {"id": 150, "vi": "Sau 10 năm phẫu thuật cắt bỏ tuyến tiền liệt đi tiểu không hết nước, tiểu nhiều lần cùng một lúc là bị làm sao?", "en": "After 10 years of prostatectomy, incomplete urination and frequent urination at the same time, what is the cause?", "zh": "术后10年，排尿不尽，一次排尿次数增多是由于什么引起的？"}, {"id": 151, "vi": "Đau nửa đầu bên phải, giật giật từng cơn sau khi uống thuốc viêm xoang có sao không?", "en": "Right-sided headache, twitching episodes after taking sinus inflammation medication?", "zh": "右侧偏头痛，服用鼻窦炎药物后出现阵发性抽搐，有问题吗？"}, {"id": 152, "vi": "Nhờ bác sỹ tư vấn. sau chuyển phôi em siêu âm có 2 túi thai. - Túi 1: CLR: 9.5 mm , nhịp tim 143 , thai 7 tuần - Túi 2: CLR: 8.9 mm ,nhịp tim 150 , thai 6 tuần kích thước thai của em như vậy có bé không ạ ,nhịp tim như vậy ổn không. Em bị nghén thỉnh thoảng buồn nôn như vậy có ảnh hưởng gì đến thai không.", "en": "Dear doctor, I would like to consult with you. After embryo transfer, I did an ultrasound and found two gestational sacs. - Sac 1: CLR: 9.5 mm, heart rate 143, gestational age 7 weeks - Sac 2: CLR: 8.9 mm, heart rate 150, gestational age 6 weeks. Is my fetus development normal like this? Is the heart rate normal? I have morning sickness occasionally with nausea, does this affect the fetus?", "zh": "感谢医生咨询。移植胚胎后，我做了超声检查，发现有两个孕囊。  \n- 孕囊1：CLR：9.5 mm，心率143次/分钟，孕周7周  \n- 孕囊2：CLR：8.9 mm，心率150次/分钟，孕周6周  \n我目前的胎儿大小是否正常？心率这样是否正常？我有轻微孕吐，偶尔会感到恶心，这对胎儿有影响吗？"}, {"id": 153, "vi": "Con em được 16 ngày tuổi. Con mấy ngày này hay khóc đêm, vặn mình nhiều, giật mình, ngủ thì rên mệt mỏi. Em có coi trên mạng bảo là con thiếu vitamin d3. Vậy bác sĩ cho em hỏi mình có cần bổ sung vitamin d3 hay không. Và nếu có thì liều lượng như thế nào ạ? Em có xem qua thuốc xịt dimao vitamin d3. Bác sĩ có thể tư vấn cho em được không ạ?", "en": "The baby is 16 days old. This baby cries at night, twists itself a lot, jerks suddenly, and breathes with difficulty when sleeping. I checked online and was told that the baby might be deficient in vitamin D3. Therefore, I would like to ask the doctor whether I should supplement vitamin D3 or not. And if so, what is the dosage? I have seen a spray called Dimao Vitamin D3. Can the doctor advise me on this?", "zh": "宝宝16天大。这几天晚上经常哭闹，身体扭动频繁，突然惊醒，睡觉时呼吸费力。我在网上看到说宝宝可能缺乏维生素D3。所以想请问医生，是否需要补充维生素D3？如果需要，剂量应该是多少？我看到有喷雾剂叫“迪茅维生素D3”，医生可以给我一些建议吗？"}, {"id": 154, "vi": "Em đi xét nghiệm máu kết quả thể tích khối hồng cầu (htc): 42,4% cao hơn so với khoảng bình thường. Bác sĩ giải đáp giúp em với.", "en": "My blood test result shows hematocrit (HCT): 42.4%, higher than the normal range. Doctor, please explain this to me.", "zh": "我做了血液检查，血细胞比容（HCT）结果为42.4%，高于正常范围。请医生帮忙解答一下。"}, {"id": 155, "vi": "Chỉ số HCV cho kết quả miễn dịch tự động là 4.07 có bị viêm gan C không?", "en": "Is there hepatitis C if the HCV index from automated immunological test is 4.07?", "zh": "HCV 指标自动免疫检测结果为 4.07 是否患有丙型肝炎？"}, {"id": 156, "vi": "Chồng em làm xét nghiệm tinh dịch đồ, kết quả không thấy có tinh trùng. Sau đó có làm xét nghiệm trên 9 nhiễm sắc thể và kết quả là mất vi đoạn trên vùng mở rộng SY1191, SY1192. Với tình trạng trên, vợ chồng em có thể có con được không và phương pháp điều trị là gì ạ?", "en": "My husband underwent semen analysis, the result showed no sperm. Then he underwent testing on the nine chromosomes, and the result was a deletion in the expanded region SY1191, SY1192. With this condition, can my husband and I have a child, and what is the treatment method?", "zh": "我丈夫做了精液检查，结果没有发现精子。之后又做了9号染色体检测，结果显示在SY1191、SY1192扩增区域存在缺失。针对这种情况，我们夫妻能否生育？治疗方法是什么？"}, {"id": 157, "vi": "Có đơn thuốc trị bệnh tăng nhãn áp không?", "en": "Is there a prescription medication for treating glaucoma?", "zh": "有治疗青光眼的处方药吗？"}, {"id": 158, "vi": "Bác sĩ cho em hỏi vợ em từng bị sốt và nhịp tim thai lên hơi cao nên có nhập viện điều trị. Sau khoảng vài tiếng nhập viện tim thai trở lại bình thường và theo dõi nhịp tim 3 lần một ngày ở bệnh viện, nhịp tim thai nhi đều bình thường. Sau khi xuất viện 2 tuần vợ em tái khám siêu âm Doppler và mọi chỉ số bình thường. Đến nay đã 7 tuần sau lần nhịp tim lên cao thì mọi chỉ số khi đi siêu âm Doppler đều bình thường. Cho em hỏi vậy khi sinh ra em bé có nguy cơ mắc bệnh tim không ạ? Do nhịp tim từng bị tăng cao rồi trở lại bình thường như vậy ạ?", "en": "Doctor, I would like to ask: my wife previously had fever and fetal heart rate increased slightly, so she was hospitalized for treatment. After a few hours in the hospital, the fetal heart rate returned to normal, and she was monitored three times a day at the hospital, and the fetal heart rate was always normal. After discharge, two weeks later she had a Doppler ultrasound check-up and all parameters were normal. To date, at 7 weeks after the episode of elevated fetal heart rate, all parameters during Doppler ultrasound examinations are normal. I would like to ask: does my baby have a risk of congenital heart disease? Since the fetal heart rate previously increased and then returned to normal like this?", "zh": "医生请问，我妻子以前有过发烧，胎儿心率稍微升高，因此住院治疗。大约几小时后，胎儿心率恢复正常，住院期间每天监测三次心率，胎儿心率均正常。出院两周后，她再次进行多普勒超声检查，所有指标均正常。到目前为止，距离胎儿心率升高已过去7周，每次进行多普勒超声检查，所有指标均正常。请问，孩子出生后是否有心脏病的风险？因为胎儿心率曾短暂升高后又恢复正常吗？"}, {"id": 159, "vi": "Điều trị dứt điểm mỡ máu kèm rối loạn lipid máu?", "en": "Treatment of definitive hyperlipidemia accompanied by lipid disorder?", "zh": "能否根治高脂血症伴血脂异常？"}, {"id": 160, "vi": "Con em năm nay 15 tuổi và đang trong độ tuổi phát triển. Chế độ ăn uống và tập luyện như thế nào để cháu có thể phát triển tốt trong giai đoạn này ạ?", "en": "My child is currently 15 years old and is in the developmental age. What kind of diet and exercise routine should my child follow to develop well during this stage?", "zh": "孩子今年15岁，正处于发育期。请问在这一阶段，孩子应该怎样饮食和锻炼才能更好地发育？"}, {"id": 161, "vi": "Em bị nổi nốt đỏ ở chân, trước đây có nổi ít xong hết nhưng thời gian gần đây em lại bị nổi nhiều hơn ở chân. Các bác sĩ tư vấn giúp em với ạ.", "en": "My daughter has red spots on her feet. Previously, she had few spots that then resolved, but recently she has developed more spots on her feet. Doctors, please advise me.", "zh": "我脚上出现了红色斑点，之前有过少量出现后来好了，但最近脚上的红斑明显增多。医生能给我一些建议吗？"}, {"id": 162, "vi": "Tôi năm nay 45 tuổi, giới tính nam. Cách đây 2 năm tôi bị hội chứng thận hư và gout, hiện vẫn đang uống thuốc ngày 1 viên. Xin hỏi trường hợp của tôi có tiêm được vắc xin phòng Covid-19 không ạ?", "en": "I am 45 years old, male. Two years ago, I was diagnosed with nephrotic syndrome and gout, and I am currently taking one pill per day. I would like to ask whether my condition allows me to receive the COVID-19 vaccine?", "zh": "我今年45岁，男性。两年前我被诊断为肾病综合征和痛风，目前每天服用1粒药。我想咨询一下，我的情况是否可以接种新冠疫苗？"}, {"id": 163, "vi": "Cơ chế rối loạn đông máu và cơ chế tạo ra cục máu đông hoàn toàn khác nhau. Có phương pháp nào xác định sớm khả năng bị cục máu đông sau tiêm vắc xin không?", "en": "The mechanisms of blood coagulation disorder and the mechanisms of thrombus formation are completely different. Is there any method to determine early the risk of thrombosis after vaccination?", "zh": "凝血功能障碍和血栓形成机制完全不同。是否有方法可以早期判断接种疫苗后发生血栓的风险？"}, {"id": 164, "vi": "Con em được 2 tháng tuổi , hôm nay em có phát hiện cháu có 3 nốt ban đỏ ở thái dương và tai , cháu không có biểu hiện gì bất thường , bác sĩ xem giúp em xem đây là bệnh da liễu hay là gì.", "en": "My child is 2 months old. Today, I noticed three red spots on the temporal region and ear of my child. The child has no other abnormal symptoms. Doctor, please have a look at this and let me know whether this is a skin disease or something else.", "zh": "2个月大的孩子，今天发现孩子太阳穴和耳朵处有3个红色斑疹，孩子没有其他异常表现，医生帮忙看一下，这是皮肤疾病还是其他疾病？"}, {"id": 165, "vi": "Thời điểm cắt thắng lưỡi cho trẻ lúc nào phù hợp nhất?", "en": "When is the most appropriate time to perform lingual frenectomy in children?", "zh": "什么时候为儿童割舌最合适？"}, {"id": 166, "vi": "Nữ giới giảm tiểu cầu nguyên nhân là gì?", "en": "What are the causes of decreased platelets in females?", "zh": "女性血小板减少的原因是什么？"}, {"id": 167, "vi": "Bác sĩ ơi, những triệu chứng nào là triệu chứng bị nhiễm virus corona vậy ạ?", "en": "Doctor, what are the symptoms of coronavirus infection?", "zh": "医生，感染冠状病毒的典型症状有哪些？"}, {"id": 168, "vi": "Cháu lấy chồng hơn năm nay chưa có em bé . Mấy chu kì trước cháu kinh cháu ra nhiều còn tháng này cháu uống thuốc bổ trứng và cháu bị nang tồn dư bên trái nên tháng này cháu ra kinh có 2 ngày đã hết và ra ít hơn mấy chu kì trứng . Liệu có làm sao không ạ .", "en": "I have been married for more than five years and have not had a child. In the previous few cycles, my periods were heavy, but this month after taking medication to improve egg quality and due to a left ovarian cyst, my period lasted only two days and was much lighter than in previous cycles. Is there anything that can be done?", "zh": "今年结婚以来一直未怀孕。之前几个月月经量都较多，而这个月我服用了促排卵药物，且左侧存在卵泡残留，因此这个月月经只持续了两天，且经量比之前几个月的排卵期要少。请问这种情况是否有办法解决？"}, {"id": 169, "vi": "Thưa bác sỹ em đã cưới vợ được 1 năm và chưa có con. Sau khi đi khám thì có kết quả vợ em bình thường . Còn em bị vô sinh nam không thấy tinh trùng trong tinh dịch. Và em đã làm xét nghiệm NST kết quả là 46,XY. Kết luận chưa phát hiện thấy bất thường di truyền ở mức độ nhiễm sắc thể. Xét nghiệm AZF là không thấy mất đoạn gen AZF a,b,c ở những gen được khảo sát tương ứng. Vậy em có bình thường không.", "en": "Dear doctor, I have been married for one year and have not had a child. After examination, my wife's results are normal. I have male infertility and no sperm in the semen. I have undergone karyotype testing, and the result is 46,XY. The conclusion is no genetic abnormalities detected at the chromosomal level. AZF gene testing shows no deletion of AZF a, b, c at the corresponding genes examined. So, am I normal?", "zh": "医生您好，我结婚一年了，一直没有孩子。检查后发现妻子一切正常，而我被诊断为男性不育，精液中未发现精子。我做了染色体检查，结果为46,XY，染色体水平未发现遗传异常。AZF基因检测显示，在所检测的AZF a、b、c基因区域均未发现缺失。那么我是否正常呢？"}, {"id": 170, "vi": "Choáng và đau nửa đầu do rò luân nhĩ có nguy hiểm không?", "en": "Is dizziness and migraine caused by sinusitis dangerous?", "zh": "迷走神经瘘引起的头晕和偏头痛危险吗？"}, {"id": 171, "vi": "Hiện tại, hàng tháng tôi bị nhiệt miệng từ 1–2 lần. Mỗi lần bị thì sau khi uống thuốc từ 5–7 ngày khỏi. Lúc bị nhiệt trên lưỡi, lúc bị ở vùng quanh miệng, 1 hoặc 2 nốt. Xin hỏi cái này có sao không ạ?", "en": "Currently, I experience mouth ulcers 1–2 times per month. Each time, after taking medication for 5–7 days, I recover. When I have mouth ulcers on the tongue, or around the mouth, there are 1 or 2 spots. I would like to ask whether this is a problem.", "zh": "目前，我每个月会发作1到2次口腔溃疡。每次发作后，服用药物5到7天就会痊愈。发作时表现为舌头或口腔周围出现1到2个溃疡点。请问这种情况严重吗？"}, {"id": 172, "vi": "Em trước khám ở bệnh viện đông y bị chẩn đoán suy tủy giảm tiểu cầu nhưng khi đi khám bệnh viện khác thì chẩn đoán mất độ cong tự nhiên 3 đốt sống lưng và có dấu hiệu thoái hóa nhẹ. Lưng đau thường xuyên khiến chất lượng giấc ngủ rất thấp, người thường xuyên mệt mỏi. Em muốn hỏi 2 bệnh trên có gì giống nhau mà lại có thể chẩn đoán nhầm được ạ? Điều trị cột sống như thế nào?", "en": "My sister was diagnosed with spinal marrow failure and thrombocytopenia at a Traditional Chinese Medicine hospital, but at another hospital, she was diagnosed with natural lumbar spine curvature loss in three vertebral segments and mild degenerative changes. Frequent back pain severely affects sleep quality, and she often feels fatigued. I would like to ask: what similarities exist between these two conditions that could lead to misdiagnosis? How is spinal treatment carried out?", "zh": "我在中医医院就诊时被诊断为骨髓抑制性血小板减少，但在另一家医院就诊时被诊断为腰椎3个节段自然曲度丧失，并伴有轻度退行性改变。经常性腰痛导致睡眠质量极差，常感疲劳。我想问一下，这两种疾病有什么相似之处，以至于可能被误诊？脊柱的治疗方式是什么？"}, {"id": 173, "vi": "Nguyên nhân mất cảm giác bắp đùi chân trái, véo không đau?", "en": "What is the cause of loss of sensation in the left thigh, no pain on palpation?", "zh": "左腿大腿感觉丧失，但触碰不痛的原因是什么？"}, {"id": 174, "vi": "Dấu hiệu tự kỷ ở trẻ 29 tháng tuổi là gì?", "en": "What are the autistic signs in a 29-month-old child?", "zh": "29个月大的儿童自闭症的征兆是什么？"}, {"id": 175, "vi": "Phương pháp keo dính phôi là bệnh nhân lựa chọn hay chỉ định? Những bệnh lý nào nên sử dụng? Hiệu quả của phương pháp này ra sao?", "en": "Is the adhesive embryo method patient-selected or indicated? Which diseases should use it? What is the efficacy of this method?", "zh": "粘膜贴片法是患者选择还是仅指征？哪些疾病应使用？该方法的效果如何？"}, {"id": 176, "vi": "Cháu bị loét ở phần lợi như hình, rất đau và rát. Vết loét lõm chứ không phải mụn thịt hay có mủ nước. Cháu bị bệnh gì và có phải do HSV không ạ?", "en": "I have a ulcer on the gum area as shown, very painful and burning. The ulcer is sunken, not a flesh tumor or with pus fluid. What disease do I have and is it due to HSV?", "zh": "我口腔内侧有类似图片的溃疡，非常疼痛且灼热。溃疡是凹陷的，而不是肉芽肿或有脓液。我得的是什么病？是HSV引起的吗？"}, {"id": 177, "vi": "Đau thần kinh sau Zona có điều trị dứt điểm được không?", "en": "Can post-zoster neuralgia be cured definitively?", "zh": "带状疱疹后神经痛能否根治？"}, {"id": 178, "vi": "Nhiệt độ giảm đột ngột khi bị sốt xuất huyết Dengue có sao không?", "en": "Is it dangerous if body temperature suddenly drops when having dengue hemorrhagic fever?", "zh": "登革热发热时体温突然下降会有问题吗？"}, {"id": 179, "vi": "Bầu bị sốt nhẹ 38,5 độ trở lại nguyên nhân do cảm mạo bình thường ( không phải do bất kỳ vi khuẩn hay virus của các bệnh nguy hiểm rubelle, thủy đậu, cúm A,...) thì có nguy hiểm cho thai nhi không? Lúc em sốt là vào ban đêm, sáng hôm đó em liền đi viện thì nhịp tim thai nhi của e có lên hơi cao 1 chút xong bác sĩ cho uống paracetamol thì giảm sốt dần. Hơn 1 ngày sau thì em không còn sốt nữa tim thai trở về bình thường kể từ khi uống thuốc trong ngày đầu đi khám khi sốt. Cho em hỏi sốt do cảm mạo vậy có ảnh hưởng gì đến bé không? Lúc đó em đang mang thai tuần 28", "en": "The mother has a mild fever of 38.5°C, which is due to a common cold (not caused by any dangerous bacteria or viruses such as rubella, varicella, or influenza A, etc.). Is this dangerous for the fetus? The fever occurs at night, and in the morning, she goes to the hospital, where the fetal heart rate is slightly elevated, then the doctor prescribes paracetamol, which gradually reduces the fever. More than one day later, she no longer has a fever, and the fetal heart rate returns to normal from the day of the first visit when she had the fever. I would like to know if a fever from a common cold has any effect on the baby? At that time, she was at 28 weeks of pregnancy.", "zh": "孕妇体温轻微升高至38.5℃，是否由普通感冒引起（非麻疹、水痘、甲型流感等危险病毒或细菌感染）对胎儿有危险吗？我发烧是在夜间，第二天早晨去医院时，胎儿心率略微升高，医生给予对乙酰氨基酚后，体温逐渐下降。一天后，我已不再发烧，胎儿心率也恢复至正常水平，从首次就诊当天服药后开始恢复正常。请问，由普通感冒引起的发烧对胎儿有影响吗？当时我处于孕28周。"}, {"id": 180, "vi": "Nữ giới viêm kết mạc dị ứng điều trị như thế nào?", "en": "How is allergic conjunctivitis in women treated?", "zh": "女性过敏性结膜炎如何治疗？"}, {"id": 181, "vi": "Hẹp một bên thận nên ăn uống và điều trị thế nào?", "en": "How should diet and treatment be managed for unilateral renal obstruction?", "zh": "一侧肾积水应如何饮食和治疗？"}, {"id": 182, "vi": "Chụp MRI có tiêm thuốc đối quang từ khi mang thai liệu có ảnh hưởng đến thai nhi?", "en": "Does receiving an iodinated contrast agent during MRI pregnancy affect the fetus?", "zh": "怀孕期间做增强MRI是否会影响胎儿？"}, {"id": 183, "vi": "Cho em hỏi là hôm trước em có đi net thì lỡ uống nhầm nước ngọt của người khác mà chai nước đó đã để được khá lâu và trong nước không còn gas nữa vậy nếu người đó bị hiv thì em có khả năng bị hiv không", "en": "May I ask: yesterday I went to a net and accidentally drank a sweet drink belonging to someone else; that bottle of drink had been stored for a relatively long time and the drink no longer contains carbonation. If that person has HIV, do I have a risk of getting HIV?", "zh": "我想问一下，昨天我去网吧时，不小心喝错了别人的一瓶饮料，那瓶饮料放了相当久，而且饮料里已经没有气了。如果那个人是HIV阳性，我有被感染HIV的风险吗？"}, {"id": 184, "vi": "Có dấu hiệu mệt mỏi và tay chân bủn rủn khi uống 2 loại thuốc cùng lúc phải làm thế nào?", "en": "There are symptoms of fatigue and trembling of hands and feet when taking two types of medication at the same time. What should I do?", "zh": "同时服用两种药物时出现疲劳和手脚发抖该怎么办？"}, {"id": 185, "vi": "Kết quả xét nghiệm HIV dương tính là do đâu?", "en": "What is the cause of a positive HIV test result?", "zh": "HIV检测结果为阳性是由于什么原因？"}, {"id": 186, "vi": "Dư ối độ sâu 89mm có nguy hiểm không và cách điều trị", "en": "Is a deep ulcer depth of 89mm dangerous and what is the treatment?", "zh": "深部水肿89毫米是否危险以及治疗方法"}, {"id": 187, "vi": "Sốt kéo dài, chán ăn phải làm sao?", "en": "What should I do for prolonged fever and loss of appetite?", "zh": "长期发热、食欲不振该怎么办？"}, {"id": 188, "vi": "Em bị dị dạng mạch máu não . Cách đây 3 năm em đã phẫu thuật xong . Chân tay em bi co cứng . Đi mất thăng bằng . Em sử dụng sản phẩm natoopes được không.", "en": "I have cerebral vascular malformation. Three years ago, I had surgery completed. My hands and feet are stiff. I lose balance. Can I use Natoopes product?", "zh": "我患有脑血管畸形。三年前我已做完手术。现在我的手脚僵硬，失去平衡。我可以使用Natoopes产品吗？"}, {"id": 189, "vi": "Ngón tay rướm máu sau khi chạm vào răng chó chết không rõ nguyên nhân có nguy cơ mắc bệnh dại không?", "en": "Does touching a dead dog's tooth and having a finger bleed afterward pose a risk of rabies infection?", "zh": "接触死狗牙齿后手指出血，且原因不明，有得狂犬病的风险吗？"}, {"id": 190, "vi": "Em có làm các siêu âm hệ tiết niệu. Kết quả ghi nhận hạ phân thuỳ V có nốt tăng âm đường kính 7mm. Bác sĩ cho em hỏi kết quả như trên là sao. Có nguy cơ bệnh gì và có nguy hiểm không và cách chữa.", "en": "I have undergone ultrasound of the urinary system. The results show a hypoechoic lesion in the posterior ureteral segment with a diameter of 7mm. Doctor, I would like to know what this result means. What diseases or risks are associated with it, and is it dangerous? What is the treatment?", "zh": "我做了泌尿系统超声检查。结果显示膀胱三角区有一个7mm的强回声斑。医生请问一下，这个结果是什么意思？有没有可能患什么疾病？严重吗？该怎么治疗？"}, {"id": 191, "vi": "Nong bao quy đầu đúng cách cho bé như thế nào?", "en": "How to properly clean the glans of a baby's penis?", "zh": "如何正确为婴儿包住阴茎？"}, {"id": 192, "vi": "Phẫu thuật đặt miếng lưới nhân tạo điều trị thoát vị gần rốn có đẻ mổ được không?", "en": "Can a cesarean section be performed in a patient who has undergone surgery to place an artificial mesh for treatment of near-navel hernia?", "zh": "腹腔镜手术植入人工网片治疗近脐疝是否可以剖宫产？"}, {"id": 193, "vi": "Nguyên nhân và cách khắc phục rụng tóc nhiều ở tuổi 19?", "en": "What are the causes and ways to remedy excessive hair loss at age 19?", "zh": "19岁脱发的原因及治疗方法是什么？"}, {"id": 194, "vi": "Tóc nhiều gàu, rụng nhiều, gãi chảy máu có phải nấm da đầu không?", "en": "Excessive dandruff, heavy hair loss, and bleeding from scratching—could this be fungal scalp infection?", "zh": "头皮多油、脱发、搔抓出血是否可能是头癣？"}, {"id": 195, "vi": "Hai bên tinh hoàn không đều nhau có sao không?", "en": "Is it problematic if both testes are unequal in size?", "zh": "两侧睾丸大小不等有没有问题？"}, {"id": 196, "vi": "Thai cháu được 13 tuần sáng sớm nay cháu thấy 1 bên bụng của cháu nó rất cứng, còn bên kia thì không. Vậy là sao vậy ạ", "en": "The baby is 13 weeks old. This morning, the child feels one side of the abdomen is very hard, while the other side is not. What could this be?", "zh": "孩子13周，今天早上孩子感觉腹部一侧非常硬，另一侧则没有。这是怎么回事？"}, {"id": 197, "vi": "Trẻ đã tiêm vắc xin viêm gan B lúc mới sinh có cần tiêm thêm không?", "en": "Does a child who received hepatitis B vaccine at birth need to receive it again?", "zh": "新生儿已接种乙肝疫苗，是否还需要再次接种？"}, {"id": 198, "vi": "Đồng thời bị viêm dây thần kinh thị giác và phồng giác mạc có điều trị được không?", "en": "Can visual nerve inflammation and corneal edema be treated simultaneously?", "zh": "同时患有视神经炎和角膜水肿能否治疗？"}, {"id": 199, "vi": "Ngủ ít, đo điện não đồ, biên độ các sóng thấp là dấu hiệu bệnh gì?", "en": "Sleeping less, EEG measurement, low amplitude of waves is a sign of what disease?", "zh": "睡眠时间少，做脑电图，波幅低是何种疾病的征兆？"}, {"id": 200, "vi": "Đốt sùi mào gà sâu bên trong hậu môn bằng cách nào?", "en": "How to remove human papillomavirus deep inside the anus?", "zh": "如何通过内服方式治疗肛门尖锐湿疣？"}, {"id": 201, "vi": "Nguyên nhân mắt bị xuất huyết kèm đau đầu, cảm giác choáng váng là gì?", "en": "What is the cause of eye bleeding accompanied by headache and dizziness?", "zh": "什么病因会导致眼底出血伴头痛、头晕？"}, {"id": 202, "vi": "Nồng độ Hb trung bình hồng cầu của em là 31.9 thấp hơn mức trung bình 32-36 nghĩa là gì và có nguy hiểm không ạ? Độ phân bố tiểu cầu 18 cao hơn mức trung bình là 10-16.5 có phải em bị bệnh gì không ạ?", "en": "What does it mean that the average hemoglobin level of my red blood cells is 31.9, which is lower than the normal range of 32-36, and is it dangerous? The platelet distribution width is 18, which is higher than the normal range of 10-16.5. Does this mean I have a disease?", "zh": "血红蛋白平均值为31.9，低于正常范围32-36的水平，这意味着什么，有没有危险？血小板分布宽度为18，高于正常范围10-16.5，是否说明你患有某种疾病？"}, {"id": 203, "vi": "Tim hồi hộp, chân tay bủn rủn, mất ngủ khi mang thai có làm sao không?", "en": "Is it a problem if I have palpitations, trembling of hands and feet, and insomnia during pregnancy?", "zh": "怀孕时出现心悸、手脚发麻、失眠，有没有关系？"}, {"id": 204, "vi": "Trường hợp bé trai thoát vị bẹn nên mổ hở hay là nội soi?", "en": "Should a boy with inguinal hernia undergo open surgery or laparoscopic surgery?", "zh": "男孩腹股沟疝应该选择开放手术还是腹腔镜手术？"}, {"id": 205, "vi": "Thoát vị đĩa đệm nặng gây tê chân kèm mất cảm giác đi châm cứu không mổ có được không?", "en": "Can acupuncture without surgery be performed for severe disc herniation causing leg numbness accompanied by loss of sensation?", "zh": "严重椎间盘脱出引起下肢麻木伴感觉丧失，能否针灸治疗而不手术？"}, {"id": 206, "vi": "Thủng màng nhĩ có chữa khỏi được không?", "en": "Can eardrum perforation be cured?", "zh": "中耳膜破裂能治愈吗？"}, {"id": 207, "vi": "Triệu chứng cảnh báo bệnh thủy đậu bội nhiễm", "en": "Warning signs of varicella superinfection", "zh": "水痘并发感染的预警症状"}, {"id": 208, "vi": "Tại sao uống thuốc kháng sinh xong thấy mệt, buồn ngủ, cảm giác buồn nôn và đau đầu?", "en": "Why do I feel tired, drowsy, nausea, and headache after taking antibiotics?", "zh": "为什么服用抗生素后感到疲倦、嗜睡、恶心和头痛？"}, {"id": 209, "vi": "Năm nay cháu 21 tuổi và đang mắc các triệu chứng như mất ngủ, lo lắng, bồn chồn, tim đập nhanh. Tình trạng này kéo dài được 2 năm rồi. Ngày ngủ được có 3 - 4 tiếng, ngủ không sâu giấc, mơ màng, hay bị thức khi có tiếng động nhỏ. Bên cạnh đó, cháu còn bị bệnh loét dạ dày ạ.", "en": "This year I am 21 years old and am experiencing symptoms such as insomnia, anxiety, restlessness, and a rapid heartbeat. This condition has lasted for 2 years. I sleep for 3-4 hours each night, have shallow sleep, experience fragmented sleep, and wake up easily to small noises. In addition, I have a gastric ulcer.", "zh": "今年我21岁，出现失眠、焦虑、烦躁、心跳加快等症状，这种情况已经持续两年了。每天睡眠时间只有3到4小时，睡眠不深，多梦，容易因轻微声响而惊醒。此外，我还患有胃溃疡。"}, {"id": 210, "vi": "Men gan tăng, HBeAg dương tính liệu có phải bị viêm gan B mãn tính không?", "en": "Elevated liver function tests, positive HBeAg, does this mean having chronic hepatitis B?", "zh": "转氨酶升高，HBeAg阳性，是否意味着患有慢性乙型肝炎？"}, {"id": 211, "vi": "Tán sỏi thận qua da bằng ống soi mềm áp dụng trong trường hợp nào?", "en": "When is percutaneous nephrolithotripsy using a flexible ureteroscope indicated?", "zh": "软性膀胱镜经皮肾石取出术适用于哪些情况？"}, {"id": 212, "vi": "Bố em có tiền sử xơ gan, hiện đang điều trị sốt siêu vi và nhiễm khuẩn đường tiết niệu. Tuy nhiên bố em bị đau cơ ở chân rất dữ dội, nhất là ban đêm đau không ngủ được. Tình trạng kéo dài đã vài tuần từ sau 1 lần bị căng cơ do đi bộ quá nhiều. Liệu có phải bố em bị viêm cơ không ạ?", "en": "My father has a history of cirrhosis and is currently being treated for viral fever and urinary tract infection. However, he is experiencing severe muscle pain in his legs, especially at night, which prevents him from sleeping. This condition has persisted for several weeks following a previous episode of muscle cramp due to excessive walking. Could my father be suffering from inflammatory myopathy?", "zh": "我父亲有肝硬化病史，目前正接受病毒性发热和尿路感染的治疗。然而，他最近几周出现严重的腿部肌肉疼痛，尤其是夜间疼痛剧烈，无法入睡。这种状况在一次因长时间步行导致的肌肉痉挛后持续存在。请问父亲是否患有肌炎？"}, {"id": 213, "vi": "Con em 1 tuổi. Đầu hơi nghiêng nhưng đi khám bác sĩ nói là ngày xưa bế 1 tay bị lệch. Giờ có cách hay dụng cụ nào tập cho thẳng lại không ạ?", "en": "My child is 1 year old. The head is slightly tilted, but the doctor said it was due to leaning one hand when carried in the past. Now, is there any good method or device to correct it?", "zh": "1岁的孩子。头稍微歪斜，但医生检查说小时候抱着一只手臂导致偏斜。现在有没有好的方法或者工具可以帮助纠正？"}, {"id": 214, "vi": "Cho trẻ ngủ vào giờ nào để tăng trưởng tốt chiều cao?", "en": "When should children sleep to grow tall properly?", "zh": "什么时候让孩子睡觉才能更好地促进身高增长？"}, {"id": 215, "vi": "Con gái em cháu được 6 tuổi rồi ạ dạo gần đây em hay thấy tóc của cháu bị gãy và thỉnh thoảng có tóc trắng ạ tóc của cháu mọc lên đều là màu vàng nhạt lắm và nhìn trong ánh sáng cảm giác như là bị trắng ạ ? Có cần cho cháu đi khám không ạ hiện cháu được 20,5 kg ạ", "en": "My niece is 6 years old. Recently, I have noticed that her hair is breaking and occasionally she has white hair. Her hair grows in a very light yellow color, and under light, it looks whitish. Should we take her for a check-up? Currently, she weighs 20.5 kg.", "zh": "我家侄女6岁了，最近我经常发现她的头发容易断裂，偶尔会有白发，她的头发长出来都是淡淡的黄色，放在光下看感觉像是发白。需要带她去医院检查吗？目前她体重20.5公斤。"}, {"id": 216, "vi": "Hắt xì nhiều lần trong ngày kèm mùi tanh và hôi, uống thuốc không khỏi là bị làm sao?", "en": "Frequent sneezing throughout the day accompanied by a fishy and foul odor, not relieved by medication, what is the condition?", "zh": "每天多次打喷嚏，伴有腥臭味，服药无效是怎样的情况？"}, {"id": 217, "vi": "Nguyên nhân lệch cằm là gì?", "en": "What is the cause of jaw deviation?", "zh": "歪嘴的原因是什么？"}, {"id": 218, "vi": "Có loại thuốc nhỏ mũi nào dành cho bà bầu bị viêm xoang mãn tính không?", "en": "Is there any nasal spray medication available for pregnant women with chronic sinusitis?", "zh": "孕期慢性鼻窦炎是否有适合的鼻喷药物？"}, {"id": 219, "vi": "Tôi có một số thắc mắc muốn được giải đáp: 1. Trẻ từ bao nhiêu tuổi trở lên thì được tiêm vắc xin Covid-19? 2. Tôi trước đây hay bị dị ứng thời tiết, nước và mỗi lần như vậy phải uống thuốc mới hết. Khoảng hơn 1 năm nay thì không thấy bị như vậy nữa. Tuy nhiên cách đây vài năm tôi bị dị ứng thuốc khá nặng. Lần 1 là do bị dị ứng thời tiết nổi mề đay khắp người nên tôi mua thuốc ở tiệm thuốc tây uống, sau khi uống 1 liều thuốc tình trạng nổi mề đay không giảm mà trong khoang miệng tôi bắt đầu sưng rộp và lở loét mấy chục nốt, đau rát không ăn uống, nói chuyện được, khi đi bệnh viện khám (bác sĩ nói là bị dị ứng thuốc, cho uống thuốc điều trị 10 ngày mới khỏi). Lần 2 tôi bị viêm nhiễm phụ khoa, cũng mua thuốc ngoài tiệm thuốc tây để uống, uống 1 liều thì mặt sưng lên, cơ thể nổi mề đay và lưỡi có biểu hiện cứng, khó nói chuyện, đi khám bác sĩ bảo dị ứng thuốc và cho nhập viện điều trị mấy ngày. Sau này rút kinh nghiệm mỗi lần mua thuốc đều báo bác sĩ biết để kê toa thuốc tránh bị dị ứng. 3. Chồng tôi 45 tuổi, thỉnh thoảng bị huyết áp cao, bị viêm quan B đã chữa xong, hiện 6 tháng tái khám 1 lần và 3 năm nay không phải uống thuốc (bác sĩ không yêu cầu uống). Vậy xin hỏi tôi và chồng tôi có tiêm ngừa vắc xin Covid-19 được không?", "en": "I have several questions that I would like to be answered: 1. From what age can children receive the Covid-19 vaccine? 2. Previously, I often suffered from weather and water allergies, and each time I had to take medication to relieve the symptoms. However, over the past year, I have not experienced such symptoms anymore. Nevertheless, several years ago, I had a severe drug allergy. The first time, due to a weather-related allergic reaction causing widespread hives all over my body, I bought medication from a Western pharmacy and took one dose; after taking the medication, the hives did not subside, and I began to develop blisters and ulcers in my mouth with dozens of lesions, experiencing pain and burning, unable to eat or speak. When I went to the hospital for examination, the doctor said I had a drug allergy and prescribed treatment for 10 days before I recovered. The second time, I had a gynecological infection and bought Western medicine from a pharmacy to take orally; after taking one dose, my face swelled up, my body developed hives, and my tongue showed signs of stiffness and difficulty speaking. When I went to the hospital for examination, the doctor diagnosed me with a drug allergy and admitted me for treatment for several days. Later, I learned from experience that every time I purchase medication, I inform the doctor so that they can prescribe medication to avoid drug allergy. 3. My husband is 45 years old, occasionally has high blood pressure, had chronic prostatitis which has been treated and cured, and currently undergoes check-ups every six months, and has not been taking any medication for the past three years (the doctor has not prescribed any medication). Therefore, I would like to ask whether I and my husband can receive the Covid-19 vaccine?", "zh": "我有一些问题想咨询：1. 从几岁开始可以接种新冠疫苗？2. 我以前经常因天气和水引起过敏，每次发作都需要吃药才能缓解。但过去一年里，我再没有出现这种情况。然而几年前我曾有过严重的药物过敏。第一次是因为天气过敏导致全身出现荨麻疹，我在药店买了西药服用，服用一剂后，皮肤上的荨麻疹没有缓解，口腔内开始出现数十个水泡和溃烂，疼痛难忍，无法进食和说话，去医院就诊（医生说属于药物过敏，给予治疗10天后才痊愈）。第二次是因为妇科感染，我也在药店买了西药服用，服用一剂后，面部肿胀，全身出现荨麻疹，舌头出现僵硬、说话困难的情况，去医院就诊后医生诊断为药物过敏，并住院治疗数天。后来我吸取了教训，每次买药都会告诉医生，让医生开药避免药物过敏。3. 我丈夫45岁，偶尔会血压升高，之前患有前列腺炎已治愈，目前每6个月复查一次，过去三年未服用药物（医生未要求服药）。请问我和我丈夫是否可以接种新冠疫苗？"}, {"id": 220, "vi": "Dùng thuốc kháng sinh Penicillin xịt mũi trị viêm xoang có được không?", "en": "Can antibiotics Penicillin be used as nasal spray to treat sinusitis?", "zh": "用青霉素滴鼻剂治疗鼻窦炎可以吗？"}, {"id": 221, "vi": "Dịch ở tinh hoàn 8.1mm có tự tiêu không", "en": "Will the testicular cyst measuring 8.1 mm resolve on its own?", "zh": "睾丸囊肿8.1毫米会不会自行吸收？"}, {"id": 222, "vi": "Em vừa tham gia khám sức khỏe và bác sĩ kết luận em hồng cầu nhỏ nhược sắc. Em không hiểu, xin giải đáp giúp em.", "en": "I just participated in a health check-up and the doctor concluded that I have microcytic hypochromic anemia. I don't understand, please explain to me.", "zh": "我刚刚参加了体检，医生诊断我红细胞体积小、淡染。我不明白，恳请解答。"}, {"id": 223, "vi": "Tôi bị đau ngực tầm 10 ngày nay và chưa đến có kinh vậy hiện tượng đó là sao ..", "en": "I have had chest pain for about 10 days and haven't had my period yet; what could this be?", "zh": "我最近有大约10天的胸痛，而且还没有来月经，这种情况是怎么回事？"}, {"id": 224, "vi": "Em đang bầu. Trước bầu em đi khám bị trĩ nội độ 1. Đi ngoài thỉnh thoảng có ít máu . Gió bầu em bị nhiều và thường xuyên hơn . Vậy bác sĩ cho em hỏi có loại thuốc bôi nào để giảm chảy máu mà ko ảnh hưởng đến em bé không.", "en": "I am pregnant. Before pregnancy, I visited a doctor and was diagnosed with grade 1 internal hemorrhoids. I occasionally have a little blood in my stool. During pregnancy, I experience more wind and it occurs more frequently. Therefore, I would like to ask the doctor if there is any ointment that can reduce bleeding without affecting the baby.", "zh": "我正在怀孕。在怀孕前我去医院检查，被诊断为1度内痔，排便时偶尔有少量出血。怀孕后我感到风寒症状更频繁了。因此我想问一下医生，有没有外用药可以减少出血，且不影响胎儿？"}, {"id": 225, "vi": "Sau nhổ răng số 8 có cần khâu lại huyệt nhổ răng?", "en": "After extraction of tooth number 8, is it necessary to suture the extraction site?", "zh": "拔第8颗牙后需要缝合拔牙窝吗？"}, {"id": 226, "vi": "Em có xét nghiệm NST và có kết quả là mang NST chuyển đoạn tương hỗ giữa NST số 4 và 17. Vậy có ảnh hưởng gì đến quá trình mang thai và gây dị tật cho thai nhi không ạ?", "en": "My NST test result shows a reciprocal translocation between chromosome 4 and chromosome 17. So, does this affect the pregnancy process and cause birth defects in the fetus?", "zh": "我做了NST检查，结果是4号染色体和17号染色体之间存在相互易位。这种情况对怀孕过程以及对胎儿是否会造成畸形有影响吗？"}, {"id": 227, "vi": "Em đã từng bị dị ứng khi ăn cua rang và gạch cua, nổi mề đay sau khoảng 30 phút. Ăn nước canh cua thì chưa bị. Lần cuối vào năm 2004, sau đó rút kinh nghiệm không ăn thì rất ít bị mề đay lại. Trời hanh khô hay bị viêm da vùng đùi, bắp chân, kích ứng ngứa. Tại cơ quan đã test nhanh một lần, đã khám sàng lọc lập danh sách tiêm tại trung tâm y tế. Em nên làm gì để tiêm an toàn?", "en": "I have previously had allergic reactions when eating grilled shrimp and crab, with hives appearing about 30 minutes later. Eating shrimp soup has not caused any reaction. The last time was in 2004, after which I learned from experience and stopped eating them, so I have very rarely had hives again. In dry and hot weather, I often suffer from skin inflammation on the thighs and calves, as well as itching and irritation. At my workplace, I have undergone a rapid test once and was screened for vaccination and placed on a list for injection at the healthcare center. What should I do to receive vaccinations safely?", "zh": "我以前吃炸虾和虾干时会出现过敏，大约30分钟后出现荨麻疹。吃虾汤则没有问题。最后一次是在2004年，之后吸取教训不再吃，因此很少再出现荨麻疹。天气干燥时，常出现大腿和小腿的皮炎，以及瘙痒刺激。在单位做过一次快速检测，已进行筛查并列入了社区卫生中心的注射名单。我应该如何才能安全地接种疫苗？"}, {"id": 228, "vi": "Ban đêm bé hay bắt chéo chân tay, rặn è è là do đâu?", "en": "Why does the child frequently cross their legs and arms at night and grunt?", "zh": "宝宝晚上经常交叉手脚、皱眉是为什么？"}, {"id": 229, "vi": "Đau bụng trên rốn kèm đi ngoài nhiều lần, phân ướt cảnh báo bệnh lý gì?", "en": "Upper abdominal pain at the navel level accompanied by frequent loose stools alerts to what disease?", "zh": "右上腹痛伴多次腹泻、稀便，提示哪些疾病？"}, {"id": 230, "vi": "Thuốc rovagi3 dùng khi bị viêm họng cấp có ảnh hưởng đến thai nhi không", "en": "Does the use of rovagi3 for acute pharyngitis affect the fetus?", "zh": "罗加吉3用于急性咽炎时会对胎儿产生影响吗"}, {"id": 231, "vi": "Sỏi tiết niệu, giãn đài bể thận độ 2, ứ đọng nước 15 mm có nguy hiểm không?", "en": "Is urinary stone, degree 2 calyceal and pelvicalyceal dilation with 15 mm urine retention dangerous?", "zh": "尿路结石，肾盂肾盏扩张2度，积水15毫米是否危险？"}, {"id": 232, "vi": "Nhịp tim tôi thường cao hơn 90, khi nghỉ ngơi, khi vận động tập thể dục có thể lên 110. Sức khỏe bình thường không thấy mệt mỏi, khó thở. Mọi người bảo tôi nhịp tim bị cao, tôi có phải đi khám không?", "en": "My resting heart rate is usually higher than 90, and it can reach up to 110 during physical exercise. I have normal health, no fatigue or shortness of breath. People tell me my heart rate is high; should I go for a check-up?", "zh": "我的静息心率通常高于90，运动时可达110。平时健康，没有疲劳、呼吸困难等症状。别人说我心率偏高，我需要去医院检查吗？"}, {"id": 233, "vi": "Uống thuốc hormon tuyến giáp có ảnh hưởng gì đến cơ thể và chức năng sinh sản không?", "en": "What effects does taking thyroid hormone medication have on the body and reproductive function?", "zh": "服用甲状腺激素会对身体和生殖功能产生什么影响？"}, {"id": 234, "vi": "Nhìn đèn tia sáng giống cầu vồng khi nheo mắt có phải là dấu hiệu bệnh về mắt?", "en": "Does seeing a rainbow-like effect when looking at a bright light with the eyes closed indicate a eye disease?", "zh": "眯眼时看灯光呈现彩虹状是否为眼部疾病的表现？"}, {"id": 235, "vi": "Em mới phát hiện có bầu, thử que lên 2 vạch, xét nghiệm HCG lên 300. Hiện tại có biểu hiện đau tức bụng dưới, có lúc đau âm ỉ. Xin hỏi như vậy có vấn đề gì không ạ? Em đang bị cúm thì có thể uống thuốc nào được ạ?", "en": "I just found out I'm pregnant, the pregnancy test shows two lines, HCG test result is 300. Currently, I have lower abdominal pain, sometimes intermittent. I would like to know if this indicates any problem. Since I have the flu, which medication can I take?", "zh": "我最近发现有怀孕，验孕棒显示两条线，HCG检测值为300。目前有下腹疼痛，有时是间歇性隐痛。请问这种情况有问题吗？我目前有感冒，可以服用什么药物吗？"}, {"id": 236, "vi": "Nam giới không tiểu được sau mổ thoát vị đĩa đệm có ảnh hưởng gì?", "en": "What impact does male inability to urinate after lumbar disc herniation surgery have?", "zh": "腰椎间盘突出术后男性尿潴留有什么影响？"}, {"id": 237, "vi": "Mắc nang buồng trứng thể giáp biên có thể thực hiện trẻ hóa da CO2 không?", "en": "Can a CO2 skin rejuvenation treatment be performed for a follicular ovarian tumor with thyroid border?", "zh": "多囊卵巢综合征是否可以进行二氧化碳嫩肤？"}, {"id": 238, "vi": "Mang thai 3 tháng bị chóng mặt đau đầu nên bổ sung gì", "en": "At 3 months of pregnancy, experiencing dizziness and headache, what should be supplemented?", "zh": "怀孕3个月出现头晕头痛应补充什么"}, {"id": 239, "vi": "Bé nhà em 10 ngày nữa là tròn 5 tháng. Bé được 7kg, bé uống sữa công thức và sữa mẹ, bé biết lẫy lúc 4 tháng và đã nhỏ xong rota Bỉ ạ. Bé đang bị đi ngoài 3 ngày nay rồi, đi phân nhầy 5 6 lần 1 ngày, trước bé chỉ đi ngày 1 lần phân đẹp. Em cho bé uống men ống entergermina 2 ngày rồi, không thấy bé đỡ nên hôm nay em cho uống men vi sinh biolactomen. Hiện bé vẫn đi ngoài nhiều lần. Bác sĩ cho em hỏi là em nên cho bé uống tiếp men biolactomen không và liều lượng như nào.", "en": "My child will be exactly 5 months old in 10 days. The child weighs 7kg, is fed both formula milk and breast milk, started crawling at 4 months, and has already completed the Rotavirus vaccine. The child has had diarrhea for the past 3 days, with 5 to 6 loose stools per day, previously having only one well-formed stool per day. I gave the child Entergermina probiotic for 2 days, with no improvement, so today I started giving Biolactomen probiotic. The child still has frequent bowel movements. I would like to ask the doctor whether I should continue giving the child Biolactomen and at what dosage.", "zh": "我家宝宝再过10天就满5个月了。宝宝现在重7公斤，喝配方奶和母乳，4个月时会爬，之前已经接种完比利时的轮状疫苗。宝宝最近3天一直腹泻，每天大便5到6次，呈黏液状，之前每天只排一次成型的大便。我给宝宝吃了2天entergermina益生菌，没有好转，今天开始给宝宝吃biolactomen益生菌。目前宝宝仍然每天排便多次。医生请问一下，我是否应该继续给宝宝服用biolactomen，剂量是多少？"}, {"id": 240, "vi": "Có thể sử dụng ống bơm trực tràng điều trị táo bón khi đang cho con bú không?", "en": "Can rectal tubing be used to treat constipation while breastfeeding?", "zh": "在哺乳期间可以使用直肠给药管治疗便秘吗？"}, {"id": 241, "vi": "Cho em hỏi là ở ngực bên tay trái thinh thoảng nó nhức nhức ở gần tim là sao", "en": "May I ask, occasionally I feel a dull pain near the heart on the left side of the chest, what could this be?", "zh": "我想问一下，我左侧胸部靠近心脏的位置偶尔会隐隐作痛，这是怎么回事？"}, {"id": 242, "vi": "Em đang điều trị trào ngược và viêm dạ dày. Đến hôm nay em đã hết các triệu chứng buồn nôn, đau bụng, miệng đắng, ợ hơi . Nhưng thỉnh thoảng còn cảm thấy ê ngực nhẹ, thoáng qua vài giây, như vậy có được coi là bình thường không?", "en": "I am undergoing treatment for gastroesophageal reflux and gastritis. Until now, I have no longer experienced nausea, abdominal pain, bitter mouth, or burping. However, occasionally I still feel a mild chest discomfort for a few seconds. Is this considered normal?", "zh": "我正在接受反流性胃炎的治疗。到目前为止，我已经没有恶心、腹痛、口苦、打嗝等症状了。但偶尔仍会感到轻微的胸闷，持续几秒钟，这种情况正常吗？"}, {"id": 243, "vi": "Có phản ứng kháng thể HIV là gì ạ? Khi có phản ứng kháng thể có phải mình đã nhiễm HIV không ạ?", "en": "What is the HIV antibody reaction? When there is an antibody reaction, does that mean I have been infected with HIV?", "zh": "HIV抗体反应是什么？出现抗体反应就意味着自己已经感染HIV了吗？"}, {"id": 244, "vi": "Bệnh chàm khô nguyên nhân do đâu và cần làm gì để hạn chế lây lan?", "en": "What is the cause of dry eczema and what should be done to prevent its spread?", "zh": "干燥性湿疹的原因是什么？如何采取措施以减少传播？"}, {"id": 245, "vi": "Bé trai nhà em nay được 7 tháng tuổi, nặng 8,3kg, hiện tại đang bị tay chân miệng, do mới lần đầu có bé nên không biết các triệu chứng để bé ủ bệnh trước đó 4 ngày kèm sốt nhẹ <37,5. Nay đi khám bác sĩ cho thuốc uống được 2 ngày, nhưng bé vẫn bị nổi thêm các mụn nước, chiều tối bé sốt <=38 độ, nhất là về khuya, em có chườm hạ nhiệt bằng khăn ấm, vậy các triệu chứng trên sau khi uống thuốc có tăng bệnh không? Em có nên cho con nhập viện không? Cách chăm sóc bé bệnh thế nào?", "en": "My son is 7 months old, weighing 8.3 kg. He is currently suffering from hand, foot, and mouth disease. Since this is his first child, I don't know the symptoms he had before, and he had mild fever <37.5°C for the past 4 days. He has been taking oral medication for 2 days, but he still has new blisters appearing. In the evening, his fever is up to <=38°C, especially at night. I have been using a warm cloth to reduce his fever. After taking the medication, do these symptoms worsen? Should I take my child to the hospital? How should I care for my sick child?", "zh": "我儿子现在7个月大，体重8.3公斤，目前患有手足口病，因为这是第一个孩子，所以不清楚之前有无过类似症状，仅知道发病前4天伴有轻度发热（<37.5℃）。现在去医院开了口服药，吃了两天，但孩子仍然出现新的水疱，下午体温达到≤38℃，尤其是晚上更明显，我用温毛巾进行物理降温。请问在服用药物后，上述症状是否会加重？我是否应该带孩子去医院住院？孩子患病期间应该如何护理？"}, {"id": 246, "vi": "An uống đầy đủ có làm khối u ung thư phát triển nhanh hơn và thực phẩm an toàn cho bệnh nhân ung thư", "en": "Does adequate nutrition cause cancer tumors to grow faster and are there safe foods for cancer patients?", "zh": "充分饮食是否会使肿瘤体积更快增长，以及哪些食物对癌症患者安全？"}, {"id": 247, "vi": "Em kết hôn đã hơn 1 năm. Nhưng đến nay vẫn chưa mang bầu. Về phần chồng em thì đã có 1 người con riêng rồi. Còn em thì chưa mang bầu lần nào. Trước đây em không uống bất kỳ thuốc tránh thai nào. Kinh nguyệt của em không điều có khi 35 , 40, 45 ngày mới có kinh trở lại. Em năm nay 31 tuổi. Trước đây em đi khám bệnh viện nói buồn trứng và tử cung em bình thường. Bác sĩ tư vấn.", "en": "I have been married for more than 1 year, but I have not become pregnant yet. My husband has one child from a previous relationship, and I have never been pregnant before. I have never taken any contraceptive pills. My menstrual cycle is irregular, and it takes 35, 40, or 45 days before my period returns. I am now 31 years old. Previously, when I visited the hospital, the doctor said my ovaries and uterus are normal. The doctor advised me.", "zh": "我结婚已超过一年，至今仍未怀孕。我丈夫已有过一个亲生孩子，而我则从未怀孕过。过去我从未服用任何避孕药。我的月经不规律，有时35、40、45天才来一次。我今年31岁。以前我去医院检查，医生说我的卵巢和子宫都正常。医生建议我。"}, {"id": 248, "vi": "Bệnh lupus ban đỏ hệ thống kèm hội chứng thận hư có thể chữa khỏi hoàn toàn không?", "en": "Can systemic lupus erythematosus with nephrotic syndrome be cured completely?", "zh": "系统性红斑狼疮伴肾病综合征能否完全治愈？"}, {"id": 249, "vi": "Mắt phải em bị BRVO, nhìn hình ảnh không rõ nét, em đã tiêm 2 mũi Avastin nhưng tình trạng không cải thiện hơn, bác sĩ cho em hỏi là em có tiếp tục nên tiêm Avastin không?", "en": "Right eye, I have BRVO, vision is blurry, I have received two injections of Avastin but my condition has not improved, doctor, I would like to ask whether I should continue receiving Avastin injections?", "zh": "我右眼患有BRVO，视力模糊，我已经注射了两针阿瓦斯汀，但病情没有改善，医生请问是否应该继续注射阿瓦斯汀？"}, {"id": 250, "vi": "Hiến 1 quả thận có ảnh hưởng tới sức khỏe không?", "en": "Does donating one kidney affect health?", "zh": "捐献一个肾脏会对健康产生影响吗？"}, {"id": 251, "vi": "Cho em hỏi có xét nghiệm nào xác định bênh lậu mãn tính không và xét nghiệm PCR có phát hiện được bệnh lậu mãn tính hoặc chlamydia mãn tính không?", "en": "May I ask if there is any test to determine chronic gonorrhea, and can PCR testing detect chronic gonorrhea or chronic chlamydia?", "zh": "请问有没有检查可以确定慢性淋病？PCR检测能否发现慢性淋病或慢性衣原体感染？"}, {"id": 252, "vi": "Tôi bị mẩn ngứa, xét nghiệm ký sinh trùng âm tính hết. Tôi muốn xét nghiệm dị nguyên.", "en": "I have itching and rash, all parasitic tests are negative. I would like to undergo allergen testing.", "zh": "我有瘙痒症状，寄生虫检测结果均为阴性。我想做过敏原检测。"}, {"id": 253, "vi": "Trẻ 4 tuổi giãn não thất, bể hố sau có khỏi được không?", "en": "Can a 4-year-old child with enlarged ventricles and posterior sulcal pool recover?", "zh": "4岁儿童脑室扩大，后颅窝池能否痊愈？"}, {"id": 254, "vi": "Mẹ có mức đường huyết cao có nên cai sữa cho bé không?", "en": "Should mother stop breastfeeding due to high blood glucose levels?", "zh": "母亲血糖偏高是否应该停止哺乳？"}, {"id": 255, "vi": "Tôi có ăn lòng lợn vào 12 giờ trưa và bữa tối 7 giờ có ăn cháo lòng. Sau khi đi vệ sinh tôi thấy có máu, phân bình thường và bụng hơi đầy. Xin tư vấn giúp.", "en": "I ate pork tripe at 12 noon and had porridge with tripe at 7 p.m. After defecation, I noticed blood, normal stool, and a slightly distended abdomen. I would like to consult.", "zh": "我中午12点吃了猪肚，晚上7点吃了猪肚粥。上厕所时发现有血，大便正常，腹部略有胀感。请给予咨询。"}, {"id": 256, "vi": "Em là nam tinh trùng yếu. Xét nghiệm lượng tinh trùng tiến tới chỉ có 17%. Bác sĩ cho em hỏi phải ương thuốc gì để tăng cường chất lượng tinh trùng. Chế độ ăn uống như thế nào để cải thiện.", "en": "I am a male with weak sperm. The sperm count test shows only 17%. Doctor, I would like to ask which herbal medicine should I take to improve sperm quality. What diet should I follow to enhance it?", "zh": "我是男性，精子质量差。精液检查显示精子数量仅达到17%。医生请问，应该服用什么药物来提高精子质量？饮食上应该如何调整？"}, {"id": 257, "vi": "Co thắt đường ruột có thể ăn thịt bò không?", "en": "Can someone with intestinal stricture eat beef?", "zh": "有肠道梗阻可以吃牛肉吗？"}, {"id": 258, "vi": "Xin bác sĩ tư vấn giúp em kết quả xét nghiệm sàng lọc trên có bình thường không ạ? Double Test PAPP-A (Cobas) 5.49 Free Beta HCG (Cobas) 84.40 Miễn dịch Hội chứng Down (Trisomy 21) 1:16410 (<1:250) Hội chứng Edward (Trisomy 18) 1:37942 (<1:250) Hội chứng Patau (Trisomy 13) 1:119584.", "en": "Dear doctor, could you please advise me whether the screening test results are normal? Double Test PAPP-A (Cobas) 5.49 Free Beta HCG (Cobas) 84.40 Immune Down Syndrome (Trisomy 21) 1:16410 (<1:250) Edward Syndrome (Trisomy 18) 1:37942 (<1:250) Patau Syndrome (Trisomy 13) 1:119584.", "zh": "医生帮忙咨询一下，这份产前筛查结果正常吗？双项检测 PAPP-A（Cobas）5.49 自由β-hCG（Cobas）84.40 染色体21三体综合征（唐氏综合征）1:16410（<1:250） 染色体18三体综合征（爱德华氏综合征）1:37942（<1:250） 染色体13三体综合征（帕塔乌综合征）1:119584。"}, {"id": 259, "vi": "Em làm beta ngày 4/11 kết quả là 420.40 U/L. Hôm nay ngày 7/11 em xét nghiệm lại beta là 2066.34 U/L. Cho em hỏi với chỉ số như vậy có ổn định không ạ? Em đã có thể đi siêu âm để xem thai vào buồng tử cung hay chưa ạ? Và với chỉ số beta như vậy thai của em được bao nhiêu tuần tuổi ạ? Ngày đầu kỳ kinh cuối của em là 31/10 ạ.", "en": "My beta level on day 4/11 was 420.40 U/L. Today, on day 7/11, I retested and my beta level is 2066.34 U/L. I would like to know if such a value is stable. Can I already undergo ultrasound to check if the fetus is in the uterine cavity? And with such a beta level, how many weeks old is my fetus? My last menstrual period was on 31/10.", "zh": "我11月4日的hCG值是420.40 U/L，今天11月7日复查hCG值为2066.34 U/L。请问这样的数值是否正常？我可以做B超检查确认胎儿是否在子宫内吗？根据这样的hCG值，我的胎儿大约是多少周龄？我的末次月经是10月31日。"}, {"id": 260, "vi": "Tôi làm gói xét nghiệm SCC cho kết quả = 2,0. Xin bác sĩ cho biết với kết quả như vậy đã có nguy cơ mắc ung thư chưa?", "en": "I have performed the SCC test and the result is = 2.0. Please advise the doctor whether such a result indicates any risk of cancer?", "zh": "我的SCC检测结果为2.0。请问医生，这样的结果是否意味着有患癌症的风险？"}, {"id": 261, "vi": "Chức năng nhai chưa hồi phục sau gãy xương hàm liệu ca phẫu thuật có thành công không? Hướng điều trị thế nào?", "en": "Is masticatory function still not recovered after mandibular fracture? Was the surgery successful? What is the treatment approach?", "zh": "咬合功能在下颌骨骨折后仍未恢复，手术是否成功？如何治疗？"}, {"id": 262, "vi": "Chuột rút, vai đau mỏi, không cầm được vật nặng là bệnh gì?", "en": "Cramp, shoulder pain and stiffness, inability to hold heavy objects - what is this disease?", "zh": "抽筋、肩部酸痛、无法举起重物是什么病？"}, {"id": 263, "vi": "Cho em hỏi em mới sinh được 3 tháng mà bị cảm lạnh, biểu hiện là nước mắt nước mũi chảy nhiều, ngơ ngác, ngạt mũi thì làm cách nào để khỏi ạ?", "en": "May I ask, my baby is 3 months old and has a cold, with symptoms of excessive nasal and eye discharge, lethargy, and nasal congestion. What should I do to recover?", "zh": "我想问一下，我宝宝刚满3个月，得了感冒，症状是流鼻涕、流眼泪，眼睛发呆，鼻子不通气，该怎么办才能好呢？"}, {"id": 264, "vi": "Lưỡi của trẻ xuất hiện từng đốm đỏ cách chữa tại nhà như thế nào?", "en": "What is the home remedy for red spots appearing on a child's tongue?", "zh": "儿童舌头出现散在红点，如何用家庭方法治疗？"}, {"id": 265, "vi": "Nay con tôi đi khám, bác sĩ trả lời là u mầm lông. Xin hỏi đây là u lành hay ác?", "en": "Now my child went for a check-up, the doctor replied it is a hair follicle tumor. I would like to ask whether this is a benign or malignant tumor?", "zh": "我孩子去检查了，医生说是有毛囊瘤。请问这是良性还是恶性？"}, {"id": 266, "vi": "Phiền bác sĩ cho em hỏi: Em bị bạch biến ở bàn chân. Em có dùng thuốc bôi Vitiskin và tacrolimus đã hơn 1 tháng rồi nhưng có vẻ như bệnh tình không tiến bộ. Bác sĩ cho em hỏi bệnh bạch biến có thể chữa khỏi không và thường phải điều trị trong thời gian bao lâu ạ?", "en": "Dear doctor, I would like to ask you: I have vitiligo on my feet. I have been using topical Vitiskin and tacrolimus for more than a month, but it seems my condition has not improved. Doctor, can vitiligo be cured, and how long does treatment usually last?", "zh": "麻烦医生帮我看一下：我脚掌有白癫疯。我已经用了超过一个月的维他命皮肤霜和他克莫司软膏，但病情似乎没有好转。医生请问白癫疯能治好吗？通常需要治疗多长时间？"}, {"id": 267, "vi": "Khó chịu khi vận động hoặc cương cứng sau cắt bao quy đầu phải làm sao?", "en": "How to deal with discomfort during movement or rigidity after circumcision?", "zh": "运动时不适或包皮环切术后阴茎变硬该怎么办？"}, {"id": 268, "vi": "Trẻ sơ sinh quấy đêm là do đâu?", "en": "What causes infant nighttime restlessness?", "zh": "新生儿夜间烦躁的原因是什么？"}, {"id": 269, "vi": "Trong nhà em đang có một cuộc tranh luận, hai phe rất rõ ràng, em muốn đưa con đi tiêm chủng cho đúng lịch nhưng nhà em lại có ý kiến cho rằng không nên đi tiêm lúc này vì lo sợ bị lây Covid-19. Bác sĩ có thể cho em ý kiến về vấn đề này không?", "en": "At home, there is a clear debate between two sides; I want to take my child for vaccinations according to the schedule, but at home, there is an opinion that vaccination should not be done now due to concern about contracting COVID-19. Can the doctor please give me an opinion on this issue?", "zh": "我家正在发生一场争论，双方观点非常明确，我想按照正确的接种时间表带孩子接种疫苗，但家里有人认为现在不应当接种，担心会感染新冠。医生能给我关于这个问题的意见吗？"}, {"id": 270, "vi": "Bản thân tôi có bị nhiễm virus viêm gan B (xét nghiệm HBsAg dương tính và HBeAg âm tính), vợ tôi không có virus viêm gan B và đã được tiêm vacxin ngừa viêm gan B. Vợ tôi đang mang thai tháng thứ tư, vợ chồng tôi rất lo lắng em bé có thể bị nhiễm virus viêm gan B. Vậy xin hỏi bác sĩ, từ giờ đến lúc sinh vợ chồng tôi phải làm gì để em bé không bị lây nhiễm virus viêm gan B. Xin cảm ơn bác sĩ!", "en": "I am infected with hepatitis B virus (HBsAg positive and HBeAg negative), my wife does not have hepatitis B virus and has received the hepatitis B vaccine. My wife is currently in her fourth month of pregnancy, and we are very worried that our baby might be infected with hepatitis B virus. Therefore, I would like to ask the doctor, from now until delivery, what steps should we take to prevent the baby from being infected with hepatitis B virus. Thank you, doctor!", "zh": "我本人患有乙型肝炎病毒（HBsAg阳性，HBeAg阴性），我的妻子没有乙型肝炎病毒，且已接种过乙型肝炎疫苗。我的妻子目前怀孕四个月，我们非常担心孩子会感染乙型肝炎病毒。因此想咨询医生，从现在起到分娩期间，我们夫妻需要采取哪些措施才能防止孩子感染乙型肝炎病毒。谢谢医生！"}, {"id": 271, "vi": "Con nhà cháu hiện nay được gần 2 tháng tuổi. Mấy hôm nay cháu bị sốt liên tục 38 độ mà không thấy giảm với các triệu chứng: ngày đầu tiên bé mệt mỏi, nhiệt độ 37,8 - 37,9 độ C; ngày thứ 2 cháu húng hắng ho 1 - 2 cái; ngày thứ 3 thì ho 3 - 4 cái kèm hắt hơi. Cháu có cho đi viện làm xét nghiệm thì mọi thứ vẫn ổn, không bị cúm A. Ngày thứ 4 cháu giảm ho, chỉ ho 1 cái nhưng vẫn hắt hơi và hôm nay cũng vậy, không ho nhưng nhiệt độ cũng không giảm, vẫn 38 độ. Cháu vẫn bú mẹ bình thường, vẫn chơi. Nhưng nhiệt độ vẫn sốt 38 - 38,2 độ. Như vậy, con cháu có bình thường không ạ? Làm thế nào để cháu có thể giảm sốt? Đặc biệt, sờ trán bé lúc nóng lúc lạnh nhưng phần gáy đặc biệt nóng ạ. Cháu dùng nhiệt kế điện tử để đo ở trán bé.", "en": "The baby at home is now nearly 2 months old. For several days, the child has had continuous fever at 38 degrees without improvement, with symptoms: on the first day, the baby was tired, temperature 37.8 - 37.9 degrees C; on the second day, the baby had 1-2 coughs; on the third day, there were 3-4 coughs accompanied by sneezing. The child was taken to the hospital for testing, and all results were normal, no influenza A. On the fourth day, the cough decreased, only 1 cough, but still sneezing, and today is the same, no cough but the temperature has not decreased, still at 38 degrees. The baby is still breastfeeding normally and playing normally. However, the temperature remains at 38-38.2 degrees. Is this normal for the child? How can the child reduce the fever? Especially, when touching the baby's forehead, it is hot and cold at different times, but the back of the neck is particularly hot. The mother uses an electronic thermometer to measure the baby's forehead temperature.", "zh": "我家孩子目前将近两个月大。最近几天持续发烧，体温达到38度，且未见缓解，伴有以下症状：第一天孩子感到疲倦，体温为37.8-37.9度；第二天孩子咳嗽1-2次；第三天咳嗽3-4次并伴有打喷嚏。我带孩子去医院做了检查，各项指标均正常，未感染甲型流感。第四天孩子咳嗽减少，仅咳嗽1次，但仍打喷嚏，今天也一样，没有咳嗽，但体温仍为38度。孩子正常哺乳，正常玩耍，但体温仍维持在38-38.2度。请问孩子这种情况是否正常？如何才能帮助孩子退烧？特别的是，摸孩子额头时有时热有时冷，而后颈部位特别热。我使用电子体温计测量孩子的额头温度。"}, {"id": 272, "vi": "Nam giới bị vật nhọn nơi công cộng đâm điều trị như thế nào?", "en": "How is treatment performed for a man who is stabbed by a sharp object in a public place?", "zh": "男性在公共场所被尖锐物刺伤如何治疗？"}, {"id": 273, "vi": "Zona thần kinh mạn tính có chữa khỏi dứt điểm được không?", "en": "Can chronic neuralgia be cured definitively?", "zh": "带状疱疹神经痛能彻底治愈吗？"}, {"id": 274, "vi": "Em năm nay 29 tuổi. Em cưới chồng được gần 1 năm rồi nhưng mãi chưa có bé. Tháng này em có đi canh trứng. Chu kỳ của em 28 ngày, ngày thứ 14 em đi siêu âm thấy có 1 nang noãn trái kích thước 18x18, chẩn đoán trứng em chưa rụng, ngày thứ 17 em đi siêu âm thì nang noãn em còn 11x13. Cho em hỏi đó là do trứng không phát triển hay đã rụng ạ? Em xin cảm ơn nhiều.", "en": "I am 29 years old. I have been married for nearly one year but have not been able to conceive. This month I underwent ovulation monitoring. My cycle is 28 days long; on day 14, ultrasound showed a left ovarian follicle measuring 18x18 mm, diagnosed as not ovulated; on day 17, the follicle size was 11x13 mm. I would like to know whether this is due to the egg not developing or has already ovulated. Thank you very much.", "zh": "我今年29岁，结婚近一年了但一直没能怀孕。这个月我做了监测排卵。我的月经周期是28天，第14天做B超发现左侧有18×18毫米的成熟卵泡，诊断为卵泡未排；第17天再次做B超，卵泡大小变为11×13毫米。我想问一下，这是卵泡没有发育还是已经排卵了呢？谢谢！"}, {"id": 275, "vi": "Rối loạn thần kinh, không kiểm soát được suy nghĩ sau khi bỏ ma túy đá nên điều trị thế nào?", "en": "Nervous disorder, uncontrollable thoughts after stopping crystal meth use, how should it be treated?", "zh": "戒毒后出现无法控制的思维障碍，应如何治疗？"}, {"id": 276, "vi": "Thai 16 tuần, em bị nứt đầu vú do cơ địa từ trước giờ đã khô sẵn. Em đi tắm và có bị bong lớp vảy sần sùi. Đi khám đầu vú thì bác sĩ dặn bôi kem dưỡng ẩm. Em cũng không sờ nắn hay cậy ti. Mấy hôm nay, thỉnh thoảng lúc ngủ dậy, chợt phát hiện trên áo có vết sữa non rỉ rỉ đã khô. Ti bên phải thỉnh thoảng giật giật nhói nhói. Ra sữa non sớm điều này có ảnh hưởng gì tới sinh non không ạ?", "en": "16 weeks pregnant, the child has cracked nipples due to a pre-existing dryness of the breast tissue. The child goes bathing and has experienced flaking, rough skin. Upon breast examination, the doctor advised applying a moisturizing cream. The child has not touched or squeezed the nipples. Recently, occasionally upon waking up, the child has noticed dried, leaking colostrum stains on the clothing. The right breast occasionally feels sharp, stabbing pain. Will early colostrum production affect preterm birth?", "zh": "16周孕妇，孩子因先天体质干燥导致乳头皲裂。她洗澡时出现脱皮现象。去医院检查乳房后，医生建议使用润肤霜。她没有触摸或挤压乳头。最近几天，偶尔醒来时发现衣服上有干涸的初乳痕迹。右侧乳头偶尔会突然出现刺痛感。早产初乳分泌是否会影响早产？"}, {"id": 277, "vi": "Tôi 44 tuổi, có tiền sử hen suyễn, hở van tim đã được chữa khỏi, tôi có nên tiêm vắc xin Covid-19 không?", "en": "I am 44 years old, have a history of asthma and previously treated mitral valve regurgitation that has been cured. Should I get the COVID-19 vaccine?", "zh": "我44岁，有哮喘病史，心脏瓣膜反流已治愈，我是否应该接种新冠疫苗？"}, {"id": 278, "vi": "Đã điều trị khỏi bệnh lậu có nguy cơ tái phát không?", "en": "Has cured syphilis, is there a risk of recurrence?", "zh": "淋病治愈后有复发的风险吗？"}, {"id": 279, "vi": "Chồng cháu đi khám thì nhận được kết quả bị lỗi mất đoạn gen AZFa, không thể sản xuất tinh trùng được, không thể có con. Liệu có phương pháp nào để giúp chồng cháu không?", "en": "The husband and son-in-law went for examination and received a result showing a deletion in the AZFa gene, making sperm production impossible, thus unable to have children. Is there any method to help the husband?", "zh": "丈夫和侄子去检查时发现AZFa基因缺失，无法产生精子，无法生育。是否有方法可以帮助他们？"}, {"id": 280, "vi": "Cháu năm nay 19 tuổi. Kinh nguyệt của cháu rất thất thường 2-3 tháng mới có kinh 1 lần thậm chí kéo dài đến 6 tháng mới có. Như vậy là có bệnh không và đó là bệnh gì ạ?", "en": "I am 19 years old now. My menstrual cycle is very irregular, having a period every 2-3 months, sometimes even up to 6 months. Is this a disease, and if so, what is it?", "zh": "我今年19岁。我的月经非常不规律，每隔2到3个月才来一次，甚至长达6个月才来一次。这种情况是否说明有病？是什么病？"}, {"id": 281, "vi": "Bác sĩ cho hỏi mặt tôi bị tăng sắc tố sau viêm lên đen xanh mặt có chữa được không? Và chữa như thế nào?", "en": "Doctor, I would like to ask: after inflammation, my face has become hyperpigmented, turning dark blue-black. Is this treatable? And how should it be treated?", "zh": "医生请问，我面部因炎症后色素沉着变黑变蓝，这种情况能治好吗？该怎么治疗？"}, {"id": 282, "vi": "Điều trị cao huyết áp 4 năm nhưng các chỉ số giảm ít nên làm gì?", "en": "Treatment for hypertension for 4 years but blood pressure indices have decreased slightly, what should be done?", "zh": "高血压治疗4年但各项指标改善不多，该怎么办？"}, {"id": 283, "vi": "Hiện tại, bé nhà mình được 3 tháng tuổi. Lúc mới sinh, bé bị xuất huyết não. Một đến hai ngày nay, bé hay khóc nhiều. Việc này có ảnh hưởng gì đến não của bé không ạ?", "en": "Currently, our child is 3 months old. At birth, the child had intracranial hemorrhage. Over the past one to two days, the child has been crying frequently. Does this affect the child's brain?", "zh": "目前我家宝宝3个月大。出生时曾有脑出血。最近一到两天，宝宝经常哭闹。这种情况会对宝宝的脑部有影响吗？"}, {"id": 284, "vi": "Vết thương vẫn há sau tiểu phẫu ngón chân út phải làm gì?", "en": "What should be done for the wound that still opens after surgery on the right big toe?", "zh": "足部小趾手术后伤口仍会裂开怎么办？"}, {"id": 285, "vi": "Em đã quan hệ tình dục và bây giờ em muốn chích ngừa ung thư tử cung liệu có còn tác dụng không", "en": "I have had sexual intercourse and now I want to get vaccinated against cervical cancer—will the vaccine still be effective?", "zh": "我已经发生过性行为，现在想接种宫颈癌疫苗，还有效吗？"}, {"id": 286, "vi": "Em vừa sinh em bé được một tháng 4 ngày, em ra sản dịch hơn 22 ngày và không ra nữa. Hai hôm nay ( tức là 1 tháng 2 ngày) em lại ra máu, lúc đầu là máu đỏ tươi và sau đó là đỏ thẫm. Em nuôi con hoàn toàn bằng sữa mẹ. Vậy bác sĩ cho em hỏi là em bị gì vậy ạ và có cần phải đi khám không", "en": "I just gave birth to my baby one month and four days ago, and I have had lochia for more than 22 days and now it has stopped. Two days ago (i.e. one month and two days) I started having bleeding again, initially bright red blood, and then darker red blood. I am exclusively breastfeeding my baby. So, I would like to ask the doctor, what might be wrong with me, and do I need to visit a doctor?", "zh": "我产后一个月零四天，产后出血已经超过22天，现在没有出血了。但两天前（即产后一个月零二天）又开始出血，最初是鲜红色，后来变成深红色。我完全用母乳喂养宝宝。请问医生我这是什么情况？需要去医院检查吗？"}, {"id": 287, "vi": "Tiêm mũi sởi đơn sau 18 tháng tiêm MMR có sao không?", "en": "Is it okay to administer monovalent measles vaccine by intranasal route after 18 months of MMR vaccination?", "zh": "18个月后接种单剂麻疹疫苗是否会有问题？"}, {"id": 288, "vi": "Sỏi tiết niệu có kích thước lớn có để lại biến chứng nguy hiểm gì không?", "en": "Can large urinary tract stones lead to dangerous complications?", "zh": "泌尿系结石较大是否会导致危险的并发症？"}, {"id": 289, "vi": "Em đã từng bị sùi mào gà nhưng giờ đã hết các biểu hiện bệnh. Em đọc mạng thấy một số thuốc bổ trợ tăng sức đề kháng đào thải hpv như ahcc, ahpc... em có nên uống không ạ? Và nơi nào bán uy tín ạ?", "en": "I have previously had genital warts but now have no signs or symptoms of the disease. I read online that some supplements, such as AHCC, AHPC, etc., can boost immunity and help eliminate HPV. Should I take them? And where can I buy them reliably?", "zh": "我以前得过尖锐湿疣，但现在症状已经消失了。我在网上看到一些辅助增强免疫力、帮助清除HPV的药物，比如AHCC、AHPC等，我该服用吗？在哪里可以买到可靠的呢？"}, {"id": 290, "vi": "Thường xuyên nghẹt mũi buổi tối có phải do khối nang vòm mũi không và điều trị thế nào?", "en": "Is frequent nasal congestion at night due to ethmoid sinus cysts, and how to treat it?", "zh": "晚上经常鼻塞是否由鼻窦囊肿引起，该如何治疗？"}, {"id": 291, "vi": "Nên kết hợp vitamin và khoáng chất như thế nào để mang lại hiệu quả tốt nhất?", "en": "How should vitamins and minerals be combined to achieve the best results?", "zh": "应该如何搭配维生素和矿物质以达到最佳效果？"}, {"id": 292, "vi": "Uống thuốc Paracetamol nhiều trong 1 lần có sao không?", "en": "Is it harmful to take paracetamol in a high dose at one time?", "zh": "一次服用对乙酰氨基酚过多会有问题吗？"}, {"id": 293, "vi": "Vừa rồi tôi có đi khám virus viêm gan B thì thấy chỉ số virus là 4200 copies / ml; chỉ số HBeAb là 0,020. Riêng số virus tăng rất nhanh(trước đó 3 tháng mới chỉ là 6000 copies/ ml). Vậy xin bác sĩ cho biết tôi phải làm gì?", "en": "Recently I went for a viral hepatitis B examination and found the viral load to be 4200 copies/ml; the HBeAb level is 0.020. The viral load has increased significantly (three months ago it was only 6000 copies/ml). Therefore, I would like to ask the doctor what I should do?", "zh": "不久前我去了乙肝病毒检查，发现病毒载量为4200拷贝/ml，HBeAb指标为0.020。特别是病毒数量增长非常快（三个月前仅为6000拷贝/ml）。请问医生我该怎么办？"}, {"id": 294, "vi": "Vợ cháu có bệnh nền giảm tiểu cầu vô căn mới đây, cho cháu hỏi tiêm vắc xin Covid-19 có bị sao không? Nhờ cho lời khuyên.", "en": "My wife has recently been diagnosed with idiopathic thrombocytopenic purpura. I would like to know if it is safe to receive the Covid-19 vaccine. I kindly request your advice.", "zh": "我妻子最近被诊断为不明原因的血小板减少症，想咨询一下是否可以接种新冠疫苗？谢谢您的建议。"}, {"id": 295, "vi": "Nguyên nhân và cách điều trị thiếu máu lên não?", "en": "What are the causes and how to treat anemia in the brain?", "zh": "脑部贫血的原因及治疗方法？"}, {"id": 296, "vi": "Vợ tôi năm nay 31 tuổi lúc ngủ hay bị tê tay hoặc chân và hay đau các khớp xương tay vào chân nữa bác sĩ cho tôi hỏi vợ tôi bị bệnh gì vậy.", "en": "My wife is 31 years old and often experiences tingling in her hands or feet, and also frequently has joint pain in her hands and feet. Doctor, may I ask what disease my wife has?", "zh": "我妻子今年31岁，睡觉时经常出现手脚麻木，以及手足关节疼痛，医生请问一下我妻子得的是什么病？"}, {"id": 297, "vi": "Đau nhói lưng sau khi điều trị zona thần kinh liệu có đáng lo?", "en": "Is it concerning to have sharp back pain after treatment for shingles neuropathy?", "zh": "带状疱疹神经治疗后出现腰痛是否需要担心？"}, {"id": 298, "vi": "Bố tôi hay hút thuốc lá, ngày khoảng 2 bao Vinataba, gần đây ông thường xuyên bị ho ra đờm xanh, đau tức ngực, khó thở. Vậy ông có thể bị ung thư phổi không?", "en": "My father often smokes, about 2 Vinataba packs per day, and recently he has been experiencing frequent coughing up of green sputum, chest pain, and shortness of breath. Could he have lung cancer?", "zh": "我父亲经常吸烟，每天大约两包Vinataba，最近他经常咳嗽咳出绿色痰液，胸痛，呼吸困难。他有可能得肺癌吗？"}, {"id": 299, "vi": "Mắt tôi đi xe máy nắng buổi sáng có đeo kính đen mà vẫn bị chói mắt, nước mắt chảy ra nhìn rất mờ và khó chịu. Tôi bị bệnh gì đấy ạ.", "en": "My eyes get dazzled when I ride a motorcycle in the morning sun, even wearing dark sunglasses, and I end up with watery eyes, blurred vision, and discomfort. Do I have a disease?", "zh": "我骑摩托车在早上阳光下即使戴了墨镜，仍然感到刺眼，流泪，视力模糊且不适。我得的是什么病啊？"}, {"id": 300, "vi": "Em bị viêm tuyến giáp mà mang thai thì có ảnh hưởng đến thai nhi không", "en": "Will a mother's thyroid inflammation during pregnancy affect the fetus?", "zh": "如果母亲患有甲状腺炎且怀孕，是否会影响胎儿？"}, {"id": 301, "vi": "Em sắp sinh và phát hiện mình nhiễm virus HPV type 58. Hiện tại không có vấn đề gì xảy ra nhưng em không biết sanh thường qua ngã âm đạo được không ạ? Có lây cho bé khi sanh ngã âm đạo không ạ? Và sanh xong có cho con bú sữa mẹ được không thưa bác sĩ? Em đang rất hoang mang. Mong bác sĩ trả lời sớm ạ.", "en": "I am about to give birth and have found out I am infected with HPV type 58. Currently, there are no issues, but I don't know if a vaginal delivery is possible. Will the baby be infected during a vaginal delivery? And can I breastfeed my baby after delivery, doctor? I am very anxious. I hope the doctor can reply soon.", "zh": "我即将分娩，检查发现感染了HPV 58型。目前没有出现任何问题，但我担心顺产是否会经过阴道？顺产时会传染给婴儿吗？产后可以母乳喂养吗？医生，我非常焦虑，恳请医生尽快回复。"}, {"id": 302, "vi": "Đau giữa ngực khi nuốt thức ăn và uống nước là dấu hiệu bệnh gì?", "en": "What disease is indicated by chest pain when swallowing food and water?", "zh": "吞咽食物和喝水时出现胸骨中部疼痛是何种疾病的征兆？"}, {"id": 303, "vi": "Tiêm nhầm thuốc Bilneuro 5000 vào tĩnh mạch có làm sao không?", "en": "Is it harmful to accidentally inject Bilneuro 5000 into the vein?", "zh": "静脉误输 Bilneuro 5000 会有问题吗？"}, {"id": 304, "vi": "Hiện tượng a-mi-đan có đốm trắng và hướng điều trị như nào?", "en": "What is the treatment for the Mallory-Weiss phenomenon with white spots?", "zh": "亚当氏现象伴有白色斑点，其治疗方案是什么？"}, {"id": 305, "vi": "Cho em hỏi chỉ số CA125 của em bằng 39. Chỉ số này có sao không", "en": "May I ask if my CA125 value of 39 is normal?", "zh": "我想问一下，我的CA125数值是39，这个数值有问题吗？"}, {"id": 306, "vi": "Cảm giác nóng trong người, khô môi, nóng bàn tay, mệt trong người em uống đồ mát nên bớt nóng nhưng còn mệt đuối sức với lại cảm thấy giảm ham muốn", "en": "Feeling hot inside the body, dry mouth, hot hands, fatigue; my daughter drinks cold foods so the heat decreases but still feels fatigued and lacks desire.", "zh": "身体发热感，口干，手心发热，乏力，服用凉性食物后发热感有所减轻，但仍然感到疲乏无力，且性欲下降"}, {"id": 307, "vi": "Xét nghiệm máu có kiểm tra được nhiễm HP không?", "en": "Does blood test detect H. pylori infection?", "zh": "血液检查能否检测出幽门螺杆菌感染？"}, {"id": 308, "vi": "Trẻ xét nghiệm TSH chỉ số bình thường nhưng kết quả xét nghiệm máu bất thường có bị bệnh suy giáp hay không?", "en": "Is it possible for a child to have hypothyroidism if the TSH test result is normal but the blood test results are abnormal?", "zh": "儿童TSH检测值正常，但血液检测结果异常，是否患有甲状腺功能减退症？"}, {"id": 309, "vi": "Thưa bác sĩ bé nhà tôi được 8 ngày tuổi nhưng từ lúc sinh ra bị lệch 1 bên má. Không biết khi bé lớn lên có bị lệch như vậy không? Lúc mới sinh bị lệch hẳn nhưng giờ đỡ hơn 1 chút. Có phương pháp nào giúp bé không bị lệch từ giai đoạn này không ạ?", "en": "Dear doctor, my baby is 8 days old but has had a facial asymmetry on one side since birth. I don't know if the child will develop this asymmetry as they grow older. At birth, the asymmetry was very obvious, but now it has slightly improved. Is there any method to help the baby not develop this asymmetry from this stage?", "zh": "医生您好，我家宝宝8天大，从出生起就有一侧脸颊偏斜。不知道以后长大会不会也这样？刚出生时偏斜很明显，现在稍微好了一些。请问有没有什么方法可以在这个阶段帮助宝宝避免继续偏斜呢？"}, {"id": 310, "vi": "Nghén nặng và ói mửa nhiều khi mang thai phải làm sao?", "en": "What should I do if I have severe nausea and vomiting during pregnancy?", "zh": "怀孕期间严重孕吐和频繁呕吐怎么办？"}, {"id": 311, "vi": "Nguyên nhân gây nổi mẩn đỏ và ngứa ngáy da đầu ở trẻ nhỏ", "en": "Causes of red rash and itchy scalp in young children", "zh": "引起婴幼儿头皮出现红斑和瘙痒的原因"}, {"id": 312, "vi": "Phương pháp khắc phục tác dụng phụ của thuốc điều trị HIV và lao phổi?", "en": "What methods to manage adverse effects of HIV and tuberculosis treatment drugs?", "zh": "治疗HIV和肺结核药物的副作用处理方法？"}, {"id": 313, "vi": "Phương pháp điều trị viêm lợi tại nhà như thế nào?", "en": "What is the method of treating gingivitis at home?", "zh": "如何在家治疗牙龈炎？"}, {"id": 314, "vi": "Sau sảy thai bao lâu nên mang thai lại", "en": "How long after miscarriage should one conceive again?", "zh": "流产之后多久可以再次怀孕"}, {"id": 315, "vi": "Con em lúc sinh ra khớp gối bị trật xương ra đằng sau đầu gối không co lên dược bác sĩ bảo là bẩm sinh .Nhưng về nhà tầm dược 3 ngày thì gối dần cử động và co lên được cái xương bị trât ra sau cũng dần vào khớp nhưng giờ chân bé rất yếu, mềm khớp gối lỏng lẻo và gối bị mất độ vững cứ bị ật ra sau so với chân kia.", "en": "The baby had a dislocated knee joint at birth, with the bone displaced backward from the back of the knee, and the doctor said it was congenital. However, at home after about 3 days, the knee gradually started to move and the dislocated bone slowly returned into the joint. But now the baby's leg is very weak, soft, the knee joint is loose, and the knee has lost stability, constantly slipping backward compared to the other leg.", "zh": "宝宝出生时膝关节出现股骨向后脱位，医生说是先天性。但回家后大约三天，膝关节逐渐可以活动，脱位的股骨也慢慢回纳到关节内，但现在宝宝腿部非常虚弱，膝关节松软，膝关节稳定性差，总是向后移位，与另一条腿不一致。"}, {"id": 316, "vi": "Phụ nữ tiêm vaccine ngừa Covid-19 sau bao lâu thì mới được mang thai ạ? Người có tiền sử huyết áp cao có tiêm vaccine ngừa Covid được không?", "en": "How long after receiving the COVID-19 vaccine can women get pregnant? Can individuals with a history of high blood pressure receive the COVID-19 vaccine?", "zh": "接种新冠疫苗后，女性需要等待多久才能怀孕？有高血压病史的人可以接种新冠疫苗吗？"}, {"id": 317, "vi": "Em đã có một bé 9 tuổi rồi, nhưng từ đó đến nay đã ba lần mang thai (tự nhiên) mà cứ được 3-4 tháng là bị sảy. Em mong muốn được khám và tìm ra nguyên nhân bất thường để điều trị và thả bầu trong năm nay ạ.", "en": "I have a 9-year-old child, but since then I have had three natural pregnancies, each ending in miscarriage after 3-4 months. I hope to visit a clinic and find out the underlying abnormal cause so that I can receive treatment and have a safe pregnancy this year.", "zh": "我已经有一个9岁的孩子了，但从那时起至今已经自然怀孕三次，每次维持3-4个月就流产。我希望可以检查并找出异常原因，以便治疗并今年终止妊娠。"}, {"id": 318, "vi": "Phổi có vết đen sau điều trị lao màng phổi có đáng lo?", "en": "Is it concerning that the lungs have black spots after treatment for tuberculous pleurisy?", "zh": "肺在胸膜结核治疗后出现黑色斑点需要担心吗？"}, {"id": 319, "vi": "Đau bụng, chán ăn, sụt cân có phải bị ung thư không?", "en": "Abdominal pain, loss of appetite, weight loss - could this be cancer?", "zh": "腹痛、食欲不振、体重下降是否可能是癌症？"}, {"id": 320, "vi": "Cho em hỏi em bị viêm gan B và bị xơ gan độ 3 và độ 4 và sau khi tôi uống thuốc được 2 tháng có đi kiểm tra lại thì giảm xuống độ 2 độ 3. Vậy cho em hỏi liệu gan của em có trở lại bình thường được không?", "en": "May I ask, I have hepatitis B and cirrhosis stage 3 and 4. After taking medication for 2 months and undergoing a follow-up check-up, my cirrhosis has decreased from stage 3 to stage 2. Therefore, I would like to know whether my liver can return to normal?", "zh": "我想问一下，我患有乙型肝炎，已经发展到3期和4期肝硬化，我在服药两个月后复查，肝硬化程度从3期和4期下降到了2期和3期。请问我的肝脏能否恢复正常？"}, {"id": 321, "vi": "Viêm mao mạch dị ứng do thời tiết điều trị như thế nào?", "en": "How is allergic vasculitis due to weather treated?", "zh": "天气诱发的血管炎如何治疗？"}, {"id": 322, "vi": "Sau phẫu thuật nối gân ở cẳng tay sấp thì bao lâu mới tập luyện được?", "en": "How long after a palm tendon repair surgery can I start exercising?", "zh": "手臂掌侧韧带手术后多久可以开始锻炼？"}, {"id": 323, "vi": "Kết quả xét nghiệm HBsAg âm tính có nguy cơ mắc bệnh viêm gan B không?", "en": "Does a negative HBsAg test result have a risk of contracting hepatitis B disease?", "zh": "乙肝表面抗原检测阴性，是否具有乙型肝炎感染的风险？"}, {"id": 324, "vi": "Con hỏi tại sao con cứ nằm xuống là nghe nóng, khó chịu vùng bụng, tim thì đập mạnh, nhanh hơn bình thường rất nhiều ạ?", "en": "The child asks why, whenever lying down, they feel hot, uncomfortable in the abdominal area, and the heart beats strongly and much faster than normal?", "zh": "孩子问为什么一躺下就感觉腹部发烫、不适，心跳也变得特别快、特别强呢？"}, {"id": 325, "vi": "Trên chân tôi có những vết tím xanh nhà bị và chạm mạnh mặc dù tôi không va chạm vào đâu các vết đỏ nó to khoảng 2-3 cm", "en": "On my foot, I have several purple-blue spots that appear after strong pressure, even though I haven't collided with anything; the red spots are about 2-3 cm in size.", "zh": "我的脚上出现了几处紫色瘀斑，即使我没有受到撞击，只是用力按压也会出现红色，这些红斑大小约为2-3厘米。"}, {"id": 326, "vi": "Bị điện cao thế 3FA phóng từ trên đỉnh đầu có thể tỉnh lại không?", "en": "After being exposed to high-voltage 3FA discharged from the top of the head, can one regain consciousness?", "zh": "遭受3FA高压电从头顶射来，能否醒来？"}, {"id": 327, "vi": "A-mi-đan sưng là tình trạng của viêm a-mi-đan hay bệnh da liễu?", "en": "Tonsillar swelling is a condition of tonsillitis or dermatological disease?", "zh": "腺样体肿大是炎症还是皮肤病？"}, {"id": 328, "vi": "Khi nhai thức ăn hoặc há miệng to, mắt bị co giật là do đâu?", "en": "What is the cause of eye twitching when chewing food or opening the mouth widely?", "zh": "咀嚼食物或张大嘴巴时，眼睛出现抽搐是由于什么原因？"}, {"id": 329, "vi": "Tôi bị vàng da tôi đã đi khám xét nghiệm máu và chụp x-quang gan tất cả đều bình thường mà da tôi vàng.", "en": "I have jaundice. I have gone for blood tests and liver X-ray, all of which are normal, yet my skin is yellow.", "zh": "我有黄疸，我去做了血液检查和肝部X光检查，所有检查结果都正常，但我的皮肤还是发黄。"}, {"id": 330, "vi": "Em có khám ở chỗ mình bác sĩ kê cho hai loại thuốc này là urofizz và femifortil từ bỏ em uống hai loại này thì không còn nghén hay chán ăn nữa mà ăn rất nhiều như trưa từng bị nghén thì có sao không.", "en": "I have visited the clinic where the doctor prescribed these two medications: Urofizz and Femifortil. After stopping the use of these two, I no longer feel nausea or loss of appetite and am eating a lot. In the afternoon, I previously felt nausea—does that matter?", "zh": "我按照医生开的两种药乌罗菲兹和费米福利进行检查，停用这两种药后，不再有孕吐或食欲不振，反而吃得很多，就像中午之前有孕吐一样，这样会有问题吗？"}, {"id": 331, "vi": "Lần trc có đi khám thì ngta chuẩn đoán bị viêm abidan . Nhưng dạo gần đây em đã uống hết thuốc nhưng có triệu chứng sốt cử động mắt thấy đau ( kiểu là liếc ...). Mong bác sỹ tư vấn giúp em.", "en": "Previously, I went for a check-up and was diagnosed with abidan inflammation. But recently, after having finished the medication, I have developed symptoms such as fever and eye movement pain (like when glancing...). I hope the doctor can advise me.", "zh": "之前去医院检查，医生诊断为阿比丹炎。但最近我吃完药后出现发热、眼球运动疼痛（就是眨眼时的感觉）。恳请医生给予指导。"}, {"id": 332, "vi": "Mẩn đỏ khắp người sau quan hệ với gái mại dâm có phải nhiễm lậu không?", "en": "Do itching and redness all over the body after having sex with a prostitute, could this be syphilis?", "zh": "与卖淫女发生性关系后全身出现瘙痒红斑是否感染淋病？"}, {"id": 333, "vi": "Sau tai tôi có một cục nhô lên nhìn có vẻ giống cục xương hầu hết ai cũng có sau vài ngày thì nó không nhô lên nó lại xẹp xuống sau vài tháng thì dưới lỗ tai gần gần đó có một cục nhỏ tôi đụng vô thì nhứt nhẹ tôi không bít có bị gì không?", "en": "After the ear, I have a bump that looks like a thyroid gland, everyone has it. After a few days, it doesn't protrude anymore, it collapses down. After a few months, near the ear hole, there is a small lump, when I touch it, it feels slightly. I don't know if I have any problem.", "zh": "我耳朵后面有一个隆起的肿块，看起来像喉部的肿块，大家都有，几天后它就不再隆起，反而瘪下去了。几个月后，在耳朵后面附近出现了一个小肿块，碰到时会有一点轻微的触感，我不确定自己有没有问题？"}, {"id": 334, "vi": "Sử dụng thuốc nào để điều trị bệnh phổi tắc nghẽn?", "en": "Which drug is used to treat obstructive lung disease?", "zh": "使用哪种药物来治疗阻塞性肺病？"}, {"id": 335, "vi": "Tụ dịch màng nuôi 15 tuần 4 ngày có ảnh hưởng thai nhi và dùng duphaston no-spa có an toàn không", "en": "15 weeks 4 days amniotic fluid accumulation - does it affect the fetus? Is it safe to use Duphaston No-Spa?", "zh": "15周4天羊水积聚是否会影响胎儿？使用杜芬斯顿诺-斯帕安全吗？"}, {"id": 336, "vi": "Mình thấy nhiều ca nhiễm không có triệu chứng, xét trên phương diện cá nhân thì những người đó có được tính là miễn dịch với vi rút không ạ? Hay là tùy biến thể?", "en": "I have seen many cases of infection without symptoms; from a personal perspective, can those individuals be considered immune to the virus or just variant?", "zh": "我看到有很多无症状感染病例，从个人角度来看，这些人是否可以被认为是病毒免疫？还是说只是针对特定变种？"}, {"id": 337, "vi": "Điều trị vôi tuyến tùng thế nào?", "en": "How to treat thyroid gland calcification?", "zh": "如何治疗嗜酸性粒细胞增多症？"}, {"id": 338, "vi": "Người ghép thận có tiếp xúc được với trẻ vừa tiêm vắc-xin thủy đậu sống không?", "en": "Can a kidney transplant recipient come in contact with a child who has just received a live varicella vaccine?", "zh": "肾移植患者能否与刚接种水痘活疫苗的儿童接触？"}, {"id": 339, "vi": "Quan hệ không an toàn với người bị viêm gan B có tiêm phòng được nữa không?", "en": "Can someone who has had unsafe sexual contact with a person with hepatitis B still receive vaccination?", "zh": "与乙肝患者发生不安全性行为后还能接种疫苗吗？"}, {"id": 340, "vi": "Bao tay của nhân viên y tế dùng tiêm vắc xin hoặc lấy mẫu có thay cho từng người hay sử dụng chung cho nhiều người? Sử dụng chung có lây nhiễm nếu có người dương tính mà họ không biết không? Vì biến chủng delta lây rất nhanh mà việc bao tay tiếp xúc với nhiều người rất nguy hiểm.", "en": "Are the gloves used by healthcare workers for injections or sample collection used individually or shared among multiple people? Could there be transmission if a positive case is present but unknown, when gloves are shared? Because the Delta variant spreads very rapidly, frequent glove contact with multiple people is very dangerous.", "zh": "医护人员接种疫苗或采样时，是每人单独使用手套，还是多人共用？如果有人检测为阳性但不知情，共用手套是否会导致交叉感染？由于德尔塔变种传播速度极快，频繁接触多人使用手套存在很大风险。"}, {"id": 341, "vi": "Tóc rụng nhiều phải làm gì?", "en": "What should I do if I have excessive hair loss?", "zh": "脱发严重该怎么办？"}, {"id": 342, "vi": "Viêm không đặc hiệu cổ tử cung, biến đổi lành tính và tạp khuẩn điều trị như thế nào?", "en": "How to treat nonspecific cervical inflammation, benign changes, and mixed infection?", "zh": "宫颈非特异性炎症、良性变异及杂菌如何治疗？"}, {"id": 343, "vi": "Lồi mắt, hay run chân tay, siêu âm tuyến giáp có Basedow thì có sao không?", "en": "Eye protrusion, or hand and foot tremors, if the thyroid ultrasound shows Basedow's disease, is there any problem?", "zh": "眼突，或手足震颤，甲亢的超声检查有格雷夫斯病，会怎样？"}, {"id": 344, "vi": "Cho tôi hỏi chích thuốc kích trứng mà trứng không lớn hoặc lớn chậm thì có nên chích tiếp không, và chích nhiều có ảnh hưởng đến buồng trứng không.", "en": "May I ask whether I should continue injecting fertility-stimulating medication if the egg does not grow or grows slowly, and does injecting frequently affect the ovaries?", "zh": "请问注射促排卵药物后卵泡不长大或生长缓慢，是否应该继续注射？频繁注射会对卵巢造成影响吗？"}, {"id": 345, "vi": "Lão hóa ở người già phải làm sao?", "en": "What should be done about aging in the elderly?", "zh": "老年人如何应对衰老？"}, {"id": 346, "vi": "Chỉ số GPT (ALT) là gì? Ảnh hưởng như thế nào khi đạt ngưỡng 51 U/L?", "en": "What is the GPT (ALT) level? How does it affect when reaching the threshold of 51 U/L?", "zh": "ALT（GPT）指标是什么？当达到51 U/L阈值时有何影响？"}, {"id": 347, "vi": "Nam giới giãn tĩnh mạch điều trị như thế nào?", "en": "How is male varicose vein treated?", "zh": "男性静脉曲张如何治疗？"}, {"id": 348, "vi": "Sáng nay sau khi ngủ dậy cháu thấy da ở vùng kẻ cổ gần vai bị rát râm ran đụng vào thì rát đau. Cháu đã đi tắm 1 lần với nước không vào buổi sáng và 1 lầm với xà bông diệt khuẩn Lifebuoy mà vẫn không hết (bình thường bị ngứa thì cháu tắm sạch là khỏi). Không biết cháu bị sao ạ cháu còn có nổi mụn cóc không biết liên quan gì không. Và cho cháu hỏi thêm là tai sao da ở vùng bầu vú nếu gãi nhiều thì hiện chấm đỏ đỏ như cạo gió ạ.", "en": "This morning, after waking up, the child felt a burning and itching sensation on the skin near the neck, close to the shoulder, which became painful when touched. The child has taken a bath once with plain water in the morning and once with Lifebuoy antibacterial soap, but the symptoms have not improved (usually, when the child gets itchy, bathing clean resolves it). I don't know what the child has. The child also has warts; I don't know if they are related. Additionally, I would like to ask: why does the skin on the breast area turn red and reddened when scratched frequently?", "zh": "今天早上醒来后，孩子发现颈侧靠近肩部的皮肤有灼烧感，触碰时会感到疼痛。孩子早上洗了1次没有添加的水，又用Lifebuoy抗菌肥皂洗了1次，但症状仍未缓解（平时一痒就会洗完就没事）。不知道孩子是什么情况，孩子还有没有出现疣的情况，是否有关联？另外想请教一下，为什么乳房区域的皮肤如果抓挠较多，会出现红色斑点，像被刮伤一样？"}, {"id": 349, "vi": "Hồi tháng 6 em có làm siêu âm tuyến giáp và phát hiện khối đồng âm có nhân ở thuỳ phải tuyến giáp, kích thước 7x3mm. Tầm tháng 9 vừa rồi em có làm xét nghiệm chức năng tuyến giáp, các chỉ số T3 FT4 và THS vẫn nằm trong ngưỡng bình thường. Em có nên đi làm xét nghiệm sinh thiết để chắc chắn đây là u lành không? Có hướng điều trị nào cho bệnh của em không?", "en": "In June, I underwent an ultrasound of the thyroid and discovered a hypoechoic lesion with a central focus in the right lobe of the thyroid, measuring 7x3 mm. In September, I had a thyroid function test, and the values of T3, FT4, and TSH remained within the normal range. Should I undergo a biopsy to confirm whether this is a benign tumor? Are there any treatment options available for my condition?", "zh": "6月份我做了甲状腺超声检查，发现右叶甲状腺有一个实性结节，大小为7x3mm。9月份我做了甲状腺功能检查，T3、FT4和TSH指标均在正常范围内。我是否应该做活检以确认这个结节是良性？是否有针对我这种情况的治疗方案？"}, {"id": 350, "vi": "Em bị chậm kinh khoảng 10 ngày trước em đi khám bác sĩ cho em uống thuốc kích thích ra kinh , đến nay thì có máu ra nhưng ít máu ra thành cục nhỏ vậy cho em hỏi em có sao không", "en": "My period is delayed by about 10 days. I went to see a doctor who prescribed me medication to induce menstruation. Until now, I have bleeding, but it is minimal and appears in small clots. I would like to know if I am in any trouble.", "zh": "我之前月经推迟了约10天，去医院看医生后，医生给我开了促经药。到现在已经有出血，但出血量很少，呈小块状，我想问一下，我有没有问题？"}, {"id": 351, "vi": "Bệnh nhân phù não sau phẫu thuật tim nên dùng thuốc nào để sớm hồi phục?", "en": "What medication should a patient with cerebral edema after cardiac surgery use to recover early?", "zh": "心脏手术后出现脑水肿的患者应使用哪种药物以促进早期恢复？"}, {"id": 352, "vi": "Mắc hồng cầu nhỏ thì có tham gia hiến máu được không và uống rượu bia có ảnh hưởng gì không?", "en": "Can someone with microcytic anemia donate blood, and does drinking alcohol have any impact?", "zh": "红细胞体积小是否可以参加献血？饮酒是否有影响？"}, {"id": 353, "vi": "Người cao tuổi bị béo phì, cao huyết áp, gan nhiễm mỡ có nên uống thực phẩm chức năng hỗ trợ giảm cân không?", "en": "Should elderly patients with obesity, hypertension, and fatty liver take dietary supplements to support weight loss?", "zh": "老年人患有肥胖、高血压、脂肪肝，是否应该服用辅助减重的膳食补充剂？"}, {"id": 354, "vi": "Bạn em 32 tuổi, hay bị trễ kinh từ 10-15 ngày. Mỗi lần có kinh bị đau bụng, đau lưng nhiều, kinh nguyệt ra nhiều. Tình trạng như vậy có ảnh hưởng đến việc vô sinh không và cách điều trị như thế nào?", "en": "My sister is 32 years old and often has menstrual delay from 10 to 15 days. Each time she has her period, she experiences severe abdominal and back pain, and heavy menstrual bleeding. Does this condition affect fertility, and what is the treatment approach?", "zh": "我妹妹32岁，月经经常推迟10-15天。每次来月经时都会出现腹痛、腰痛，经量较多。这种情况是否会影响不孕，该如何治疗？"}, {"id": 355, "vi": "Bác sĩ làm ơn cho em hỏi: em làm nghiệm pháp dung nạp glucose khi mang thai ở tuần 28. Kết quả: lúc đói: 6.3 mmol/L; sau uống 1 giờ: 5.7 mmol/L; sau uống 2 giờ: 12.1 mmol/L. Em đang thắc mắc tại sao có kết quả không thuận như vậy hả bác sĩ? (Sao lại sau uống glucose 1 giờ thì đường huyết lại thấp hơn lúc chưa uống và sau 2 giờ lại cao như vậy ạ?). Như thế em được kết luận tiểu đường thai kỳ như thế nào ạ? Em cần điều trị bằng thuốc không?", "en": "Doctor, please help me with the following question: I performed a glucose tolerance test during pregnancy at week 28. The results were: fasting: 6.3 mmol/L; one hour after drinking glucose: 5.7 mmol/L; two hours after drinking glucose: 12.1 mmol/L. I am confused as to why the results are not normal—(why is the blood glucose level lower after one hour of drinking glucose than at fasting, and then high again after two hours?). Based on these results, how should I be diagnosed with gestational diabetes? Do I need to be treated with medication?", "zh": "医生您好，请问一下：我怀孕28周时做了葡萄糖耐量试验。结果如下：空腹时6.3 mmol/L；饮用葡萄糖1小时后5.7 mmol/L；饮用葡萄糖2小时后12.1 mmol/L。我非常疑惑，为什么结果会这样呢？（为什么喝葡萄糖1小时后血糖反而比空腹时还低，而2小时后又升高这么多呢？）这样的话，我是否可以被诊断为妊娠糖尿病呢？我需要用药治疗吗？"}, {"id": 356, "vi": "Bị đau nhức sau khi bó bột gãy xương mác là do đâu?", "en": "What is the cause of pain and discomfort after casting a torn medial malleolus fracture?", "zh": "骨折后包扎出现疼痛的原因是什么？"}, {"id": 357, "vi": "Xét nghiệm máu cho số lượng bạch cầu 9440 có bất thường không?", "en": "Is there any abnormality in the blood test for white blood cell count of 9440?", "zh": "血常规白细胞计数为9440是否异常？"}, {"id": 358, "vi": "Bác sĩ cho em hỏi chu kì kinh của em không đều. Từ tháng trước có kinh tính đến thời điểm này e bị chậm kinh 26 ngày rồi, tổng ngày kéo dài là 52 ngày rồi chưa thấy có kinh nguyệt, hiện tại thì e thấy biểu hiện đau 2 bầu vú, trước đó thì đi tiểu đau và dắt.Em vẫn quan hệ đều ko sử dụng bao cao su. Trc đó e đã đi siêu âm xét nghiệm ko có thai. Bác sĩ tư vấn cho em cần phải làm những xn gì ạ.", "en": "Doctor, I would like to ask you about my irregular menstrual cycle. From last month until now, I have been experiencing a delayed period by 26 days, and the total duration has reached 52 days without menstruation. Currently, I am experiencing breast pain; previously, I had painful urination and frequent urination. I still have regular intercourse without using condoms. Previously, I underwent ultrasound and pregnancy testing, and the results showed no pregnancy. Doctor, could you please advise me on which further examinations I should undergo?", "zh": "医生请问一下，我的月经周期不规律。上个月开始到现在，月经推迟了26天，总共持续了52天，至今还没有来月经。目前我感觉乳房胀痛，之前有过小便疼痛和尿频的情况。我仍然有性生活，但没有使用安全套。之前我做过超声检查，未发现怀孕。医生建议我需要做哪些检查呢？"}, {"id": 359, "vi": "Sổ mũi kéo dài có phải dấu hiệu của viêm xoang không?", "en": "Does prolonged nasal congestion indicate sinusitis?", "zh": "慢性鼻塞是否是鼻窦炎的征兆？"}, {"id": 360, "vi": "Viêm nướu ở người cao tuổi nên uống thuốc nào?", "en": "What medication should elderly patients with gingivitis take?", "zh": "老年人牙龈炎应该服用什么药？"}, {"id": 361, "vi": "Em xem kết quả khám bệnh, huyết áp của em là 100/60, vậy cho em hỏi có phải em bị huyết áp thấp không ạ? Vì thời gian gần đây e rất hay hoa mắt, chóng mặt ạ?", "en": "I reviewed my medical examination results, my blood pressure is 100/60, so I would like to ask if this means I have low blood pressure? Because recently I have been experiencing frequent dizziness and lightheadedness.", "zh": "我看了体检结果，我的血压是100/60，所以想问一下，这是否说明我患有低血压？因为最近我经常感到头晕、眼花。"}, {"id": 362, "vi": "Em đang mang thai tuần thứ 37 nhưng chưa làm dung nạp đường huyết thai kỳ thì giờ có nên làm xét nghiệm này hay không ạ?", "en": "I am at 37 weeks of pregnancy but have not yet done the gestational diabetes tolerance test. Should I do this test now?", "zh": "我目前怀孕37周，但尚未进行妊娠糖耐量检测，现在做这个检查还来得及吗？"}, {"id": 363, "vi": "Kết quả xét nghiệm men G6PD 3,7 lần có nghĩa là gì?", "en": "What does the test result of G6PD enzyme 3.7 times mean?", "zh": "G6PD酶检测结果为3.7倍是什么意思？"}, {"id": 364, "vi": "Dạo này cháu hay đi bơi hôm qua về mắt cháu bị đỏ cháu có sử dụng thuốc nhỏ mắt nhưng cho đến hôm nay mắt cháu sưng đỏ lên bác sĩ có thể cho cháu cách khắc phục không?", "en": "Recently, my child often goes swimming. Yesterday, after swimming, my child's eyes became red. My child used eye drops, but until today, the eyes are swollen and red. Doctor, can you please advise me on how to remedy this?", "zh": "最近孩子经常游泳，昨天游泳回来后孩子眼睛发红，孩子用了眼药水，但到今天眼睛仍然肿胀发红，医生能给孩子一些处理方法吗？"}, {"id": 365, "vi": "Có cách nào điều trị thủng màng nhĩ lớn đã lâu không khỏi không?", "en": "Is there any way to treat a large eardrum perforation that has not healed for a long time?", "zh": "对于久治不愈的较大鼓膜穿孔，有没有治疗方法？"}, {"id": 366, "vi": "Trẻ 10 tuổi hay làm tổn thương bản thân, nói chuyện 1 mình, la hét, ghét trêu đùa và cầm dao kéo có dấu hiệu của bệnh tự kỷ không?", "en": "Does a 10-year-old child who self-harms, talks to themselves, screams, dislikes teasing, and holds a knife have signs of autism?", "zh": "10岁儿童经常自伤，独自说话，大喊大叫，厌恶开玩笑，并持有刀具，是否提示自闭症？"}, {"id": 367, "vi": "Các hạch di động ở gáy, sau tai, cổ có kích thước to dần liệu bình thường không?", "en": "Are gradually increasing in size cervical lymph nodes at the base of the skull, behind the ear, and neck normal?", "zh": "耳后、颈部及枕部可触及的活动性淋巴结，其大小逐渐增大是否正常？"}, {"id": 368, "vi": "Thường xuyên bị xây xẩm mặt mày, cứng hàm, đỏ mặt, ù tai phải làm sao?", "en": "Frequently experiencing facial flushing, jaw stiffness, facial redness, and right ear fullness - what should I do?", "zh": "经常出现面部发麻、下颌僵硬、面部发红、右耳耳鸣怎么办？"}, {"id": 369, "vi": "Cháu đi khám thì thấy bảo u tuyến giáp. Uống thuốc 1 thời gian rồi thấy cũng đỡ nhưng mặt và tay chân lại bị phù lên rất to không biết có bị sao không ạ", "en": "I went for a check-up and was told I have a thyroid nodule. I took medication for some time and felt better, but my face and arms and legs have become very swollen. I don't know if there's anything wrong with me.", "zh": "我去检查时被告知有甲状腺结节。吃药一段时间后感觉有所好转，但面部和四肢却明显肿胀，不知道是否有什么问题。"}, {"id": 370, "vi": "Viêm da cơ địa xuất hiện các mụn mủ và đau nhức là bị làm sao?", "en": "Atopic dermatitis appears with pus blisters and pain - what should be done?", "zh": "特应性皮炎出现脓疱和疼痛怎么办？"}, {"id": 371, "vi": "Đầu lồi lõm có vết lõm ở đỉnh đầu có sao không", "en": "Is there a problem with a prominent forehead with a depression at the vertex of the head?", "zh": "头顶有凹陷的前额突出是否有问题"}, {"id": 372, "vi": "Bé 5 tháng tuổi bị zona có nguy hiểm không?", "en": "Is it dangerous for a 5-month-old child to have shingles?", "zh": "5个月大的婴儿患带状疱疹危险吗？"}, {"id": 373, "vi": "Em vừa sinh đc 2 tháng nhưng ra tháng em đã có kinh trở lại thì cho em hỏi em muốn ngừa tranh thai biện pháp nào là an toàn và tạm thời để em ngừa thai bằng cách uống thuốc thì em uống thuốc loại nào thì được.Vì em rất dễ có bầu nên em cần biện pháp ngừa hiện tại.Bác sỹ cho em chút tư vấn và ý kiến.", "en": "I just gave birth 2 months ago, but my period has already returned. I would like to ask: what is the safest and temporary method of contraception for me to use orally? Which oral contraceptive should I take? Since I am very prone to getting pregnant, I need a current contraceptive method. Doctor, please give me some advice and opinion.", "zh": "我产后两个月，月经就恢复了，想咨询一下，我想要一种安全且暂时的避孕方法，如果选择口服避孕药，应该服用哪种？因为我很容易怀孕，所以需要目前的避孕措施。医生请给我一点建议和意见。"}, {"id": 374, "vi": "Nguyên nhân dẫn đến thiếu hụt D3 là gì?", "en": "What is the cause of vitamin D3 deficiency?", "zh": "导致维生素D3缺乏的原因是什么？"}, {"id": 375, "vi": "Trẻ không phân biệt được màu sắc và ghi nhớ khi học nên làm gì?", "en": "What should be done when a child cannot distinguish colors and remember during learning?", "zh": "孩子不能分辨颜色且学习时记忆力差该怎么办？"}, {"id": 376, "vi": "Chồng tôi bị yếu tinh trùng, tinh trùng ít. Vậy cho tôi hỏi có biện pháp hay cách điều trị giúp vợ chồng tôi có con được không?", "en": "My husband has low sperm count and poor sperm quality. Thus, I would like to ask if there are any methods or treatments that can help my husband and I have a child?", "zh": "我丈夫有精子活力差、精子数量少的问题。我想咨询一下，是否有办法或治疗方法可以帮助我们夫妻生育孩子？"}, {"id": 377, "vi": "Trẻ ngại tiếp xúc với người lạ, ít phản kháng nên làm thế nào?", "en": "The child is shy around strangers and has little resistance, so what should be done?", "zh": "孩子害怕与陌生人接触，反应较弱，该怎么办？"}, {"id": 378, "vi": "Bác sĩ cho em hỏi nốt vôi hóa thận có thể dẫn tới sỏi thận ạ? Và nang thùy trái có ảnh hưởng nhiều không ạ?", "en": "Doctor, I would like to ask whether calcified kidney nodules can lead to kidney stones? And does the left lobe cyst have a significant impact?", "zh": "医生请问，肾钙化灶会不会导致肾结石？左肺囊肿会有很大影响吗？"}, {"id": 379, "vi": "Hôm nay em có làm một số xét nghiệm soi tươi dịch âm đạo, pap,... kết quả hầu như bình thường. Nhưng hôm nay là ngày gần cuối của chu kì kinh nguyệt thì kết quả có chính xác không? Hay phải khám khi sạch kinh 3-5 ngày thì kết quả mới chính xác?", "en": "Today I did some tests including wet mount of vaginal fluid, Pap test, etc., and the results are almost normal. But today is near the end of my menstrual cycle, so are the results accurate? Or should I visit the clinic after my period is clean for 3-5 days to get accurate results?", "zh": "今天我做了阴道分泌物涂片检查、Pap检查等，结果几乎都正常。但今天是月经周期的最后几天，结果准确吗？还是需要等到月经干净3-5天后再检查，结果才更准确？"}, {"id": 380, "vi": "Bé 4 tuổi chỉ nói được một số từ đơn giản, ít giao tiếp với người khác là bị làm sao?", "en": "What's wrong with a 4-year-old child who can only say a few simple words and has little communication with others?", "zh": "4岁孩子只能说出一些简单词汇，与他人交流少，是有什么问题吗？"}, {"id": 381, "vi": "Trước em có uống thuốc tráng thai cẩn cấp khoảng 3 tháng liền .Cách đấy 3 năm rồi bây giờ em muốn có con thì có được không .Có ảnh hưởng gì không", "en": "Previously, I took emergency pregnancy pills continuously for about 3 months. Three years ago, now I want to have a child, is that possible? Will there be any impact?", "zh": "我之前连续三个月服用过紧急保胎药。三年前的事了，现在我想怀孕，可以吗？会有影响吗？"}, {"id": 382, "vi": "Đau và sôi bụng, đi ngoài phân lỏng khi đang cho con bú liệu ảnh hưởng đến bé không?", "en": "Abdominal pain and bloating, having loose stools while breastfeeding, could this affect the baby?", "zh": "哺乳期间出现腹痛和腹胀，以及腹泻，是否会影响婴儿？"}, {"id": 383, "vi": "Bệnh viêm tụy cấp tái phát nhiều lần ở trẻ có thể điều trị được không?", "en": "Can children with recurrent acute pancreatitis be treated?", "zh": "反复发作的儿童急性胰腺炎能否治疗？"}, {"id": 384, "vi": "Có nên uống vitamin E hằng ngày không?", "en": "Should I take vitamin E daily?", "zh": "每天应该服用维生素E吗？"}, {"id": 385, "vi": "Em đang có bầu 6 tuần. Hôm qua em đi khám thì có kết luận là tụ dịch màng nuôi. Hôm nay em bị ra một ít máu nâu. Cho em hỏi là có ảnh hưởng gì đến thai không ạ?", "en": "I am 6 weeks pregnant. Yesterday when I went for a check-up, I was diagnosed with amniotic fluid accumulation. Today I have a little brown blood. I would like to know if this affects the fetus.", "zh": "我目前怀孕6周。昨天去医院检查，医生诊断为羊膜腔积液。今天我出现少量棕褐色出血。我想问一下，这会对胎儿有影响吗？"}, {"id": 386, "vi": "Đặt sonde dạ dày sau khi mổ ruột thừa liệu có đáng lo ngại?", "en": "Is it concerning to place a gastric tube after appendectomy?", "zh": "阑尾手术后放置胃管是否值得担忧？"}, {"id": 387, "vi": "Con em 13 tháng bú mẹ hoàn toàn mà táo bón từ lúc 2 tháng. Hiện tại phân thành khuôn và rất to. Tuần ỉa 1 lần . Vẫn tự ỉa dc đầu phân như phân dê thì giờ em nên làm gì. Em có bổ sung men mà vẫn vậy . Cũng ăn nhiều rau mà vẫn không ăn thua.", "en": "My 13-month-old child has had constipation since 2 months of age and is exclusively breastfed. Currently, the stool is formed and very large. There is one bowel movement per week. The child still passes stool normally, like cow's dung. What should I do now? I have supplemented probiotics but with no improvement. The child eats a lot of vegetables but still does not eat enough.", "zh": "13个月大的孩子从2个月开始就完全母乳喂养，一直有便秘问题。目前大便呈块状且非常大，每周排便1次，仍能自行排出类似牛粪的初始大便。现在应该怎么办？我已经补充了益生菌，但情况仍未改善，也增加了蔬菜摄入，但仍然不改善。"}, {"id": 388, "vi": "Xuất hiện sẹo lồi nhỏ sau bấm khuyên tai có sao không?", "en": "Is it normal to have a small raised scar after ear piercing?", "zh": "耳后按压后出现小凸起疤痕有问题吗？"}, {"id": 389, "vi": "Em gái của em nói là khi bịt mắt bên trái lại thì mắt phải thấy vật ở khoảng cách từ khoảng 70-80 cm là bị mờ, nhưng khi không bịt mắt trái lại thì nó vẫn thấy rõ bình thường. Cho em hỏi đây có phải là cận thị một mắt không?", "en": "My sister says that when she covers her left eye, her right eye becomes blurry when viewing an object at a distance of about 70-80 cm, but when she does not cover her left eye, she can see clearly normally. I would like to know if this could be unilateral myopia?", "zh": "我妹妹说，当遮住左眼时，右眼在70-80厘米距离处看物体就会变得模糊，而当不遮住左眼时，视力则正常。我想问一下，这是否属于单眼近视？"}, {"id": 390, "vi": "Trẻ bị viêm tụy cấp và viêm gan cấp khám có tìm ra nguyên nhân không?", "en": "Does a child with acute pancreatitis and acute hepatitis have the cause identified during examination?", "zh": "儿童患有急性胰腺炎和急性肝炎就诊时能否查出病因？"}, {"id": 391, "vi": "Tôi hiện đang điều trị bệnh trầm cảm, rối loạn thần kinh thực vật và tiền đình thì có tiêm vaccine Covid-19 được không?", "en": "I am currently being treated for depression, autonomic nervous system disorder and vestibular disorder. Can I receive the COVID-19 vaccine?", "zh": "我目前正在接受抑郁症、植物神经功能紊乱和前庭功能障碍的治疗，可以接种新冠疫苗吗？"}, {"id": 392, "vi": "Chữa zona ở môi như thế nào?", "en": "How to treat herpes zoster around the mouth?", "zh": "嘴唇带状疱疹如何治疗？"}, {"id": 393, "vi": "Bé 4 tháng ngày ho 7,8 tiếng ho có đờm có sao không.", "en": "4-month-old child, ho for 7.8 hours per day with cough and sputum, is this normal?", "zh": "4个月大的宝宝每天咳嗽7到8小时，有痰，需要担心吗？"}, {"id": 394, "vi": "Chỉ số xét nghiệm RPR âm tính cho biết điều gì?", "en": "What does a negative RPR test result indicate?", "zh": "RPR 检验指标阴性意味着什么？"}, {"id": 395, "vi": "Em ăn thịt bò thỉnh thoảng bị ngứa. Tình trạng của em vậy có được tiêm ngừa Covid-19 được không?", "en": "The child eats beef occasionally and gets itchy. Can the child receive the Covid-19 vaccine?", "zh": "我偶尔吃牛肉会痒。我这种情况可以接种新冠疫苗吗？"}, {"id": 396, "vi": "Tỷ lệ điều trị thành công ung thư thực quản di căn đến gan cao không?", "en": "Is the treatment success rate for advanced esophageal cancer metastasized to the liver high?", "zh": "晚期食管癌肝转移的治疗成功率高吗？"}, {"id": 397, "vi": "Bệnh viêm màng não nếu phát hiện sớm có khả năng điều trị khỏi hoàn toàn không?", "en": "Can meningitis be completely cured if detected early?", "zh": "如果早期发现脑膜炎，能否完全治愈？"}, {"id": 398, "vi": "Xét nghiệm đờm có ý nghĩa gì trong chẩn đoán lao?", "en": "What is the significance of sputum testing in the diagnosis of tuberculosis?", "zh": "痰检对结核病的诊断有什么意义？"}, {"id": 399, "vi": "Nam giới sưng, đau nhức một bên quai hàm nguyên nhân là gì?", "en": "What is the cause of swelling and pain on one side of the mandibular angle in men?", "zh": "男性一侧下颌角肿痛的原因是什么？"}, {"id": 400, "vi": "Em có hút thai cách đây được 4 tuần. Em có đi xét nghiệm máu cách đây 3 tuần thì nồng độ hcg của em vẫn còn. Và em mới có kinh nguyệt lại đc 1 ngày. Vậy em muốn đi xét nghiệm máu thì nồng độ beta HCG của em đã trở về bình được chưa", "en": "I had an abortion four weeks ago. I had a blood test three weeks ago and my hCG level was still present. And I just started having my period again for one day. So I want to go for a blood test to check if my beta HCG level has returned to normal.", "zh": "我四周前做了人工流产。三周前我去查了血，HCG水平仍然存在。我最近才恢复来月经，只有一天。我想去查血，我的beta-HCG水平是否已经恢复正常了？"}, {"id": 401, "vi": "Kết quả xét nghiệm IgE là 312 có nghiêm trọng không?", "en": "Is a IgE test result of 312 serious?", "zh": "IgE 检测结果是 312，严重吗？"}, {"id": 402, "vi": "Em là nam giới, 26 tuổi, em xuất hiện các triệu chứng đau lưng(chỗ cột sống ngang eo), và hay bị mỏi gối, đặc biệt là gối trái. Hai triệu chứng này thường xuyên xuất hiện, đặc biệt là những khi thay đổi thời tiết, khi mệt mỏi, ngồi lâu hay đi xe đường dài. Em đã đến bệnh viện xét nghiệm máu và siêu âm ổ bụng, thì siêu âm không có vấn đề gì, còn máu thì axit uric có chỉ số khá cao 450umol/L. Bác sĩ cho em hỏi đó có phải triệu chứng bệnh Gút không ạ?", "en": "I am a male, 26 years old. I have been experiencing back pain (at the lumbar spine level around the waist) and frequent knee fatigue, especially on the left knee. These two symptoms occur frequently, particularly when the weather changes, when I am tired, or when I sit for long periods or travel by long car rides. I went to the hospital for blood tests and abdominal ultrasound; the ultrasound showed no issues, but the blood test revealed a relatively high uric acid level of 450 umol/L. The doctor asked me whether these symptoms could be signs of gout.", "zh": "我是一名26岁的男性，出现腰背疼痛（腰椎水平部位），以及经常感到膝盖乏力，尤其是左膝。这两个症状经常出现，特别是在天气变化、疲劳、长时间坐着或长途乘车时。我曾去医院做了血液检查和腹部超声，腹部超声没有发现问题，但血液检查中尿酸水平较高，为450umol/L。医生请问，这是否是痛风的症状？"}, {"id": 403, "vi": "Em đã sinh 1 cháu 4 tuổi. 2 năm nay thả để có bầu mà chưa có. Em mới đi khám sơ sơ là bị mỏng niêm mạc 5mm. Em đã uống thuốc nhưng vẫn chưa có.", "en": "I have a 4-year-old child. For the past two years, I have been trying to conceive without success. I recently went for a preliminary check-up and was diagnosed with 5mm thinning of the mucosa. I have been taking medication, but I still haven't conceived.", "zh": "我有一个4岁的孩子。近两年来一直尝试怀孕但未成功。我最近去检查了一下，发现黏膜变薄5mm。我服用了药物，但仍然没有怀孕。"}, {"id": 404, "vi": "Cấy ghép implant nên chọn gây tê hay gây mê?", "en": "Should implant surgery be performed with local anesthesia or general anesthesia?", "zh": "种植手术应选择局部麻醉还是全身麻醉？"}, {"id": 405, "vi": "Trẻ 10 tháng tuổi cần tiêm phòng mũi nào?", "en": "What nasal vaccine is needed for a 10-month-old child?", "zh": "10个月大的儿童需要接种哪种疫苗？"}, {"id": 406, "vi": "Hư men răng có cách nào điều trị không?", "en": "Is there any treatment for tooth enamel damage?", "zh": "牙齿松动有治疗方法吗？"}, {"id": 407, "vi": "15 tuổi còn tiêm được vắc-xin viêm màng não mủ không?", "en": "Can a 15-year-old still receive meningococcal vaccine?", "zh": "15岁还能接种流脑疫苗吗？"}, {"id": 408, "vi": "Nguyên nhân ho kéo dài sau mở thông khí quản là do đâu?", "en": "What is the cause of prolonged cough after tracheostomy?", "zh": "长期气管切开后咳嗽的原因是什么？"}, {"id": 409, "vi": "Cho em hỏi vùng nách em có lông rậm và cứng mặc dù em là nữ. Hiện nó lan xuống phần vú , lan nhiều lắm ạ ? Vậy em có bị bệnh gì không ạ ? Tay chân em thì lông mọc bình thường ạ. Nếu mà phần nách của em bị viêm nang lông thì phải chữa trị như thế nào ạ ?", "en": "May I ask, I have thick and hard hair in my armpit area, although I am female. Currently, it is spreading down to my breast and spreading a lot. So, do I have any disease? My hands and feet have normal hair growth. If the armpit area is affected by folliculitis, how should it be treated?", "zh": "我想问一下，我作为女性，腋下有浓密且硬的毛发，而且已经蔓延到乳房区域，情况很严重吗？我是否得了什么病？我的手足毛发是正常的。如果我的腋下发生毛囊炎，应该如何治疗？"}, {"id": 410, "vi": "Chỉ số sinh hóa trong máu: glucose là 6,5 mmol/L, HbA1c là 6,8%, triglyceride 2,6 mmol/L so với bình thường thì ở mức nào và cần làm gì?", "en": "Blood biochemistry values: glucose is 6.5 mmol/L, HbA1c is 6.8%, triglyceride is 2.6 mmol/L. Compared to normal, what is the level and what should be done?", "zh": "血生化指标：血糖为6.5 mmol/L，HbA1c为6.8%，甘油三酯为2.6 mmol/L，与正常值相比处于什么水平，需要采取什么措施？"}, {"id": 411, "vi": "Chỉ số HBV-DNA PCR định lượng là 1.336.364.000. Như vậy có nguy hiểm không và hướng điều trị thế nào.", "en": "The quantified HBV-DNA PCR level is 1,336,364,000. Is this dangerous and what is the treatment approach?", "zh": "定量HBV-DNA PCR检测值为1,336,364,000。这样是否危险，该如何治疗？"}, {"id": 412, "vi": "Tôi bị đau nửa sườn trái âm ỉ cả ngày suốt 10 năm nay. Căn bệnh này không chữa khỏi được hả?", "en": "I have had a dull, persistent pain on the left side of my chest for 10 years. Is this condition curable?", "zh": "我已有10年每天持续隐隐作痛的左侧胸骨旁疼痛，这种病能治好吗？"}, {"id": 413, "vi": "Gần tháng nay trong khoang miệng của em có đốm đỏ màu nhạt ở gần chỗ hàm răng hôm qua em lấy tăm em chọc vào chỗ đốm đỏ ý ,không đau ,không chảy máu mà đốm đỏ nó lại đậm hơn mong các bác sĩ tư vấn.", "en": "Recently, in my mouth, there has been a pale red spot near the tooth. Yesterday, I used a cotton swab to prick the red spot, it was not painful, no bleeding, but the red spot became darker. I hope doctors can advise.", "zh": "近一个月来，我口腔内靠近牙齿处出现了一个淡红色的斑点。昨天我用棉签戳了这个红色斑点，没有疼痛，也没有出血，但红色斑点反而变深了。恳请医生给予指导。"}, {"id": 414, "vi": "Yếu thần kinh thị giác mắt phải, thị lực 1/10 có cách nào điều trị không?", "en": "Visual neurological weakness of the right eye, visual acuity 1/10, is there any treatment?", "zh": "右眼神经性视力障碍，视力为1/10，有什么治疗方法？"}, {"id": 415, "vi": "Em có khám sức khoẻ theo công ty và kết quả cyfra21-1 là 3.98. Hiện tại em khoẻ mạnh bình thường và cũng không có triệu trứng gì về hô hấp. Em có bị viêm mũi dị ứng trong những lần khám trc. Bác sỹ tư vấn giúp em.", "en": "I have had a health check-up through the company and the CYFRA21-1 result is 3.98. Currently, I am in good health and do not have any respiratory symptoms. I have had allergic rhinitis in previous check-ups. Doctor, please advise me.", "zh": "我参加了公司健康检查，CYFRA21-1 检测结果为 3.98。目前我身体状况良好，没有呼吸系统的任何症状。之前几次体检时曾有鼻炎过敏史。请医生给予建议。"}, {"id": 416, "vi": "Tại sao người già truyền dịch hay bị vỡ mạch máu?", "en": "Why do elderly patients undergoing fluid infusion often suffer from blood vessel rupture?", "zh": "为什么老年人输液容易发生血管破裂？"}, {"id": 417, "vi": "Cháu hiện tại được 5.5 tháng, nặng 10kg. Thứ 6 tuần trước cháu có đi ngoài khoảng 10 lần, những ngày sau đó đến hôm nay cháu đi mỗi ngày khoảng 4-5 lần. Phân đi hơi loãng, có chất nhầy, có lợn cợn hạt (bình thường cháu đi phân sệt sệt), màu vàng. Hôm nay cháu có lấy mẫu phân và xét nghiệm thì có kết luận là LOẠN KHUẨN NHẸ. Vậy nguyên nhân và cách điều trị là gì?", "en": "I am currently 5.5 months old, weighing 10kg. About a week ago, I had about 10 bowel movements, and since then, up to today, I have had about 4-5 bowel movements per day. The stool is slightly loose, contains mucus, has small particle-like substances (normally my stool is firm), and is yellow. Today I collected a stool sample and underwent testing, which showed a mild Clostridium difficile infection. What is the cause and how should I treat it?", "zh": "目前宝宝5.5个月，体重10kg。上周六开始，宝宝大便约10次，之后到今天每天大便约4-5次。大便偏稀，有黏液，有细小颗粒（平时大便较成形），颜色为黄色。今天宝宝取了大便样本进行检测，结果显示为轻度艰难梭菌感染。请问病因及治疗方案是什么？"}, {"id": 418, "vi": "Liệu đã bị HPV có cách chữa hết bệnh dù đã ăn sâu vào máu không?", "en": "Can HPV be completely cured even though it has deeply entered the blood?", "zh": "是否已经感染了HPV，即使病毒已经深入血液，是否还能彻底治愈？"}, {"id": 419, "vi": "Nguyên nhân mọc nhiều lông sau lưng?", "en": "What is the cause of excessive hair growth on the back?", "zh": "背部毛发增多的原因？"}, {"id": 420, "vi": "Gần đây em thấy móng tay mình ngả vàng phần giữa móng (chỉ 3 móng). Đó có phải là bệnh không ạ?", "en": "Recently I have noticed my nails turning yellow in the middle part (only 3 nails). Is this a disease?", "zh": "最近我注意到自己的指甲中部（只有3个指甲）变黄了，这是否是某种疾病呢？"}, {"id": 421, "vi": "Cho em hỏi tử cung bị nghiêng về phía trước có thể thụ thai không?", "en": "May I ask whether a uterus that is tilted forward can conceive?", "zh": "请问子宫前倾是否可以怀孕？"}, {"id": 422, "vi": "Em sinh mổ bé được 17 tháng tuổi và bây giờ em đang mang thai 34 tuần rồi. Khoảng bao lâu thì mình nên đến bệnh viện và sinh mổ có dấu hiệu sanh hay đau bụng hông? Và khi nào mình cần đến bệnh viện ạ?", "en": "My child was delivered by cesarean section at 17 months of age, and now I am 34 weeks pregnant. How long should I go to the hospital and have a cesarean section when signs of labor or lower abdominal or hip pain occur? And when should I go to the hospital?", "zh": "孩子剖腹产出生后17个月，现在怀孕34周。什么时候应该去医院，出现分娩征兆或腹痛、髋部疼痛时怎么办？什么情况下需要去医院？"}, {"id": 423, "vi": "Đi ngoài ngay sau khi ăn cảnh báo bệnh lý gì?", "en": "Fecal incontinence after eating may indicate which disease?", "zh": "进食后立即排便提示何种疾病？"}, {"id": 424, "vi": "Em mới 15 tuổi đã quan hệ tình dục thì có sao không?", "en": "Is it problematic for a 15-year-old girl to have sexual intercourse?", "zh": "我15岁开始性生活会有什么问题吗？"}, {"id": 425, "vi": "Người mắc xuất huyết não nên điều trị thế nào cho phù hợp?", "en": "How should a patient with intracerebral hemorrhage be treated appropriately?", "zh": "脑出血患者应如何进行合适的治疗？"}, {"id": 426, "vi": "Nang thận ảnh hưởng như thế nào trong cơ thể, và những biến chứng có thể?", "en": "How does renal cyst affect the body, and what complications may occur?", "zh": "肾囊肿对身体有什么影响，以及可能的并发症？"}, {"id": 427, "vi": "Cách đây 2 ngày tôi thấy rất mệt, mặt có cảm giác díu vào, má trái gần mắt bì bì khó chịu. Bác sĩ cho hỏi tôi bị làm sao và nên đi khám như thế nào.", "en": "Two days ago I felt very tired, my face had a sensation of tightness, and the left cheek near the eye felt uncomfortable and tingling. Doctor, could you please advise me on what might be wrong and how I should go for a check-up.", "zh": "两天前我感到非常疲倦，面部有紧绷感，左脸靠近眼睛处有隐隐不适。医生请问一下我是什么情况，应该去怎么检查？"}, {"id": 428, "vi": "Em vừa mới bị trào ngược lần đầu tiên vào tuần trước. Dùng thuốc 3 ngày kết hợp với chế độ sinh hoạt thì gần một tuần sau các triệu chứng bắt đầu giảm, chỉ còn triệu chứng bị nuốt vướng và cảm thấy họng hơi khô thôi ạ. Uống nước gừng ấm thì bớt vài giờ sau lại bắt đầu bị lại. Cho em hỏi có bị sao không ạ? Làm gì để chấm dứt được việc nuốt vướng ạ?", "en": "I just experienced my first episode of acid reflux last week. I took medication for 3 days combined with lifestyle changes, and after nearly one week, the symptoms started to improve, leaving only the sensation of throat tightness and a slightly dry throat. Drinking warm ginger water helps for a few hours, but the symptoms return afterward. I would like to know if there is any problem. What can I do to completely eliminate the sensation of throat tightness?", "zh": "我上周刚刚第一次出现反流。用了三天药，结合生活习惯，大约一周后症状开始减轻，只剩下吞咽不适和喉咙稍干的感觉。喝温姜水后几小时又会复发。请问我会有什么问题吗？如何才能彻底解决吞咽不适的问题？"}, {"id": 429, "vi": "Mắt em bị lác ngoài có chữa được không ạ phẫu thuật xong sau mấy ngày có thể làm việc được ạ sau mấy ngày là khỏi hẳn ạ", "en": "Is external strabismus in my child treatable? After surgery, how many days can I return to work? After how many days will I be completely recovered?", "zh": "我孩子有外斜视，能治好吗？手术后几天可以工作？几天能完全康复？"}, {"id": 430, "vi": "Quan hệ tình dục có thai, có bị nhiễm viêm gan B và chữa khỏi được không?", "en": "Is there a risk of contracting hepatitis B during sexual intercourse, and can it be cured?", "zh": "性行为导致怀孕，是否会感染乙型肝炎，能否治愈？"}, {"id": 431, "vi": "Bác sĩ cho hỏi, huyết áp 150/110 có bình thường không, tôi thường xuyên bị huyết áp như vậy.", "en": "Doctor, I would like to ask, is a blood pressure of 150/110 normal? I often have blood pressure like this.", "zh": "医生请问一下，血压150/110正常吗？我经常会出现这样的血压。"}, {"id": 432, "vi": "Mẹo tâm lý, chăm sóc hỗ trợ trẻ không còn tiểu dầm khi đã lớn là gì?", "en": "What are the psychological tips and supportive care measures for children who no longer wet the bed when they grow older?", "zh": "如何为长大后不再尿床的儿童提供心理支持和护理建议？"}, {"id": 433, "vi": "Phân gầy như bút chì là bệnh gì ạ?", "en": "What disease causes pencil-thin stools?", "zh": "像铅笔芯一样的大便是什么病？"}, {"id": 434, "vi": "Tuyến tiền liệt tăng lớn, hồi âm không đồng đều là dấu hiệu của bệnh gì?", "en": "Enlarged prostate gland with heterogeneous echo is a sign of what disease?", "zh": "前列腺增大，回声不均匀是哪种疾病的征兆？"}, {"id": 435, "vi": "Viêm tai giữa mạn tính thủng màng nhĩ có cần thiết phải vá màng nhĩ không?", "en": "Does chronic middle ear inflammation with eardrum perforation require eardrum repair?", "zh": "慢性中耳炎伴鼓膜穿孔是否需要鼓膜成形术？"}, {"id": 436, "vi": "Cây đâm vào mắt gây xước lòng đen có sao không?", "en": "Does a plant penetrating the eye causing corneal abrasion lead to complications?", "zh": "眼睛被植物刺伤导致视网膜裂伤会有问题吗？"}, {"id": 437, "vi": "Hiện tại, tôi đã tiêm vaccine AstraZeneca mũi một được một tuần. Sau bao lâu nữa, tôi có thể triệt lông bằng ánh sáng laser và uống collagen được chiết xuất từ cá và có thành phần chủ yếu là vitamin C, các loại hoa quả?", "en": "Currently, I have received the first dose of the AstraZeneca vaccine one week ago. After how many more weeks can I undergo laser hair removal and take collagen extracted from fish and primarily containing vitamin C, various fruits?", "zh": "目前，我已接种了阿斯利康疫苗第一剂，一周后，我多久后可以进行激光脱毛，并服用从鱼中提取、主要成分为维生素C以及各种水果的胶原蛋白？"}, {"id": 438, "vi": "Em năm nay 39 tuổi, cách đây một tháng em bị khó thở em đc bác sĩ chẩn đoán là bị rối loạn tiền đình. Sau khi uống thuốc xong em bị nhức đầu tê bì nữa người bên phải, em được bác sĩ cho đi chụp CT, chụp điện tim, xét nghiệm máu và chụp phim thì bác sĩ kết luận là bị thần kinh, em đc uống thuốc thần kinh vẫn không đỡ. Vậy bác sĩ tư vấn em với ạ", "en": "I am 39 years old. One month ago, I started experiencing shortness of breath. I was diagnosed by a doctor with vestibular disorder. After taking the medication, I developed headache and numbness on the right side of my body. I was advised to undergo CT scan, electrocardiogram, blood tests, and X-ray, and the doctor concluded that I have a neurological condition. I have been taking neurological medication, but my symptoms have not improved. Therefore, I would like the doctor's advice.", "zh": "我今年39岁，一个月前开始出现呼吸困难，医生诊断为前庭功能障碍。服药后，我出现了右侧身体麻木、头痛的症状，医生建议我做CT、心电图、血常规和X光检查，最终诊断为神经系统疾病，我服用神经类药物后症状仍无改善。请问医生该如何建议？"}, {"id": 439, "vi": "Điều trị lao phổi bao lâu thì không lây nhiễm cho người khác?", "en": "How long after treatment for pulmonary tuberculosis does a person no longer become infectious to others?", "zh": "结核病治疗多久后不再传染给他人？"}, {"id": 440, "vi": "Em thường xuyên bị nhiệt miệng kéo dài, cứ hết lần này lại đến lần khác lại bị, có phải do chức năng gan không ? Liệu xét nghiệm gan và máu thì có thể kiểm tra được không ?", "en": "My child frequently suffers from prolonged oral heat, and every time it resolves, it recurs. Is this due to liver function? Can liver and blood tests detect this?", "zh": "我经常反复出现口腔溃疡，一次好了又复发，这是肝功能问题吗？肝功能和血常规检查可以检查出来吗？"}, {"id": 441, "vi": "Em bị thai ngừng phát triển khi mới được 5,6 tuần. Em có xét nghiệm chỉ số NK của em là 10744, trong khi đó chỉ số bình thường em biết là 200-5000. Vậy trường hợp của em có thể có bầu sinh con bình thường được không ạ? Em phải làm gì để giảm chỉ số NK để quá trình mang thai được tốt không ạ? Phương pháp IVF có thể giúp em có một thai kỳ khỏe mạnh không ạ?", "en": "My pregnancy stopped developing at just 5.6 weeks. I have my NK cell count test result of 10744, while I know the normal range is 200-5000. So, can my case still result in a normal pregnancy? What should I do to reduce my NK cell count in order to have a better pregnancy process? Can IVF help me have a healthy pregnancy?", "zh": "我怀孕5.6周时胚胎停止发育。我的NK细胞计数是10744，而我知道的正常范围是200-5000。请问我的情况是否有可能正常怀孕？我应该怎么做才能降低NK细胞计数，以改善怀孕过程呢？试管婴儿（IVF）能否帮助我拥有一个健康的妊娠呢？"}, {"id": 442, "vi": "Chụp X-quang răng khoảng 10 lần trong ngày có ảnh hưởng nghiêm trọng đến sức khỏe không?", "en": "Does taking 10 dental X-rays per day have a serious impact on health?", "zh": "每天拍10次牙片会对健康造成严重危害吗？"}, {"id": 443, "vi": "Người quen của tôi vừa tiêm vắc xin AstraZeneca được khoảng 3 tuần thì phát hiện mang thai (trễ chu kỳ khoảng một tuần). Liệu có khả năng gây ảnh hưởng cho thai nhi và trẻ sau này không?", "en": "My friend just received the AstraZeneca vaccine about 3 weeks ago and then discovered she was pregnant (with a delay of about one week). Could this have any impact on the fetus and the child later on?", "zh": "我朋友刚接种阿斯利康疫苗约三周，就发现自己怀孕了（月经推迟约一周）。请问这会对胎儿及以后的儿童产生影响吗？"}, {"id": 444, "vi": "17 tuổi mọc nhiều lông ở khắp cơ thể là do đâu?", "en": "What is the cause of excessive hair growth all over the body in a 17-year-old?", "zh": "17岁全身毛发增多的原因是什么？"}, {"id": 445, "vi": "Thoái hóa nhu mô não có nguy hiểm không và điều trị thế nào?", "en": "Is cerebral parenchymal degeneration dangerous and how is it treated?", "zh": "脑组织退化危险吗？如何治疗？"}, {"id": 446, "vi": "Em mới nhận được kết quả xét nghiệm máu của bên mình. Có chỉ số bilirubin trực tiếp là hơi cao. Em chưa hiểu và cần được tư vấn rõ hơn.", "en": "I have just received my blood test results. My direct bilirubin level is slightly elevated. I do not understand and need clearer advice.", "zh": "我刚收到自己的血液检查结果。直接胆红素的数值略高。我不太理解，需要更详细的咨询。"}, {"id": 447, "vi": "Cô tôi bị u tuyến giáp ác tính và ở giai đoạn 2 rồi. Phẫu thuật có khỏi được không? Hay chỉ kéo dài được thời gian thôi?", "en": "My sister has developed a malignant thyroid tumor and is at stage 2. Can surgery cure it? Or will it only prolong the time?", "zh": "我姐姐患有恶性甲状腺肿瘤，已进入二期。手术能治愈吗？还是只是延长生存时间？"}, {"id": 448, "vi": "Một tháng nay em luôn có hiện tượng đau mỏi lưng, vai gáy, người có cảm giác ớn lạnh và cảm giác thiếu máu lên não. Như thế là bị làm sao ạ?", "en": "For the past month, I have always had the symptom of back and shoulder pain, feeling cold all over the body, and a sensation of blood deficiency in the brain. What is wrong with me?", "zh": "最近一个月，我总是有腰背酸痛、肩颈疼痛，身体有发冷感，还感觉大脑缺血。这是怎么回事啊？"}, {"id": 449, "vi": "Khó di chuyển, khó cầm nắm, không tập trung có nguy cơ mắc bệnh gì?", "en": "Difficulty moving, difficulty grasping, inability to concentrate – what diseases are at risk?", "zh": "移动困难、抓握困难、注意力不集中可能患有哪些疾病？"}, {"id": 450, "vi": "Đau đầu sau va đập mũi có sao không?", "en": "Is there any problem after head pain following a nose bump?", "zh": "鼻子撞伤后会头痛吗？"}, {"id": 451, "vi": "Nhức đầu kèm lạnh chân tay có phải sốt xuất huyết không?", "en": "Does headache accompanied by cold hands and feet indicate hemorrhagic fever?", "zh": "伴有手脚冰冷的头痛是出血热吗？"}, {"id": 452, "vi": "Mụn thịt hình thành thế nào, điều trị ra sao?", "en": "How is flesh pustule formed and treated?", "zh": "肉粒是如何形成的，该如何治疗？"}, {"id": 453, "vi": "Nữ giới sau khi cắt tuyến giáp sử dụng collagen dạng nước có ảnh hưởng gì không?", "en": "Does women using water-based collagen after thyroidectomy have any effect?", "zh": "女性甲状腺切除术后使用水性胶原蛋白有什么影响？"}, {"id": 454, "vi": "Bác sĩ cho em hỏi em cao 1m74 nặng 73kg thì đã phù hợp hay thừa cân ạ?", "en": "Doctor, I would like to ask you: I am 174 cm tall and weigh 73 kg; is this considered normal or overweight?", "zh": "医生请问一下，我身高1米74，体重73公斤，属于正常体重还是超重呢？"}, {"id": 455, "vi": "Xin cho em biết con em 5 tuổi bị chó con 45 ngày tuổi cào hay cắn không xác định được, vết xước nhẹ không chảy máu vậy cần tiêm phòng không ạ?", "en": "Please tell me if my 5-year-old child was scratched or bitten by a 45-day-old puppy, the exact source is unknown, with mild scratches without bleeding, whether vaccination is needed?", "zh": "请问一下，我5岁的孩子被一只45天大的小狗抓伤或咬伤，但无法确定具体是抓还是咬，伤口较轻且没有出血，这种情况需要打疫苗吗？"}, {"id": 456, "vi": "Kết quả xét nghiệm của tôi là virut dưới ngưỡng phát hiện, men gan không cao lắm , vậy tôi có phải uống tiếp thuốc kháng virut không ?", "en": "My test result shows virus below detectable threshold, liver enzymes are not high, so should I continue taking antiviral medication?", "zh": "我的检查结果是病毒低于检测下限，肝酶水平并不高，那么我还需要继续服用抗病毒药物吗？"}, {"id": 457, "vi": "Cháu bị gù lưng (lưng tôm) giờ cháu phải điều trị bằng phương pháp nào để hết gù trong thời gian ngắn?", "en": "My back curves (lumbar scoliosis); what treatment method should I use to eliminate the curvature in a short period of time?", "zh": "我患有脊柱侧弯（虾腰），现在该如何治疗才能在短时间内治愈？"}, {"id": 458, "vi": "Làm những xét nghiệm gì để biết nguyên nhân gây thiếu máu?", "en": "What tests are done to find the cause of anemia?", "zh": "做哪些检查以确定贫血的原因？"}, {"id": 459, "vi": "Trẻ bị viêm ruột do virus cần chăm sóc như thế nào?", "en": "How should a child with viral gastroenteritis be cared for?", "zh": "病毒性肠炎患儿需要如何护理？"}, {"id": 460, "vi": "Trẻ 15 tháng đi ngoài mấy lần 1 ngày là bình thường?", "en": "How many times per day does a 15-month-old child have diarrhea considered normal?", "zh": "15个月大的孩子每天拉几次大便属于正常？"}, {"id": 461, "vi": "Bị rát mắt lúc đi nắng là do đâu? Khắc phục thế nào?", "en": "What is the cause of eye burning when exposed to sunlight? How to remedy it?", "zh": "晒太阳时眼睛发烫是什么原因？如何缓解？"}, {"id": 462, "vi": "Đau đột ngột vùng thận phải kèm buồn nôn, vã mồ hôi là bệnh gì?", "en": "What is a sudden right renal pain accompanied by nausea and sweating?", "zh": "突发右肾区疼痛伴恶心、出汗是什么疾病？"}, {"id": 463, "vi": "Cháu cắt trọn tuyến giáp cách đây 4 năm, đang uống thuốc bổ sung hormone giáp hằng ngày 3/4 viên. Cháu có tiêm vaccine Covid-19 được không? Bệnh lý của cháu có được coi là bệnh lý nền không? Nếu chẳng may bị Covid-19 có nguy hiểm tính mạng như những bệnh nhân bị tiểu đường không? Cháu năm nay 40 tuổi.", "en": "I had my entire thyroid gland removed four years ago and am currently taking daily thyroid hormone replacement therapy of 3/4 tablet. Can I receive the COVID-19 vaccine? Is my condition considered a comorbid condition? If I accidentally contract COVID-19, will it be as life-threatening as in patients with diabetes? I am currently 40 years old.", "zh": "我四年前做了甲状腺全切手术，目前每天服用半粒至四分之三粒甲状腺激素替代药物。我可以接种新冠疫苗吗？我的疾病是否属于基础疾病？如果不幸感染新冠，其致死风险是否与糖尿病患者相当？我今年40岁。"}, {"id": 464, "vi": "Sau sinh 3 tháng đi kiểm tra em phát hiện bị cường giáp nay đã điều trị được 4 tháng, chỉ số TSH 0.37. Sáng nay em xét nghiệm thêm chỉ số TRAP bên mình kết quả <0.3 . kết quả TRAP cho biết thêm gì không thưa bác sĩ.", "en": "Three months postpartum, during my check-up, I was found to have hyperthyroidism. I have been treated for four months now, with TSH level at 0.37. This morning, I additionally tested for TRAP level, and the result is <0.3. Doctor, what does the TRAP result indicate further?", "zh": "产后3个月检查发现患有甲亢，目前治疗已4个月，TSH水平为0.37。今早我自行额外检测了TRAP指标，结果小于0.3。请问医生，TRAP指标能提供更多信息吗？"}, {"id": 465, "vi": "Em bị tai nạn ở môi cách đây 1 năm. Nay vết thương đã lành và ở vết thương đỏ, gân máu chằng chịt là sẹo loại gì ạ? Em bôi kem trị sẹo càng bôi có dấu hiệu to ra ạ. Bệnh viện nào uy tín để trị sẹo được ạ?", "en": "My injury occurred one year ago. Now the wound has healed and at the wound site, there is redness, tangled ligaments and blood vessels—what type of scar is this? I apply scar treatment cream, and the area seems to enlarge when I apply it. Which reputable hospital can treat scars?", "zh": "我一年前遭遇了外伤。现在伤口已经愈合，伤口处发红，肌腱和血管交织成疤痕，这是什么类型的疤痕？我一直在使用祛疤膏，但越用疤痕反而变得越大。请问哪家医院治疗疤痕比较权威？"}, {"id": 466, "vi": "Trĩ ngoại mức độ nhẹ điều trị dứt điểm thế nào?", "en": "How to treat external hemorrhoids of mild degree curatively?", "zh": "外痔轻度如何彻底治疗？"}, {"id": 467, "vi": "Có trường hợp nào bé bị tay chân miệng mà chỉ mọc nốt ở miệng còn chân tay không có dấu hiệu gì?", "en": "Is there a case of a child with hand, foot, and mouth disease that only has spots in the mouth, with no signs on the hands and feet?", "zh": "有没有孩子患手足口病，但只有口腔出疹，手脚没有症状？"}, {"id": 468, "vi": "Tôi tự đo huyết áp bằng máy điện tử omron chỉ số tối đa thường từ 110-125 và huyết áp tối thiểu thường từ 60-68. Vậy huyết áp tối thiểu như vậy có thấp quá không, điều đó nói lên vấn đề gì?", "en": "I measure my blood pressure myself using an Omron electronic machine, with a maximum reading usually ranging from 110-125 and a minimum reading usually ranging from 60-68. Is this minimum blood pressure too low? What does this indicate?", "zh": "我用欧姆龙电子血压计自测血压，最高值通常在110-125之间，最低值通常在60-68之间。这样的最低血压是否过低？这说明了什么问题？"}, {"id": 469, "vi": "Đang uống thuốc điều trị cường giáp sử dụng thuốc tránh thai hàng ngày Rigevidon thì có ảnh hưởng gì không?", "en": "Is there any effect when taking daily oral contraceptive Rigevidon while on treatment for hyperthyroidism?", "zh": "正在服用治疗甲亢的药物，同时每日服用Rigevidon避孕药，会有影响吗？"}, {"id": 470, "vi": "Em đã lập gia đình được hơn 1 năm nhưng vợ chưa mang bầu (vợ đã đi khám và có kết quả trứng to nhỏ không đều). Xin tư vấn của bác sĩ ạ.", "en": "We have been married for more than one year but my wife has not become pregnant (my wife has undergone examination and the results show irregular ovarian follicle size). I would like to consult with the doctor.", "zh": "我结婚一年多了，妻子一直没有怀孕（妻子已经检查过，结果显示卵泡大小不均）。恳请医生给予指导。"}, {"id": 471, "vi": "Cách đây hơn năm em đã không còn đi khám lại viêm gan B do công việc quá bận rộn. Ngày mới phát hiện em có virus trong người, em đi khám được bác sĩ thông báo virus viêm gan B không hoạt động. Em sẽ đi khám lại trong thời gian sớm nhất khi công việc không quá nhiều. Em muốn hỏi bác sĩ khi viêm gan B hoạt động thì triệu chứng trong người như thế nào, có phải suy thận dẫn đến nửa đêm đi vệ sinh nhiều lần không, riêng em một đêm cứ 2-3 lần đi vệ sinh, ăn uống kém sút cân và thiếu máu. Với lại trong ăn uống đối với người bị viêm gan B nên cần bổ sung gì nhiều nhất?", "en": "Five years ago, I stopped visiting the doctor for hepatitis B due to being too busy with work. Recently, when I discovered the virus in my body, the doctor informed me that the hepatitis B virus was inactive. I will visit the doctor again as soon as possible when my work is not too heavy. I would like to ask the doctor what symptoms occur when hepatitis B becomes active, whether kidney failure leads to frequent nighttime urination, as I have to go to the bathroom 2-3 times every night, have poor appetite, weight loss, and anemia. Also, in terms of diet, what should people with hepatitis B supplement the most?", "zh": "五年前由于工作太忙，我已不再复查乙型肝炎。最近才发现体内有病毒，医生告知我乙型肝炎病毒目前不活跃。我将在工作不繁忙时尽快再次就诊。我想咨询一下医生，当乙型肝炎病毒活跃时，身体会出现哪些症状？是否会出现肾功能衰竭导致夜间频繁排尿？我每晚排尿2-3次，食欲差、体重下降且有贫血。另外，对于乙型肝炎患者，在饮食上最需要补充哪些营养成分？"}, {"id": 472, "vi": "Mấy năm gần đây con bị rụng tóc nhiều mà tóc mọc chậm lắm ạ, da dầu thường hay bị bết, đỉnh đầu thưa tóc. Có cách nào giảm rụng tóc và giúp tóc mọc nhanh không ạ?", "en": "In the past few years, my child has been experiencing excessive hair loss with very slow hair regrowth. The scalp is oily and often feels greasy, with thinning hair at the top of the head. Is there any way to reduce hair loss and promote faster hair growth?", "zh": "近几年孩子脱发很多，头发生长很慢，头皮油脂分泌多，经常发油，头顶稀疏。请问有什么方法可以减少脱发并促进头发快速生长吗？"}, {"id": 473, "vi": "Rát môi, nổi hạt ở viền lưỡi khi niềng răng thép có sao không?", "en": "Is it normal to have a burning sensation and bumps at the edge of the tongue when wearing steel braces?", "zh": "戴钢牙套时，嘴唇发烫、舌头边缘出现小疙瘩，有没有关系？"}, {"id": 474, "vi": "Vì sao viêm màng não lại là loại bệnh nguy hiểm nhất?", "en": "Why is bacterial meningitis the most dangerous disease?", "zh": "为什么化脓性脑膜炎是最危险的疾病？"}, {"id": 475, "vi": "Em xét nghiệm máu kết quả bạch cầu ái toan tăng, bác sĩ tư vấn uống thuốc tẩy giun vậy em nên uống loại thuốc nào thì tốt? Tư vấn giúp em.", "en": "My blood test result shows increased eosinophils. The doctor advised taking antihelminthic medication. Which type of medication should I take? Please advise me.", "zh": "我血检结果显示嗜酸性粒细胞升高，医生建议我服用驱虫药，我该服用哪种药物比较好？请帮我看一下。"}, {"id": 476, "vi": "Mệt mỏi về chiều, khi nói khàn tiếng hụt hơi, ăn uống khó nuốt nguyên nhân.", "en": "Fatigue in the afternoon, hoarseness with shortness of breath when speaking, difficulty swallowing; cause.", "zh": "下午疲倦，说话嘶哑，吞咽困难，原因是什么？"}, {"id": 477, "vi": "Cho em hỏi dạo gần đây hai bên má hàm trên có nổi hạt như u hột . Em lấy lưỡi rờ thì hơi nhức tí không đâu lắm, hai bên má nổi như nhau cùng vị trí trên hàm. Nguyên nhân do đâu mà bị.", "en": "I would like to ask, recently I have noticed small bumps or nodules on both upper jaw areas. When I rub my tongue against them, there is a slight pain, not severe. Both sides of the face are equally affected and appear in the same position on the upper jaw. What could be the cause of this?", "zh": "最近我上下颌两侧都长了像小疙瘩一样的颗粒，用舌头轻轻触碰时有点轻微刺痛，但不严重，两侧脸颊在同一位置都出现类似情况。这是什么原因导致的？"}, {"id": 478, "vi": "30 tuổi hút thuốc nhiều có nguy cơ ung thư phổi cao không?", "en": "Does a 30-year-old who smokes have a high risk of lung cancer?", "zh": "30岁吸烟者肺部癌症的风险高吗？"}, {"id": 479, "vi": "Bé nhà con nay được 3 tháng 14 ngày trước bé uống sữa riso của nuti thì đi ị đều ngày 1 lần nhưng uống đc khoảng 2 tháng thì bé tự khi bị bón phân cứng con có đổi qua dielac alpha thì 3 ngày không ị con phải bơm thì bé mới ị mà khóc to lắm rặn mãi không ra. Con phải làm sao ? Có cần đổi sữa nữa không?", "en": "My child is now 3 months and 14 days old. Previously, when my child was drinking NUTI's riso milk, they had one bowel movement per day. However, after drinking it for about two months, when my child started experiencing hard constipation, I switched to Dielac Alpha. Since then, my child has not had a bowel movement for three days and needs to be pumped to pass stool, which causes them to cry loudly and strain for a long time without success. What should I do? Do I need to switch to another milk formula?", "zh": "孩子现在3个月14天，之前喝Nuti的Riso奶粉，每天排便一次，能喝大约两个月后，孩子自己出现便秘，便便变硬，便改为Dielac Alpha，3天才排便一次，需要灌肠才能排便，排便时哭闹厉害，使劲很久才排出。该怎么办？还需要换奶粉吗？"}, {"id": 480, "vi": "Cho em biết cách tính tuổi thai IVF như thế nào ạ? Em ngày kinh cuối là 17/8/2019 và ngày chuyển phôi là 3/9/2019 (phôi em ngày 3). Vậy cho tới hôm nay thì thai của em được mấy tuần mấy ngày rồi ạ?", "en": "Could you please tell me how to calculate IVF pregnancy age? My last menstrual period was on 17/8/2019 and the embryo transfer was on 3/9/2019 (day 3 embryo). So, until today, how many weeks and days old is my pregnancy?", "zh": "请问一下试管婴儿的孕期是如何计算的吗？我的末次月经是2019年8月17日，移植日期是2019年9月3日（胚胎为3天期）。到今天为止，我的胚胎已经怀孕多少周多少天了？"}, {"id": 481, "vi": "Làm thế nào để cải thiện tình trạng da và mắt vàng do thiếu men G6PD?", "en": "How to improve the skin and eye yellowing due to G6PD deficiency?", "zh": "如何改善由G6PD缺乏引起的黄疸皮肤和眼睛发黄状况？"}, {"id": 482, "vi": "Gần đây cháu để ý hay có dịch màu vàng nâu tiết ra băng vệ sinh hàng ngày dùng, và khí hư có màu đen ạ. Cháu chỉ bị thay đổi màu khí hư, ngoài ra không có bị đau ngứa rát mùi hay bất cứ biểu hiện gì bất thường, bác sĩ có thể chẩn đoán cháu bị sao không ạ?", "en": "Recently, I have noticed yellow-brown discharge coming out of my sanitary pad every day, and my vaginal discharge is black. I only have a change in vaginal discharge color; I do not have pain, itching, odor, or any other unusual symptoms. Doctor, can you diagnose what condition I might have?", "zh": "最近我注意到每天使用卫生巾时都有黄色褐色的分泌物，阴道分泌物呈黑色。除了白带颜色改变外，我没有出现疼痛、瘙痒、异味或其他任何异常症状，医生能否为我诊断一下是什么问题？"}, {"id": 483, "vi": "Hôm trước em có khám thì kết quả Prolactin là trên 800, hôm nay khám lại là 436. Vậy chỉ số Prolactin của em đã bình thường chưa ạ?", "en": "Last day I had a check-up and the prolactin result was above 800, today I had another check-up and the value is 436. So, is my prolactin level now normal?", "zh": "昨天检查泌乳素是800以上，今天复查是436。请问我的泌乳素水平已经正常了吗？"}, {"id": 484, "vi": "Tôi sinh mổ được 24 ngày nhưng đến nay lượng sữa vẫn chưa nhiều. vậy bác sĩ cho tôi gợi ý về cách kích sữa, cách cho con bú bao lâu 1 lần là hợp lí. chế độ ăn như thế nào giúp sữa về nhanh và nhiều hơn", "en": "I had a cesarean section 24 days ago, but until now my milk supply is still low. Therefore, I would like the doctor's advice on how to stimulate milk production, how often I should breastfeed, and what diet can help increase milk supply quickly and effectively.", "zh": "我剖腹产24天了，到现在奶量仍然很少。医生能否给我一些关于如何刺激乳汁分泌、每次哺乳时间应该多久的建议？什么样的饮食有助于更快、更多地产生乳汁？"}, {"id": 485, "vi": "Trường hợp nào có thể tiêm hoóc-môn tăng trưởng được không?", "en": "In which case is it possible to administer growth hormone?", "zh": "哪些情况可以注射生长激素？"}, {"id": 486, "vi": "Tôi xét nghiệm có kết quả PH=5, so với chỉ số an toàn là 5,5-7,5, nhờ bác sỹ tư vấn và giúp cho tôi", "en": "I have a test result with pH = 5, compared to the safe range of 5.5-7.5, I kindly request the doctor's advice and assistance.", "zh": "我的检查结果显示pH=5，而安全范围是5.5-7.5，麻烦医生给我咨询和指导一下。"}, {"id": 487, "vi": "Có thể điều trị dứt điểm bệnh lý thần kinh ngoại biên không?", "en": "Can peripheral nervous system disorders be cured completely?", "zh": "可以根治周围神经系统疾病吗？"}, {"id": 488, "vi": "Tiểu đường thai kỳ ảnh hưởng thế nào đến thai nhi? Kiêng khem ra sao?", "en": "How does gestational diabetes affect the fetus? What dietary restrictions should be followed?", "zh": "妊娠期糖尿病对胎儿有什么影响？需要忌食什么？"}, {"id": 489, "vi": "Tiểu một ngày nhiều lần, màu bất thường là dấu hiệu của bệnh gì?", "en": "Frequent urination once a day with abnormal color - what disease is this indicative of?", "zh": "每天排尿多次，尿液颜色异常，可能是哪种疾病？"}, {"id": 490, "vi": "Nổi hạch ở cẳng tay và dưới háng có nguy hiểm không?", "en": "Is it dangerous to have lymph node swelling in the forearm and under the armpit?", "zh": "手臂和腋下出现淋巴结肿大是否有危险？"}, {"id": 491, "vi": "Em bị đau nửa người bên trái. Lúc đi khám bệnh bác sĩ cho e đi xét nghiệm máu và chụp XQ ngực thẳng. Các chỉ số xét nghiệm huyết học tế bào và sinh hoá máu của em đều nằm trong chỉ số bình thường trừ cái AST của em là 32. Còn chụp XQ thì bác sĩ ghi là bóng tim không to, nốt mờ tròn đều vùng thấp phổi và (P) to núm vú. Bác sĩ chuẩn đoán là em bị đau thần kinh và cho thuốc về uống. Nhưng giờ em lại bị đau", "en": "My left side is painful. When I went for a check-up, the doctor advised me to have a blood test and a frontal chest X-ray. All the blood cell and blood chemistry test results are within normal limits except for my AST, which is 32. Regarding the X-ray, the doctor noted that the heart shadow is not enlarged, there are evenly blurred round opacities in the lower lung fields, and (P) enlarged breast nodules. The doctor diagnosed me with nerve pain and prescribed medication to take at home. But now I am experiencing pain again.", "zh": "我左边半边身体疼痛。去医院检查时，医生让我做血检和胸部正位X光检查。我的血常规和生化检查结果均在正常范围内，除了AST为32。X光检查显示心脏大小正常，肺部下野有均匀的圆形模糊影，以及（P）乳腺增大。医生诊断为神经痛，开了药让我回家服用。但现在我又开始疼痛了。"}, {"id": 492, "vi": "Tôi sinh mổ từ tháng 10 năm 2018 tới nay sức khỏe bình thường nhưng tới nay đã 16 tháng mà bụng tôi vẫn to như có bầu 5 tháng, không biết cách gì khắc phục.", "en": "I had a cesarean section in October 2018 and have been in good health since then, but now, after 16 months, my abdomen is still enlarged as if I were pregnant for 5 months. I don't know what to do to remedy this.", "zh": "我从2018年10月剖腹产至今身体状况正常，但至今已16个月，我的腹部仍然像怀孕5个月一样大，不知道该如何改善。"}, {"id": 493, "vi": "Suy nghĩ linh tinh, khó tập trung có phải dấu hiệu loạn thần?", "en": "Fanciful thoughts, difficulty concentrating—could these be signs of delirium?", "zh": "思绪混乱，注意力不集中是否是谵妄的表现？"}, {"id": 494, "vi": "Vấn đề thứ nhất là cách tính tuổi thai ivf như thế nào ạ? Vấn đề thứ hai là : em chuyển phôi ngày 5 , 12 ngày sau chuyển phôi em xét nghiệm beta HCG được 999.5 , đến hôm nay là 14 ngày sau chuyển phôi em xét nghiệm beta được 1329. Vậy sự tăng chỉ số beta như vậy có bình thường hay chậm ạ bác sĩ không kê đơn thuốc mới mà vẫn duy trì đơn thuốc cũ cho em, như vậy có đảm bảo cho sự phát triển của thai không ạ?", "en": "The first issue is how to calculate IVF pregnancy age? The second issue is: I transferred the embryo on day 5. Twelve days after embryo transfer, I tested beta-HCG and got 999.5. Today, at 14 days after embryo transfer, I tested beta-HCG and got 1329. Is such an increase in beta value normal or slow, doctor? You haven't prescribed any new medication and have continued with the previous medication. Is this sufficient to ensure fetal development?", "zh": "第一个问题是试管婴儿的孕周如何计算？第二个问题是：我移植日为第5天，移植后12天我测出β-hcg值为999.5，到今天即移植后14天，β-hcg值为1329。这样的β-hcg值增长是否正常或偏慢？医生没有开新药，而是继续使用原来的药物，这样是否能保证胎儿的正常发育？"}, {"id": 495, "vi": "Người nhà tôi đang điều trị hóa chất ung thư vú. Tôi nghe mọi người xung quanh nói vitamin D rất tốt cho bệnh nhân ung thư vú, cho tôi hỏi thông tin đó có đúng không, nếu đúng thì uống liều như thế nào?", "en": "My family member is undergoing chemotherapy for breast cancer. I have heard people around me say that vitamin D is very beneficial for breast cancer patients. I would like to know if this information is accurate, and if so, what dosage should be taken?", "zh": "我家人正在接受乳腺癌的化疗。我听说维生素D对乳腺癌患者很有好处，我想咨询一下这个说法是否正确，如果正确，应该服用多少剂量？"}, {"id": 496, "vi": "Chỉ số AST của trẻ là 55,2 U/L liệu có đáng lo?", "en": "Is the child's AST level of 55.2 U/L concerning?", "zh": "患儿的AST值为55.2 U/L，是否需要担心？"}, {"id": 497, "vi": "Lúc em tập võ về cơ bụng xong em tưởng đâu là bị đau vài ngày sẽ cho đến nay hơn tuần rồi mà bụng em mõi lần căng ra hay vun vai là đau là bị sao vậy ạ?", "en": "After practicing martial arts, I thought I would just have a few days of abdominal pain, but now it's been more than a week, and every time my abdomen stretches or contracts, I feel pain. What could be wrong?", "zh": "练完武术后，我本以为只是肚子疼几天，结果到现在已经超过一周了，每次肚子一紧或一用力就疼痛，这是怎么回事啊？"}, {"id": 498, "vi": "Điều trị bướu cổ cho người bị bệnh cường giáp như nào?", "en": "How to treat goiter in patients with hyperthyroidism?", "zh": "甲亢患者的甲状腺肿如何治疗？"}, {"id": 499, "vi": "Tôi cắt bỏ túi mật và đặt ống stent nhưng gan vẫn tạo ra sỏi thì cần điều trị như thế nào?", "en": "I had my gallbladder removed and a stent placed, but the liver still forms stones—what should I do?", "zh": "我已切除胆囊并放置了支架，但肝脏仍产生结石，应如何治疗？"}, {"id": 500, "vi": "Suy giảm thị lực có phải do đau đầu hay tác dụng phụ của Topiramate?", "en": "Is vision loss due to headache or a side effect of Topiramate?", "zh": "视力下降是由于头痛还是托吡酯的副作用所致？"}, {"id": 501, "vi": "Độ phân bố TC (PDW) 17.3. Có ảnh hưởng gì không bác sĩ", "en": "PDW 17.3. What impact does it have, doctor?", "zh": "PDW 值为 17.3，医生，这会有影响吗？"}, {"id": 502, "vi": "Chồng em hay bị đau dạ dày mỗi khi uống rượu, và gần đây hay đau bụng giữa, ợ chua là bị làm sao và có thuốc gì ạ?", "en": "My husband often suffers from gastric pain whenever he drinks alcohol, and recently he has been experiencing mid-abdominal pain and acid reflux. What could be causing this and what medications are available?", "zh": "我丈夫每次喝酒都会出现胃痛，最近经常出现中上腹痛和反酸，这是怎么回事，有什么药物可以治疗吗？"}, {"id": 503, "vi": "Phương pháp cải thiện đầu nhỏ theo di truyền?", "en": "Methods to improve small head due to genetic inheritance?", "zh": "如何改善遗传性小头症？"}, {"id": 504, "vi": "Đổ mồ hôi nách nhiều do nguyên nhân gì?", "en": "What is the cause of excessive underarm sweating?", "zh": "为什么腋下多汗？"}, {"id": 505, "vi": "Em đã uống thuốc tránh thai khẩn cấp loại 72 giờ vào sáng hôm sau, khoảng 10 giờ sau khi có nguy cơ mang thai ngoài ý muốn. Thuốc có còn hiệu quả không ạ? Thuốc có ảnh hưởng đến chu kỳ kinh nguyệt không và có tác dụng ngăn rụng trứng trong bao lâu?", "en": "She took the 72-hour emergency contraceptive pill the next morning, about 10 hours after having an unintended pregnancy risk. Is the medication still effective? Does the medication affect the menstrual cycle, and for how long does it prevent ovulation?", "zh": "她在次日早上服用了72小时紧急避孕药，于有意外怀孕风险后约10小时服用。药物还有效吗？药物会影响月经周期吗？药物能抑制排卵多久？"}, {"id": 506, "vi": "Liệt hầu họng do nhồi máu não có thể hồi phục khả năng nuốt được không và điều trị thế nào?", "en": "Can dysphagia due to cerebral infarction recover swallowing ability, and how is it treated?", "zh": "脑梗导致的吞咽困难能否恢复吞咽功能，如何治疗？"}, {"id": 507, "vi": "Vì sao kết quả xét nghiệm bilirubin 2 ngày liên tiếp lại khác nhau?", "en": "Why do the test results for bilirubin differ on two consecutive days?", "zh": "为什么连续两天的胆红素检测结果不同？"}, {"id": 508, "vi": "Phương pháp FNA tuyến giáp là gì?", "en": "What is the method of thyroid FNA?", "zh": "甲状腺细针穿刺活检的方法是什么？"}, {"id": 509, "vi": "Gây mê nội khí quản là gì? Thực hiện thế nào?", "en": "What is endotracheal anesthesia? How is it performed?", "zh": "什么是经气管麻醉？如何实施？"}, {"id": 510, "vi": "Chưa trễ kinh nhưng lưng em đau và đầu nhức nhói lắm. Ra dịch nhầy trắng sữa và trắng trong nhiều. Tuần trước có xét nghiệm beta có 20.8 ạ. Thử que thì nhìn kỹ mới thấy vạch 2. Liệu em có khả năng có thai không ạ? Tại em bị sảy 2 lần rồi nên giờ rất lo lắng ạ.", "en": "No delay in menstruation but I have severe back pain and intense headache. I have had a lot of white, milky and clear discharge. Last week, the beta test result was 20.8. The test strip showed a second line only when examined carefully. Is there a possibility of pregnancy? I have had two miscarriages before, so I am very worried now.", "zh": "月经还没来，但我的背很痛，头痛得很厉害。有大量白色奶酪样和透明的分泌物。上个星期查了beta值，是20.8。用试纸测试时，仔细看才看到第二条线。我有没有可能怀孕呢？因为我之前流产了两次，所以现在非常担心。"}, {"id": 511, "vi": "Tôi là nam, 36 tuổi. Đợt vừa qua tôi có căng thẳng xung đột trong gia đình khoảng 2 tháng. Sau đó tôi thường xuyên thấy hồi hộp trống ngực trong ngày, không thấy khó thở gì cả. tôi đi khám bác sĩ chỉ kết luận nhịp nhanh xoang. tôi đã làm xét nghiệm máu, men gan, thận bình thường. Siêu âm ổ bụng có polyp túi mật. Điện tâm đồ có nhịp nhanh xoang. Bác sĩ chỉ kê thuốc bổ và bảo không cần thuốc điều trị. Sau 2 tháng thì 10 phần tôi đỡ được 2-3 phần. Bác sĩ cho hỏi tôi có cần đi khám lại không?", "en": "I am male, 36 years old. Recently, I experienced family stress and conflict for about 2 months. After that, I have frequently felt palpitations and chest tightness during the day, without any shortness of breath. I visited a doctor who only diagnosed sinus tachycardia. I underwent blood tests, liver and kidney function tests, which were normal. Abdominal ultrasound revealed gallbladder polyps. ECG showed sinus tachycardia. The doctor prescribed supplements and advised no need for treatment. After two months, I feel improvement in 10 parts, with 2-3 parts improved. The doctor asks if I need to be re-evaluated?", "zh": "我是一名36岁的男性。最近两个月，我家庭中出现了持续的紧张和冲突。之后，我每天都会感到胸闷，但没有呼吸困难。我去了医生那里就诊，医生仅诊断为窦性心动过速。我做了血常规、肝功能、肾功能检查，结果均正常。腹部超声检查发现胆囊息肉。心电图显示窦性心动过速。医生开了些补品，建议不需要治疗药物。两个月后，我感觉症状改善了约20%至30%。医生问我是否需要再次就诊？"}, {"id": 512, "vi": "Viêm gan B cấp nên điều trị thuốc gì?", "en": "What drugs should be used to treat acute B hepatitis?", "zh": "乙型肝炎急性应使用什么药物治疗？"}, {"id": 513, "vi": "Uống bổ sung vitamin C khi đang điều trị suy giáp có ảnh hưởng gì không?", "en": "Does taking vitamin C supplementation during treatment of hypothyroidism have any effect?", "zh": "在治疗甲状腺功能减退期间服用维生素C会有何影响？"}, {"id": 514, "vi": "Tôi mọc răng khôn cả 2 bên liệu tôi có bị nhổ cả răng hàm trên không", "en": "I have wisdom teeth growing on both sides, will I need to have my upper molars extracted?", "zh": "我两边都长智齿，我是否需要拔掉上颌磨牙？"}, {"id": 515, "vi": "Đợt vừa rồi e rối loạn kinh nguyệt và được kê thuốc thúc ra kinh. Em có kinh hôm nay là ngày kinh thứ 7. Hôm nay em chỉ còn ra 1 ít dịch màu nâu nhạt. Chiều mai em đã có thể đi chụp vòi trứng được chưa ạ?", "en": "Recently, I had menstrual disturbance and was prescribed medication to induce menstruation. Today, which is day 7 of my period, I am only seeing a little pale brown discharge. Can I go for hysterosalpingography tomorrow afternoon?", "zh": "上次月经紊乱，医生开了促排卵药物。今天是月经的第7天，目前只有一点淡棕色分泌物。明晚可以去照输卵管吗？"}, {"id": 516, "vi": "Em năm nay 22 tuổi sinh bé đướcj hơn 2,5 tuổi em có đặt vòng được hơn 1 năm, mới mấy ngày em tự ý tháo vòng ra vì thấy nó kiểu lồi ra và muốn rơi khỏi tử cung, do công việc bận rộn nên em không có thời gian nghỉ đi khám, không biết tình trạng tự ý tháo vòng của em ra có bị ảnh hưởng gì đến sinh sản sau này không? Hay có bị các vấn đề khác nữa không?", "en": "I am 22 years old, had a child more than 2.5 years ago, and have had an IUD for over one year. Just a few days ago, I removed the IUD on my own because I noticed it was protruding and wanted to fall out of the uterus. Due to a busy work schedule, I did not have time to take a break and visit a doctor. I do not know if removing the IUD on my own has affected my future fertility or if there are any other issues.", "zh": "我今年22岁，生育孩子后超过2.5年，放置宫内节育器已超过一年。最近几天我自行取出了节育器，因为感觉它突出出来，想要从子宫脱落。由于工作繁忙，我没有时间休息去检查，不知道自行取出节育器会对今后的生育造成影响吗？或者还会引发其他问题吗？"}, {"id": 517, "vi": "Giãn tĩnh mạch có điều trị được bằng phương pháp phẫu thuật và laser không?", "en": "Can varicose veins be treated by surgical and laser methods?", "zh": "静脉曲张能否通过手术和激光治疗？"}, {"id": 518, "vi": "Điều trị Basedow khi cho con bú như thế nào?", "en": "How to treat Basedow's while breastfeeding?", "zh": "哺乳期间如何治疗格雷夫斯病？"}, {"id": 519, "vi": "Xét nghiệm máu bệnh tay chân miệng có phát hiện được vi rút gây bệnh không?", "en": "Can blood testing for hand, foot, and mouth disease detect the virus causing the disease?", "zh": "手足口病血检能否检测出致病病毒？"}, {"id": 520, "vi": "Dùng thuốc augmentin và alphachoay khi đang cho con bú có ảnh hưởng không có cần ngừng cho bú không", "en": "Using Augmentin and Alphachoay while breastfeeding has any effects? Is it necessary to stop breastfeeding?", "zh": "在哺乳期间使用阿莫西林克拉维酸钾和阿尔法可乐定是否有影响？是否需要停止哺乳？"}, {"id": 521, "vi": "Chụp vòi trứng có đau không và chụp thì nên bao nhiêu lâu có con", "en": "Does ovarian follicle ultrasound cause pain, and how long after the scan should one conceive?", "zh": "输卵管造影会痛吗？做输卵管造影后多久能怀孕"}, {"id": 522, "vi": "Có nên điều trị bệnh tai biến và trĩ cùng một lúc không?", "en": "Should stroke and hemorrhoids be treated at the same time?", "zh": "是否应该同时治疗中风和痔疮？"}, {"id": 523, "vi": "Tôi thì hay thủ dâm không biết khi lớn lên sẽ có bị sao không", "en": "I often masturbate, I don't know whether I will have any problems when growing up.", "zh": "我经常自慰，不知道长大后会不会有什么问题？"}, {"id": 524, "vi": "Thoái hóa võng mạc ngoại vi có chữa được không?", "en": "Can peripheral retinal degeneration be treated?", "zh": "外周视网膜变性能治好吗？"}, {"id": 525, "vi": "Suy thận độ 1 có điều trị được không?", "en": "Can stage 1 kidney failure be treated?", "zh": "1度肾功能不全能否治疗？"}, {"id": 526, "vi": "Cách điều trị triệu chứng tê yếu chân và giảm khả năng giao tiếp do nhồi máu não là gì?", "en": "What is the treatment for the symptoms of leg numbness and weakness and reduced communication ability due to ischemic stroke?", "zh": "治疗由脑梗死引起的下肢麻木无力及交流能力下降的症状的方法是什么？"}, {"id": 527, "vi": "Chồng tôi có kháng nguyên viêm gan B, vậy có được tiêm vaccine Covid-19 không ?", "en": "My husband has hepatitis B surface antigen; can he receive the COVID-19 vaccine?", "zh": "我丈夫有乙型肝炎病毒表面抗原，是否可以接种新冠疫苗？"}, {"id": 528, "vi": "Em bị chậm kinh hơn 2 tháng, hiện tại thì em nên làm gì ạ?", "en": "I have missed my period by more than 2 months. What should I do now?", "zh": "我月经推迟了两个月，现在应该做什么呢？"}, {"id": 529, "vi": "Viêm khớp thái dương hàm có nguy hiểm không?", "en": "Is temporomandibular joint inflammation dangerous?", "zh": "颞下颌关节炎危险吗？"}, {"id": 530, "vi": "Cho em hỏi con em được 2 tháng tuổi em vừa cho con đi khám bác sĩ cho uống những thuốc như trên ảnh cho em hỏi có vừa uống mà vẫn tiêm được vắc xin 6 trong 1 không ạ?", "en": "May I ask if my 2-month-old child, who just had a doctor's visit and was prescribed the medications shown in the photo, can still receive the 6-in-1 vaccine?", "zh": "我想问一下，我家宝宝两个月大，医生刚刚开了上面图片里的药，我想问一下，吃了这些药之后，还能打六联疫苗吗？"}, {"id": 531, "vi": "Điều trị lao phổi đa kháng thuốc vẫn bị khạc đờm mỗi sáng có sao không?", "en": "Is it problematic to have morning cough after treatment for multidrug-resistant pulmonary tuberculosis?", "zh": "抗药性肺结核治疗期间每天早上仍然有咳嗽，有问题吗？"}, {"id": 532, "vi": "Lúc trước em ăn cay không hề có dấu hiệu gì, nhưng gần đây em ăn cay lại bị tiêu chảy, nguyên do vì sao ạ? Có do bệnh gì không ạ?", "en": "Previously, I did not have any symptoms when eating spicy food, but recently, I have been experiencing diarrhea after eating spicy food. What could be the cause? Is it due to any disease?", "zh": "以前我吃辛辣食物没有任何症状，但最近吃辛辣食物就会出现腹泻，这是什么原因呢？有没有可能是某种疾病引起的？"}, {"id": 533, "vi": "Men gan tăng kèm ngứa, nổi mẩn khi uống thuốc điều trị lao có nên uống nữa không?", "en": "Should I continue taking medication for tuberculosis if I have increased liver enzymes along with itching and hives?", "zh": "肝功能升高伴服抗结核药后出现瘙痒、皮疹，是否还应继续服药？"}, {"id": 534, "vi": "21 tuổi còn tăng chiều cao được không?", "en": "Can a 21-year-old still grow in height?", "zh": "21岁还能长高吗？"}, {"id": 535, "vi": "Bà nội em hơn 70 tuổi mấy năm nay đau vùng khớp gối 2 bên, đi lại khó khăn, thỉnh thoảng có uống thuốc nam thì đỡ nhưng ko khỏi hẳn, bác sĩ cho em hỏi bà em bị bệnh gì và điều trị như thế nào , có đỡ được không.", "en": "My mother-in-law is over 70 years old and has had pain in both knee joints for several years, making walking difficult. Occasionally, she takes traditional herbal medicine, which provides some relief but does not completely resolve the issue. Doctor, I would like to ask you what disease she has and how it should be treated, and whether it will improve.", "zh": "我妹妹的奶奶今年超过70岁，近几年双侧膝关节疼痛，行动困难，偶尔服用中药会有所缓解，但无法彻底治愈。医生请问一下，她得的是什么病，怎么治疗，能否好转？"}, {"id": 536, "vi": "Bàn chân thường xuyên lạnh khi đi chân đất hoặc quạt thổi vào là biểu hiện bệnh gì?", "en": "What disease is indicated by frequently cold feet when walking on ground or when blown by a fan?", "zh": "经常在走路或风扇吹风时脚部发冷是何种疾病的征兆？"}, {"id": 537, "vi": "Từ hôm qua đến nay em có biểu hiện sốt, đau họng, ngứa ngáy, thi thoảng hay nôn mửa thì em bị sao", "en": "Since yesterday, I have had fever, sore throat, itching, and occasionally vomiting. What is wrong with me?", "zh": "从昨天到现在，我出现了发烧、喉咙痛、瘙痒，偶尔会呕吐，这是怎么回事？"}, {"id": 538, "vi": "Hôm nay tôi đã đi chụp X-quang vú để tầm soát ung thư vú. Kết luận là có hai vôi hoá thô kích thước 1x2 cm bên vú trái. Vôi hoá thô có tiến triển thành ung thư không?", "en": "Today I went for a mammography to screen for breast cancer. The conclusion is there are two coarse calcifications measuring 1x2 cm on the left breast. Can coarse calcifications progress into breast cancer?", "zh": "今天我做了乳腺X光检查以筛查乳腺癌。检查结果为左侧乳腺有两处粗大钙化灶，大小为1x2厘米。粗大钙化灶会发展成乳腺癌吗？"}, {"id": 539, "vi": "Kết quả xét nghiệm nội tiết ngày 2 chu kỳ của tôi chỉ số prolactin là 525 như vậy có cao quá không ạ? Có cần uống thuốc để giảm không ạ?", "en": "The endocrine lab result on day 2 of my cycle shows a prolactin level of 525. Is this considered high? Do I need to take medication to reduce it?", "zh": "我月经周期第2天的内分泌检查结果显示催乳素水平为525，这样算高吗？需要吃药来降低吗？"}, {"id": 540, "vi": "Nồng độ ferritin cao có gây ảnh hưởng đến sức khỏe không?", "en": "Does high ferritin level affect health?", "zh": "血清铁蛋白水平高会对健康造成影响吗？"}, {"id": 541, "vi": "Nhiệt miệng không có mủ kèm đau nhưng điều trị không khỏi là dấu hiệu của bệnh gì?", "en": "Mouth ulcers without pus accompanied by pain but not responding to treatment are signs of what disease?", "zh": "口腔溃疡无脓液伴疼痛但治疗无效是何种疾病的征兆？"}, {"id": 542, "vi": "Tất cả các triệu chứng thông thường có thể xảy ra sau tiêm vaccine Covid? Làm sao để phân biệt triệu chứng đó là do tác dụng của tiêm ngừa hay do nguyên nhân khác. Ví dụ như sau tiêm ngừa không thấy biểu hiện gì bất thường, nhưng đến 2 ngày sau (hoặc 3, 4 ngày sau) lại cảm thấy các biểu hiện như: sốt, ngẹt mũi, đau họng, đau cơ... thì các biểu hiện này còn có thể xem là do tiêm ngừa không hay phải quan tâm đến nguyên nhân khác?", "en": "What are all the common symptoms that may occur after receiving the Covid vaccine? How to differentiate whether these symptoms are due to the vaccine effect or due to other causes? For example, after vaccination, no unusual symptoms are observed, but two days later (or three, four days later), symptoms such as fever, nasal congestion, sore throat, muscle pain appear—can these symptoms still be considered as due to vaccination or should other causes be considered?", "zh": "接种新冠疫苗后可能出现的所有常见症状是什么？如何区分这些症状是疫苗的正常反应，还是由其他原因引起？例如，接种疫苗后没有出现任何异常表现，但2天后（或3天、4天后）开始出现发热、鼻塞、咽喉痛、肌肉疼痛等症状，这些症状是否仍可归因于接种疫苗，还是需要考虑其他原因？"}, {"id": 543, "vi": "Vì sao phải làm xét nghiệm tinh dịch đồ? Những yếu tố nào sẽ ảnh hưởng đến kết quả xét nghiệm tinh dịch đồ? Cần lưu ý gì trước khi thực hiện xét nghiệm này?", "en": "Why is a semen analysis test required? What factors may affect the results of a semen analysis? What should be noted before undergoing this test?", "zh": "为什么需要进行精液检查？哪些因素会影响精液检查的结果？进行此项检查前需要注意什么？"}, {"id": 544, "vi": "Nhiễm trùng máu có nguy hiểm đến tính mạng không và điều trị như thế nào?", "en": "Is sepsis dangerous to life and how is it treated?", "zh": "败血症是否危及生命，如何治疗？"}, {"id": 545, "vi": "Áp xe hậu môn tự vỡ liệu có bị rò hậu môn không?", "en": "Does a spontaneously ruptured perianal abscess lead to fecal incontinence?", "zh": "肛门脓肿自行破溃是否会引发肛瘘？"}, {"id": 546, "vi": "Phụ nữ mang thai bị thủy đậu phải làm sao?", "en": "What should be done for a pregnant woman with chickenpox?", "zh": "孕妇患水痘怎么办？"}, {"id": 547, "vi": "Em có quan hệ với bạn gái em, và xuất tinh bên trong ,bạn gái em có biểu hiện mắc ói, nhưng không ói đc với lại đau bụng. Có phải là có thai không?", "en": "I have sexual intercourse with my girlfriend, and I ejaculated inside. My girlfriend has shown symptoms of nausea, but she cannot vomit, and she has abdominal pain. Could this be pregnancy?", "zh": "我与女友有性关系，并在体内射精，女友出现恶心，但没有呕吐，同时伴有腹痛。是否可能是怀孕了？"}, {"id": 548, "vi": "Điều trị ổ cặn màng phổi không phẫu thuật như thế nào?", "en": "How to treat non-surgical pleural effusion?", "zh": "非手术性胸膜增生的治疗方式是什么？"}, {"id": 549, "vi": "Em đang mang thai 26 tuần 4 ngày, hôm nay em có xét nghiệm lượng đường trong máu kết quả là 4,52. Kết quả thế này có cần phải làm xét nghiệm dung nạp đường nữa không ạ?", "en": "I am currently 26 weeks and 4 days pregnant. Today I had a blood glucose test, and the result is 4.52. Should I undergo a glucose tolerance test based on this result?", "zh": "我目前怀孕26周4天，今天做了血糖检测，结果是4.52。这个结果是否还需要做糖耐量检测呢？"}, {"id": 550, "vi": "Tôi là nam hiện 36 tuổi, bị mỡ máu cao dẫn đến thi thoảng hay đau cả đầu. Nhờ bác sĩ tư vấn giúp về chế độ ăn và thuốc uống nếu cần.", "en": "I am a male, currently 36 years old, with high lipid levels leading to occasional headaches. I kindly request the doctor's advice regarding diet and medication if needed.", "zh": "我是一名36岁的男性，患有高血脂，导致偶尔会头痛。希望医生能给我一些关于饮食和是否需要用药的建议。"}, {"id": 551, "vi": "Tôi sinh con được 25 ngày. Khi sinh cháu nặng 3.6kg. Cháu ti mẹ 100% và đều đặn.Hôm nay tôi cân thì cháu được 4.2kg. Còn 5 hôm nữa là cháu đầy tháng. Vậy với cân nặng trên con tôi có bị còi quá không ạ", "en": "I gave birth 25 days ago. At birth, the baby weighed 3.6 kg. The baby is breastfeeding 100% and regularly. Today, I weighed the baby and the weight is 4.2 kg. Five days from now, the baby will be one month old. So, with the above weight, is my baby suffering from rachitic condition?", "zh": "我产后25天。孩子出生时体重3.6公斤，目前母乳喂养，喂养充分且规律。今天称重，孩子体重为4.2公斤，还有5天就满月了。根据这个体重，我家孩子会不会得佝偻病呢？"}, {"id": 552, "vi": "Bé nhà em được 50 ngày lúc bé còn trong tháng giấc tối bé ngủ rất ngon khi đói thì ọ ẹ đưa ti mẹ vào là ti no ngủ tiếp , nhưng khoảng 2 ngày gần đây đầu hôm khoảng 18h đến 21h là bé đói đi chăng nữa đưa ti mẹ vào là bé khóc ngất không vỗ được thêm bé không chịu ngủ nữa ạ. Giấc tối cũng vậy nữa ạ bé ngủ không sâu giấc", "en": "My child is 50 days old. When the baby was in the month, during nighttime the baby slept very well. When hungry, the baby would cry and then suckle from the mother's breast and then become full and continue sleeping. However, in the past two days, around 18:00 to 21:00 in the evening, the baby is hungry again. When the mother puts the baby to suckle, the baby cries intensely and cannot be soothed further, and refuses to sleep. The same thing happens during nighttime; the baby does not sleep deeply.", "zh": "我家宝宝50天大，之前宝宝在夜间睡眠很好，饿了会哭闹，喂奶后就会吃饱并继续睡觉。但最近两天，大约在每天18点到21点之间，宝宝又开始饿了，喂奶后宝宝会哭闹不止，无法安抚，也不肯睡觉。夜间睡眠也一样，睡眠不深。"}, {"id": 553, "vi": "Xuất hiện nhiều đốm xuất huyết dưới da là bệnh gì?", "en": "What disease causes numerous subcutaneous hemorrhagic macules?", "zh": "出现多处皮下出血点是什么病？"}, {"id": 554, "vi": "Phì đại tiền liệt tuyến kích thước 40gr, PSA 4.9 mol/l có sao không?", "en": "Prostate enlargement with a size of 40 grams, PSA 4.9 mol/l, is there any problem?", "zh": "前列腺增生，体积40克，PSA 4.9 mol/L，有问题吗？"}, {"id": 555, "vi": "Thường xuyên đau đầu sau chảy máu dưới nhện là vì sao?", "en": "Why does frequent headache occur after subarachnoid hemorrhage?", "zh": "经常在蛛网下出血后出现头痛的原因是什么？"}, {"id": 556, "vi": "Cơ chế gây bệnh Kwashiorkor là gì?", "en": "What is the pathogenesis of kwashiorkor?", "zh": "克瓦希奥尔克病的发病机制是什么？"}, {"id": 557, "vi": "Tình trạng hẹp động mạch trong và ngoài sọ nguy hiểm như thế nào?", "en": "How dangerous is the stenosis of intracranial and extracranial arteries?", "zh": "颅内和颅外动脉狭窄的危害性如何？"}, {"id": 558, "vi": "Đau đầu, nổi hạch tai trái là dấu hiệu bệnh gì?", "en": "Headache, enlarged lymph node on the left ear is a sign of what disease?", "zh": "头痛、左耳淋巴结肿大是何种疾病的征兆？"}, {"id": 559, "vi": "Môi bị méo do bỏng a-xít lâu năm có phẫu thuật chỉnh hình khuôn mặt được không?", "en": "Can patients with long-term acid burn causing facial asymmetry undergo facial reconstructive surgery?", "zh": "长期酸烧伤导致的口唇畸形能否进行面部整形手术？"}, {"id": 560, "vi": "Nữ giới tai rỉ mủ, có mùi hôi nguyên nhân là gì?", "en": "What is the cause of foul-smelling purulent ear discharge in women?", "zh": "女性耳流脓，有臭味，原因是什么？"}, {"id": 561, "vi": "Em có lưu thai 32 tuần cách đây hơn 3 tháng. Lưu không rõ nguyên nhân. Em có đi xét nghiệm để tìm hiểu nguyên nhân, mọi thứ không vấn đề gì. Nhưng kết quả xét nghiệm nhiễm sắc thể đồ kết luận: 46,XX,21ps+, tăng kích thước vệ tinh nhiễm sắc thể số 21. Vậy đây có phải là nguyên nhân gây thai lưu không bác sĩ? Em xét nghiệm Rubella thấy trước đây đã từng bị, nhưng trong 3 tháng đầu mang thai em không bị cúm ạ.", "en": "I had a spontaneous abortion at 32 weeks ago, more than three months ago. The cause of the miscarriage is unclear. I underwent testing to investigate the cause, and everything appeared normal. However, the chromosomal test result concluded: 46,XX,21ps+, increased size of the satellite of chromosome 21. Is this the cause of the miscarriage, doctor? I tested for Rubella and previously had it, but during the first three months of pregnancy, I did not have influenza.", "zh": "我三个月前有32周的流产，流产原因不明。我做过检查以了解原因，各项检查均无异常。但染色体检查结果为：46,XX,21ps+，21号染色体卫星体增大。医生，这是否是导致流产的原因？我之前检查过风疹病毒，曾有感染史，但在怀孕前三个月内我没有患过流感。"}, {"id": 562, "vi": "Tê bì ở vùng da gây tê sau phẫu thuật cắt kén màng phổi có sao không?", "en": "Is there paresthesia in the skin area that became numb after spinal cord surgery?", "zh": "胸膜腔穿刺术后出现皮肤麻木是否有问题？"}, {"id": 563, "vi": "Tự nhiên tôi bị đau đầu gối, đi lại cà nhắc. Xin cho biết nguyên nhân và cách chữa.", "en": "Naturally, I have knee pain and feel unsteady when walking. Please inform me of the causes and treatments.", "zh": "我自然地感到膝关节疼痛，行走不稳。请问原因及治疗方法。"}, {"id": 564, "vi": "Viêm nang lông mạn tính kéo dài từ nhỏ đến lớn nặng hơn vào mùa đông điều trị như thế nào", "en": "Chronic folliculitis progressing from small to large, worsening in winter, how to treat it?", "zh": "慢性毛囊炎持续从小到大发展，冬季加重，应如何治疗"}, {"id": 565, "vi": "Bắp tay bị sưng sau khi tiêm thuốc chống cơn gò bụng do đau dạ dày nên làm gì?", "en": "Arm swelling after injection of anti-epigastric drug due to gastric pain - what should be done?", "zh": "注射抗胃痉挛药物后手臂肿胀该怎么办？"}, {"id": 566, "vi": "Em bị buồng trứng đa nang cả hai bên trái và phải. Em có xét nghiệm nhiễm sắc thể nhưng chưa có kết quả. Em muốn hỏi là em bị bệnh đa nang như vậy thì có khả năng sinh con tự nhiên không ạ? Bệnh đa nang có chữa khỏi được không ạ?", "en": "I have polycystic ovaries on both left and right sides. I have undergone chromosomal testing but have not yet received the results. I would like to ask whether having polycystic ovary disease like this means I have a chance of conceiving naturally? Can polycystic disease be cured?", "zh": "我两边卵巢都有多囊卵巢。我做了染色体检查，但还没有结果。我想问一下，我这种情况是否有可能自然怀孕？多囊卵巢能治愈吗？"}, {"id": 567, "vi": "Em có kết quả xét nghiệm nhiễm sắc thể là 46,XY,Yqh+, người nam có tăng vùng dị nhiễm sắc nhánh dài nhiễm sắc thể Y.", "en": "The patient's karyotype result is 46,XY,Yqh+, a male with an increased region of Y chromosome long arm numerical abnormality.", "zh": "我有染色体检测结果为46,XY,Yqh+，男性存在Y染色体长臂异倍体区域。"}, {"id": 568, "vi": "Đau nửa đầu dữ dội nên điều trị thế nào?", "en": "How should severe migraine be treated?", "zh": "如何治疗剧烈偏头痛？"}, {"id": 569, "vi": "Nốt ban thủy đậu to, nổi nhiều, vỡ ra có ảnh hưởng gì không?", "en": "Large, numerous varicella lesions that break open – what impact does this have?", "zh": "水痘疹块大、数量多，破溃会有影响吗？"}, {"id": 570, "vi": "Dùng thuốc tẩy giun Fugacar nhưng không hết giun có cần sử dụng liều nhắc lại sau 2-4 tuần không?", "en": "Should I use a repeat dose of the deworming drug Fugacar after 2-4 weeks if the worm infection is not completely eliminated?", "zh": "使用驱虫药Fugacar后蛔虫未根除，是否需要在2-4周后重复用药？"}, {"id": 571, "vi": "Lách to và đau sau sốt rét phải làm sao?", "en": "Enlarged spleen and pain after malaria - what should be done?", "zh": "疟疾后脾大和疼痛怎么办？"}, {"id": 572, "vi": "Xin cho em hỏi kinh nguyệt của em rất thưa, thường thì 6 tháng mới có 1 lần. Có cách nào điều chỉnh cho nó ngắn lại không bác sĩ?", "en": "Dear doctor, my menstrual cycle is very irregular, usually occurring only once every 6 months. Is there any way to regulate it to become shorter?", "zh": "医生，我月经非常稀少，通常六个月才来一次，有没有办法让它变得规律一些？"}, {"id": 573, "vi": "Làm sao để phụ nữ sau sinh ngủ ngon?", "en": "How can women after childbirth have good sleep?", "zh": "如何让产后女性睡得更好？"}, {"id": 574, "vi": "Tụ máu đuôi mắt sau gãy thành xoang trước và sau bên trái điều trị thế nào?", "en": "How to treat orbital hematoma after fracture of the left anterior and posterior ethmoid walls?", "zh": "左侧额窦和蝶窦骨折后眼窝内血肿如何治疗？"}, {"id": 575, "vi": "Sau khi sinh ngừng thuốc protevir do men gan bình thường và virus dưới ngưỡng, sau 3 tháng định lượng virus tăng mạnh 10 mũ 7 có nên tiếp tục dùng thuốc không", "en": "After delivery, stopped protevir due to normal liver enzymes and virus below threshold, after 3 months, viral load increased sharply to 10^7, should medication be continued?", "zh": "产后肝功能恢复正常且病毒低于检测下限后，停用普乐可勃，三个月后病毒定量突然升高10^7，是否应继续用药？"}, {"id": 576, "vi": "Ăn chung bát đũa với người nhiễm viêm gan B có khả năng lây nhiễm không?", "en": "Can sharing a plate and utensils with someone infected with hepatitis B lead to infection?", "zh": "与乙肝患者共用碗筷会有感染风险吗？"}, {"id": 577, "vi": "Phù người, rối loạn kinh nguyệt khi dùng thuốc tăng cân có phải do suy thượng thận?", "en": "Water retention, menstrual disorders when using weight gain medication—could this be due to adrenal insufficiency?", "zh": "服用增重药物后出现水肿和月经紊乱是否由肾上腺功能减退引起？"}, {"id": 578, "vi": "Chỉ số bạch cầu bao nhiêu là bình thường?", "en": "What is the normal white blood cell count?", "zh": "白细胞计数正常值是多少？"}, {"id": 579, "vi": "Cách đây 1 năm, sau 1 lần đi cắt tóc cạo mặt thì da mặt em sưng đỏ và xuất hiện các lớp da chết bong tróc. Sau đó em mua mỡ về bôi thì đỡ hẳn. Nhưng thời gian sau ở 2 bên gò má của em hay bị mẩn đỏ và có các lớp da chết, da khô như vảy, đặc biệt là vào mùa đông. Cho đến bây giờ vẫn bị đỏ hồng 2 bên gò má. Em bị bệnh gì về da không và phương pháp điều trị là gì?", "en": "One year ago, after a haircut involving shaving my face, my face swelled up and became red, with layers of dead skin flaking off. Afterwards, I bought oil and applied it, and the condition improved significantly. However, later on, both sides of my cheekbones frequently became itchy and red, with layers of dead skin and dry, scaly skin, especially during winter. Until now, both sides of my cheekbones still remain red and pink. What skin disease do I have, and what is the treatment method?", "zh": "一年前，有一次剃掉面部头发后，面部出现红肿，并伴有脱皮、死皮脱落。之后我买了油脂涂抹，症状明显缓解。但后来两侧太阳穴经常出现发红和死皮，皮肤干燥如鳞屑，尤其在冬季明显。到现在，两侧太阳穴仍然发红。我患有什么皮肤疾病？治疗方法是什么？"}, {"id": 580, "vi": "Dạ bé nhà em được 5 tháng 21 ngày ba hôm nay đêm nào cũng không ngủ được chỉ khóc em thấy nướu của bé có nhô ra hai răng cửa cho em hỏi dấu hiệu của mọc răng là gì ạ", "en": "My baby is 5 months and 21 days old. For the past three nights, she cannot sleep and only cries. I notice her gums have protruded with two front teeth. I would like to know what are the signs of teething.", "zh": "我女儿5个月21天，最近三天每晚都睡不着，只哭，我看到她牙龈有突出两个乳牙，想请问一下，长牙的征兆是什么？"}, {"id": 581, "vi": "Hiện em đang mang thai tuần 35, và có hiện tượng 1 bên chân phù to hơn chân còn lại và thấy đau nhức hơn chân kia Vậy bác sĩ cho em hỏi là em có bị làm sao không ạ?", "en": "I am currently 35 weeks pregnant and have noticed that one leg is swollen more than the other, and I feel more pain and discomfort in that leg. So, I would like to ask the doctor whether I am experiencing any problem.", "zh": "我目前怀孕35周，一侧脚肿得比另一侧明显，而且感觉一侧脚更痛。请问医生，我这种情况有问题吗？"}, {"id": 582, "vi": "Hoóc-môn GH do thùy trước tuyến yên tiết ra nhiều hơn bình thường sẽ gây hiện tượng gì ở người?", "en": "What phenomenon will occur in people due to excessive normal secretion of GH hormone from the anterior pituitary gland?", "zh": "垂体前叶分泌的生长激素（GH）超过正常水平会使人出现什么现象？"}, {"id": 583, "vi": "Trẻ bị són tiểu nhiều lần nhưng khám và xét nghiệm nước tiểu bình thường phải làm sao?", "en": "The child has frequent urinary incontinence but urine examination and tests are normal. What should be done?", "zh": "孩子多次出现尿失禁，但尿液检查和体检均正常，该怎么办？"}, {"id": 584, "vi": "Làm thế nào để kiểm tra vòng tránh thai có đúng vị trí không?", "en": "How to check if the contraceptive ring is in the correct position?", "zh": "如何检查避孕环是否处于正确位置？"}, {"id": 585, "vi": "Trẻ dùng núm vú giả có dễ bị viêm tai không? Và có cách nào để hạn chế viêm tai nếu vẫn dùng ti giả không?", "en": "Do babies using artificial nipples have an increased risk of otitis? And are there any ways to reduce the risk of otitis if artificial nipples are still used?", "zh": "使用奶嘴的婴儿是否更容易发生中耳炎？如果仍然使用奶嘴，是否有办法减少中耳炎的发生？"}, {"id": 586, "vi": "Chảy máu, có đờm sau cắt a-mi-đan là bị làm sao?", "en": "Bleeding with sputum after AMI surgery - what is the cause?", "zh": "割除腺样体后出现出血和咳痰是怎样的情况？"}, {"id": 587, "vi": "Bệnh tiểu đường có bị biến chứng thần kinh ngoại vi hay không?", "en": "Does diabetes mellitus have peripheral nervous system complications?", "zh": "糖尿病是否会出现周围神经病变？"}, {"id": 588, "vi": "Mắt bị đóng vôi không còn thấy đường có điều trị được không?", "en": "Can eyes with calcium deposits still see the path be treated?", "zh": "眼睛有钙化，看不见道路，有没有治疗方法？"}, {"id": 589, "vi": "Em muốn hỏi là xét nghiệm combo HIV ở tuần thứ 12 và 17 âm tính, giang mai 12 tuần âm tính, bệnh lậu không triệu chứng 17 tuần âm tính nhưng em bị đau họng và trắng lưỡi 2 tháng rồi, nghi bị nấm thì có ảnh hưởng đến xét nghiệm HIV và giang mai không ạ? Em bị viêm gan B nhưng làm định lượng dưới ngưỡng phát hiện thì có cần điều trị gì không ạ?", "en": "I would like to ask: the combo HIV test at week 12 and week 17 is negative, syphilis at 12 weeks is negative, and asymptomatic gonorrhea at 17 weeks is negative, but I have had sore throat and white tongue for two months, suspecting fungal infection—would this affect the HIV and syphilis test results? I have hepatitis B, but the quantification is below the detection limit—do I need any treatment?", "zh": "我想问一下，第12周和第17周的HIV组合检测为阴性，梅毒12周检测为阴性，无症状淋病17周检测为阴性，但我已经连续两个月有喉咙痛和舌苔白的症状，怀疑是真菌感染，这种情况会影响HIV和梅毒的检测结果吗？我有乙型肝炎，但定量检测结果低于检测下限，这种情况需要治疗吗？"}, {"id": 590, "vi": "Em xét nghiệm HIV khẳng định ở bệnh viện tại thời điểm 6 tháng sau nguy cơ mà âm tính thì khả năng âm tính giả có hay không và có phải xét nghiệm lại nữa không?", "en": "My HIV test was confirmed at the hospital at 6 months after exposure with a negative result—is there a possibility of a false negative, and do I need to get retested?", "zh": "在6个月高危期结束后于医院进行HIV确认检测，结果为阴性，是否存在假阴性的情况？是否需要再次检测？"}, {"id": 591, "vi": "Thường xuyên co giật sau khi sử dụng rượu bia có nguy cơ mắc bệnh gì?", "en": "Does frequent seizures after consuming alcohol and beer increase the risk of developing any disease?", "zh": "经常在饮酒后出现抽搐，有什么疾病风险？"}, {"id": 592, "vi": "Đi làm đồng dẫm phải mảnh chai nhưng không tiêm vắc-xin uốn ván, đã liệt chân tay đang nằm viện truyền kháng sinh còn cơ hội chữa trị không", "en": "Going to work and stepping on a broken glass but not receiving tetanus vaccine, now have paralysis of arms and legs, currently hospitalized receiving antibiotics, is there still a chance of treatment?", "zh": "上班时踩到玻璃碎片，但未接种破伤风疫苗，现已出现四肢瘫痪，正在医院接受抗生素治疗，还有治愈的机会吗？"}, {"id": 593, "vi": "Trẻ xuất hiện nốt nhỏ ở cổ có phải hạch không? Điều trị thế nào?", "en": "Does the child have small nodules in the neck? Are they lymph nodes? How to treat?", "zh": "孩子脖子上出现小疙瘩是不是淋巴结？怎么治疗？"}, {"id": 594, "vi": "Hai vợ chồng đã cưới nhau được ba năm. Đi khám ba lần, đều được bảo hai vợ chồng bình thường nhưng không hiểu lý do vì sao lại vô sinh, có đặt thuốc và canh chu kỳ nhưng vẫn chưa hiệu quả và có kết quả. Xin tư vấn giúp, tại sao hai vợ chồng đi khám kết quả bình thường, không bị gì mà vẫn vô sinh hiếm muộn, có đi canh chu kỳ để thụ thai mà vẫn chưa hiệu quả.", "en": "A married couple has been married for three years. They have visited the clinic three times, and each time they were told that both partners are normal, but they do not understand why they are still infertile. They have taken medication and monitored their cycle, but without success, and have had test results. We kindly request advice: why do both partners have normal examination results, showing no abnormalities, yet still suffer from infertility and subfertility? Why is conception not achieved despite monitoring the cycle for pregnancy?", "zh": "夫妻结婚三年。已检查三次，每次都被告知夫妻双方均正常，但不明白为何仍不育，曾服用药物并监测排卵，但效果不佳且未见成效。恳请咨询，为何夫妻双方检查结果均正常、无任何疾病，却仍不育，且通过监测排卵尝试怀孕仍未成功。"}, {"id": 595, "vi": "Bé gái lớp 4 xuất hiện kinh nguyệt có nên tiêm hormon kìm hãm dậy thì sớm ở trẻ không?", "en": "Should a 4-year-old girl who has started menstruation receive a hormone to suppress premature puberty in children?", "zh": "4年级的女孩出现月经，是否应该给儿童注射抑制性激素以治疗早熟？"}, {"id": 596, "vi": "Em bị mòn cổ chân răng và sâu răng hiện tại bị ê buốt", "en": "My child has worn-down tooth enamel and tooth decay, and currently experiences tooth sensitivity.", "zh": "我有牙槽骨吸收和蛀牙，目前有刺痛感"}, {"id": 597, "vi": "Khi mang thai thì khí hư như thế nào là bình thường ạ?", "en": "What should normal vaginal discharge be like during pregnancy?", "zh": "怀孕期间白带正常应该是怎样的呢？"}, {"id": 598, "vi": "Chỉ tiêm mũi đầu vắc-xin 3 trong 1 đã mang thai có ảnh hưởng gì không?", "en": "Is it safe to receive the first dose of the 3-in-1 vaccine during pregnancy?", "zh": "只接种了三联疫苗第一针是否会影响怀孕？"}, {"id": 599, "vi": "Em có đi khám sức khỏe tổng quát ở ngày 24/10. Mục khám phụ khoa bác sĩ có kết luận lộ tuyến quanh lỗ cổ tử cung. Vậy cho em hỏi cần điều trị như thế nào, và đơn thuốc ra sao ạ?", "en": "I had a general health check-up on October 24. The gynecology examination concluded with a finding of ectopic cervical canal. Thus, I would like to ask: what treatment is needed, and what medication should be prescribed?", "zh": "我于10月24日进行了全身健康检查，妇科医生检查发现宫颈口周围有糜烂。我想问一下，这种情况需要如何治疗，开什么药？"}, {"id": 600, "vi": "Đau bàn chân trong nhiều năm dấu hiệu của bệnh gì?", "en": "What disease are chronic foot pains a sign of?", "zh": "多年足部疼痛是什么疾病的症状？"}, {"id": 601, "vi": "Tuần trước tôi đi siêu âm thì nói não thất bên thai nhi là 6mm. Hôm nay tôi đi siêu âm thì nói thai của tôi được 29 tuần 4 ngày đo được não thất bên là 8mm, hố sau 8mm. Vậy cho tôi hỏi mức độ não thất bên tăng lên như vậy có ảnh hưởng như thế nào ạ? Và tôi cần làm gì không ạ?", "en": "Last week I had an ultrasound which stated the fetal brain ventricle was 6mm. Today I had another ultrasound which stated my fetus is 29 weeks and 4 days, with a brain ventricle measurement of 8mm and a posterior fossa measurement of 8mm. So I would like to ask, does such an increase in ventricular size have any impact? And what should I do?", "zh": "上周我做超声检查时，医生说胎儿的脑室侧为6毫米。今天我再次做超声检查，胎儿为29周4天，脑室侧测得为8毫米，后颅窝为8毫米。我想问一下，脑室侧这样明显增加，会有什么影响吗？我需要做什么吗？"}, {"id": 602, "vi": "Điều trị viêm da tiết bã như thế nào hiệu quả?", "en": "How to effectively treat seborrheic dermatitis?", "zh": "如何有效治疗脂溢性皮炎？"}, {"id": 603, "vi": "Chỉ số Ferritin tăng cao 742 có nguy hiểm không?", "en": "Is a ferritin level of 742 dangerous?", "zh": "铁蛋白指标升高至742有危险吗？"}, {"id": 604, "vi": "U xơ thần kinh có điều trị bằng cách chỉnh sửa gen không?", "en": "Is nerve fibrosis treated by gene editing?", "zh": "神经瘤能否通过基因编辑进行治疗？"}, {"id": 605, "vi": "Ông em bị ung thư. Trong quá trình chăm sóc, khi ông đi đại tiện em có vô tình ngửi mùi phân thì có khả năng bị mắc bệnh liên quan không ạ?", "en": "My uncle has cancer. During the care process, when my uncle has a bowel movement, I accidentally smell the stool. Is there a possibility that I might get a related disease?", "zh": "我叔叔得了癌症。在照顾他的过程中，当我看他上厕所时，无意中闻到了粪便的气味，我有没有可能因此感染相关疾病呢？"}, {"id": 606, "vi": "Người tiểu đường tuýp 2 thường xuyên xuất hiện cảm giác mệt mỏi.", "en": "People with type 2 diabetes often experience a feeling of fatigue.", "zh": "2型糖尿病患者通常会经常感到疲劳。"}, {"id": 607, "vi": "Tiêm viêm gan B ở Nhật Bản, chuyển sang tiêm tại Việt Nam có được không?", "en": "Can I switch from injecting hepatitis B vaccine in Japan to injecting it in Vietnam?", "zh": "在日本接种乙肝疫苗后，转而在越南接种可以吗？"}, {"id": 608, "vi": "Khi điều trị đái tháo đường, chỉ số glucose và hba1c về ngưỡng bình thường thì có được dừng thuốc không? Nếu được dừng thuốc thì bao lâu kiểm tra lại. Nếu không được dừng thuốc thì tiếp tục duy trì uống thuốc đang dùng hay phải thay đổi thuốc với liều nhẹ hơn?", "en": "When treating diabetes, can medication be stopped when glucose and HbA1c levels reach normal thresholds? If medication can be stopped, how often should it be rechecked? If medication cannot be stopped, should the currently used medication be continued or changed to a lower dose?", "zh": "在治疗糖尿病时，当血糖和HbA1c达到正常阈值时，是否可以停药？如果可以停药，需要多久复查一次？如果不能停药，是继续服用目前的药物，还是需要改为剂量更小的药物？"}, {"id": 609, "vi": "Em có virut viêm gan B đang ở thể tĩnh thì cần chế độ ăn uống, sinh hoạt như thế nào thì tốt ạ? Và em có cần uống thêm thuốc bổ gan không? Dùng loại gì thì tốt ?", "en": "I have hepatitis B virus in the inactive phase. What should my diet and lifestyle be like? And do I need to take additional liver-protective medication? If so, which type is best?", "zh": "我目前患有乙型肝炎病毒，处于静止期，应该怎样饮食和生活？还需要额外服用护肝药吗？用哪种比较好？"}, {"id": 610, "vi": "Đau chân, teo cơ là dấu hiệu của bệnh lý gì? Điều trị thế nào?", "en": "What disease causes leg pain and muscle atrophy? How to treat it?", "zh": "脚痛、肌肉萎缩是哪些疾病的症状？如何治疗？"}, {"id": 611, "vi": "Em có 1 đứa em gái 16 tuổi, trong người thấy bình thường không có gì ngoài việc đi tiểu buốt. Một vài nguyên nhân dẫn đến tiểu buốt và cách phòng tránh.", "en": "I have a 16-year-old sister, who is otherwise healthy except for a burning sensation when urinating. Several causes of burning sensation during urination and ways to prevent it.", "zh": "我有一个16岁的妹妹，她身体状况正常，唯一的症状就是小便刺痛。导致小便刺痛的几种原因以及如何预防。"}, {"id": 612, "vi": "Chú tôi 54 tuổi bị nhiễm vi rút HIV 5 năm rồi và đang uống thuốc điều trị, sức khỏe chú bình thường, thỉnh thoảng có hay đau bao tử và huyết áp thấp. Chú tôi có thể chích vắc xin AstraZeneca được không? Nếu được thì chú chích tại điểm chích tập trung hay phải vào viện ạ?", "en": "My uncle, 54 years old, has been infected with HIV for 5 years and is currently on antiretroviral therapy. His health is generally good, but he occasionally experiences stomach pain and low blood pressure. Can my uncle receive the AstraZeneca vaccine? If so, should he get vaccinated at a centralized injection site or should he go to the hospital?", "zh": "我伯伯54岁，已有5年HIV病毒感染史，目前正在服用抗病毒治疗，身体状况良好，偶尔会感到胃痛和低血压。请问他可以接种阿斯利康疫苗吗？如果可以，是在社区接种点接种还是需要去医院接种？"}, {"id": 613, "vi": "Hay ngạt mũi, dịch nhiều chảy xuống họng, ngứa và ù tai kèm đau nhức vai gáy có phải do viêm xoang gây ra không?", "en": "Is nasal congestion, excessive mucus flowing down to the throat, itching and ear fullness accompanied by shoulder and neck pain due to sinusitis?", "zh": "是否有鼻塞、大量分泌物流入咽喉、瘙痒和耳闷伴肩颈疼痛是由于鼻窦炎引起的？"}, {"id": 614, "vi": "Chỉ số ALT là 80 U/L liệu có nguy hiểm?", "en": "Is an ALT level of 80 U/L dangerous?", "zh": "ALT 指标是 80 U/L，是否危险？"}, {"id": 615, "vi": "Phương pháp điều trị vẹo vách ngăn mũi không cần phẫu thuật", "en": "Non-surgical treatment methods for nasal septal deviation", "zh": "非手术治疗鼻中隔偏曲的方法"}, {"id": 616, "vi": "Bị viêm nhẹ nhưng mãi vẫn không thấy ngừng tiết dịch màu xanh liệu có thành nặng không", "en": "Having mild inflammation but still not seeing cessation of green discharge, could it progress to severe?", "zh": "轻微炎症但一直未见停止分泌绿色液体，是否会发展为严重？"}, {"id": 617, "vi": "Trên môi em xuất hiện hạt trắng nhỏ nhiều là bị gì ạ! Em thấy tăng dần theo năm!", "en": "White small bumps appear on the lip; what is this? I have noticed them increasing over the years!", "zh": "我脸上出现很多小白色颗粒，而且逐年增多！"}, {"id": 618, "vi": "Em năm nay 24t, hay bị đi tiểu nhiều mặc dù ít dùng chất kích thích như rượu, bia, cafe. Đồng thời em cũng bị xuất tinh sớm. Xin bác sĩ tư vấn giúp về tình trạng của em ạ?", "en": "I am 24 years old and frequently experience frequent urination, although I do not consume much stimulants such as alcohol, beer, or coffee. At the same time, I also suffer from premature ejaculation. I would like to consult the doctor regarding my current condition.", "zh": "我今年24岁，经常出现多尿的情况，尽管我很少摄入酒精、啤酒和咖啡。同时我也患有早泄。恳请医生为我的这种情况提供咨询。"}, {"id": 619, "vi": "Em bị ngứa đầu và gầu rất nhiều, bây giờ còn ngứa cả mặt và có những vết đỏ", "en": "My daughter has had a lot of itching on her head and scalp, and now she has itching all over her face and some red patches.", "zh": "我头痛、头皮瘙痒很多，现在脸上也发痒，而且有红斑。"}, {"id": 620, "vi": "Em nội soi bị viêm phù nề và xuất huyết hang môn vị nhưng không hiểu sao trên cơ thể em lại nóng như lửa đốt, có khi nóng ở chân, có khi ở mặt, có khi ở tay và bất cứ chỗ nào trên cơ thể! Em không biết mình bị gì!", "en": "My endoscopy showed mucosal edema and hemorrhage in the gastric antrum, but I don't understand why my body feels so hot as if being burned, sometimes in the feet, sometimes on the face, sometimes on the hands, and sometimes anywhere on my body! I don't know what's wrong with me!", "zh": "我做了胃镜，发现幽门部位有水肿和出血，但我不明白为什么身体会像被火烧一样发热，有时是脚发热，有时是脸发热，有时是手发热，有时是身体任何部位都发热！我不知道自己得了什么病！"}, {"id": 621, "vi": "Điều trị viêm mũi họng tái lại như thế nào?", "en": "How to treat recurrent pharyngitis?", "zh": "如何治疗反复的咽炎？"}, {"id": 622, "vi": "Bệnh cao huyết áp có thể điều trị dứt điểm không?", "en": "Can hypertension be cured completely?", "zh": "高血压能否根治？"}, {"id": 623, "vi": "Em bị viêm phế quản cấp và viêm a-mi-đan, em uống kết hợp các loại thuốc của cả 2 bệnh này có ảnh hưởng gì không ạ? Thuốc bicebid 100 Vinacode Acc 200 My para 509 Curam tab 1000mg Incepdazol 250.", "en": "My child has acute bronchitis and pharyngitis. Can I give combination of medicines for both of these diseases? Bicebid 100 Vinacode, Acc 200, My para 509, Curam tab 1000mg, Incepdazol 250.", "zh": "我患有急性支气管炎和扁桃体炎，同时服用这两种疾病的药物会有影响吗？药物：Bicebid 100 Vinacode、Acc 200、My para 509、Curam 1000mg、Incepdazol 250。"}, {"id": 624, "vi": "Bệnh tăng động giảm chú ý ở trẻ trẻ có thể tập trung làm được 1 việc gì đó không?", "en": "Can children with attention deficit hyperactivity disorder focus on doing one thing?", "zh": "多动注意力缺陷儿童能否集中注意力做一件事？"}, {"id": 625, "vi": "Nên làm gì khi đang điều trị suy giáp phát hiện có thai?", "en": "What should be done when treating hypothyroidism and a pregnancy is detected?", "zh": "在治疗甲减期间发现怀孕该怎么办？"}, {"id": 626, "vi": "Trẻ bị rối loạn tiêu hóa tái đi tái lại nguyên nhân do đâu?", "en": "What is the cause of child with recurrent digestive disorders?", "zh": "儿童反复出现消化紊乱的原因是什么？"}, {"id": 627, "vi": "Chu kỳ kinh cuối là ngày 27/11/2019 dương lịch, bác sĩ dự kiến sinh ngày 20/8/2020. Vậy có đúng không ạ?", "en": "The last menstrual period was on 27/11/2019, and the doctor estimated the due date as 20/08/2020. Is this correct, sir?", "zh": "末次月经是2019年11月27日，医生预计预产期是2020年8月20日，对吗？"}, {"id": 628, "vi": "Thưa bác sĩ chỉ số beta của em ngày 14 sau chuyển phôi là 2954 mà siêu âm chỉ thấy hình ảnh 1 túi thai. Chỉ số như vậy có bình thường không thưa bác sĩ", "en": "Dear doctor, my beta value on day 14 after embryo transfer is 2954, but the ultrasound only shows one gestational sac. Is this value normal, dear doctor?", "zh": "医生您好，我移植后第14天的β值是2954，但超声检查只看到一个孕囊。这样的β值正常吗？医生您好"}, {"id": 629, "vi": "Các hạt nhỏ xung quanh quầng vú bị sưng đau và có hạt dưới da thì có phải tắc tuyến sữa hay phải điều trị kháng sinh không ạ? Có thể tự khắc phục bằng việc cho con bú thường xuyên không ạ?", "en": "Small lumps around the nipple area that are swollen, painful, and have lumps under the skin—could this be milk duct blockage or do I need antibiotic treatment? Can this be resolved by regularly breastfeeding?", "zh": "乳房周围出现小肿块，伴有疼痛和皮下结节，是乳腺堵塞还是需要使用抗生素治疗？是否可以通过频繁哺乳自行恢复？"}, {"id": 630, "vi": "Tự mua và ngừng kháng sinh chữa cảm cúm có bị kháng kháng sinh không?", "en": "Can self-purchasing and stopping antibiotics for the common cold lead to antibiotic resistance?", "zh": "自行购买并停用抗生素治疗普通感冒是否会引发抗生素耐药性？"}, {"id": 631, "vi": "Trẻ bị nổi mụn trắng trên đùi và dưới rốn dấu hiệu bệnh gì?", "en": "What condition does a child having white pimples on the thighs and under the navel indicate?", "zh": "儿童在大腿和腹股沟出现白色皮疹，是什么病的表现？"}, {"id": 632, "vi": "Khạc ra máu dạng sợi lẫn đờm trắng kèm đau thắt vùng lưng có phải bị viêm phổi không?", "en": "Is coughing up blood strands mixed with white sputum accompanied by back pain indicative of pneumonia?", "zh": "是否咳出丝状血痰伴白色痰液和背部绞痛就是肺炎？"}, {"id": 633, "vi": "Em muốn biết chỉ số AMH bao nhiêu là bình thường? Của em là 1,23 thì có phải là quá thấp không? Có thể mang thai tự nhiên được không? Làm sao để khắc phục? Có thuốc nào có thể làm tăng chỉ số AMH lên không?", "en": "I would like to know what AMH level is considered normal? My AMH level is 1.23—does that mean it is too low? Can I conceive naturally? How can I improve it? Are there any medications that can increase AMH levels?", "zh": "我想知道AMH的正常值是多少？我的AMH值是1.23，这是否偏低？是否可以自然怀孕？应该如何改善？有没有药物可以提高AMH值？"}, {"id": 634, "vi": "Không uống rượu, bia nhưng chỉ số GGT tăng do đâu?", "en": "Not drinking alcohol or beer, but GGT level is elevated - what could be the cause?", "zh": "不饮酒、不喝啤酒，但GGT值升高是由于什么原因？"}, {"id": 635, "vi": "Tôi đang có bầu tháng thứ 7, mấy hôm nay khớp háng đau nhức buốt dữ dội, cử động bị giật rất đau. Tôi muốn nhờ tư vấn làm sao tôi biết được cụ thể bệnh gì?", "en": "I am in my seventh month of pregnancy, and lately I have been experiencing severe pain, aching, and sharp pain in my hip joints, with movements causing sudden, very painful jerks. I would like to ask for advice on how I can specifically determine what illness I have?", "zh": "我目前怀孕第七个月，最近几天髋关节剧烈疼痛、刺痛，活动时会突然抽痛，我想咨询一下如何明确具体是哪种疾病？"}, {"id": 636, "vi": "Bố tôi bị đau khớp ở chân, không có sưng tấy chỉ thấy buốt bên trong xương.Tháng trước thì bị bên chân phải, giờ lại đau sang chân trái, đau đặc biệt ở khớp gối và chỗ mu bàn chân.Bác sĩ cho hỏi liệu bố tôi bị bệnh gì.Làm các xét nghiệm gì để chẩn đoán xác định.", "en": "My father has joint pain in his foot, without swelling, only with internal bone pain. Last month it was on the right leg, now it has shifted to the left leg, with particularly severe pain at the knee joint and the ball of the foot. Doctor, could you please advise what disease my father might have? What tests should be done to confirm the diagnosis?", "zh": "我父亲出现脚部关节疼痛，没有肿胀，只是感到骨内刺痛。上个月是右脚，现在疼痛转移到左脚，尤其是膝关节和脚背内侧疼痛。医生请问一下，我父亲可能是什么病？需要做哪些检查来明确诊断？"}, {"id": 637, "vi": "Cho em hỏi cu nhóc của em sửa công thức pha ấm bé không ti bình nhưng sữa để lạnh trong ngăn mát tủ lạnh thì bé ti vậy có như vậy mỗi khi be ti sữa lạnh di không có tốt với bổ sung dinh dưỡng như sữa ấm không.", "en": "May I ask whether my baby, who does not drink from the bottle when warm milk is prepared but drinks when cold milk stored in the refrigerator is given, has this behavior every time the baby drinks cold milk, and is it not good for nutrient supplementation compared to warm milk?", "zh": "我想问一下，我家宝宝喝温奶时不会喝，但把奶放冰箱冷藏后就能喝，这种情况每次喝冷奶是否对营养补充不利，和温奶相比是否有区别？"}, {"id": 638, "vi": "Năm nay cháu 17 tuổi. Thời gian gần đây cháu thấy một bên ngực của mình nổi cục u to, cứng hơn hẳn bên còn lại nhưng không thấy đau hay có triệu chứng gì khác. Cháu có bị làm sao không ạ?", "en": "This year I am 17 years old. Recently, I have noticed a lump on one side of my chest that is larger and much harder than the other side, but I do not feel any pain or any other symptoms. Am I in any trouble?", "zh": "今年我17岁。最近我发现一侧胸部出现一个肿块，比另一侧硬得多，但没有疼痛或其他症状。我是不是有问题啊？"}, {"id": 639, "vi": "Nhiễm nấm đường tiểu ở trẻ nhỏ liệu có ảnh hưởng đến khả năng sinh sản?", "en": "Does urinary fungal infection in children affect reproductive ability?", "zh": "小儿尿路真菌感染是否会影响生育能力？"}, {"id": 640, "vi": "Cần bổ sung acid folic trước khi mang thai bao lâu? Uống liều bao nhiêu ạ?", "en": "How long before pregnancy should folic acid be supplemented? What dose should be taken?", "zh": "怀孕前需要补充叶酸多久？服用多少剂量呢？"}, {"id": 641, "vi": "Người già uống vitamin E liều cao có sao không?", "en": "Is it harmful for the elderly to take high-dose vitamin E?", "zh": "老年人服用高剂量维生素E会有问题吗？"}, {"id": 642, "vi": "Sau điều trị giang mai 2 năm, chỉ số RPR âm tính, TPPA dương tính đã khỏi bệnh chưa?", "en": "After 2 years of syphilis treatment, RPR negative, TPPA positive, has the disease been cured?", "zh": "治疗梅毒2年后，RPR阴性，TPPA阳性，是否已痊愈？"}, {"id": 643, "vi": "Cháu bị u tuyến yên cháu đã dùng thuốc hạ prolactin được hơn 1 tháng. Cháu chậm kinh hơn 1 tháng rồi, hôm nay cháu bị ra 1 ít máu đông màu đỏ và đi tiểu xong bị buốt. Cháu có bị sao không ạ?", "en": "I have a pituitary adenoma. I have been taking prolactin-lowering medication for more than one month. I have had a delayed menstruation for over one month, and today I noticed a little clotted blood with red color, and after urination I feel pain. Am I experiencing any problems?", "zh": "我患有垂体瘤，已经使用降催乳素药物超过一个月。月经延迟超过一个月，今天出现少量红色血块，排尿后有尿痛感。我这种情况会有什么问题吗？"}, {"id": 644, "vi": "Cần làm những xét nghiệm sàng lọc và tiêm phòng như thế nào trước khi mang thai?", "en": "What screening tests and vaccinations are needed before pregnancy?", "zh": "怀孕前需要进行哪些筛查和接种？"}, {"id": 645, "vi": "Lưỡi em xuất hiện nhiều gai có màu giống lưỡi, không bị đau rát và vướng víu là bị sao ạ? Gần đây em có bị stress và ăn uống nhiều đồ nóng như mận và xoài ạ.", "en": "My tongue has many spines with a color similar to the tongue, without pain or discomfort. Recently, I have been under stress and eating a lot of hot foods such as plums and papayas.", "zh": "我舌头出现很多类似舌苔的刺状物，没有疼痛或不适感，这是什么问题啊？最近我压力大，经常吃一些辛辣食物，比如李子和猕猴桃。"}, {"id": 646, "vi": "Em 27 tuổi bị THA vô căn độ 1, rối loạn lipid máu. Huyết áp em thỉnh thoảng lên 130-140/90-100. Bác sỉ cho hỏi có cách nào ổn định huyết áp mà không cần dùng thuốc cả đời không? Vì em chưa có sinh em bé mà bị THA sợ sau này có thai rồi ảnh hưởng gì không?", "en": "I am 27 years old and have stage 1 essential hypertension with lipid disorder. My blood pressure occasionally reaches 130-140/90-100. The doctor asks if there is any way to stabilize blood pressure without needing lifelong medication? Since I have not given birth yet, I am worried that having hypertension might affect me if I get pregnant later.", "zh": "我27岁，患有原发性高血压1级，血脂异常。我的血压偶尔会升高到130-140/90-100。医生想咨询一下，有没有不需终身服药就能稳定血压的方法？因为我还没有生育，担心以后怀孕会有什么影响？"}, {"id": 647, "vi": "Phương pháp điều trị sùi mào gà ở dương vật tại nhà?", "en": "What is the treatment method for genital warts at home?", "zh": "阴茎人乳头瘤病毒感染的居家治疗方法？"}, {"id": 648, "vi": "Tôi có vài viên sỏi mật, viên lớn nhất là 9,5 mm. Xin bác sĩ cho biết: 1. Nguyên nhân hình thành sỏi mật và tác hại của sỏi mật đối với sức khỏe? 2. Phương pháp làm tan sỏi mật tự nhiên? 3. Cách phòng tránh sỏi mật?", "en": "I have several gallstones, the largest being 9.5 mm. I would like to ask the doctor: 1. What are the causes of gallstone formation and the health hazards of gallstones? 2. Natural methods to dissolve gallstones? 3. Ways to prevent gallstones?", "zh": "我有几颗胆结石，最大的是9.5毫米。请问医生：1. 胆结石形成的病因及其对健康的危害？2. 能否通过自然方法溶解胆结石？3. 如何预防胆结石？"}, {"id": 649, "vi": "Kết quả xét nghiệm sàng lọc thai nhi triệu chứng patau 1:138092 là cao hay thấp", "en": "Is the screening test result for Down syndrome 1:138092 high or low?", "zh": "胎儿筛查唐氏综合征检测结果1:138092是高还是低"}, {"id": 650, "vi": "Em vừa đi khám phụ khoa sau sinh gần 2 tháng thì bị kết luận sa thành âm đạo chứ không phải sa tử cung. Rồi họ nói để rứa sau tự lành và dặn về co cơ âm đạo nhưng em co cơ mấy lần thì cứ viêm tiết niệu nên thôi. Sa thành âm đạo có nguy hiểm như sa tử cung không và để vậy sau bao lâu lành.", "en": "I just visited gynecology examination two months after delivery and was diagnosed with vaginal prolapse rather than uterine prolapse. Then they said to wait for it to heal on its own and advised me to do pelvic floor exercises, but every time I do the exercises, I get urinary tract infection, so I stopped. Is vaginal prolapse as dangerous as uterine prolapse, and how long does it take to heal if left untreated?", "zh": "我产后近两个月去妇科检查，被诊断为盆腔器官脱垂至阴道，而不是子宫脱垂。医生说会自行恢复，嘱咐我做阴道收缩锻炼，但我做了几次后总是出现尿路感染，就放弃了。阴道壁脱垂是否和子宫脱垂一样危险？这样放着多久能自愈？"}, {"id": 651, "vi": "Vào buổi tối em thường có triệu chứng sôi bụng , bùng kêu ùng ục khó chịu , thỉnh thoảng còn đau râm ran . Bác sĩ tư vấn giúp em ạ", "en": "In the evening, I usually have the symptom of abdominal distension, gurgling noises, and discomfort, occasionally accompanied by dull aching pain. Doctor, could you please advise me?", "zh": "晚上我常常有腹部灼热感，伴有咕噜作响、不适感，偶尔还会感到隐痛。医生能给我一些建议吗？"}, {"id": 652, "vi": "Cho em hỏi là đi đặt vòng như thế nào, có đau không và ra máu bao nhiêu ngày?", "en": "May I ask how to insert an IUD, will it hurt, and for how many days will there be bleeding?", "zh": "请问放环是怎么操作的？会疼吗？出血持续几天？"}, {"id": 653, "vi": "Tôi năm nay 45 tuổi. Tôi lập gia đình muộn nên mới bắt đầu quan hệ cách đây hai năm. Tôi có đi khám và có kết quả các chỉ số hormone, tử cung và ống dẫn trứng bình thường. Duy có lượng trứng dự trữ còn thấp AMH 0,6 ng/mL. Chồng tôi tinh trùng bình thường. Tôi muốn hỏi xem chúng tôi nên làm gì để có thể có con bằng trứng của tôi.", "en": "I am 45 years old. I had a late marriage, so I only started having intercourse two years ago. I have undergone examination and the hormone levels, uterus, and fallopian tubes are normal. The only issue is the low ovarian reserve with AMH 0.6 ng/mL. My husband's semen analysis is normal. I would like to know what we should do to have a child using my eggs.", "zh": "我今年45岁。我结婚较晚，因此两年前才开始性生活。我曾去医院检查，激素水平、子宫和输卵管检查结果均正常，唯独卵泡储备量偏低，AMH为0.6 ng/mL。我的丈夫精液检查正常。我想咨询一下，我们该如何才能通过我的卵子实现生育？"}, {"id": 654, "vi": "Vàng mắt, đau hông là dấu hiệu bệnh lý gì?", "en": "Yellowing of the eyes, pain in the hip - what is the underlying disease?", "zh": "黄疸、腰痛是哪些疾病的表现？"}, {"id": 655, "vi": "Kết quả đo điện não đồ giấc ngủ nói lên điều gì?", "en": "What does the sleep EEG test results indicate?", "zh": "睡眠脑电图检查结果说明了什么？"}, {"id": 656, "vi": "Xăm lông mày chung với người bị viêm gan B có sao không?", "en": "Is it safe to get eyebrow threading if you have hepatitis B?", "zh": "眉毛纹身是否会传染乙肝患者？"}, {"id": 657, "vi": "Bệnh thoái hóa chất trắng nên dùng thuốc gì?", "en": "What medication should be used for white substance degeneration disease?", "zh": "白质退化症该用什么药？"}, {"id": 658, "vi": "Đâu là nguyên nhân trẻ có hiện tượng giật rùng mình?", "en": "What is the cause of children having a startle reaction?", "zh": "什么会导致儿童出现抽搐？"}, {"id": 659, "vi": "Đau cơ do HIV có diễn ra độc lập hay kèm theo triệu chứng khác ạ?", "en": "Does HIV-related muscle pain occur independently or accompanied by other symptoms?", "zh": "HIV引起的肌肉疼痛是独立发生还是伴随其他症状？"}, {"id": 660, "vi": "Bà bầu 5 tháng móng tay có vết lằn ngang có đáng lo?", "en": "Is it concerning for a 5-month pregnant woman to have horizontal lines on her nails?", "zh": "孕妇5个月指甲出现横向条纹是否需要担心？"}, {"id": 661, "vi": "Có khối u ở chân răng vừa trám là do đâu?", "en": "What is the cause of a tumor at the root of a tooth that has been filled?", "zh": "磨牙处有肿块是什么原因？"}, {"id": 662, "vi": "Em bị lupus ban đỏ hẹ thống biến chứng ê bị tăng áp động mạch phổi và bị xơ gan điều trị ở bệnh viện chợ rẫy. Bác sĩ khoa tim cho em uống bosentan điều trị tăng áp phổi. Em bị xơ gan có nên uống hay không", "en": "My daughter has systemic lupus erythematosus with pulmonary hypertension and liver cirrhosis, treated at Chợ Rẫy Hospital. The cardiologist prescribed bosentan for pulmonary hypertension. Since my daughter has liver cirrhosis, should she take it?", "zh": "我患有系统性红斑狼疮，出现肺动脉高压和肝硬化，目前在车公庄医院治疗。心脏科医生给我开了波生坦（bosentan）来治疗肺动脉高压。我有肝硬化，是否可以服用此药？"}, {"id": 663, "vi": "Tiêm kháng sinh nhiều có ảnh hưởng gì đến sức khỏe và sự phát triển của bé không?", "en": "What are the effects of administering antibiotics frequently on the health and development of the child?", "zh": "频繁注射抗生素对婴儿的健康和发育有什么影响？"}, {"id": 664, "vi": "Đau thắt lưng phần dưới cơ quan sinh dục nam là dấu hiệu của bệnh gì?", "en": "Lower back pain in the male genital area is a sign of what disease?", "zh": "男性生殖器下方腰痛是哪种疾病的症状？"}, {"id": 665, "vi": "Lạc nội mạc tử cung có làm khả năng sinh sản bị hạn chế không?", "en": "Does endometrial adenomyosis limit fertility?", "zh": "子宫内膜异位症是否会导致生育能力受限？"}, {"id": 666, "vi": "Đã phẫu thuật lõm ngực được 3 năm muốn hoãn lại có ảnh hưởng đến sức khỏe không?", "en": "Has undergone 3 years of chest surgery, wanting to delay the procedure, will it affect health?", "zh": "胸骨成形术术后3年，是否可以推迟手术？"}, {"id": 667, "vi": "Có nên cho trẻ bú sữa của mẹ khác không?", "en": "Should a child be fed on another mother's milk?", "zh": "是否可以让婴儿喝其他母亲的奶？"}, {"id": 668, "vi": "Kết quả xét nghiệm HBsAg 0,45 có nghĩa là thế nào ạ?", "en": "What does the test result of HBsAg 0.45 mean?", "zh": "乙肝表面抗原检测结果为0.45意味着什么？"}, {"id": 669, "vi": "Hôm 5-5 em đi siêu âm thai được 6 tuần và có tim thai. Thi thoảng em hay bị chóng mặt và buồn nôn, còn hay bị đau bụng 1 lát như kiểu đến tháng. Vậy có ảnh hưởng gì đến thai không.", "en": "On May 5th, the baby was 6 weeks old and had a fetal heart detected by ultrasound. I occasionally experience dizziness and nausea, and also have abdominal pain for a short time, similar to what I would feel in the first month. Will this affect the baby?", "zh": "5月5日，宝宝6周做B超检查，有胎心。有时会感到头晕和恶心，还经常出现类似来月经的腹痛。这种情况对胎儿有影响吗？"}, {"id": 670, "vi": "Các chỉ số hbv-dna 231 iu/ml và 1344 copies/ml cho biết gì", "en": "What do the HBV-DNA levels of 231 IU/ml and 1344 copies/ml indicate?", "zh": "乙肝DNA检测值为231 IU/ml和1344 copies/ml意味着什么"}, {"id": 671, "vi": "Bệnh quai bị có tái phát không?", "en": "Can mumps recur?", "zh": "腮腺炎会复发吗？"}, {"id": 672, "vi": "Làm thế nào để em có thể trị dứt điểm bệnh hạ canxi và nhu cầu dinh dưỡng như thế nào cho phù hợp", "en": "How can I treat hypocalcemia completely and what is the appropriate nutritional supplementation?", "zh": "如何才能彻底治愈低钙症，并如何提供合适的营养补充？"}, {"id": 673, "vi": "Trẻ 4 tuổi chưa tiêm vắc xin viêm gan B có tiêm được nữa không?", "en": "Can a 4-year-old child who has not received hepatitis B vaccine still get the vaccine?", "zh": "4岁未接种乙肝疫苗的孩子还能再接种吗？"}, {"id": 674, "vi": "Viêm gan B có cần chữa trị ngay không?", "en": "Does hepatitis B need immediate treatment?", "zh": "乙型肝炎需要立即治疗吗？"}, {"id": 675, "vi": "Cho em hỏi, tác dụng phụ của vắc-xin 3 trong 1 (sởi + quai bị + rubella) có gây tác dụng phụ gây viêm buồng trứng và vô sinh ở nữ không ạ?", "en": "May I ask, do the side effects of the 3-in-1 vaccine (measles + mumps + rubella) cause ovarian inflammation and infertility in women?", "zh": "请问，三联疫苗（麻疹+腮腺炎+风疹）的副作用会不会导致女性输卵管炎和不孕？"}, {"id": 676, "vi": "Em có bầu được 5.5 tháng, mà trước đây em không bị gì hết. Nhưng hai tuần nay em thường bị ợ chua và cảm giác như mắc thức ở cổ. Đôi khi còn bị ợ chua kèm theo thức ăn lên trên miệng. Em đổi loại thuốc sắt mà vẫn không được. Bác sĩ tư vấn cho em nguyên nhân và cách chữa với ạ.", "en": "I am 5.5 months pregnant, and previously I had no issues. However, for the past two weeks, I have frequently experienced acid reflux and a sensation of something stuck in my throat. Sometimes, I also experience acid reflux accompanied by food coming up into my mouth. I have changed my iron supplement, but it hasn't helped. Doctor, could you please advise me on the cause and treatment?", "zh": "我怀孕5.5个月，之前一直没任何问题。但最近两周经常感到胃酸反流，感觉食物堵在喉咙处。有时还会出现胃酸反流伴随食物上涌到嘴里。我换了铁剂药物，但效果不佳。医生能帮我分析一下原因，并给出治疗建议吗？"}, {"id": 677, "vi": "Dấu hiệu của bệnh tim gồm những gì", "en": "What are the signs of heart disease?", "zh": "心脏疾病的症状包括哪些"}, {"id": 678, "vi": "Nữ 17 tuổi đi ngoài ra máu có nguy cơ mắc bệnh gì?", "en": "A 17-year-old female with rectal bleeding—what diseases is she at risk for?", "zh": "17岁女性排便带血，可能患有哪种疾病？"}, {"id": 679, "vi": "Cho em hỏi với ạ. Em đi thử beta và siêu âm cùng ngày. Xem kết quả giúp em xem có vấn đề gì không với ạ.", "en": "May I ask you a question? I took a beta test and ultrasound on the same day. Please help me review the results to see if there is any issue.", "zh": "请问一下，我今天做了beta检测和超声检查，帮忙看一下检查结果有没有问题吗？"}, {"id": 680, "vi": "Em có làm xét nghiệm beta hcg. Và kết quả là 2114 ạ. Em so thì kết quả trong khoảng 4-5 tuần. Bác sĩ cho em hỏi là kết quả này lệch so với thực tế bao nhiêu ngày ạ? Em bị từ ngày 31/3 đến 27/4 em có đi tiêm sởi, quai bị, rubella. Sau hôm đó thì vợ em quan hệ đều xuất trong. Và ngày 29/5 em phát hiện có thai ạ. Bác sĩ cho em hỏi mới tiêm vài ngày mà em đã dính bầu ngay có ảnh hưởng gì nhiều không?", "en": "I have done a beta-hCG test. The result is 2114. I compared it with the range of 4-5 weeks. Doctor, could you please tell me how many days off from the actual date is this result? I got vaccinated against measles, mumps, and rubella from March 31 to April 27. After that, my wife had regular intercourse. And on May 29, I discovered I was pregnant. Doctor, could you please tell me if getting vaccinated just a few days ago has significantly affected my pregnancy?", "zh": "我做了β-hCG检查，结果是2114。我对照了4-5周的参考值。医生请问一下，这个结果比实际天数偏离了多少天？我从3月31日到4月27日打了麻疹、腮腺炎、风疹疫苗。之后妻子一直有性生活，5月29日我发现自己怀孕了。医生请问，打了几天疫苗就怀孕了，会不会有影响？"}, {"id": 681, "vi": "Em sinh mổ nên sau sinh bác sĩ chỉ định cho em tiêm truyền Tinidazol 500mg/100ml và Cefotaxim 1g liên tục trong 5 ngày, mỗi ngày 3 lần. Sau mỗi lần truyền khoảng 1 tiếng em có cho bé bú luôn vì không biết nguy cơ gây ung thư cho bé khi lớn lên của hai loại kháng sinh này. Giờ em rất lo lắng, mong bác sĩ tư vấn dùm phương pháp nào giảm thiểu nguy cơ cho con em không.", "en": "My child was delivered by cesarean section, and after birth, the doctor prescribed me to receive intravenous tinidazole 500mg/100ml and cefotaxime 1g continuously for 5 days, three times daily. After each infusion, I feed my baby immediately because I am unsure about the risk of cancer in my child later in life from these two antibiotics. Now I am very worried and hope the doctor can advise me on a method to minimize the risk for my child.", "zh": "我剖腹产，医生建议我连续注射5天，每天3次，每次注射500mg/100ml的替硝唑和1g的头孢他啶。每次输液后大约1小时我就给婴儿喂奶，因为我不确定这两种抗生素在孩子长大后是否会增加患癌风险。我现在非常担心，希望医生能给我一些建议，帮助我减少对孩子可能的风险。"}, {"id": 682, "vi": "Năm nay em 25 tuổi. Hay bị đau nửa người bên phải từ thắt lưng lên cổ vai gáy. Dù có nằm nghỉ ngơi nhưng vẫn thường xuyên bị đau vai gáy phải. Em bị bệnh gì ạ?", "en": "This year I am 25 years old. I often experience pain on the right side of my body, starting from the lower back up to the neck and shoulder. Even when lying down and resting, I still frequently suffer from right shoulder and neck pain. What disease do I have?", "zh": "今年我25岁。经常感到右侧腰部至颈部、肩部和后颈的疼痛。即使休息躺卧，右侧肩颈疼痛仍经常发作。我得的是什么病？"}, {"id": 683, "vi": "Chồng em xét nghiệm tinh trùng, PH 7.5,thể tích 3.8,tổng số tinh trùng 254, mật độ 67, di động tiến tới 42, không di động 49, tỉ lệ sống 75, hình dạng bình thường 0%. kết luận 84% tinh trùng đầu có nhiều không bào. Vậy chồng em có sinh sản bình thường được không.", "en": "My husband underwent semen analysis: pH 7.5, volume 3.8, total sperm count 254, concentration 67, progressive motility 42, non-progressive motility 49, viability 75%, normal morphology 0%. Conclusion: 84% of sperm heads have numerous vacuoles. Therefore, can my husband have normal fertility?", "zh": "我丈夫做了精液检查，pH值为7.5，体积为3.8，精子总数为254，密度为67，前向运动精子为42，非运动精子为49，存活率75%，形态正常率为0%。结论：84%的精子头部有较多的类泡。请问我丈夫是否能正常生育？"}, {"id": 684, "vi": "Mệt mỏi hay đau vùng bụng mạn sườn trái, đi ngoài thất thường nên điều trị thế nào?", "en": "How should chronic fatigue or left upper abdominal pain and irregular bowel movements be treated?", "zh": "是否有慢性疲劳或左上腹疼痛，排便不规律，应如何治疗？"}, {"id": 685, "vi": "Chu kỳ kinh của em khoảng 31 ngày và khá đều. Vậy ngày rụng trứng là ngày nào để dễ mang thai ạ?", "en": "My menstrual cycle is about 31 days and quite regular. So, on which day does ovulation occur to facilitate pregnancy?", "zh": "我的月经周期大约是31天，而且比较规律。那么排卵日是哪一天才能更容易怀孕呢？"}, {"id": 686, "vi": "Điều trị viêm phế quản như thế nào?", "en": "How to treat bronchitis?", "zh": "支气管炎如何治疗？"}, {"id": 687, "vi": "Các vết zona ở phía tay phải và giữa ngực, còn có vết loét ở môi trên bên trong miệng, đi bệnh viện về và các vết zona đã lành 90% nhưng bác sĩ không kê đơn để điều trị vết loét trong miệng khiến ăn uống gặp khó khăn và đau nhức. Các vết zona đã lành hẳn nhưng nó để lại biến chứng nóng và đau trong tay, tôi có tiền sử bị ung thư tôi tìm hiểu là bệnh này đề kháng yếu nên khó khỏi. Mong bác sĩ tư vấn điều trị.", "en": "Shingles lesions on the right arm and mid-chest, along with a ulceration on the inner upper lip; visited the hospital and the shingles lesions have healed by 90%, but the doctor did not prescribe treatment for the oral ulcer, causing difficulty in eating and pain. The shingles lesions have completely healed but left complications of heat and pain in the arm; I have a history of cancer, and I researched that this condition has weak resistance and is therefore difficult to recover from. I hope the doctor can advise on treatment.", "zh": "右臂和胸骨中部有带状疱疹皮损，口腔内上唇有溃疡。去医院检查后，带状疱疹已愈合90%，但医生未开具治疗口腔溃疡的药物，导致进食困难和疼痛。带状疱疹已完全愈合，但留下了手臂的灼热和疼痛症状。我有癌症病史，查阅资料得知该病对治疗反应差，恢复困难。恳请医生给予治疗建议。"}, {"id": 688, "vi": "Bác sĩ chẩn đoán em bị zona, cho thuốc em uống 2 liều/ngày. Nhưng em uống liên tục được không, 5-6 tiếng/1 liều (trong thuốc có kháng sinh Lincomycin và Aciclovir dạng vừa uống vừa thoa chỗ vết zona)? Thì có mau khỏi không? Uống như vậy thì có ảnh hưởng và có tác dụng phụ gì không? Có đúng không? Ảnh có mủ chụp hồi sáng, ảnh hết mủ là hiện tại vừa mới chụp.", "en": "Doctor diagnosed me with shingles, prescribed me to take 2 doses per day. But can I take it continuously, every 5-6 hours per dose (the medication contains lincomycin and aciclovir, both oral and topical for the shingles lesion)? Will it improve faster? Will this cause any side effects or interactions? Is this correct? There are X-ray images showing pus from early morning, and the current X-ray shows no pus.", "zh": "医生诊断我得了带状疱疹，开了药，每天吃两剂。但我可以连续吃吗？每剂间隔5到6小时服用（药物中含有林可霉素和阿昔洛韦，口服和外涂患处）？这样会好得快吗？这样吃会有影响或副作用吗？这样做对吗？有早晨拍摄的图片，显示有脓液，目前拍摄的图片已无脓液。"}, {"id": 689, "vi": "Gần đây tôi có triệu chứng sưng viêm và nổi hạch ở quy đầu, đây có phải triệu chưng ung thư không?", "en": "Recently I have had symptoms of swelling and inflammation and enlarged lymph nodes at the glans, is this a sign of cancer?", "zh": "最近我出现了冠状沟处的肿胀、发炎和淋巴结肿大，这是否是癌症的症状？"}, {"id": 690, "vi": "Khó chịu do đau sau zona thần kinh điều trị như thế nào?", "en": "How to treat discomfort due to pain after zoster neuralgia?", "zh": "带状疱疹神经痛如何治疗？"}, {"id": 691, "vi": "Hồi lớp 12 chán em đã có dấu hiệu bị hói nhẹ. Lên đại học thì do ăn uống ngủ nghỉ không đúng nên tóc bắt đầu rụng rất nhiều tầm 1 tháng nay rồi. Dạo này em có điều chỉnh lại sinh hoạt cho hợp lý. Cộng thêm là gội đầu bằng Bồ kết và vỏ bưởi nhưng nó vẫn rụng được cái là đỡ hơn nhưng lại ngứa da đầu.Sợi tóc của em rụng thì nó có đốm trắng và trên tóc có những hạt như gàu. Bác sĩ hãy cho em lời khuyên.", "en": "In grade 12, I already started showing mild hair loss. When I entered university, due to improper eating, sleeping, and resting habits, my hair began to fall out heavily about one month ago. Recently, I have adjusted my lifestyle to be more reasonable. Additionally, I have been washing my hair with Bồ kết and pomelo peel, but it still falls out, although it's less than before. However, my scalp now itches. The hair strands that fall out have white spots, and there are small particles like dandruff on my hair. Doctor, please give me some advice.", "zh": "高二时我就开始有轻微脱发的迹象。上大学后由于饮食作息不规律，导致头发最近一个月大量脱落。最近我调整了生活作息，同时用王不留行和橙皮洗头，虽然脱发情况有所缓解，但头皮仍然发痒。我的头发脱落时带有白色斑点，头发上还有类似头屑的颗粒。医生请给我一些建议。"}, {"id": 692, "vi": "Bị lác tái phát có thể phẫu thuật điều trị không?", "en": "Can recurrent strabismus be treated with surgery?", "zh": "反复性斜视能否手术治疗？"}, {"id": 693, "vi": "Hiện tại em và vợ em đang kế hoạch sinh con muộn. Vợ em đang sử dụng thuốc tránh thai hằng ngày drosperin được 3 năm. Em bị dị ứng với bao cao su. Việc uống thuốc drosperin trong thời gian dài có ảnh hưởng đến việc sinh sản sau này không ạ? Nếu có thì còn biện pháp tránh thai nào hiệu quả ngoài việc sử dụng bao cao su không ạ? (Vợ em vẫn chưa sinh em bé).", "en": "Currently, I and my wife are planning for a delayed pregnancy. My wife has been using daily oral contraceptive drospirenone for 3 years. I am allergic to condoms. Does long-term use of drospirenone affect future fertility? If so, are there effective contraceptive methods other than using condoms? (My wife has not given birth yet).", "zh": "目前我和妻子正在计划晚育。妻子已连续三年每天服用左炔诺孕酮避孕药。我对其避孕套有过敏反应。长期服用左炔诺孕酮会对以后的生育能力产生影响吗？如果有，除了使用避孕套外，还有哪些有效的避孕措施吗？（妻子至今尚未生育孩子）。"}, {"id": 694, "vi": "Mỗi lần bị đau răng cổ 2 lại nổi hạch ở cổ, ở hai bên và phía sau đều có hạch? Bác sĩ cho em hỏi hạch đấy có nguy hiểm không ạ?", "en": "Each time I get pain in the upper molars, lymph nodes appear in the neck, on both sides and behind? Doctor, I would like to ask if these lymph nodes are dangerous?", "zh": "每次出现颈部牙齿疼痛，都会在颈部两侧及后方出现淋巴结肿大？医生请问这些淋巴结危险吗？"}, {"id": 695, "vi": "Thuốc Turbuhaler có dùng được cho phụ nữ có thai không và cần lưu ý điều gì?", "en": "Can Turbuhaler be used in pregnant women and what precautions should be taken?", "zh": "泰布haler药物是否可以用于孕妇，需要注意什么？"}, {"id": 696, "vi": "Em mổ thăm dò kẹp 1 bên buồng trứng. Mổ xong có khả năng chửa tự nhiên không ạ? Em bị vô sinh thứ phát.", "en": "My sister underwent exploration surgery with ovarian fixation on one ovary. After the surgery, is there a possibility of natural conception? I have secondary infertility.", "zh": "做了单侧卵巢探查粘连手术。手术后有自然受孕的可能吗？我患有继发性不孕。"}, {"id": 697, "vi": "Cách đây 3 năm, em bị dị ứng mẩn ngứa tôm, cua, cá... uống thuốc tây mới khỏi. Sau đó, em uống thuốc nam thì bớt hẳn. Nhưng hiện tại cơ thể em rất dễ phát nhiệt, lỗ tai hay lùng bùng, da nóng ran do ăn, uống đồ nóng như nước mắm cay, trà gừng hòa tan, vải thiều... Cơ thể em hơi gầy, hay bị đau đầu, choáng mặt và buồn nôn. Em đã từng uống thuốc thiếu máu não, rối loạn tiền đình, viêm xoang... Sắp tới công ty em có đợt chích vắc xin Covid-19 tập trung, em có thể chích trong dịp này không?", "en": "Three years ago, I developed allergic itching rash due to shrimp, crab, fish, etc.; I took Western medicine and recovered. Afterwards, I took traditional Chinese medicine and my symptoms significantly improved. However, currently, my body easily becomes hot, my ears often become blocked, and my skin becomes hot and prickly after eating or drinking hot foods such as spicy fish sauce, dissolved ginger tea, and loquat. My body is slightly underweight and I often experience headaches, dizziness, and nausea. I have previously taken medications for cerebral anemia, vestibular disorder, and sinusitis. Soon, my company will have a concentrated COVID-19 vaccine campaign; can I receive the vaccine during this period?", "zh": "三年前，我曾对虾、蟹、鱼等食物过敏，出现瘙痒风疹，服用西药后才好转。之后服用中药后症状明显减轻。但目前身体很容易发热，耳朵常有耳鸣，食用辛辣食物如辣椒酱、溶化的姜茶、荔枝等后皮肤发烫发痒。身体偏瘦，常有头痛、头晕和恶心症状。我曾服用过脑供血不足、前庭功能障碍、鼻窦炎等药物。公司即将开展集中接种新冠疫苗，我能否在此期间接种？"}, {"id": 698, "vi": "Tuần trước cháu có đi khám phụ khoa ở bệnh viện. Cháu có làm xét nghiệm HPV và kết quả dương tính với HPV tuýp 68. Bác sĩ cho cháu hỏi nguyên nhân bị nhiễm HPV tuýp 68 và cách điều trị được không ạ? Với HPV tuýp 68 thì nó có tự đào thải được không ạ?", "en": "Last week, my daughter went to see a gynecologist at the hospital. She underwent an HPV test and the result was positive for HPV type 68. The doctor asked me about the cause of HPV type 68 infection and how to treat it. Does HPV type 68 naturally clear itself on its own?", "zh": "上周孩子去医院做了妇科检查。孩子做了HPV检测，结果为HPV 68型阳性。医生想问问孩子感染HPV 68型的原因以及治疗方法？对于HPV 68型，它是否可以自行清除？"}, {"id": 699, "vi": "Tôi bị chẩn đoán là Block nhánh phải hoàn toàn. Gần đây, tôi cảm thấy hơi tức ngực nên đi kiểm tra. Sau kho điện tim đồ 3 lần thì nhịp tim lần lượt là 93, 89 và 76. Bác sỹ kết luận là bình thường trong khi nhịp tim có sự thay đổi như vậy và tôi cảm thấy tức ngực. Năm nay tôi 25 tuổi, mong bác sỹ tư vấn.", "en": "I have been diagnosed with complete right bundle branch block. Recently, I have felt slight chest tightness, so I went for a check-up. After three electrocardiogram tests, my heart rate was 93, 89, and 76 respectively. The doctor concluded that it is normal, although there is such a change in heart rate and I feel chest tightness. I am 25 years old this year, and I hope the doctor can advise.", "zh": "我被诊断为完全右束支传导阻滞。最近我感到轻微胸闷，于是进行了检查。三次心电图检查显示心率分别为93、89和76。医生认为这是正常的，但心率有如此变化，且我仍感到胸闷。今年我25岁，希望医生能给予指导。"}, {"id": 700, "vi": "Hôm nay em có làm xét nghiệm HIV ag/ab nhưng khi trả kết quả lại thấy ghi combi PT, vậy 2 cái này cho kết quả giống nhau không ạ? Và em xét nghiệm đến thời điểm này là 5 tháng ạ kết quả âm tính vậy tỷ lệ em không mắc bệnh có cao không ạ?", "en": "Today I had an HIV Ag/Ab test, but when I received the results, it showed combi PT. So, are these two tests giving the same result? And I have been tested up to this point for 5 months, and the result is negative. Is the probability that I do not have the disease high?", "zh": "今天我做了HIV Ag/Ab检测，但拿到结果时发现写着combi PT，这两个项目的结果一样吗？我到现在已经检测了5个月，结果为阴性，那么我没有感染的概率高吗？"}, {"id": 701, "vi": "Em bị thối móng và em cắt càng ngày càng sâu như thế này. Đây là bệnh gì và chữa trị như thế nào ạ?", "en": "My nails are deteriorating and I am cutting them deeper and deeper like this. What is this disease and how should it be treated?", "zh": "我指甲腐烂，而且我越剪越深，这是什么病，怎么治疗？"}, {"id": 702, "vi": "Em hiện nay đang mang thai tuần thứ 32 , đến hẹn lịch Tiêm phòng uốn ván mũi số 2 mà em bị nghẹt mũi chảy Nước mũi nhưng không ho và không đau đầu. vậy em có tiêm được không bác sĩ, em lo quá vì sợ quá thời gian Tiêm phòng.", "en": "I am currently 32 weeks pregnant and have scheduled for the second intranasal tetanus vaccine dose, but I am experiencing nasal congestion and clear nasal discharge without cough or headache. Therefore, can I still receive the vaccine, doctor? I am very worried because I am afraid of missing the vaccination timing.", "zh": "我目前怀孕32周，到了接种百日咳疫苗的第二针时间，但我出现了鼻塞、流鼻水，没有咳嗽，也没有头痛。医生，我能不能接种呢？我非常担心，害怕错过接种时间。"}, {"id": 703, "vi": "Phương pháp vô cảm nào cho bệnh nhân tháo lồng ruột bằng hơi?", "en": "Which anesthesia method is used for the patient undergoing pneumoperitoneum-based bowel resection?", "zh": "腹腔镜肠切除术患者采用哪种麻醉方法？"}, {"id": 704, "vi": "Trẻ 2 tuần tuổi thiếu men G6PD nên điều trị thế nào và cần tránh những gì?", "en": "What treatment should be given and what should be avoided for a 2-week-old child with G6PD deficiency?", "zh": "2周龄婴儿缺乏G6PD应如何治疗，需要避免什么？"}, {"id": 705, "vi": "Chu kỳ kinh nguyệt của em không đều, 6 tháng hoặc một năm mới có một lần, tình trạng như vậy có thể sinh con được không?", "en": "My menstrual cycle is irregular, occurring once every 6 months or once a year. Can a person with this condition still have a child?", "zh": "我妹妹月经周期不规律，六个月或一年才来一次，这种情况可以怀孕吗？"}, {"id": 706, "vi": "Nguy cơ của em được 6 tháng . Em tiêm vacxin uốn ván được 5 ngày rồi em đi xét nghiệm HIV Elisa kết quả âm tính . Như vậy kết quả xét nghiệm HIV có bị ảnh hưởng sai lệch bởi vacxin làm cho âm tính giả không bác sĩ", "en": "My risk is 6 months. I received the tetanus vaccine 5 days ago, and then I had an HIV Elisa test, which came back negative. Therefore, could the HIV test result be falsely negative due to the vaccine, doctor?", "zh": "我患病6个月，5天前接种了破伤风疫苗，之后进行了HIV ELISA检测，结果为阴性。医生，这样HIV检测结果是否可能因接种破伤风疫苗而出现假阴性呢？"}, {"id": 707, "vi": "Trẻ 16 tháng tuổi chậm nói, nghịch ngợm là dấu hiệu bệnh gì?", "en": "What disease is indicated by delayed speech and hyperactivity in a 16-month-old child?", "zh": "16个月大的儿童语言发育迟缓，好动是何种疾病的征兆？"}, {"id": 708, "vi": "Bà bầu có lỗ nhỏ ở vành tai rỉ mủ, chạm vào đau, nên làm gì để không ảnh hưởng đến sức khỏe mẹ và bé?", "en": "A pregnant woman has a small hole at the ear canal draining pus, and it hurts when touched. What should be done to not affect the mother's and baby's health?", "zh": "孕妇在耳廓处有一个小孔流出脓液，触碰时疼痛，应如何做才能不影响母亲和胎儿的健康？"}, {"id": 709, "vi": "Em bị xuất huyết giảm tiểu cầu, khi em uống lá đinh lăng thì tiểu cầu tụt, vậy có phải tụt do đinh lăng không ạ hay do em? Và có thảo dược gì giúp tăng tiểu cầu không ạ?", "en": "I have thrombocytopenia with bleeding. When I consume centella leaves, my platelet count drops. Is the drop due to centella or due to me? And is there any herbal medicine that can help increase platelets?", "zh": "我患有血小板减少性出血，当我服用鱼腥草时血小板下降，那么这是鱼腥草导致的，还是我自身的原因？有没有草药可以帮助提升血小板？"}, {"id": 710, "vi": "Răng cửa thừa mới mọc có nên niềng không?", "en": "Should a newly erupted supernumerary tooth be orthodontically treated?", "zh": "新长出的切牙是否需要戴牙套？"}, {"id": 711, "vi": "Em đang bầu 30 tuần và có tiêm uốn ván mũi 2. Hiện vùng tiêm sưng 1 cục và nhức buốt cả bắp tay. Em muốn hỏi phản ứng này có nghiêm trọng không ạ và em nên làm gì để giảm đau ạ?", "en": "I am at 30 weeks of pregnancy and received a tetanus intranasal dose 2. Currently, the injection site is swollen in a lump and I have pain and soreness in the entire arm. I would like to ask whether this reaction is serious and what I should do to relieve the pain?", "zh": "我目前怀孕30周，刚接种了破伤风疫苗（鼻内注射）。注射部位出现了一个肿块，且整个手臂都感到疼痛。我想问一下这种情况严重吗？我应该怎么做才能缓解疼痛？"}, {"id": 712, "vi": "Hiện tại em đang bầu hơn 3 tuần. Em nên chọn những bài thể dục nào cho phù hợp để đảm bảo sức khoẻ cho mẹ và không ảnh hưởng tới bé ạ?", "en": "Currently I am more than 3 weeks pregnant. Which exercises should I choose that are suitable to ensure my health and not affect the baby?", "zh": "目前我怀孕超过3周。我应该选择哪些运动项目，以确保母亲健康，且不影响胎儿？"}, {"id": 713, "vi": "Điều trị bệnh quai bị bao lâu thì hết hẳn và không lây sang người khác?", "en": "How long does it take to completely treat measles and when is it no longer contagious to others?", "zh": "治疗腮腺炎需要多久才能完全康复，且不再传染给他人？"}, {"id": 714, "vi": "CRP định lượng tăng 17,43, WBC 17,6, NEU 12,7 có nguy hiểm không?", "en": "Is a quantitative CRP of 17.43, WBC of 17.6, NEU of 12.7 dangerous?", "zh": "C反应蛋白定量升高17.43，白细胞计数17.6，中性粒细胞12.7，是否有危险？"}, {"id": 715, "vi": "Dấu hiệu nhận biết và cách điều trị nhiễm Hp dứt điểm là gì?", "en": "What are the signs of recognizing and how to treat Hp infection completely?", "zh": "如何识别并彻底治疗幽门螺杆菌感染？"}, {"id": 716, "vi": "Tôi thỉnh thoảng bị dị ứng ngứa nổi mẩn, đặc biệt là khi ăn nhộng tằm hoặc nhộng ong. Tôi muốn tiêm vắc xin Covid-19 thì có ảnh hưởng gì không?", "en": "I occasionally experience allergic itching and hives, especially when eating silkworm or bee larvae. I want to get the COVID-19 vaccine—will this have any effect?", "zh": "我偶尔会出现瘙痒性皮疹过敏，尤其是在食用蚕蛹或蜂蛹时。我想接种新冠疫苗，这会有影响吗？"}, {"id": 717, "vi": "Bác sĩ ơi, cho em hỏi em mới ngưng uống thuốc tránh thai hàng ngày thì khi nào có kinh lại vậy bác?", "en": "Doctor, may I ask you something? I just stopped taking daily oral contraceptive pills, when will my period return, doctor?", "zh": "医生，您好，我刚刚停用了每日口服避孕药，那么月经什么时候会恢复呢？"}, {"id": 718, "vi": "Làm thế nào để cải thiện tình trạng phù gây nặng nề, nhức mỏi chân ở người già?", "en": "How to improve the condition of severe swelling and foot pain in the elderly?", "zh": "如何改善老年人引起的严重水肿和下肢疼痛乏力？"}, {"id": 719, "vi": "Em có sử dụng 1 máy SpO2 Kaneko kẹp ngón tay, theo em biết chỉ số bình thường là 3-9% nhưng của em đôi lúc nằm lại lên tới 20-30% ạ, như vậy có bình thường không ạ?", "en": "I am using a Kaneko SpO2 device that clips onto the finger. According to my knowledge, the normal range is 3-9%, but my readings sometimes go up to 20-30%. Is this normal?", "zh": "我使用的是一个Kaneko手指夹式血氧仪，据我所知，正常值是3-9%，但我的数值有时会高达20-30%，这种情况正常吗？"}, {"id": 720, "vi": "Con trai tôi 20 tuổi, con gái 14 tuổi. Tôi muốn tiêm vắc xin HPV cho các cháu thì nên tiêm loại nào? Theo tôi tìm hiểu thì Cervarix phòng HPV 16,18 có thể tiêm cho nữ còn Gardasil phòng HPV 6,11,16,18 nên nam phải tiêm loại này có đúng không ạ?", "en": "My son is 20 years old, my daughter is 14 years old. I want to give my children the HPV vaccine. Which type should I give them? According to my research, Cervarix prevents HPV 16, 18 and can be given to girls, while Gardasil prevents HPV 6, 11, 16, 18, so boys must receive this one—is that correct, sir?", "zh": "我的儿子20岁，女儿14岁。我想为孩子们接种HPV疫苗，应该接种哪种？据我了解，Cervarix可预防HPV 16、18，适用于女性，而Gardasil可预防HPV 6、11、16、18，因此男性必须接种Gardasil，对吗？"}, {"id": 721, "vi": "Mắc tiểu đường có ảnh hưởng đến sinh lý nam giới không?", "en": "Does having diabetes affect male physiology?", "zh": "糖尿病会影响男性生理功能吗？"}, {"id": 722, "vi": "Rãnh bao quy đầu nổi mạch cứng là dấu hiệu của bệnh lý gì?", "en": "Are there any diseases indicated by hard, visible veins in the glans penis?", "zh": "阴茎包皮系带出现硬性血管团是什么疾病的征兆？"}, {"id": 723, "vi": "Có được đi cùng chồng đến lớp học tiền sản không?", "en": "Can I go to the pre-natal class with my husband?", "zh": "可以和丈夫一起前往产前课程吗？"}, {"id": 724, "vi": "Sốt run người, nôn khan và tiêu chảy là dấu hiệu của bệnh gì?", "en": "Fever with shivering, dry vomiting, and diarrhea are signs of what disease?", "zh": "发热寒战、干呕和腹泻是哪种疾病的症状？"}, {"id": 725, "vi": "F0 dương tính, F1 âm tính, nguy cơ mắc Covid F2 như thế nào?", "en": "F0 positive, F1 negative, what is the risk of contracting Covid for F2?", "zh": "F0 阳性，F1 阴性，Covid 感染风险 F2 是多少？"}, {"id": 726, "vi": "Thuốc gây tê ngoài màng cứng để giảm đau khi chuyển dạ có an toàn cho mẹ và bé không?", "en": "Is it safe for mother and baby to use spinal anesthesia medication to reduce pain during labor?", "zh": "硬膜外麻醉药用于分娩时减轻疼痛，对母亲和婴儿是否安全？"}, {"id": 727, "vi": "Điều trị liệt toàn thân do viêm màng não như thế nào?", "en": "How to treat complete paralysis due to bacterial meningitis?", "zh": "如何治疗由脑膜炎引起的全身性瘫痪？"}, {"id": 728, "vi": "Khi tiêm vắc xin Covid-19 có tác dụng phụ nào đối với người mắc bệnh vảy nến hay không?", "en": "Are there any adverse effects from receiving the Covid-19 vaccine in individuals with psoriasis?", "zh": "接种新冠疫苗是否会对银屑病患者产生副作用？"}, {"id": 729, "vi": "Hút thuốc lá có gây lao kháng thuốc không?", "en": "Does smoking cause drug-resistant tuberculosis?", "zh": "吸烟会引发耐药结核病吗？"}, {"id": 730, "vi": "Xét nghiệm máu có kết quả HBsAg: 0,33 và HBsAb: 540 có ý nghĩa gì?", "en": "What does the blood test result of HBsAg: 0.33 and HBsAb: 540 mean?", "zh": "血清学检查结果：HBsAg：0.33，HBsAb：540 表示什么？"}, {"id": 731, "vi": "Mình có quan hệ tình dục không an toàn, xuất hiện một số triệu chứng trắng lưỡi, đau khớp. Đây có phải vết ban dạng giai đoạn sơ nhiễm HIV không ạ?", "en": "I had unsafe sexual contact and have developed some symptoms including white tongue and joint pain. Is this a rash in the acute infection stage of HIV?", "zh": "我有过不安全的性行为，出现了一些症状，如舌苔白、关节痛。这是否是HIV初感染期的皮疹呢？"}, {"id": 732, "vi": "Thận‌ ‌có‌ ‌cặn ‌có‌ ‌phải‌ ‌bị‌ ‌viêm‌ ‌cầu‌ ‌thận‌ ‌không?‌ ‌", "en": "Does kidney stone cause glomerulonephritis?", "zh": "肾有结石是否会导致肾小球肾炎？"}, {"id": 733, "vi": "Xác định giai đoạn của bệnh HIV như thế nào?", "en": "How to determine the stage of HIV disease?", "zh": "如何确定HIV的分期？"}, {"id": 734, "vi": "Em sinh mổ được 39 ngày. Mấy ngày trước sản dịch em ít đi. Nhưng hôm qua em ra nhiều máu như kinh nhưng em không thấy đau bụng hay gì cả. Em vẫn cho con bú mẹ hoàn toàn. Như vậy có sao không.", "en": "My baby was delivered by cesarean section 39 days ago. A few days ago, the lochia decreased. However, yesterday I had a lot of bleeding similar to menstruation, but I did not feel any abdominal pain or other symptoms. I am still exclusively breastfeeding. Is this normal?", "zh": "宝宝剖腹产39天。几天前恶露减少，但昨天突然出血很多，像月经一样，但没有腹痛或其他不适。我仍然完全母乳喂养。这种情况有问题吗？"}, {"id": 735, "vi": "Có thể gây tê khi chọc hút trứng không?", "en": "Can numbing be caused when aspirating an egg?", "zh": "抽卵时会麻醉吗？"}, {"id": 736, "vi": "Mũi và hai bên má đỏ, nóng rát mỗi khi thời tiết thay đổi là bị làm sao và điều trị như thế nào?", "en": "Nose and both cheeks red, hot and burning whenever weather changes - what is the cause and how to treat it?", "zh": "鼻子和两侧脸颊在天气变化时出现红肿、灼热感，是由于什么引起的，该如何治疗？"}, {"id": 737, "vi": "Bị dau ở vùng xương vai phải làm thế nào.", "en": "How to treat pain in the right shoulder bone?", "zh": "如何治疗右侧肩骨疼痛？"}, {"id": 738, "vi": "Nếu một người bị nhiễm Covid-19 (nhưng do không biết) mà vẫn đi tiêm vắc xin thì vấn đề gì sẽ xảy ra và mũi tiêm này có được tính không?", "en": "If a person is infected with COVID-19 (but unaware of it) still receives a vaccine, what will happen and will this dose be counted?", "zh": "如果一个人感染了新冠病毒（但自己不知情）仍去接种疫苗，会有什么问题？这针疫苗是否算作有效接种？"}, {"id": 739, "vi": "Cháu năm nay 18 tuổi, hiện tại mắt bên trái của cháu có dấu hiệu mờ nhìn nhòe đi sau khi bị viêm kết mạc và lành đi được 2-3 ngày. Mắt trái cháu nhìn xa có dấu hiệu nhòe đi giống như sương vậy. Bác sĩ có thể cho cháu biết nguyên nhân được không ạ?", "en": "I am currently 18 years old. Currently, my left eye has a blurred vision symptom that appears after bacterial conjunctivitis and resolves after 2-3 days. My left eye has a blurred vision when looking at distant objects, similar to fog. Can the doctor please tell me the cause?", "zh": "我今年18岁，目前左眼在结膜炎痊愈2-3天后出现视力模糊、视物模糊，看远处像有雾一样。医生能否告知我具体原因？"}, {"id": 740, "vi": "Các bác sĩ cho cháu hỏi cháu bị viêm đường tiết liệu có ảnh hưởng gì đến việc sinh sản không", "en": "Doctors, I would like to ask: will having urethral inflammation affect fertility?", "zh": "医生们请问一下，孩子患有泌尿系统炎症是否会影响生育？"}, {"id": 741, "vi": "Em bị tê, đau 2 bên chân, kể cả khi nằm nghỉ ngơi. Hiện tại em làm PSA 7,4 và fPSA/PSA total 0,09! Em đã theo dõi hon 1 năm. Hiện em 33 tuổi, liệu bệnh trẻ chân của em có liên quan đến tuyến tiền liệt (em chưa sinh tiết)", "en": "I have tingling and pain on both legs, even when lying down and resting. Currently, my PSA is 7.4 and fPSA/PSA total is 0.09! I have been following up for over 1 year. I am 33 years old. Is my young leg condition related to the prostate gland (I have not had ejaculation)?", "zh": "我双腿有麻木、疼痛感，即使在休息时也如此。目前我的PSA为7.4，fPSA/PSA总值为0.09！我已经追踪观察一年了。我现在33岁，我想知道我的年轻腿部症状是否与前列腺有关（我尚未有前列腺液分泌）"}, {"id": 742, "vi": "Cách điều trị tiểu nhiều lần, tiểu khó, tiểu không hết?", "en": "How to treat frequent urination, difficult urination, and incomplete urination?", "zh": "如何治疗尿频、排尿困难、排尿不尽？"}, {"id": 743, "vi": "Huyết áp 1 tháng trước của tôi là trên 150, sau khi đi khám, uống thuốc thì không có dấu hiệu giảm, vậy tôi có nên đi khám lại không?", "en": "My blood pressure one month ago was above 150, after visiting the doctor and taking medication, there is no sign of improvement, so should I go for another consultation?", "zh": "我一个月前的血压高于150，去医院检查并服药后，没有出现下降迹象，我是否应该再次就诊？"}, {"id": 744, "vi": "Hội chứng đuôi ngựa có thuộc danh mục bệnh lý nền không và có tiêm vaccine ngừa Covid-19 được không?", "en": "Does equine syndrome belong to the category of underlying diseases and can it receive the COVID-19 vaccine injection?", "zh": "马尾综合征是否属于基础疾病，是否可以接种新冠疫苗？"}, {"id": 745, "vi": "Uống sữa bí đỏ nhiều có bị vàng da không", "en": "Will drinking a lot of red yam milk cause yellow skin?", "zh": "多喝红甘薯奶会引发黄疸吗"}, {"id": 746, "vi": "Khoai tây có ngừa tiểu đường được không?", "en": "Can potatoes prevent diabetes?", "zh": "红薯能预防糖尿病吗？"}, {"id": 747, "vi": "Phẫu thuật điều trị viêm mũi dị ứng có hết không?", "en": "Does surgical treatment for allergic rhinitis completely cure it?", "zh": "过敏性鼻炎手术治疗能根治吗？"}, {"id": 748, "vi": "Em bị nhiễm herpes sinh dục đã 2 năm nay. Tần suất hơn 10 lần 1 năm. Mỗi đợt bùng phát em có uống thuốc acyclovir. Em đọc thấy trên mạng là có thể uống L-Lysine 1000 mỗi ngày trong vòng 6 tháng để hạn chế mụn rộp đúng không ạ? Vậy giờ em có thể uống kết hợp acyclovir 400mg /2 lần / ngày và L-Lysine 1000mg /1 lần / ngày để ức chế những đợt phát không ạ?", "en": "My daughter has had genital herpes for the past two years. It occurs more than 10 times per year. Each outbreak, I give her acyclovir. I read online that it is possible to take L-Lysine 1000 mg daily for six months to reduce outbreaks, correct? So now, can I take acyclovir 400 mg twice daily and L-Lysine 1000 mg once daily together to suppress future outbreaks?", "zh": "我已有两年患生殖器疱疹病史，每年发作超过10次。每次发作我都会服用阿昔洛韦。我在网上看到，每天服用1000毫克L-赖氨酸，持续6个月，可以减少疱疹发作，对吗？因此，我现在可以每天服用阿昔洛韦400毫克，每日两次，同时每天服用L-赖氨酸1000毫克，以抑制发作吗？"}, {"id": 749, "vi": "Ho ra máu kèm chảy máu cam là bệnh lý gì?", "en": "What is the disease characterized by hemoptysis accompanied by epistaxis?", "zh": "咳血伴鼻出血是什么疾病？"}, {"id": 750, "vi": "Bệnh tiểu đường có thể chữa khỏi bệnh được không thưa bác sĩ?", "en": "Can diabetes be cured, doctor?", "zh": "医生，糖尿病可以治愈吗？"}, {"id": 751, "vi": "Số lượng bạch cầu Monocyte 0.56 ảnh hưởng đến sức khỏe không? Có cần đi kiểm tra lại?", "en": "Does the monocyte white blood cell count of 0.56 affect health? Do I need to have it rechecked?", "zh": "单核细胞计数为0.56是否影响健康？需要复查吗？"}, {"id": 752, "vi": "Vì sao dùng thuốc điều trị tăng huyết áp lâu ngày có thể gây tác dụng phụ liệt dương?", "en": "Why can long-term use of antihypertensive medication cause side effects such as erectile dysfunction?", "zh": "为什么长期服用降压药可能导致阳痿副作用？"}, {"id": 753, "vi": "Tôi huyết áp 150/90, va gout nhẹ. Vậy tôi có thểm tiêm vaccine Covid-19 thế nào?", "en": "My blood pressure is 150/90, and I have mild gout. So, how can I receive the COVID-19 vaccine?", "zh": "我的血压是150/90，有轻度痛风。我该如何接种新冠疫苗？"}, {"id": 754, "vi": "Cho tôi hỏi kết quả CA 125 là 64.80 sẽ nghi ngờ bệnh gì. Bước tiếp theo làm gì", "en": "May I ask, if the CA 125 result is 64.80, what disease should be suspected? What is the next step?", "zh": "请问CA125检测值为64.80，应怀疑哪些疾病？下一步该怎么做？"}, {"id": 755, "vi": "Thưa bác sĩ, năm nay em 23 tuổi, tháng vừa rồi em có chậm kinh 7 ngày, sau em thử que lên 1 vạch đỏ đậm 1 vạch mờ. Song cách 3 ngày em thấy ra máu tươi nhưng không nhiều. Bác sĩ có thể tư vấn cho em được không ạ?", "en": "Dear doctor, this year I am 23 years old. Last month I had a 7-day delay in my period, after which I tested with a pregnancy test and got one clear, dark line and one faint line. However, three days later I noticed fresh blood bleeding but not much. Could the doctor please advise me?", "zh": "医生您好，今年我23岁，上个月我延迟了7天来月经，之后用验孕棒检测，一条是深红色，一条是模糊的。但三天后我出现了鲜红色的少量出血。医生可以给我一些建议吗？"}, {"id": 756, "vi": "Người bị lao phổi cần bổ sung những chất gì để tăng cân?", "en": "What nutrients should a patient with pulmonary tuberculosis take to gain weight?", "zh": "肺结核患者需要补充哪些营养物质以增加体重？"}, {"id": 757, "vi": "Mẹ em năm nay 55 tuổi. Cách đây 2 tháng mẹ em bị tăng men gan. Đã chữa trị ổn định. Tuy nhiên dạo gần đây mẹ em lại hay bị nổi mụn nhọt to khắp cơ thể. Nổi nhiều nhất ở mặt, cứ cách thời gian lại nổi mụn. Vậy có phải tình trạng mụn nhọt này là do gan không ổn định không?", "en": "My mother is now 55 years old. Two months ago, she was diagnosed with elevated liver enzymes. She has been treated and stabilized. However, recently she has been frequently developing large pustules all over her body, most commonly on the face, with recurrent outbreaks. Is this pustule condition due to unstable liver function?", "zh": "我母亲今年55岁。两个月前她出现肝功能异常，经过治疗后已稳定。然而最近她频繁出现全身性脓疱，最严重的是面部，每隔一段时间就会反复发作。这种情况是否与肝脏功能不稳定有关？"}, {"id": 758, "vi": "Tôi xét nghiệm dương tính cúm a nên xử lý thế nào?", "en": "I have a positive influenza A test result; what should I do?", "zh": "我流感A检测呈阳性，应该如何处理？"}, {"id": 759, "vi": "Tôi đã tiêm vắc xin mũi một Astrazeneca, vậy mũi 2 tôi có thể tiêm vắc xin của Pfizer được không?", "en": "I have received the first dose of Astrazeneca as a nasal injection, so can I receive the second dose of Pfizer vaccine?", "zh": "我已接种了阿斯利康的鼻喷疫苗，那么我的第二针是否可以接种辉瑞疫苗？"}, {"id": 760, "vi": "Em có chị năm nay 40 tuổi đã chuyển phôi nhiều lần và không có chỉ số hcg. Toàn phôi ngày 5 loại 1 vừa oy chị có làm xét nghiệm chỉ số NK thì được 249 chỉ số như thế có thấp và có ảnh hưởng đến quá trình chuyển phôi nhiều lần thất bại không. Chị em đã mổ thăm dò tử cung mọi thứ bình thường.", "en": "I have a sister who is 40 years old and has undergone multiple embryo transfers without HCG levels. On day 5, the embryo quality is grade 1. My sister has done a NK cell count test and got a result of 249. Is this low, and does it affect multiple failed embryo transfer procedures? My sister has had a hysteroscopy and all findings are normal.", "zh": "我姐姐今年40岁，多次进行胚胎移植但都没有HCG值。第5天的胚胎为1级，她的NK细胞计数为249，这个数值是否偏低，是否会影响多次胚胎移植失败？姐姐已经做过子宫探查，结果均正常。"}, {"id": 761, "vi": "Xơ gan giai đoạn 4 có thể điều trị khỏi không?", "en": "Can cirrhosis stage 4 be cured?", "zh": "4期肝硬化可以治愈吗？"}, {"id": 762, "vi": "Thận có nang 0,8 cm có đáng lo? Cách điều trị như thế nào?", "en": "Is a 0.8 cm renal cyst worth worrying about? What is the treatment approach?", "zh": "0.8厘米的肾囊肿需要担心吗？该如何治疗？"}, {"id": 763, "vi": "Người bệnh cường giáp có thể uống thuốc tăng nở ngực DHC được không?", "en": "Can a patient with hyperthyroidism take DHC breast enlargement medication?", "zh": "甲亢患者可以服用DHC增乳药吗？"}, {"id": 764, "vi": "Cho em hỏi bé nhà e 20 ngày tuổi quấy khóc khó ngủ, theo giới thiệu của người thân bảo mua loại thực phẩm chức năng soki tium. Cho em hỏi có được nên uống không ạ?", "en": "May I ask if it's safe for my 20-day-old baby to take Sokitium, a functional food recommended by family members, due to the baby's restlessness, difficulty sleeping?", "zh": "请问我家宝宝20天大，总是烦躁哭闹，睡眠困难。亲戚推荐我买一种保健食品Soki Tium。请问可以服用吗？"}, {"id": 765, "vi": "Trẻ sơ sinh ho có đờm kèm hắt hơi sổ mũi phải điều trị như thế nào?", "en": "How should a newborn with cough, phlegm, runny nose, and nasal congestion be treated?", "zh": "新生儿咳嗽有痰伴鼻塞、流涕应如何治疗？"}, {"id": 766, "vi": "Thường xuyên đánh trống ngực kèm mệt mỏi có sao không?", "en": "Is it dangerous to frequently have chest knocking accompanied by fatigue?", "zh": "经常胸闷伴乏力有事吗？"}, {"id": 767, "vi": "Mấy năm gần đây em hay gặp triệu chứng khó thở, nhưng cũng không thường xuyên. Em có đi hàn quốc cùng với công ty, nhiễm lạnh bên hàn. sau đó trở về thì triệu chứng khó thở của em ngày một tăng dần, và gần đây hầu như ngày nào cũng bị. em có cảm giác như có gì đó nghẹn ở cổ, không ăn thì vẫn thở được, nhưng chỉ cần ăn hoặc uống gì đó vào thì sẽ bị nhợn và hay nghẹn ở cổ. Đôi khi bị bắt ép mình phải tự ợ lên mới có cảm giác dễ thở hơn một chút. Đôi lúc có thể ói ra chút bọt bèo. Em không bị sốt, không bị khó tiêu", "en": "In the past few years, I have often experienced shortness of breath, but not frequently. I went to Korea with my company and got a cold there. After returning, my shortness of breath has gradually worsened and now I experience it almost every day. I feel as if there is something obstructing my throat; I can breathe even without eating, but as soon as I eat or drink anything, I feel choking and have a sensation of obstruction in the throat. Sometimes I have to force myself to cough up to feel slightly easier. Occasionally, I vomit a small amount of frothy substance. I do not have fever or difficulty in digestion.", "zh": "近几年我经常出现呼吸困难的症状，但并不频繁。我曾和公司一起去韩国，期间在韩国受了风寒。回国后，我的呼吸困难症状逐渐加重，最近几乎每天都会发作。我感觉喉咙有异物感，不进食也能呼吸，但只要进食或饮水就会出现呛咳和喉咙堵塞感。有时必须自己用力咳嗽才能稍微缓解呼吸困难。偶尔会呕出一些泡沫。我没有发烧，也没有消化不良。"}, {"id": 768, "vi": "Dây thần kinh giữa ở lòng bàn tay bị đứt 15 năm có phục hồi lại như trước không?", "en": "Can a severed median nerve in the palm after 15 years recover to its previous state?", "zh": "手掌正中神经损伤15年能否恢复如前？"}, {"id": 769, "vi": "Đau xương sườn trái âm ỉ là bệnh gì?", "en": "What is chronic dull left rib pain?", "zh": "左侧肋骨隐痛是什么病？"}, {"id": 770, "vi": "Cách đây khoảng 2 tháng em có đi khám và bác sĩ kết luận em bị tiểu đuòng tip 2 và được điều trị tại bệnh viện 1 tuần rồi được ra viện. Bác sĩ kê đơn trajenta 5mg uống 1viên/ngày và glucofage 500mg uống 2 viên/ngày. Sau 1 tháng em đi xét nghiem lại đường huyết 6.5 lúc đói và 9.1 sau ăn. Em không mua được thuốc trajenta .Có loại thuốc nào thay thế được cho trajenta không. Bác sĩ có thể tư vấn cho em môt đơn thuốc khác phù họp vói em.", "en": "About two months ago, I went for a check-up and the doctor diagnosed me with type 2 diabetes and treated me at the hospital for one week before discharge. The doctor prescribed Trajenta 5mg to be taken once daily and Glucophage 500mg to be taken twice daily. After one month, I had a blood glucose test again, showing a fasting level of 6.5 mmol/L and a postprandial level of 9.1 mmol/L. I am unable to purchase Trajenta. Is there any alternative medication available for Trajenta? Can the doctor provide me with another suitable prescription?", "zh": "大约两个月前，我去了医院检查，医生诊断我患有空腹血糖升高2型糖尿病，并在医院治疗一周后出院。医生开了特拉杰塔5mg，每日1片，以及格列齐特500mg，每日2片。一个月后，我复查了血糖，空腹血糖为6.5 mmol/L，餐后血糖为9.1 mmol/L。我买不到特拉杰塔了。有没有可以替代特拉杰塔的药物？医生能否为我开一个合适的替代药物处方？"}, {"id": 771, "vi": "Tôi gần đây quan hệ tình dục nhưng bị xuất tinh sớm tầm 2 phút. Không biết có cách điều trị không.", "en": "I have recently had sexual intercourse but experienced premature ejaculation around 2 minutes. I don't know if there is a way to treat it.", "zh": "我最近有性生活，但会出现射精过早的情况，大约2分钟。不知道是否有治疗方法。"}, {"id": 772, "vi": "Trẻ 23 ngày tuổi chỉ bị vàng da ở mặt có phải hiện tượng sinh lý bình thường không?", "en": "Is jaundice only on the face in a 23-day-old infant a normal physiological phenomenon?", "zh": "23天大的婴儿仅出现面部黄疸是否属于正常生理现象？"}, {"id": 773, "vi": "Đau tinh hoàn kèm chảy dịch dương vật thường xuyên là bệnh gì?", "en": "Pain in the testicle accompanied by frequent discharge from the penis – what is this disease?", "zh": "经常伴有阴茎分泌物的睾丸疼痛是什么病？"}, {"id": 774, "vi": "Giảm đau hạch ở cổ như thế nào?", "en": "How to relieve neck lymph node pain?", "zh": "如何缓解颈部淋巴结疼痛？"}, {"id": 775, "vi": "Cho em hỏi là bộ phận sinh dục của em có phần môi bé bên trái to hơn so với môi bé bên phải, liệu em có bị bệnh về đường sinh dục không ạ? Em không bị ngứa, không có khí hư ạ!", "en": "May I ask if my genital area has a smaller labia on the left side that is larger than the smaller labia on the right side? Do I have a genital disease? I don't have itching, no vaginal discharge!", "zh": "我想问一下，我生殖器左侧的小阴唇比右侧的大，我是否患有生殖器疾病？我没有任何瘙痒，也没有白带！"}, {"id": 776, "vi": "Xin tư vấn giúp cháu. Cháu phát hiện mình bị viêm gan B 2 năm nay. Cháu đã đếm vi rút hai lần. Lần đầu vi rút cao và cháu đã uống thuốc ngừa vi rút 3 tháng sau đó cháu xét nghiệm lại. Vi rút dưới ngưỡng cháu lại mua thêm thuốc uống 3 tháng nữa vậy cháu xin hỏi vi rút trong máu có hết được không? Cháu phải uống trong bao lâu và cháu uống thuốc bảo hiểm có được không? Mong tư vấn giúp cháu cách điều trị.", "en": "I would like to consult you regarding my child. My child has been diagnosed with chronic hepatitis B for the past two years. The child has tested for virus load twice. The first time, the virus load was high, and the child took antiviral medication for three months, after which the child retested. The virus load was below the detectable level, so the child purchased additional antiviral medication and took it for another three months. I would like to ask whether the virus in the blood can be completely eliminated? How long should the child take the medication, and is it acceptable for the child to take insurance-covered medication? I kindly request your advice on the treatment approach.", "zh": "请帮忙咨询一下。我近两年发现患有乙型肝炎。我已检测过病毒两次。第一次病毒量高，之后我服用了抗病毒药物3个月，再进行检测，病毒量降至检测下限，又购买了药物服用3个月。我想咨询一下，血液中的病毒能否彻底清除？需要持续服药多久？可以服用医保药物吗？恳请帮忙咨询一下治疗方案。"}, {"id": 777, "vi": "Hai bên quai hàm phát triển không đồng đều phải làm sao?", "en": "What should I do if both femoral heads develop unevenly?", "zh": "双侧髂骨发育不均该怎么办？"}, {"id": 778, "vi": "Uống quá liều thuốc Cozaar XQ 5/50mg có sao không?", "en": "Is it dangerous to take an overdose of Cozaar XQ 5/50mg?", "zh": "过量服用科泽尔XQ 5/50mg会有问题吗？"}, {"id": 779, "vi": "Tôi mổ nang thoát vị đĩa đệm ( 4 nang), mổ xong tôi không đi vệ sinh được trong vòng 1 tuần ( gần như không có cảm giác bí tiểu tiện hay đại tiện gì cả). Tôi đã tiến hành phẫu thuật lại lần nữa nhưng kết quả vẫn không khả quan hơn là mấy. Tôi đang ở nhà và phải dùng tay để thông ống tiểu tiện cũng như đại tiện. Hiện giờ tôi đang rất bối rối không biết phải làm sao bây giờ?", "en": "I underwent surgery for herniated disc nodules (4 nodules). After the surgery, I was unable to have bowel or bladder movements for one week (almost no sensation of urinary or fecal retention at all). I have undergone a second surgical procedure, but the results are still not much better. I am currently at home and have to manually catheterize for both urinary and fecal elimination. Currently, I am very confused about what to do next?", "zh": "我进行了4个椎间盘脱出囊肿的手术，手术后一周内无法排便（几乎没有任何排尿或排便的感觉）。我再次进行了手术，但结果仍然没有明显改善。我现在在家，必须用手来导尿和排便。我现在非常困惑，不知道该怎么办。"}, {"id": 780, "vi": "Xin tư vấn cách tính ngày 2 chu kỳ kinh chính xác. Tại sao phải xét nghiệm nội tiết vào ngày 2 chu kỳ kinh? Nếu qua ngày 3 thì em có được làm không?", "en": "Please advise on how to accurately calculate the exact day 2 of the menstrual cycle. Why is hormone testing performed on day 2 of the menstrual cycle? If it is after day 3, can I still undergo the test?", "zh": "请咨询如何准确计算月经周期的第2天。为什么需要在月经周期的第2天进行内分泌检查？如果过了第3天还能做吗？"}, {"id": 781, "vi": "Tôi thỉnh thoảng hắt xì hơi, sổ mũi lát sau tự hết, không có bệnh nền thì có thể tiêm vaccine Covid-19 được không?", "en": "Sometimes I have sneezing and runny nose that resolves on its own, and without any underlying diseases, can I receive the COVID-19 vaccine?", "zh": "我偶尔会打喷嚏、流鼻涕，但症状过一会儿就会自行消失，如果没有基础疾病，可以接种新冠疫苗吗？"}, {"id": 782, "vi": "Nam giới 32 tuổi thường xuyên đau đầu kéo dài", "en": "32-year-old male with chronic prolonged headache", "zh": "32岁男性经常出现持续性头痛"}, {"id": 783, "vi": "Viêm đại tràng và hậu môn kéo dài nên dùng thuốc gì?", "en": "What drugs should be used for chronic colitis and anal fissure?", "zh": "慢性结肠和肛门炎应使用什么药物？"}, {"id": 784, "vi": "Sáng nay em có đi làm cái xét nghiệm SCC chỉ số của em là 1.22, vậy sao chỉ số của em cao thế ạ?", "en": "This morning I went to work for the SCC test, my index is 1.22, so why is my index high?", "zh": "今天早上我去做了一个SCC检测，我的数值是1.22，为什么我的数值会这么高呢？"}, {"id": 785, "vi": "Hội chứng Crouzon có ảnh hưởng đến sự phát triển của trẻ không?", "en": "Does Crouzon syndrome affect child development?", "zh": "颅缝早闭症会影响儿童的发育吗？"}, {"id": 786, "vi": "Bị dính buồng tử cung mổ nội soi có thành công cao không", "en": "Is the success rate of laparoscopic hysterotomy for adhesion of the uterine cavity high?", "zh": "腹腔镜下宫腔粘连是否治疗效果良好"}, {"id": 787, "vi": "Tôi muốn chích vaccine cúm mùa thì sau bao lâu chích được vaccine Covid-19 ?", "en": "I would like to get the seasonal flu vaccine; how long should I wait before getting the COVID-19 vaccine?", "zh": "我想要接种流感疫苗，那么多久后可以接种新冠疫苗？"}, {"id": 788, "vi": "Cho cháu hỏi con cháu được 4 tháng mấy ngày nay không đi đại tiện. Bé vẫn vui vẻ đi tiểu đánh dắm bình thường ạ. Làm cách nào cho bé đi được ạ", "en": "May I ask about my child who has not had a bowel movement for the past 4 months and a few days. The baby is still happy and has normal urination and defecation. What should I do to help the baby have a bowel movement?", "zh": "请问一下，我家宝宝4个月多了，这几天没有排便。宝宝其他方面都正常，排尿也正常。请问该怎么办才能让宝宝排便呢？"}, {"id": 789, "vi": "Dạo này, khi vệ sinh vùng hậu môn, em thấy có một mụn, sờ giống như bong bóng cá, không đau và không rát. Có phải em bị lòi trĩ không ạ? Với những biểu hiện trên, em bị trĩ nội hay trĩ ngoại và đang ở cấp độ mấy ạ?", "en": "Recently, when cleaning the anal area, I have noticed a bump; when touching it, it feels like a fish bubble, without pain or burning. Do I have hemorrhoids? Based on these symptoms, do I have internal or external hemorrhoids, and at which stage am I?", "zh": "最近在清洁肛门区域时，我发现有一个肿块，摸起来像鱼泡一样，不痛不痒。请问我是不是得了痔疮？根据上述症状，我属于内痔还是外痔，处于几期呢？"}, {"id": 790, "vi": "Tôi hay bị dị ứng một số thuốc kháng sinh. Xin hỏi nếu tôi tiêm phòng Covid-19 có sao không?", "en": "I often have allergies to some antibiotics. I would like to know if it is problematic to get the COVID-19 vaccine?", "zh": "我经常对一些抗生素产生过敏。请问如果我接种新冠疫苗会有问题吗？"}, {"id": 791, "vi": "Tôi 56 tuổi, là nữ, không dị ứng thuốc uống, tiêm. Nhưng có lần hít phải thuốc xịt chống sốt xuất huyết của xã do đứng gần nên bị co thắt phế quản, không thở được, lúc ấy tôi không thể thở, may sao khi uống nước chanh lập tức hết thắt, thở được. Sự việc đó đến nay làm tôi quá ám ảnh và sợ hãi. Vậy tôi có thể tiêm ngừa vaccine Covid-19 được không?", "en": "I am 56 years old, female, no drug allergy to oral or injectable medications. However, once I inhaled the fever-reducing spray from the village because I stood nearby, I developed bronchospasm and was unable to breathe; at that time, I could not breathe, but luckily after drinking lemon water immediately, the spasm disappeared and I could breathe again. This incident has been deeply traumatic and frightening for me ever since. Therefore, can I receive the COVID-19 vaccine injection?", "zh": "我56岁，女性，没有药物口服或注射过敏史。但曾因站在附近吸入村里的防登革热喷雾，导致支气管痉挛，无法呼吸，当时我无法呼吸，幸好喝了一杯柠檬水后立即缓解，能够呼吸。这件事至今让我非常恐惧和焦虑。请问，我可以接种新冠疫苗吗？"}, {"id": 792, "vi": "Em bắt đầu có kinh từ 29/11 và hết kinh 3/12, 2/12 em test không thấy có bầu nên uống thuốc điều trị viêm đường tiết niệu Tavanic 0,5. 3/1 em thấy chậm kinh 7 ngày và test thì thấy có thai, em đi siêu âm thì thai đã xuống tử cung rồi ạ. Cho em hỏi là có bị ảnh hưởng gì không ạ?", "en": "I started my period on 29/11 and it ended on 3/12. On 2/12, I tested negative for pregnancy, so I took Tavanic 0.5 to treat urinary tract infection. On 3/1, I noticed a 7-day delay in my period and tested positive for pregnancy. I went for ultrasound and the embryo had already descended into the uterus. I would like to know if this has caused any harm.", "zh": "我从11月29日开始来月经，12月3日结束。12月2日我做了验孕，没有怀孕，于是服用了治疗尿路感染的Tavanic 0.5药物。12月3日我感觉月经推迟了7天，验孕显示怀孕了。我去做了B超，发现胚胎已经进入子宫了。我想问一下，这样会不会有影响呢？"}, {"id": 793, "vi": "Bị cường giáp suy giảm tóc rụng nhiều có phải không?", "en": "Does hyperthyroidism cause hair loss and thinning?", "zh": "甲状腺功能亢进导致脱发吗？"}, {"id": 794, "vi": "Thử thai sau chậm kinh 15 ngày có kết quả chính xác không", "en": "Is a pregnancy test 15 days after missed period accurate?", "zh": "怀孕试纸在延迟15天经期后检测结果准确吗"}, {"id": 795, "vi": "Vợ chồng tôi đã lập gia đình được hơn 3 năm nhưng chưa từng có chuyện sinh hoạt vợ chồng vì vợ tôi mắc chứng co thắt âm đạo. Có giải pháp nào cho đời sống tình dục không? Nếu can thiệp sinh học để có con có phải là một biện pháp cần thiết.", "en": "My husband and I have been married for more than 3 years but have never had sexual intercourse because I have a vaginal spasm condition. Are there any solutions for sexual intercourse? If biological intervention is needed to have children, is it a necessary measure?", "zh": "我们结婚超过3年，但从未有过夫妻生活，因为我的妻子患有阴道痉挛。是否有改善性生活的方案？如果需要进行生物干预以怀孕，是否是必要措施？"}, {"id": 796, "vi": "Vì sao vi khuẩn gây viêm tai mũi họng lại gây ra bệnh viêm cầu thận?", "en": "Why do bacteria causing pharyngitis and sinusitis cause glomerulonephritis?", "zh": "为什么引起上呼吸道感染的细菌会导致肾小球肾炎？"}, {"id": 797, "vi": "Hiện tại quanh đầu dương vật và bao quy đầu em xuất hiện mẩn đỏ. Bao quy đầu đỏ và có dấu sưng. Cảm giác đau khi chạm vào. Em mới có cảm giác đó từ trưa nay ạ. Vậy xin hỏi bác sĩ là em bị bệnh gì? Và điều trị tại nhà bằng thuôc nào.", "en": "Currently, around the glans and foreskin, I am experiencing red rash. The foreskin is red and swollen. There is pain upon touching. I have just started feeling this since this afternoon. Therefore, I would like to ask the doctor what disease I have, and which medication can be used for home treatment.", "zh": "目前阴茎头部和包皮周围出现红疹，包皮发红且有肿胀，触碰时有疼痛感。这种情况是从今天中午开始出现的。请问医生我得了什么病？可以在家中用什么药物治疗？"}, {"id": 798, "vi": "Em đi khám bác sĩ da liễu nói bị mề đay và cho thuốc về uống, em uống thấy bớt, sau hết thuốc lại bị nổi lại ra nhiều mảng ngứa lắm. Em phải làm sao ạ?", "en": "My doctor said I have hives and prescribed medication to take at home. After taking the medicine, the symptoms improved, but after finishing the medication, I started having severe itching with many patches appearing again. What should I do?", "zh": "医生皮肤科说我得了荨麻疹，开了药让我回家吃，吃了之后症状减轻了，但吃完药后又出现了很多大片的瘙痒疹。我该怎么办呢？"}, {"id": 799, "vi": "Em muốn đổi biện pháp tránh thai từ cơ học sang tránh thai hằng ngày bằng thuốc, em vừa hết kinh 10 ngày thì lại có tiếp, bây giờ em đã được uống tránh thai hằng ngày chưa?", "en": "I want to change my contraceptive method from mechanical to daily oral contraception. I just finished my period for 10 days and then it came back. Now, have I started taking daily oral contraception?", "zh": "我想从机械避孕改为每日口服避孕药，我刚停经10天又来了月经，现在已经开始吃每日口服避孕药了吗？"}, {"id": 800, "vi": "Trẻ 12 tuổi môi thâm tím, mặt tái nhợt, không hồng hào có phải dấu hiệu mắc bệnh tim?", "en": "Is cyanotic lips, pale and non-ruddy face in a 12-year-old child a sign of heart disease?", "zh": "12岁儿童口唇青紫，面色苍白无华，是否为患有心脏病的表现？"}, {"id": 801, "vi": "Trẻ 5 tuổi bị giun kim gây ngứa hậu môn nên điều trị thế nào?", "en": "What treatment should be given to a 5-year-old child with pinworm causing perianal itching?", "zh": "5岁儿童因钩虫引起肛门瘙痒，应如何治疗？"}, {"id": 802, "vi": "Xét nghiệm Double test các nguy cơ nhỏ hơn 1/10000 có nguy hiểm không?", "en": "Is it dangerous to perform a double test for risks less than 1/10000?", "zh": "双项检测中，风险小于1/10000是否危险？"}, {"id": 803, "vi": "Hiện tại em đang có 1 bé 13 tháng tuổi, tới giờ vẫn chưa có kinh, có bất thường không? Hôm nay em đi làm xét nghiệm tầm soát ung thư cổ tử cung, lúc cầm cái hủ đựng bệnh phẩm em có thấy giống máu mà không để ý mấy, rồi đi về, về tới nhà cho tới chiều tối thì thấy có máu ra rỉ rả, vậy dấu hiệu đó có nguy hiểm không?", "en": "Currently, I have a 13-month-old child who has not had menstruation yet, is there anything abnormal? Today, I went for a cervical cancer screening test, and when holding the container for the specimen, I noticed something resembling blood but didn't pay attention to it, then went home. Back home, until late afternoon, I noticed continuous bleeding, is this sign dangerous?", "zh": "目前我有一个13个月大的孩子，到现在还没有来月经，这正常吗？今天我去做了宫颈癌筛查，取样本时我看到容器里有类似血液的东西，但没有注意，后来回家，直到晚上才发现有持续出血的情况，这种情况有危险吗？"}, {"id": 804, "vi": "Sau sạch kinh 7-8 ngày niêm mạc tử cung 5mm là mỏng hay bình thường. Nếu niêm mạc tử cung mỏng thì cách điều trị là gì", "en": "After menstruation clearance at 7-8 days, endometrial thickness of 5mm is thin or normal. If the endometrial thickness is thin, what is the treatment approach?", "zh": "月经干净后7-8天，子宫内膜5mm是偏薄还是正常？如果子宫内膜偏薄，该如何治疗？"}, {"id": 805, "vi": "Độ tuổi nào dễ mắc bệnh hiếm muộn ở nam giới? Đâu là yếu tố nguy cơ khiến nam giới dễ mắc các bệnh hiếm muộn?", "en": "At what age are men more likely to develop rare late-onset diseases? What are the risk factors that make men more susceptible to rare late-onset diseases?", "zh": "男性在哪一年龄段更容易罹患罕见不育症？哪些因素会增加男性罹患罕见不育症的风险？"}, {"id": 806, "vi": "Hồi sáng em bán hàng cho khách vì sợ không kịp giao nên em có sơ xuất bị té đập mông xuống đất mà em thì mới bơm tinh trùng được 6 ngày. Cho em hỏi có bị ảnh hưởng gì không ạ.", "en": "In the morning, I was selling goods to customers because I was afraid I wouldn't have time to deliver, so I made a mistake and fell, hitting my buttocks on the ground, and I only started ejaculating 6 days ago. I would like to ask if this has affected me.", "zh": "早上我因为怕来不及送货，所以去给客户送货时不小心摔倒，臀部撞到地上，我这次输精只有6天。请问会有什么影响吗？"}, {"id": 807, "vi": "Thiếu máu não cần làm những xét nghiệm gì để xác định nguyên nhân?", "en": "What tests are needed to determine the cause of cerebral anemia?", "zh": "需要做哪些检查来确定脑性贫血的原因？"}, {"id": 808, "vi": "Mặt sưng phồng một bên có liên quan đến dây thần kinh không?", "en": "Is unilateral swelling and puffiness related to a nerve?", "zh": "一侧面部肿胀是否与神经有关？"}, {"id": 809, "vi": "Bệnh giãn tĩnh mạch thừng tinh nhẹ có chữa được không?", "en": "Can mild varicocele disease be treated?", "zh": "精索静脉曲张轻度能治好吗？"}, {"id": 810, "vi": "Có thể điều trị dứt điểm rối loạn thần kinh thực vật không?", "en": "Can neurological vegetative disorder be cured completely?", "zh": "可以根治植物神经紊乱吗？"}, {"id": 811, "vi": "Em có đi siêu âm thai được 4 tuần 4 ngày bác sĩ ghi là thai nằm ở góc tử cung theo dõi và hẹn 1 tuần sau khám lại. Liệu thai có dịch chuyển về giữa không.", "en": "My ultrasound at 4 weeks and 4 days showed the embryo located at the uterine angle, and the doctor advised follow-up examination one week later. Will the embryo move to the mid-uterus?", "zh": "我做了4周4天的产检，医生记录是胚胎位于子宫角部，需要观察，一周后复查。胚胎会不会移向子宫中央？"}, {"id": 812, "vi": "Khi bình thường tôi thủ dâm có thời gian trung bình từ 10 - 30 phút thì xuất tinh, nhưng khi quan hệ thì tôi chưa xuất tinh lần nào. Lần lâu nhất tôi quan hệ khoảng gần 2 giờ. Vậy tôi có bệnh lý gì không?", "en": "When normally I masturbate, I ejaculate after an average time of 10-30 minutes, but during intercourse, I have never ejaculated. The longest time I had intercourse was approximately 2 hours. So, do I have any pathology?", "zh": "平时我自慰时平均持续10到30分钟就会射精，但发生性关系时我从未射精过，最长的一次性生活持续了接近2小时。我是否有疾病？"}, {"id": 813, "vi": "Đứt dây hãm bao quy đầu có ảnh hưởng gì không?", "en": "Does cutting the frenulum of the glans have any impact?", "zh": "包皮系带断裂有什么影响吗？"}, {"id": 814, "vi": "Em lúc ăn thường xuyên chan cơm với nước canh rồi đến lúc em đi ngủ bụng em ồn lên nhưng em không đói vậy là sao hả bác sĩ?", "en": "My child usually eats rice with soup frequently, but when my child goes to sleep, the abdomen becomes noisy, yet my child is not hungry. What should I do, doctor?", "zh": "我吃饭时经常吃粥加汤，到睡觉时肚子会发出响声，但我不饿，这是怎么回事啊，医生？"}, {"id": 815, "vi": "75 tuổi mắc ung thư tiền liệt tuyến có nên cắt tinh hoàn không?", "en": "Should a 75-year-old patient with prostate cancer have their testicles removed?", "zh": "75岁患有前列腺癌是否需要切除睾丸？"}, {"id": 816, "vi": "Em có thai 8 tuần bị lưu em đã đi hút nhưng bị sót phải hút lại. Em hút lại đến nay được 17 ngày rồi. Hôm qua em đi siêu âm lại thì trong tử cung còn 2mm dịch, niêm mạc dày 4mm nhưng siêu âm về em lại có kinh hôm sau luôn. Bác sĩ cho em hỏi niêm mạc của em như vậy có mỏng không? Như vậy có ảnh hưởng đến mang thai sau này không?", "en": "I am 8 weeks pregnant and had an abortion, but there was residual tissue, so I had a second abortion. I had the second abortion and it has been 17 days since then. Yesterday I did an ultrasound and there is still 2mm fluid in the uterus, with a 4mm thick endometrium. However, the ultrasound report says I had menstruation the day after. Doctor, can you tell me if my endometrium is thin like this? Will this affect my future pregnancy?", "zh": "我怀孕8周，做了清宫手术，但有残留，所以又进行了第二次清宫。到现在已经17天了。昨天我做了B超检查，子宫内还残留2mm液体，子宫内膜厚4mm，但B超报告说下次就会来月经。医生请问一下，我现在的子宫内膜是偏薄吗？这样会影响以后的怀孕吗？"}, {"id": 817, "vi": "Chỉ số creatinin của tôi là 1,54 mg/dL, thận có vấn đề gì không và nên uống thuốc gì để bồi bổ thận?", "en": "My creatinine level is 1.54 mg/dL. Is there any problem with my kidneys, and what medication should I take to supplement my kidneys?", "zh": "我的肌酐值是1.54 mg/dL，我的肾脏有问题吗？应该服用什么药物来补肾？"}, {"id": 818, "vi": "Điều trị lệch vách ngăn, cuốn mũi phì đại và viêm xoang xuất tiết không khỏi phải làm gì?", "en": "What should be done for treatment of deviated nasal septum, enlarged nasal turbinates, and chronic sinusitis with purulent discharge that does not improve?", "zh": "中鼻甲肥大、鼻中隔偏曲及慢性鼻窦炎伴分泌性鼻炎反复不愈该如何治疗？"}, {"id": 819, "vi": "Tôi xét nghiệm tiền ung thu được chỉ số CA 19-9 là 67.8 trong khi ngưỡng là 37. Xin bác sĩ tư vấn giúp với kết quả như vậy tiếp theo tôi phải làm gì ạ?", "en": "I had a cancer screening test and my CA 19-9 level is 67.8, while the threshold is 37. I would like to ask the doctor what I should do next with this result?", "zh": "我的肿瘤筛查检测结果显示CA 19-9值为67.8，而正常阈值为37。医生请问，针对这样的结果，我接下来应该做什么？"}, {"id": 820, "vi": "Em mới tiêm phòng dại sau phơi nhiễm mũi cuối cách đây 1 tháng. Vậy nếu bị chuột cắn ở đầu thì phải làm sao ạ?", "en": "My child received rabies vaccination after exposure at the nasal end one month ago. So, what should I do if my child is bitten by a mouse on the head?", "zh": "我一个月前在鼻腔末段暴露后接种了狂犬病疫苗。如果现在头部被老鼠咬伤该怎么办？"}, {"id": 821, "vi": "Bé nhà em 4 tháng vẩn chưa tự lẫy được. Em để bé nằm nghiêng thì bé mới lẫy ạ. Và nằm sấp thì cổ bé nghẩng cao và nhìn xung quanh bình thường. Nhưng khi bế bé dựng lên thì cổ bé chúi chúi xuống ạ. Và nghiêng bên này bên kia kiểu không tự chủ được. Cho em hỏi như vậy cổ bé có vấn đề gì không?", "en": "My child is 4 months old and has not yet started to sit up independently. When I place the baby on their side, the baby can then sit up. When lying on their stomach, the baby's neck is naturally extended and they can look around normally. However, when holding the baby upright, the baby's neck bends downward, and the head tilts side to side in an uncontrolled manner. I would like to know if this indicates any problem with my baby's neck.", "zh": "我家宝宝4个月还不会自主抬头。我让宝宝侧躺时，宝宝才能抬头；俯卧时，宝宝的脖子能自然挺直，能正常看四周。但当我抱着宝宝站立时，宝宝的脖子会低头，而且左右歪斜，像是没有自主控制一样。请问这种情况宝宝的脖子有没有问题？"}, {"id": 822, "vi": "Con nhà em bị ngã sứt môi như hình, đã khâu 6 mũi, phải làm gì để vết thương nhanh lành và ít để lại sẹo nhất?", "en": "My child fell and has a mouth laceration like in the picture, has already had 6 stitches. What should I do to help the wound heal quickly and leave the least scar?", "zh": "我家人摔倒导致嘴唇撕裂，已经缝了6针，该如何做才能让伤口尽快愈合，并尽量减少疤痕？"}, {"id": 823, "vi": "Tôi hay bị đau bụng do ăn đồ lạnh và thỉnh thoảng bị đau bụng không rõ nguyên nhân.", "en": "I often experience abdominal pain after eating cold food and occasionally have abdominal pain of unknown cause.", "zh": "我经常因为吃冷食而出现腹痛，有时也会无缘无故地感到腹痛。"}, {"id": 824, "vi": "U nhú thực quản có triệu chứng và cách điều trị như thế nào", "en": "What are the symptoms and treatment options for esophageal polyps?", "zh": "食管息肉有症状和治疗方法吗？"}, {"id": 825, "vi": "Tôi muốn hỏi vừa qua tôi có đi khám được bác sĩ chẩn đoán là bị suy tĩnh mạch hiển bé hai bên ở chân và khuyên đi phẫu thuật, vậy tôi muốn nhờ bác sĩ tư vấn giúp", "en": "I would like to ask, recently I visited a doctor who diagnosed me with bilateral superficial venous insufficiency in the legs and advised surgery. Therefore, I would like to ask the doctor for advice.", "zh": "我想咨询一下，最近我去医院检查，医生诊断我患有双侧下肢浅静脉曲张，并建议我进行手术治疗，因此我想请医生给我一些建议。"}, {"id": 826, "vi": "Đỏ mắt có nên sử dụng thuốc Bratorex-Dexa để tra mắt không?", "en": "Should red eyes be treated with Bratorex-Dexa eye drops?", "zh": "眼睛发红是否可以使用Bratorex-Dexa滴眼液？"}, {"id": 827, "vi": "Triệu chứng ớn lạnh không kèm sốt là do bệnh gì?", "en": "What disease causes chills without fever?", "zh": "伴有寒战但无发热的病因是什么？"}, {"id": 828, "vi": "Em bi trào ngược dạ dày, sau đó đi khám bác sĩ tư thì kê thuốc uống được 15 ngày thì hết. Sau đó do uống nước đá, ăn kim chi nên triệu chứng bị lại. uống gaviscon thấy ổn. Nhưng về đêm lại khó chịu, gây ho khi nằm xuống. Hiện đang sử dụng thuốc Atussin Tablets, Nystatin, Halez Tablets, Azithromycin 500mg, trước khi ăn uống Soares. Em mới mua khi tái lại, nên uống thuốc này ko ạ.", "en": "My daughter has gastroesophageal reflux. After visiting a doctor, she was prescribed oral medication and the symptoms disappeared after 15 days. However, after drinking ice water and eating kimchi, the symptoms returned. Taking Gaviscon helps stabilize the condition. But at night, she feels uncomfortable and develops coughing when lying down. Currently, she is using Atussin Tablets, Nystatin, Halez Tablets, Azithromycin 500mg, and Soares before meals. I just started buying this medication when the symptoms returned, so I am not sure if I should take it.", "zh": "我有胃食管反流病，后来去医生那里就诊，医生开了药吃了15天后就好了。之后因为喝了冰水、吃了泡菜，症状又复发了。服用Gaviscon后感觉稳定，但晚上躺下时会感到不适，躺下时会咳嗽。目前在使用Atussin片、Nystatin、Halez片、阿奇霉素500mg，饭前服用Soares。我最近才开始复发，所以现在还没开始吃这个药。"}, {"id": 829, "vi": "Phụ nữ mang thai bị hen phải làm gì?", "en": "What should women with asthma during pregnancy do?", "zh": "孕妇患有哮喘该怎么办？"}, {"id": 830, "vi": "Ngủ nghiến răng mạnh phải làm sao?", "en": "How to deal with strong teeth grinding while sleeping?", "zh": "如何治疗严重的磨牙症？"}, {"id": 831, "vi": "Bé nhà em lúc tầm 9 tháng em có phát hiện sau tai phải của bé có nổi một hạch nhỏ, không biết là mới nổi hay là nổi từ lúc nào mà do em không để ý, em có dắt bé đi khám ở bệnh viện vài lần, bác sĩ bảo hạch không sao từ từ lớn sẽ hết, nhưng đến nay bé đã được 2 tuổi mà một hạch vẫn còn vẫn y như vậy. Bác sĩ cho em hỏi em phải làm sao để hạch đó hết và liệu hạch đó có nguy hiểm lắm không ạ?", "en": "My child developed a small lymph node behind the right ear around 9 months of age. I don't know if it's a new appearance or has been there all along, as I didn't notice it. I have taken my child to the hospital several times for examination, and the doctors said the lymph node is fine and will gradually shrink and disappear. However, now that my child is 2 years old, the lymph node still exists and looks exactly the same. The doctor asks me what I should do to make this lymph node disappear and whether this lymph node is dangerous?", "zh": "我孩子大约9个月大时，发现她右耳后出现一个小小的淋巴结，不知道是最近才长出来的，还是之前就存在而我没注意。我带孩子去过医院几次，医生说这个淋巴结没问题，会慢慢长大然后自行消失，但到现在孩子已经2岁了，这个淋巴结依然存在，情况和之前一样。医生请问一下，我该如何让这个淋巴结消失？这个淋巴结会不会很危险呢？"}, {"id": 832, "vi": "Tôi có đi khám phụ khoa và kết quả tôi bị nội mạc tử cung mỏng 5mm và viêm tử cung. Tôi được cho thuốc bôi và đặt. Tôi muốn hỏi tôi phải làm gì để nội mạc tử cung bình thường trở lại?", "en": "I went for a gynecological examination and the results showed thin endometrium at 5mm and uterine inflammation. I was prescribed topical and intrauterine medications. I would like to ask what I should do to restore the endometrium to normal?", "zh": "我去了妇科检查，结果显示子宫内膜厚度为5毫米，伴有子宫炎。医生给我开了阴道用药和宫腔用药。我想问一下，我应该怎么做才能让子宫内膜恢复正常？"}, {"id": 833, "vi": "Trẻ mắc bệnh vàng da có bướu huyết thanh ở đầu cần chiếu đèn đến khi nào?", "en": "When should a child with jaundice and a hepatic cyst at the head be exposed to light?", "zh": "患有黄疸并伴有颅内血肿的儿童需要照射灯光到什么时候？"}, {"id": 834, "vi": "Hiện tại, tôi luôn trong tình trạng mệt mỏi và buồn ngủ vào ban ngày. Ngoài ra, tôi còn bị đau họng. Tôi đang mắc bệnh gì và cách điều trị ra sao?", "en": "Currently, I am always in a state of fatigue and daytime drowsiness. In addition, I have sore throat. What disease am I suffering from and how should I treat it?", "zh": "目前，我每天白天都感到疲倦和嗜睡。此外，我还患有咽喉痛。我得了什么病，该如何治疗？"}, {"id": 835, "vi": "Nữ giới sốt, ho, mất vị giác là dấu hiệu bệnh gì?", "en": "Female patient with fever, cough, loss of taste—what is the disease?", "zh": "女性发热、咳嗽、味觉丧失是何种疾病的表现？"}, {"id": 836, "vi": "Tràn dịch màng phổi có nguy cơ bị ung thư không? Điều trị bao lâu mới khỏi?", "en": "Is there a risk of lung cancer with pleural effusion? How long does it take to recover from treatment?", "zh": "胸腔积液有肺癌的风险吗？治疗多久才能痊愈？"}, {"id": 837, "vi": "Tôi bị ngứa da 5, 6 năm nay, tự hết rồi lại bị, hiện tại chủ yếu ngứa ở vùng da đầu và cũng không nhiều như trước, tôi đang uống thuốc huyết áp, bị trĩ, ù tai (giống tiếng u i như tiếng giun buổi đêm yên tĩnh ở vùng quê).", "en": "I have had skin itching for 5 to 6 years, it goes away on its own and then comes back; currently, the itching is mainly on the scalp and not as severe as before. I am currently taking blood pressure medication, I have hemorrhoids, and hearing loss (similar to the sound of worms moving at night in quiet rural areas).", "zh": "我已有5到6年皮肤瘙痒病史，自行缓解后又复发，目前主要在头部皮肤瘙痒，程度比以前轻。我目前正在服用降压药，患有痔疮，耳鸣（类似夜晚乡村安静时蚯蚓蠕动的声音）。"}, {"id": 838, "vi": "Hiện tôi chưa có gia đình. Tôi đi khám và được chẩn đoán mắc chứng rối loạn hỗn hợp lo âu và trầm cảm. Bệnh này có ảnh hưởng đến ham muốn không? Tôi cảm thấy trong người không có ham muốn gì cả, trong khi trước đây tôi vẫn bình thường.", "en": "I currently do not have a family. I went for a check-up and was diagnosed with a mixed anxiety and depression disorder. Does this condition affect libido? I feel completely lacking in desire, while previously I was normal.", "zh": "我目前没有家庭。我去医院检查，被诊断为混合性焦虑和抑郁障碍。这种病会影响性欲吗？我现在完全没有性欲，而以前是正常的。"}, {"id": 839, "vi": "Muốn có bầu tháng sau có nên tiêm phòng trước không", "en": "Do I need to get vaccinated before trying to get pregnant next month?", "zh": "下个月想要怀孕，是否应该提前接种疫苗？"}, {"id": 840, "vi": "Tôi xét nghiệm cho HbsAg Dương tính (7,71) khoảng tháng 6/2019. Đến tháng 9/2019 xét nghiệm lại thì cho kết quả Âm tính. Đến tháng 2/2020 tiếp tục xét nghiệm lại cho kết quả HbsAg Âm tính (0,46),HbeAg âm tính (0,54), HbcAb Dương tính total (0,02). Tôi muốn hỏi với kết quả này tôi có thể đi hiến máu được không?", "en": "I had a test for HbsAg positive (7.71) in June 2019. In September 2019, the test result was negative. In February 2020, the test result was HbsAg negative (0.46), HbeAg negative (0.54), HbcAb positive total (0.02). I would like to ask whether with this result I can donate blood?", "zh": "我于2019年6月检查发现HBsAg阳性（7.71）。到2019年9月复查时结果为阴性。2020年2月再次检查，结果为HBsAg阴性（0.46），HBeAg阴性（0.54），HbcAb总抗体阳性（0.02）。我想咨询一下，根据这个结果，我是否可以献血？"}, {"id": 841, "vi": "Siêu âm hình thể giúp phát hiện các dị tật thai nhi nào?", "en": "What fetal anomalies can ultrasound morphology detect?", "zh": "胎儿结构超声检查能发现哪些先天畸形？"}, {"id": 842, "vi": "Đánh giá kết quả điều trị bệnh tuyến giáp tirads 3, tirads 4", "en": "Assessment of the treatment outcome for thyroid disease TIRADS 3, TIRADS 4", "zh": "评估甲状腺扫描分级3级和4级的治疗效果"}, {"id": 843, "vi": "Cháu đau bụng đi ngoài uống thuốc berberin được không?", "en": "Can I take berberin if I have abdominal pain and diarrhea?", "zh": "可以吃伯Berberin治疗腹痛腹泻吗？"}, {"id": 844, "vi": "Cháu thường bị nổi mề đay vào buổi chiều và tối nhưng ngủ dậy vào buổi sáng lại khỏi. Cháu có đi khám, bác sĩ họ kê thuốc nhưng cháu không thấy khỏi.", "en": "I usually get hives in the afternoon and evening, but they disappear when I wake up in the morning. I have visited a doctor, and they prescribed medication, but I haven't seen any improvement.", "zh": "我通常在下午和晚上出现荨麻疹，但早上醒来时就会消失。我曾去医院就诊，医生开了药，但我没有好转。"}, {"id": 845, "vi": "Bệnh ù tai có liên quan đến co thắt mạch máu không?", "en": "Is hearing loss related to blood vessel constriction?", "zh": "耳聋是否与血管痉挛有关？"}, {"id": 846, "vi": "Có thể chữa khỏi bệnh teo dây thần kinh thị giác không?", "en": "Can visual nerve atrophy be cured?", "zh": "视觉神经萎缩这种病能治愈吗？"}, {"id": 847, "vi": "Tôi bị đau gáy bên trái, đau tức rất khó chịu, nhất khi khi ăn thức ăn như lòng lợn, uống bia, rượu nếp... Tôi đã đi chụp X-quang, tôi bị thoái hóa đốt sống lưng nên bị chèn ép. Tư vấn giúp tôi có phải bị thế không? Cách chữa như thế nào, có khỏi được không?", "en": "I have left-sided throat pain, severe dull pain, especially when eating pork, drinking beer, or rice wine. I have undergone X-ray examination, and I was diagnosed with lumbar spine degeneration causing compression. Please advise whether I have this condition. What is the treatment method, and can I recover fully?", "zh": "我左侧喉咙疼痛，有压迫感，非常难受，尤其在吃猪肉、喝啤酒、米酒等食物时更明显。我做了X光检查，发现有腰椎退行性病变，导致椎体受压。请问这种情况是否属实？该如何治疗？能否痊愈？"}, {"id": 848, "vi": "Xuất hiện sỏi 4x8 mm nằm ở niệu quản, điều trị thế nào?", "en": "A 4x8 mm stone is located in the ureter. How to treat it?", "zh": "在输尿管处发现一个4x8 mm的结石，应如何治疗？"}, {"id": 849, "vi": "Tôi sinh mổ nay được 1 tháng liệu có ăn được trứng gà chưa ạ?", "en": "I had a cesarean section one month ago, can I eat chicken eggs now?", "zh": "我剖腹产一个月了，可以吃鸡蛋吗？"}, {"id": 850, "vi": "Bị tụ máu quanh màng nhện và phình mạch máu não do ngã xe có làm nút mạch được không?", "en": "Will having subdural hematoma and cerebral aneurysm due to car accident lead to vessel occlusion?", "zh": "脑出血和脑动脉瘤因车祸引起，这种情况能做血管栓塞吗？"}, {"id": 851, "vi": "Thai em 28 tuần em có đi xét nghiệm nước tiểu. Em bị viêm đường tiết niệu bệnh đó ảnh hưởng tới thai nhi không? Bác sĩ có kê đơn dùng thuốc kháng sinh trong 1 tuần. Uống kháng sinh như thế có ảnh hưởng tới thai nhi không? Ngoài dùng kháng sinh ra em có thể dùng thêm thuốc gì cho khỏi được không?", "en": "My daughter is at 28 weeks and has undergone urine testing. She has a urinary tract infection. Does this condition affect the fetus? The doctor has prescribed antibiotic treatment for one week. Will taking antibiotics affect the fetus? Apart from antibiotics, are there any other medications I can take to recover?", "zh": "我女儿28周时做了尿液检查。她患有尿路感染，这种情况会影响胎儿吗？医生开了为期一周的抗生素。服用抗生素会对胎儿有影响吗？除了用抗生素外，我还可以用其他药物来治疗吗？"}, {"id": 852, "vi": "Đau nhức vùng chậu và khớp háng bên trái là do đâu?", "en": "What is the cause of left pelvic and hip joint pain?", "zh": "左侧盆腔和髋关节疼痛的原因是什么？"}, {"id": 853, "vi": "Tôi bị lao phổi đã điều trị khỏi năm 2016 có phù hợp để tiêm vaccine Covid-19 hay không?", "en": "I have had pulmonary tuberculosis treated and cured in 2016. Is it appropriate to receive the COVID-19 vaccine?", "zh": "我于2016年治愈了肺结核，是否可以接种新冠疫苗？"}, {"id": 854, "vi": "Thường bị đổ mồ hôi nhớt trên da mặt là triệu chứng bệnh gì?", "en": "What disease has excessive oily sweating on the face as a symptom?", "zh": "经常在面部出现湿性出汗是哪种疾病的症状？"}, {"id": 855, "vi": "Phụ nữ bị chấn thương cột sống D12 khi nào có thể mang thai?", "en": "When can women with spinal D12 trauma become pregnant?", "zh": "脊柱第12节段受伤的女性何时可以怀孕？"}, {"id": 856, "vi": "Chân em bị nổi những lằn máu đỏ tím ở dưới chân giống như gân máu vậy ạ.Em không biết là em bị sao vậy ạ", "en": "My foot has developed red-brown lines under the foot, resembling blood vessels. I don't know what's wrong with me.", "zh": "我的脚出现了类似静脉的红色到紫红色条状出血，我不清楚自己得了什么病。"}, {"id": 857, "vi": "Em tôi bị viêm phổi đã được bác sĩ cho uống thuốc tuy đã đỡ nhưng giờ còn hay nôn mửa và đi ngoài vậy cần phải tránh những gì?", "en": "My child has pneumonia and has been given medication by the doctor; although symptoms have improved, they are still frequently vomiting and having diarrhea. What should we avoid?", "zh": "我孩子得了肺炎，医生已经开了药，虽然症状有所缓解，但现在仍然经常呕吐和腹泻，需要避免什么？"}, {"id": 858, "vi": "Chảy máu cam là dấu hiệu cảnh báo bệnh lý gì?", "en": "What disease is nosebleed a warning sign of?", "zh": "流鼻血是何种疾病的预警信号？"}, {"id": 859, "vi": "Trẻ khi khóc hoặc chạy nhảy, bìu thường to một bên nên điều trị thế nào?", "en": "What should be done when a child's scrotum becomes enlarged on one side during crying or running and jumping?", "zh": "孩子哭闹或跑跳时，阴囊常一侧变大，应如何治疗？"}, {"id": 860, "vi": "Dương tính Hp, viêm loét manh tràng, dạ dày nên điều trị thế nào?", "en": "Hp positive, ulcerative ileitis, gastritis - how should it be treated?", "zh": "幽门螺杆菌阳性，回肠末端炎，胃炎应如何治疗？"}, {"id": 861, "vi": "Sưng cứng nướu bên dưới là dấu hiệu bệnh gì?", "en": "What is the condition indicated by swollen and hard gums?", "zh": "下方牙龈肿胀硬结是何种疾病的征兆？"}, {"id": 862, "vi": "Nguyên nhân gây suy nhược cơ thể sau cơn sốt là do đâu?", "en": "What is the cause of physical weakness after a fever?", "zh": "发热后出现全身虚弱的原因是什么？"}, {"id": 863, "vi": "Bác sĩ cho em hỏi tình trạng rau bám mép có nguy hiểm không ạ?", "en": "Doctor, I would like to ask you whether marginal placenta attachment is dangerous?", "zh": "医生请问胎盘边缘着床危险吗？"}, {"id": 864, "vi": "Chỉ số xét nghiệm là 6,6 mol, liệu có nguy cơ mắc tiểu đường hay không?", "en": "The test result is 6.6 mmol/L, is there a risk of developing diabetes?", "zh": "检测值为6.6 mmol/L，是否具有糖尿病风险？"}, {"id": 865, "vi": "Đau đầu kéo dài uống thuốc không khỏi điều trị như thế nào?", "en": "Chronic headache not relieved by medication - how to treat?", "zh": "持续头痛，用药无效，该如何治疗？"}, {"id": 866, "vi": "Huyết áp thấp có ảnh hưởng đến kết quả lưu huyết não không?", "en": "Does low blood pressure affect cerebral perfusion outcomes?", "zh": "低血压会影响脑血流灌注结果吗？"}, {"id": 867, "vi": "Em năm nay 21 tuổi, em hay bị sôi bụng và đau bụng ạ. Vậy cho em hỏi là em bị gì ạ?", "en": "I am 21 years old and often experience abdominal bloating and abdominal pain. So, I would like to ask, what is my condition?", "zh": "我今年21岁，经常出现腹胀和腹痛，想问问我是得了什么病？"}, {"id": 868, "vi": "Tối hôm đó em có đi học về muộn, trời mưa khá lớn. Không hiểu sao khi về có vài chỗ trên tay và một ít vết lằn đỏ ở mặt. Vài ngày sau, mấy chỗ đó bắt đầu hóa mủ. Em đã mua thuốc uống và thuốc bôi nhưng không giảm, thậm chí những vết đó còn lây lan ra những chỗ khác. Vậy em bị gì ạ?", "en": "That evening, I came back from school late, and it rained heavily. I don't know why, but when I got home, I noticed a few spots on my hands and a little red streaks on my face. A few days later, those spots started to become purulent. I bought oral and topical medications, but they didn't help; in fact, the lesions spread to other areas. So, what illness do I have?", "zh": "那天晚上我晚自习回家，天下大雨。回家后手上出现几处红斑，脸上也有少量红色条纹。几天后，这些部位开始化脓。我买了口服药和外用药，但症状没有缓解，反而这些红肿部位还扩散到了其他地方。我这是得了什么病？"}, {"id": 869, "vi": "Cục nhỏ màu vàng dưới lông mi là gì", "en": "What is a small yellow spot under the eyelash?", "zh": "眼睫毛根部的小黄色结节是什么"}, {"id": 870, "vi": "Một bên mắt bé dần lại có phải triệu chứng của bệnh nhược cơ không?", "en": "Is gradual eye weakness a symptom of muscle weakness disease?", "zh": "孩子一侧眼睛逐渐变小，是否是肌无力病的症状？"}, {"id": 871, "vi": "Em sờ ở dưới hàm bên trái, thấy có một dạng như hạch nhỏ. Bình thường không đau gì cả, chỉ khi em lấy tay chạm vào, sờ thì nó nhức nhẹ. Em không rõ triệu chứng này thế nào. Em không uống rượu bia, hút thuốc gì cả.", "en": "I palpated under the left mandible and found a small lump. It does not hurt normally, only slightly painful when I touch it with my hand. I am not sure about this symptom. I do not consume alcohol or smoke anything.", "zh": "我在左下颌处摸到一个类似淋巴结的小肿块。平时没有疼痛，只有用手触摸或按压时才会感到轻微的疼痛。我不清楚这种症状是什么情况。我从不饮酒，也不吸烟。"}, {"id": 872, "vi": "Em muốn hỏi cách điều trị bệnh loạn sản sụn xương.", "en": "I would like to ask about the treatment of cartilage osteochondrosis disease.", "zh": "我想问一下骨软骨病的治疗方法。"}, {"id": 873, "vi": "Điều trị viêm mũi dị ứng hiệu quả bằng cách nào?", "en": "How to effectively treat allergic rhinitis?", "zh": "如何有效治疗过敏性鼻炎？"}, {"id": 874, "vi": "Tôi bị cao huyết áp vô căn, mỡ máu, như vậy có đủ điều kiện để tiêm vaccine Covid-19 hay không?", "en": "I have essential hypertension and high lipid levels. Is this sufficient to receive the COVID-19 vaccine?", "zh": "我患有原发性高血压和高血脂，这种情况是否符合接种新冠疫苗的条件？"}, {"id": 875, "vi": "Dùng thuốc điều trị thiếu máu do thiếu sắt như thế nào không bị dư sắt?", "en": "How to use iron deficiency anemia treatment drugs without excess iron?", "zh": "如何使用铁剂治疗缺铁性贫血而不致铁过量？"}, {"id": 876, "vi": "Hiện tại em đang bầu ở tuần thứ 24. Em bị nẻ tay, em có sử dụng thuốc mỡ tra mắt tetracyclin bôi tay 2 lần qua đêm thì có ảnh hưởng gì tới em bé k ạ. Em rất lo lắng và muốn nghe lời khuyên của bác sĩ.", "en": "Currently, I am at 24 weeks of pregnancy. I have cracked hands, and I have been using tetracycline ointment applied to hands twice nightly. Will this affect my baby? I am very worried and would like to hear the doctor's advice.", "zh": "目前我怀孕24周，我有手掌皲裂的情况，我用了两次夜间涂抹四环素眼膏于手掌，这对胎儿有影响吗？我非常担心，想听听医生的建议。"}, {"id": 877, "vi": "Em lấy chồng được hơn một năm, kinh nguyệt không đều. Em muốn kiểm tra nội tiết có vấn đề gì không, vậy muốn kiểm tra nội tiết thì làm xét nghiệm gì ạ?", "en": "I have been married for more than one year, and my menstrual cycle is irregular. I want to check if there is any hormonal issue, so which tests are needed for hormonal evaluation?", "zh": "我结婚一年多，月经不规律。我想检查一下内分泌是否有问题，那么想检查内分泌需要做哪些检查呢？"}, {"id": 878, "vi": "Mẹ đang cho con bú có bị giới hạn loại vaccine ngừa Covid-19 hay cân nhắc cai sữa cho bé trước khi tiêm không? Nếu không thì em bé bú sữa mẹ sau khi tiêm có nguy hiểm gì không? Em bé 7 tháng trở lên có được tiêm vaccine Covid-19 không?", "en": "Is there a restriction on administering COVID-19 vaccines to mothers who are breastfeeding, or should breastfeeding be discontinued before vaccination? If not, is there any risk to the infant after vaccination if the baby continues to breastfeed? Can infants aged 7 months and older receive the COVID-19 vaccine?", "zh": "母亲在哺乳期间是否需要限制接种新冠疫苗，或者是否需要在接种前停止哺乳？如果不需要，接种疫苗后继续哺乳对婴儿是否有危险？7个月以上的婴儿可以接种新冠疫苗吗？"}, {"id": 879, "vi": "Ánh sáng xanh có hại như thế nào đối với điểm vàng của mắt?", "en": "What harm does blue light cause to the macula of the eye?", "zh": "蓝光对眼睛黄斑区有什么危害？"}, {"id": 880, "vi": "Em được chẩn đoán bị hạt ở thành tử cung và phải cắt. Bác sĩ cho em hỏi bị hạt là bị gì, nguyên nhân gây bệnh và cách điều trị.", "en": "The patient is diagnosed with a polyp in the uterine cavity and needs to undergo surgery. The doctor asks the patient: What is a polyp, what are the causes of the disease, and what is the treatment?", "zh": "医生告知我被诊断为子宫内膜息肉，需要手术切除。医生想问一下，什么是子宫内膜息肉，病因是什么，以及治疗方法是什么。"}, {"id": 881, "vi": "Nổi mẩn đỏ kèm khó thở do dị ứng thuốc kháng sinh có nên ngưng không?", "en": "Should we discontinue red rash with difficulty breathing due to antibiotic allergy?", "zh": "是否应停止使用抗生素引起的药物过敏导致的皮疹和呼吸困难？"}, {"id": 882, "vi": "Âm tính với HBV có thể dùng tỏi đen không?", "en": "Can people with HBV use black garlic?", "zh": "乙肝表面抗原阳性可以吃黑蒜吗？"}, {"id": 883, "vi": "Kết quả siêu âm chẩn đoán thận trái có 2 quả, vì sao?", "en": "What is the reason for the ultrasound diagnosis showing two kidneys on the left side?", "zh": "左肾超声检查显示有2个肾脏，为什么？"}, {"id": 884, "vi": "Thường xuyên đau đầu, chóng mặt, mất ngủ là biểu hiện của những bệnh gì?", "en": "Frequent headache, dizziness, insomnia - what diseases are these symptoms indicative of?", "zh": "经常头痛、头晕、失眠是哪些疾病的症状？"}, {"id": 885, "vi": "Em có đi khám và bác sĩ nói rằng em bị viêm âm đạo do cầu khuẩn Gram dương (+) và trực khuẩn Gram âm (+). Cho em hỏi trường hợp của em thế này có bị nặng không, và có khả năng lây sang người có quan hệ với em không? Em có nên đưa người có quan hệ với em đi thăm khám hay không, và có nên làm các xét nghiệm về các bệnh lây qua đường tình dục như lậu, HPV, giang mai hay không?", "en": "My doctor said I have vaginal inflammation caused by Gram-positive cocci (+) and Gram-negative bacilli (+). I would like to know whether my condition is severe, and whether there is a risk of transmission to my sexual partner. Should I have my sexual partner examined, and should I get tested for sexually transmitted infections such as syphilis, HPV, and gonorrhea?", "zh": "我去了医院检查，医生说我的阴道感染是由革兰氏阳性菌（+）和革兰氏阴性菌（+）引起的。我想问一下，我这种情况严重吗？会传染给和我有性关系的人吗？我是否应该让和我有性关系的人也去检查？是否应该做性传播疾病的检查，比如淋病、HPV、梅毒？"}, {"id": 886, "vi": "Bác sỹ cho em hỏi em xét nghiệm máu kiểm tra sức khoẻ thì thấy chỉ số beta crosslaps ( cobas) bị tăng là 463 ( giới hạn cho phép <300). Vậy em có cần đi khám chuyên sâu hơn và cần uống thuốc gì không?", "en": "Doctor, I would like to ask you: when I had a blood test for health check-up, I found my beta crosslaps (cobas) level increased to 463 (the permitted limit <300). Therefore, do I need to see a specialist and take any medication?", "zh": "医生请问一下，我做健康体检的血液检查时，发现beta crosslaps（cobas）指标升高，为463（允许范围<300）。那么我是否需要进一步深入检查，以及需要服用什么药物吗？"}, {"id": 887, "vi": "Tôi mới mổ thay van hai lá cơ học và sửa, đặt vòng van ba lá được 12 tuần. Gần đây, tôi cảm thấy nhịp tim không đều, lúc nhanh lúc chậm, thỉnh thoảng kèm theo trống ngực. Tôi vẫn tái khám và sử dụng thuốc theo chỉ định. Gần đây nhất, bác sĩ có bổ sung thuốc nhưng tình trạng trên vẫn không giảm. Triệu chứng trên có nguy hiểm không và điều trị như thế nào để khắc phục?", "en": "I recently underwent mechanical mitral valve replacement and received repair with a tricuspid valve ring placement 12 weeks ago. Recently, I have felt an irregular heartbeat, alternating between fast and slow, occasionally accompanied by chest discomfort. I continue to follow up with my doctor and take medications as prescribed. Recently, my doctor has added new medications, but these symptoms have not improved. Are these symptoms dangerous, and how should they be treated to resolve them?", "zh": "我刚接受了机械二尖瓣置换及三尖瓣成形术和三尖瓣环成形术，术后12周。最近我感觉心跳不规则，时快时慢，偶尔伴有胸闷。我仍按医嘱定期复查并服药。最近医生增加了药物治疗，但上述症状仍未缓解。这些症状是否危险？应如何治疗以改善？"}, {"id": 888, "vi": "Sau khi em sử dụng một lượng rượu bia nhỏ dù chỉ là một ngụm nhỏ cũng đỏ hết cả mặt. Vậy có phải do em thuộc nhóm máu O do mọi người truyền tai nhau không, hay em đang mắc một bệnh gì đó? Có cách nào khắc phục được không?", "en": "After I consume a small amount of alcohol, even just a sip, my entire face turns red. So, is this because I belong to blood group O, as people often say, or am I suffering from some disease? Is there any way to remedy this?", "zh": "在我喝下少量酒精（哪怕只是小口）后，整个面部都会变得发红。这是否是因为我属于O型血，大家口耳相传的缘故，还是因为我患有某种疾病？有没有办法可以解决这个问题？"}, {"id": 889, "vi": "Mỗi lần quan hệ xong là âm đạo đau bụng và ra máu, nguyên nhân chính là gì?", "en": "Each time after intercourse, there is vaginal pain, abdominal pain, and bleeding. What is the main cause?", "zh": "每次性生活后出现阴道痛、腹痛和出血，最可能的原因是什么？"}, {"id": 890, "vi": "Tư vấn chỉ số HbA1c = 8%, glucose = 7 mmol/L, đề kháng insulin.", "en": "Advice for HbA1c = 8%, glucose = 7 mmol/L, insulin resistance.", "zh": "咨询HbA1c为8%，血糖为7 mmol/L，胰岛素抵抗。"}, {"id": 891, "vi": "Đau người, mệt mỏi sau điều trị Covid tại nhà có sao không?", "en": "Do I have a problem with fatigue and body pain after treating COVID at home?", "zh": "居家治疗新冠后出现全身疼痛、乏力是否正常？"}, {"id": 892, "vi": "Đa nhân hai thùy và eo tuyến giáp có nguy hiểm không?", "en": "Is multinodular goiter of both lobes and isthmus dangerous?", "zh": "双叶和甲状腺峡部多发性是否危险？"}, {"id": 893, "vi": "Em năm nay 27 tuổi có đi siêu âm thì phát hiện có nang dạng lac nội mạc tử cung buồng trứng trái kích thước 28×31mm, cho em hỏi bị như này sẽ ảnh hưởng đến khả năng sinh con nhiều không? Kích thước nang có bị lớn không? Với lại em tính mua dùng Nga Phụ Khang được không ?", "en": "I am 27 years old and found a ovarian endometrioma on the left ovary during ultrasound, measuring 28×31 mm. I would like to know if this condition will affect my fertility? Will the size of the cyst increase? Also, can I purchase and use Nga Phụ Khang?", "zh": "我今年27岁，做B超检查发现左侧卵巢内膜囊肿，大小为28×31mm，想咨询一下这种情况会影响生育能力吗？囊肿会继续增大吗？另外，我可以购买并使用俄罗斯的“诺帕康”吗？"}, {"id": 894, "vi": "Bệnh nhân nữ 45 tuổi cắt hoàn toàn tuyến giáp 1 năm nay ,đang uống 1 viên Berlthyrox 100. Xét nghiệm hiện tại TSH 0,001 uIU/ml.FT4 23pg/ml. T 3 1,1 ng/ml.khám lâm sàng bình thường, sút nhẹ cân. Xin hỏi có cần điều chỉnh thuốc cho bệnh nhân không?", "en": "45-year-old female patient who underwent total thyroidectomy one year ago, currently taking 1 tablet of Levothyroxine 100. Current laboratory results: TSH 0.001 uIU/ml, FT4 23 pg/ml, T3 1.1 ng/ml. Clinical examination is normal, with mild weight loss. Please ask whether medication adjustment is needed for this patient?", "zh": "45岁女性患者已接受甲状腺全切手术1年，目前服用100μg的左甲状腺素钠1片。目前检查结果：TSH 0.001 mIU/ml，FT4 23 pg/ml，T3 1.1 ng/ml，体格检查正常，体重略有下降。请问是否需要调整药物剂量？"}, {"id": 895, "vi": "Xét nghiệm sàng lọc sau sinh nguy cơ cao là thể mang alpha thalassemia vậy thể mang là mắc bệnh hay mang gen bệnh", "en": "Screening test after birth for high risk of alpha thalassemia carrier status: is a carrier affected with the disease or just carrying the disease gene?", "zh": "产后的筛查显示高风险为α地中海贫血携带者，那么携带者是会得病还是只是携带致病基因？"}, {"id": 896, "vi": "Quầng vú bị khô, bong da là dấu hiệu bệnh gì và điều trị thế nào?", "en": "Dry, flaking nipple is a sign of what disease and how to treat it?", "zh": "乳房出现干燥、脱皮是何种疾病的征兆，该如何治疗？"}, {"id": 897, "vi": "Bé em được 15 tháng, bé gái,ba ngày trước bị ho, đi khám song về cho uống thuốc, uống được hai ngày ho bớt đi rất nhiều, nhưng bị khò khè,thở hơi mệt, giờ em phải làm gì", "en": "My daughter is 15 months old, female. Three days ago she had a cough, visited the doctor who prescribed medication, and after taking the medicine for two days, the cough has significantly improved, but she now has wheezing and breathlessness. What should I do now?", "zh": "孩子15个月大，女孩，三天前开始咳嗽，去医院看过后开了药，吃了两天咳嗽明显减轻，但出现喘息、呼吸费力，现在该怎么办？"}, {"id": 898, "vi": "Có cách nào điều trị u đầu tụy tắc nghẽn cho bệnh nhân bị tràn dịch phổi không?", "en": "Is there any treatment for obstructive pancreatic ductal obstruction for a patient with pleural effusion?", "zh": "对于患有肺积液的患者，是否有治疗胰腺梗阻性囊肿的方法？"}, {"id": 899, "vi": "Viêm sụn vành tai kèm chảy mủ có chữa khỏi được không?", "en": "Can cartilage inflammation of the ear and purulent discharge be cured?", "zh": "耳廓软骨炎伴脓性分泌物能否治愈？"}, {"id": 900, "vi": "Tim thai đập không đều có ảnh hưởng gì không ạ?", "en": "Does an irregular heartbeat affect anything?", "zh": "胎儿心律不齐有什么影响吗？"}, {"id": 901, "vi": "Chân em khoảng gần 1 tháng trở lại đây, đá bóng có 5 phút là bị tê đoạn gần cổ chân. Bị cả 2 chân, ngồi nghỉ 5 phút lại đỡ. Em không biết bị gì nhờ tư vấn ạ.", "en": "My legs have been experiencing tingling for nearly one month. After playing football for 5 minutes, I get tingling in the area near the ankle of both legs. It subsides after resting for 5 minutes. I don't know what's wrong, so I need your advice.", "zh": "我大约一个月前开始出现脚部麻木，踢足球5分钟后近脚踝处会麻木，双侧脚都有，休息5分钟后会缓解。我不清楚是什么问题，麻烦咨询一下。"}, {"id": 902, "vi": "Con mình bị tiêu chảy hơn 3 ngày hôm nay. Phân con đi có màu xanh vàng, có bọt. Ngày đi 3 4 lần lúc thì sệ, lúc thì nước. Con vẫn chơi ngoan ăn ngoan, không sốt. Mình đã cho con uống Smecta+men entergermina mà tình trạng không đỡ. Vậy có thuốc nào điều trị tiêu chảy tốt hơn không?", "en": "My child has had diarrhea for more than 3 days today. The stool is green-yellow in color and contains bubbles. The child has bowel movements 3 to 4 times a day, sometimes loose, sometimes watery. The child is still active, eating well, and does not have a fever. I have given the child Smecta + Men Entergermina, but the condition has not improved. Is there any other medication that treats diarrhea better?", "zh": "我家宝宝今天腹泻超过3天。大便呈黄绿色，有泡沫，每天排便3到4次，时而稀，时而水样。宝宝仍然活泼，食欲正常，没有发烧。我已经给宝宝服用过思密达+蒙脱石散，但症状没有改善。请问还有哪些更好的治疗腹泻的药物吗？"}, {"id": 903, "vi": "Phân bị nhạt màu (vàng nhạt) có đốm đen thì có sao không ạ?", "en": "Is it a problem if the stool is pale yellow (light yellow) with black spots?", "zh": "大便颜色变浅（呈淡黄色）并有黑色斑点，会怎样？"}, {"id": 904, "vi": "Xử trí cơn đau tức ngực tại nhà như thế nào?", "en": "How to manage chest pain at home?", "zh": "如何在家中处理胸痛发作？"}, {"id": 905, "vi": "Đau vùng giữa hai lông mày và buốt sau đầu sau sinh 1 năm là triệu chứng của bệnh gì?", "en": "Pain in the area between the two eyebrows and headache behind the head one year postpartum - what disease is this symptom of?", "zh": "产后一年出现眉间疼痛及头部刺痛的症状是什么病的表现？"}, {"id": 906, "vi": "Tiết dịch đầu vú kèm máu có phải do tác dụng phụ của thuốc điều trị viêm gan B không?", "en": "Is milk production from the breast accompanied by blood a side effect of medication for hepatitis B?", "zh": "乳头溢液伴出血是否为抗乙型肝炎病毒药物的副作用？"}, {"id": 907, "vi": "Điều trị giang mai như thế nào?", "en": "How is syphilis treated?", "zh": "梅毒如何治疗？"}, {"id": 908, "vi": "Mẹ cháu có đi khám và được bác sĩ kết luận là bị đa u hỗn hợp hai thùy và eo tuyến giáp, có chỉ định mổ. Nhưng do mẹ cháu lo sợ nên vẫn chưa mổ sau hơn 1 năm khám. Nên cháu muốn hỏi bác sĩ là bệnh của mẹ cháu nếu để lâu thì có nguy hiểm gì không?", "en": "My mother has been examined and the doctor diagnosed her with mixed multinodular goiter involving both lobes and isthmus, with surgical indication. However, due to her fear, she has not undergone surgery yet, more than one year after the examination. Therefore, I would like to ask the doctor whether my mother's condition could become dangerous if left untreated?", "zh": "我母亲已经看过医生，被诊断为甲状腺双叶及峡部多发性混合结节，医生建议手术。但由于母亲非常担心，已经过去一年多仍未接受手术。因此我想咨询一下医生，如果这种情况长期不治疗，会对母亲的健康造成什么危险？"}, {"id": 909, "vi": "Em đang mang thai tuần 37, trước đây khi đầu và giữa thai kỳ em thường xuyên súc miệng bằng Listerine và sử dụng thuốc xịt thơm miệng Oralan, không biết có ảnh hưởng gì không vì giờ em nhớ ra nên khá lo lắng. Không biết có vấn đề gì với thai nhi không?", "en": "I am currently 37 weeks pregnant. In the first and second trimesters, I frequently used Listerine to rinse my mouth and used the oral spray Oralan. I am now wondering if this could have any effect, as I recently remembered doing this and am quite concerned. Is there any problem with the fetus?", "zh": "我目前怀孕37周，以前在孕早期和中期经常用Listerine漱口，并使用Oralan口腔喷雾剂，现在想起来有点担心，不知道这样会对胎儿有影响吗？胎儿会不会有问题？"}, {"id": 910, "vi": "Tôi bị tai nạn giao thông chấn thương mắt mổ cách đây 10 năm nay mắt đục thuỷ tinh thể nhưng bác sĩ chuẩn đoán bệnh do sẹo giác mạc, theo bác sỹ có cách nào giải quyết không", "en": "I suffered a traffic accident causing ocular trauma and had eye surgery 10 years ago. My lens is cloudy, but the doctor diagnosed the disease as due to corneal scar. According to the doctor, is there any way to resolve this?", "zh": "我10年前因交通事故导致眼部创伤，接受了手术治疗，目前存在玻璃体混浊，医生诊断为角膜瘢痕所致，医生是否有办法解决？"}, {"id": 911, "vi": "Vì sao răng bị nứt", "en": "Why does a tooth crack?", "zh": "为什么牙齿会裂开"}, {"id": 912, "vi": "Em có xuất hiện vài cái hạch ở cánh tay và hai hố nách. Em đi siêu âm bác sĩ nói là hạch viêm bai không cho làm sinh thiết. Bây giờ em phải làm thế nào. Em có nguy cơ mắc ung thư không.", "en": "I have noticed a few lymph nodes in my arm and in both armpits. I went for an ultrasound, and the doctor said they are inflamed lymph nodes and did not recommend a biopsy. Now, what should I do? Do I have a risk of developing cancer?", "zh": "我手臂和腋窝出现了几个淋巴结，医生做了超声检查，说这是炎症性淋巴结，没有建议做活检。现在我该怎么办？我有患癌症的风险吗？"}, {"id": 913, "vi": "Căng tức bàng quang vào buổi sáng có bất thường không?", "en": "Is it abnormal to have bladder discomfort in the morning?", "zh": "早晨膀胱不适是否异常？"}, {"id": 914, "vi": "Tôi năm nay 26 tuổi đã lập gia đình. Tôi xuất hiện gai nhú tại môi bé âm đạo, ngứa khô không chảy dịch, xét nghiệm HPV âm tính vậy tôi có khả năng bị sùi mào gà không?", "en": "I am 26 years old and have been married. I have noticed papular growths at the vaginal introitus, with itching and dryness without discharge. The HPV test is negative. Am I likely to have genital warts?", "zh": "我今年26岁，已结婚。我外阴小阴唇出现赘生物，有瘙痒干涩，无分泌物，HPV检测为阴性，我有尖锐湿疣的可能吗？"}, {"id": 915, "vi": "Chỉ số bạch cầu trung tính 6,56 G/L có bình thường không?", "en": "Is a neutrophil count of 6.56 G/L normal?", "zh": "中性粒细胞计数 6.56 ×10⁹/L 是否正常？"}, {"id": 916, "vi": "Cháu bị trào ngược dạ dày thực quản điều trị được khoảng 1 năm rưỡi thấy cũng gần hết nhưng khoảng một tháng nay cháu ăn cơm không được ăn là nhợn ối lên họng, biếng ăn, ăn cơm không ngon. Bụng thì đau âm ỉ, miệng đắng cho cháu hỏi cháu bị đau dạ dày lại hay bị bệnh nào khác và cần làm những xét nghiệm gì", "en": "I have been suffering from gastroesophageal reflux disease treated for about 1.5 years, and the symptoms have nearly disappeared, but for about a month now, when eating rice, I feel sourness rising to the throat, loss of appetite, and poor taste when eating rice. I also have intermittent abdominal pain and a bitter taste in my mouth. I would like to know whether I have developed gastric pain again or another condition, and which tests I need to undergo.", "zh": "我患有胃食管反流病，治疗大约一年半，症状已接近缓解，但最近一个月吃米饭时会感觉食物反流到喉咙，食欲差，吃米饭没有味道。腹部有隐隐作痛，口苦。我想咨询一下，我是否又出现了胃炎，或者是否有其他疾病，需要做哪些检查？"}, {"id": 917, "vi": "Bố mẹ và em trai đều mang gien bệnh beta thalassime", "en": "Both parents and the brother are carriers of the beta thalassemia gene.", "zh": "父母和弟弟均携带β-地中海贫血基因"}, {"id": 918, "vi": "Mấy ngày nay em thấy hơi đau ở ngực bên trái, mức độ đau nhẹ nhưng em thấy khác so với đau khi gần đến chu kì kinh nguyệt, mà chỉ ngực bên trái đau và hơi căng, còn ngực bên phải thì ko sao. Em sờ cũng ko thấy có cục cứng hay khối u gì cả, tuy nhiên em vẫn cảm thấy lạ và bất thường. Bác sĩ cho em hỏi đau ngực vậy là dấu hiệu của những bệnh gì và e có phải đi khám không?", "en": "Recently, I have felt a slight pain on the left side of my chest, a mild pain but different from the pain I usually experience near my menstrual cycle; only the left side of the chest hurts with a feeling of tightness, while the right side is fine. I have not felt any lumps or masses upon examination, yet I still feel strange and unusual. Doctor, could you please advise me on what conditions this chest pain might indicate and should I go for a check-up?", "zh": "最近几天我感觉左侧胸部有点疼痛，疼痛程度较轻，但和月经前的疼痛感觉不同，只有左侧胸部疼痛并有轻微紧绷感，右侧胸部没有不适。我摸了摸也没有发现硬块或肿块，但仍然觉得异常。医生请问一下，这种胸痛可能是哪些疾病的征兆，我需要去医院检查吗？"}, {"id": 919, "vi": "Đái rắt khi đặt sonde JJ trong niệu quản có sao không?", "en": "Is it bad to have frequent urination when placing JJ catheter into the ureter?", "zh": "插入JJ导管到输尿管时会出现尿频吗？"}, {"id": 920, "vi": "Em vừa chích xong mũi thứ 3 của HPV. Nếu công ty cho chích vắc xin Covid-19 thì em có chích luôn được không ạ? Em không có bị vấn đề bị dị ứng thuốc hay sốt sau khi tiêm thuốc bao giờ.", "en": "I just finished the third dose of HPV vaccine. If the company offers the COVID-19 vaccine, can I get it as well? I have never had any drug allergy or fever after receiving any injection.", "zh": "我刚刚完成了HPV的第三针接种。如果公司提供新冠疫苗，我可以立即接种吗？我从未对药物产生过过敏或接种后发热的情况。"}, {"id": 921, "vi": "Viêm lưỡi kéo dài có sao không?", "en": "Can prolonged lingual inflammation lead to complications?", "zh": "长期舌炎会有影响吗？"}, {"id": 922, "vi": "Mổ đục thủy tinh thể (cườm khô) ở mắt phải kiêng quan hệ tình dục bao lâu?", "en": "How long after right eye cortical cataract surgery (dry cataract) should sexual intercourse be avoided?", "zh": "右眼玻璃体混浊（干性）手术后多久禁止性生活？"}, {"id": 923, "vi": "Làm sao để biết trẻ đang tự kỷ ở mức độ nhẹ hay nặng?", "en": "How to determine whether a child has autism at a mild or severe level?", "zh": "如何判断儿童自闭症属于轻度还是重度？"}, {"id": 924, "vi": "Em bị sa trực tràng hay nặng vùng đi nặng bây giờ phải làm sao trực tràng mới rút lên được ạ?", "en": "My child has severe rectal prolapse, now the rectum cannot be pulled up anymore. What should I do now?", "zh": "我患有严重直肠脱垂，现在肛门处脱出严重，该怎么办？直肠怎么才能缩回去呢？"}, {"id": 925, "vi": "Ngứa khi mang thai có ảnh hưởng sức khỏe không và cách giảm", "en": "Does itching during pregnancy affect health and how to reduce it?", "zh": "怀孕时瘙痒会影响健康吗以及如何缓解"}, {"id": 926, "vi": "Mắc quai bị khi nào có thể vận động mạnh?", "en": "When can someone with mumps engage in strenuous activity?", "zh": "患腮腺炎时在什么情况下可以进行剧烈运动？"}, {"id": 927, "vi": "Nam giới sụt cân nhanh liên tục nên làm gì?", "en": "What should a man do if he has continuous rapid weight loss?", "zh": "男性持续体重下降应怎么办？"}, {"id": 928, "vi": "Xét nghiệm feritin có thể tăng trong các trường hợp nào ?", "en": "In which cases can ferritin levels increase?", "zh": "铁蛋白检查在哪些情况下可能升高？"}, {"id": 929, "vi": "83 tuổi có điều trị sa trực tràng và sa sinh dục được nữa không?", "en": "Can a 83-year-old patient still be treated for rectal prolapse and pelvic organ prolapse?", "zh": "83岁还能继续治疗直肠脱垂和生殖器脱垂吗？"}, {"id": 930, "vi": "Em cháu hay bị đau đầu chóng mặt hay bị nhức đầu", "en": "The child often suffers from headache, dizziness, or head pain.", "zh": "孩子经常出现头痛、头晕或头痛的情况"}, {"id": 931, "vi": "Người tôi thường hay bị ho kéo dài rất khó chữa, làm cách nào để chữa khỏi cơn ho?", "en": "The person I often suffers from prolonged cough that is very difficult to treat; how can I cure the cough episode?", "zh": "我经常患有难以治愈的慢性咳嗽，该如何治疗才能痊愈？"}, {"id": 932, "vi": "Ngón chân đau nhức kèm bầm tím ở người tiểu đường có sao không?", "en": "Does foot pain and bruising in diabetics pose a problem?", "zh": "糖尿病患者脚趾疼痛伴瘀斑是否有问题？"}, {"id": 933, "vi": "Đường khớp vành của trẻ gồ lên có phải bất thường?", "en": "Is the aortic arch elevation in children abnormal?", "zh": "儿童的主动脉弓凸起是否异常？"}, {"id": 934, "vi": "Viêm màng bồ đào từ nhỏ có ghép được giác mạc hay thủy tinh thể không?", "en": "Can a patient with early pigmentary vitritis undergo corneal or lens transplantation?", "zh": "从小就有葡萄膜炎是否可以进行角膜或晶状体移植？"}, {"id": 935, "vi": "Tôi là nam 36 tuổi, ngày 21/10 tôi đi nội soi, kết quả viêm dạ dày, HP dương tính. Tôi được kê đơn diệt HP và thuốc có tên omeprazon, đơn thuốc cho thuốc này là 1 tháng, sau đó nghỉ thuốc 14 ngày mới đến khám lại. Tôi uống sắp hết thuốc nhưng vẫn còn đau dạ dày thì nên uống tiếp hay ngừng thuốc 14 ngày để đi khám lại? Nếu tiếp tục uống thuốc thì uống những loại nào. Đơn thuốc của tôi có omeprazon, gastropugite, amoxicilin, bismuth, klacid.", "en": "I am a 36-year-old male. On October 21, I underwent endoscopy, and the results showed gastritis with positive H. pylori. I was prescribed H. pylori eradication therapy and omeprazole. The prescription for omeprazole is for one month, followed by a 14-day break before follow-up. I am almost finished with the medication but still have stomach pain—should I continue taking the medication or stop for 14 days to see the doctor again? If I continue taking the medication, which types should I take? My prescription includes omeprazole, gastropugite, amoxicillin, bismuth, klacid.", "zh": "我是一名36岁的男性，10月21日进行了胃镜检查，结果显示患有胃炎，幽门螺杆菌（HP）阳性。医生开具了根除幽门螺杆菌的药物，其中包括奥美拉唑，处方为一个月，之后休息14天再复诊。我快喝完药了，但胃痛仍然存在，我应该继续服药还是停药14天后再复诊？如果继续服药，应该服用哪些药物？我的处方包括奥美拉唑、胃复安、阿莫西林、铋剂、克拉霉素。"}, {"id": 936, "vi": "Cách đây hơn 2 tháng em có quan hệ tình dục không an toàn. Sau tầm một tháng em có biểu hiện sốt nhẹ và ho ( cũng có thể do đổi môi trường làm việc), gần đây tay em hay nổi mụn nhỏ khi tắm xong như dị ứng thời tiết nhưng em hoang mang đấy có phải triệu chứng của bệnh. Em có thể đi xét nghiệm trong khoảng thời gian này không?", "en": "More than two months ago, I had unprotected sexual intercourse. After about one month, I started experiencing mild fever and cough (which might also be due to changing work environment). Recently, my hands often develop small bumps after bathing, resembling weather-related allergy, but I am worried whether this could be a symptom of an illness. Can I get tested during this period?", "zh": "两个月前我有过不安全的性行为。一个月后我出现轻微发热和咳嗽（也可能是因为工作环境改变），最近洗澡后双手经常出现小疙瘩，像天气过敏一样，但我很担心这是否是疾病的症状。我现在可以进行相关检查吗？"}, {"id": 937, "vi": "Lợi trùm răng khôn hàm dưới gây khó chịu nên làm gì?", "en": "Impacted lower third molar causes discomfort; what should be done?", "zh": "下颌智齿冠周炎引起不适该怎么办？"}, {"id": 938, "vi": "U tuyến yên dùng thuốc có ảnh hưởng đến khả năng có thai không và làm thế nào để dễ có bầu?", "en": "Does pituitary hormone medication affect fertility and how to increase the chances of conception?", "zh": "垂体瘤用药会影响生育能力吗？如何提高受孕几率？"}, {"id": 939, "vi": "Làm gì để phục hồi sau mổ u tủy ở cổ?", "en": "What should be done to recover after surgery for cervical medullary tumor?", "zh": "如何恢复颈椎髓内肿瘤手术后的状态？"}, {"id": 940, "vi": "Nhiễm trùng máu, hạ huyết áp ảnh hưởng đến mổ sỏi mật tụy ngược dòng?", "en": "Blood infection, hypotension affect retrograde pancreatobiliary lithotripsy?", "zh": "败血症、低血压是否会影响胆总管结石手术？"}, {"id": 941, "vi": "Cháu tôi 6,5 tháng tuổi có hiện tượng tóc trên đầu cháu có hai vùng phân biệt phần đỉnh đầu tóc đen đậm và cứng hơn tóc hai bên thái dương và vùng gáy. Xin hỏi bác sĩ vì sao có hiện tượng này? Như vậy có cần khám chữa không?", "en": "My nephew is 6.5 months old and has a phenomenon where the hair on his head has two distinct areas: a dark, thicker, and harder hair area at the vertex, and hair on the sides of the forehead and nape that is different. I would like to ask the doctor why this occurs? Does this require a visit and treatment?", "zh": "我6.5个月大的孙子头部出现两种不同区域的头发：头顶部分头发颜色深、质地较硬，而两侧太阳穴和后脑部位的头发则较浅、质地较软。请问医生这是什么原因？这种情况需要检查治疗吗？"}, {"id": 942, "vi": "Trẻ bị sốt liên tục khi điều trị viêm VA có sao không?", "en": "Will a child have any complications if fever persists during treatment for viral infection?", "zh": "持续治疗VA炎患儿会有什么问题吗？"}, {"id": 943, "vi": "Điều trị táo bón, chướng bụng kéo dài.", "en": "Treatment of chronic constipation and bloating.", "zh": "治疗慢性便秘、腹胀。"}, {"id": 944, "vi": "Em bị viêm da dị ứng cơ địa. Làm sao để bệnh không tái lại? Viêm da dị ứng có gây nổi mụn không ạ? Mụn có tự hết không ạ, nhất là mụn đầu đen?", "en": "My child has atopic dermatitis. How can the condition be prevented from recurring? Can atopic dermatitis cause acne? Does acne go away on its own, especially blackheads?", "zh": "我患有特应性皮炎。如何才能防止病情复发？特应性皮炎会引发痘痘吗？痘痘会自行消失吗，尤其是黑头？"}, {"id": 945, "vi": "Em có đi siêu âm em được 29 tuần 1170g bác sĩ nói đầu thai nhi hơi to so với bụng của bé BDP là 7.55cm chu vi bụng AC 24.42cm thì có nguy hiểm không? Làm cách nào để cân bằng đầu với bụng của em bé", "en": "My ultrasound was done at 29 weeks, 1170g. The doctor said the fetal head is a bit larger than the abdomen. BDP is 7.55 cm, abdominal circumference AC is 24.42 cm. Is this dangerous? How can I balance the head size with the abdomen of my baby?", "zh": "我做了29周的B超，体重1170克，医生说胎儿头围偏大，头围（BPD）为7.55厘米，腹围（AC）为24.42厘米，这样有危险吗？如何才能使胎儿头围与腹围平衡？"}, {"id": 946, "vi": "Nguyên nhân tỷ lệ bạch cầu ưa axit cao hơn so với mức trung bình?", "en": "What is the cause of a higher ratio of acidophilic granulocytes compared to the average?", "zh": "为何嗜酸性粒细胞比率高于平均水平？"}, {"id": 947, "vi": "Khoảng một tuần trở lại đây, mình cảm thấy thị lực giảm một cách rõ rệt nhưng không rõ nguyên nhân vì sao. Mắt luôn có cảm giác cộm và nhức. Mình có tiền sử bị bệnh xoang. Nhờ bác sĩ tư vấn giúp mình nguyên nhân có thể là gì?", "en": "For about a week now, I have noticed a significant decline in my vision, but I don't know the cause. My eyes always feel gritty and painful. I have a history of sinus disease. I kindly ask the doctor to advise me on possible causes.", "zh": "最近一周，我明显感到视力下降，但不清楚原因。眼睛总是有胀痛感。我有慢性鼻窦炎病史。请医生帮我咨询一下可能的原因。"}, {"id": 948, "vi": "Ba tôi năm nay 69 tuổi, có bệnh lý nền: tiểu đường 10 năm nay (đang uống thuốc đều và chỉ số ổn định 5–7 mmol/L), rung tim (mới phát hiện từ cuối năm ngoái), hở van 3 lá (đang theo dõi, chưa kết luận chính xác) vì cần nội soi tim thêm.", "en": "My father is 69 years old and has comorbidities: diabetes for 10 years (currently on medication and blood glucose levels stable at 5–7 mmol/L), atrial fibrillation (recently diagnosed at the end of last year), mitral valve regurgitation (under follow-up, exact diagnosis not yet confirmed) due to need for further cardiac endoscopy.", "zh": "我父亲今年69岁，有基础疾病：糖尿病10年（一直规律服药，血糖稳定在5–7 mmol/L），心律失常（去年年底新发现），三尖瓣反流（正在随访，尚未明确诊断），因需进一步行心脏内镜检查。"}, {"id": 949, "vi": "Thị lực suy giảm, nhìn xa không thấy rõ, kèm theo thường xuyên mỏi mắt, nhức đầu là dấu hiệu của cận thị?", "en": "Decreased vision, inability to see distant objects clearly, accompanied by frequent eye fatigue and headache, are signs of myopia?", "zh": "视力下降，远视不清，常伴有眼疲劳、头痛，是近视的表现吗？"}, {"id": 950, "vi": "Hạt ngọc dương vật có nguy hiểm không?", "en": "Is penis stone dangerous?", "zh": "阴茎珍珠是否有危险？"}, {"id": 951, "vi": "Lao phổi đã chữa khỏi có thể bị lại không?", "en": "Can pulmonary tuberculosis that has been cured recur?", "zh": "肺结核治愈后会复发吗？"}, {"id": 952, "vi": "97 tuổi bị trĩ nội nên uống thuốc điều trị hay phẫu thuật thì tốt hơn?", "en": "For a 97-year-old patient with internal hemorrhoids, should medication be taken for treatment or is surgery better?", "zh": "97岁患有内痔，应该服用药物治疗还是手术治疗更好？"}, {"id": 953, "vi": "Gãy xương ống tay đã điều trị khỏi nhưng bị sưng và ngả sang màu tím, do đâu?", "en": "The forearm fracture has been treated and healed but now has swelling and turned purple. What is the cause?", "zh": "手腕骨已治愈，但出现肿胀并变紫，是什么原因？"}, {"id": 954, "vi": "Lúc sáng em vừa đi chụp X-quang phổi, kết quả ghi là tổn thương kẽ hai bên phổi. Vậy đó là bệnh gì và có nguy hiểm không ạ?", "en": "In the morning, I just had a chest X-ray, and the result states bilateral interstitial lesions. So, what is this condition and is it dangerous?", "zh": "早上我刚做了胸部X光检查，结果显示双肺有间质病变。这是什么病，严重吗？"}, {"id": 955, "vi": "Loét bàn chân do tiểu đường có nguy hiểm không?", "en": "Is diabetic foot ulcer dangerous?", "zh": "糖尿病足溃疡危险吗？"}, {"id": 956, "vi": "Bé 9 tháng bị bệnh não chất trắng và thiếu men G6PD có can thiệp điều trị được không?", "en": "Can a 9-month-old child with white matter brain disease and G6PD deficiency be treated?", "zh": "9个月大的孩子患有白质脑病和G6PD缺乏症，能否进行治疗干预？"}, {"id": 957, "vi": "Làm thế nào để chữa hết bệnh đau dây thần kinh tọa?", "en": "How to cure the condition of sciatica pain completely?", "zh": "如何彻底治愈坐骨神经痛？"}, {"id": 958, "vi": "Kết quả xét nghiệm phân của bé hồng cầu +, bạch cầu ++, không có ký sinh trùng đường ruột, chỉ số bình thường âm tính. Mong bác sĩ giải thích kết quả.", "en": "The stool test results show red blood cells +, white blood cells ++, no intestinal parasites, normal values negative. I hope the doctor can explain the results.", "zh": "患儿粪便检查显示红细胞+，白细胞++，肠道无寄生虫，各项指标正常，阴性。恳请医生解释结果。"}, {"id": 959, "vi": "Tôi bị thoái hoá khớp gối ( đa chụp X-quang). Các loại thuốc điều trị?. Môn thể thao phu hợp?", "en": "I have knee osteoarthritis (confirmed by multiple X-rays). What medications are used for treatment? Which sports are suitable?", "zh": "我患有膝关节退行性病变（多处X光检查）。有哪些药物治疗？适合从事哪些运动？"}, {"id": 960, "vi": "Điều trị chồi trong bàng quang như thế nào?", "en": "How to treat bladder papilloma?", "zh": "膀胱内异物的治疗方式是什么？"}, {"id": 961, "vi": "Tại sao khi nhìn thì thấy những vật nhiễu trước mắt theo em tìm hiểu là hiện tượng tuyết thị giác, bệnh đó có nguy hiểm không và cách chữa trị ạ?", "en": "Why do I see disturbances before my eyes when looking? According to my research, this is snow vision phenomenon. Is this condition dangerous and what is the treatment?", "zh": "为什么看东西时会看到眼前有漂浮的物体呢？我了解到这是雪盲现象，这种病危险吗？该怎么治疗？"}, {"id": 962, "vi": "Cho em hỏi em thở bình thường mà cứ chập là phải lấy hơi dài", "en": "May I ask, I breathe normally, but every time I feel a sudden urge, I have to take a deep breath.", "zh": "我想问一下，我正常呼吸，但每次呼吸都得深吸一口气。"}, {"id": 963, "vi": "Con trai em 3 tuổi rưỡi, bé có cơ địa nóng, tiền sử sốt co giật lần đầu, cơ địa dị ứng không ăn được hải sản biển, vậy bé có tiêm vaccine Covid-19 được không? Có an toàn về phản ứng sau tiêm không?", "en": "My 3.5-year-old boy has a hot constitution, has a history of first-time fever seizure, and has an allergic constitution that cannot tolerate sea seafood. Can he receive the COVID-19 vaccine? Is it safe regarding post-vaccination reactions?", "zh": "我儿子3岁半，体质偏热，有首次发热惊厥史，对海产品过敏，不能吃海鲜，那么他可以接种新冠疫苗吗？接种后是否有安全的不良反应？"}, {"id": 964, "vi": "Bố em sau khi phẫu thuật ung thư thực quản được khoảng 1 năm thì bị khàn, yếu giọng. Đi khám thì bác sĩ ở đấy bảo là bị liệt dây thanh quản trái, không có biểu hiện của di căn, cần phải theo dõi thêm. Bác sĩ cho em hỏi, liệt dây thanh quản sau phẫu thuật có thể chữa được không.", "en": "My brother, about one year after esophagectomy for esophageal cancer, developed hoarseness and weak voice. Upon examination, the doctor there diagnosed him with left vocal cord paralysis, with no signs of metastasis, and advised further monitoring. The doctor asks me, can left vocal cord paralysis after surgery be treated?", "zh": "我弟弟在食管癌手术后大约一年出现声音嘶哑、声带无力。去医院检查，医生说他患左侧声带麻痹，没有转移迹象，需要继续观察。医生想问一下，手术后出现声带麻痹能否治愈？"}, {"id": 965, "vi": "Đái dầm ở người lớn tuổi là biểu hiện của bệnh gì?", "en": "Urinary incontinence in the elderly is a sign of what disease?", "zh": "老年人尿失禁是何种疾病的表現？"}, {"id": 966, "vi": "Tiêm pentaxim xong có tiêm phế cầu sau 2 ngày được không", "en": "Can I get vaccinated with pneumococcal vaccine two days after receiving pentaxim?", "zh": "接种百白破疫苗后两天可以接种肺炎球菌疫苗吗"}, {"id": 967, "vi": "Nữ giới đau nửa bên đầu là dấu hiệu bệnh gì?", "en": "In women, pain on one side of the head is a sign of what disease?", "zh": "女性单侧头痛是何种疾病的征兆？"}, {"id": 968, "vi": "Con 20 tuổi vài ngày gần đây con cảm thấy âm đạo của mình nóng,ngứa và rát rất khó chịu...Con nghĩ là bình thường nhưng tới hôm nay là gần 1 tuần rồi con tự kiểm tra thì thấy mép trong cô bé của con có nổi sần và có mụn thịt gần âm đạo.Nửa năm trước con có quan hệ không an toàn.Cuối tháng 1 vừa rồi con có quan hệ dùng bao nhưng vì sự cố nên con phải mua thuốc tránh thai khẩn cấp uống.Mới đầu con nghĩ là dị ứng thuốc tránh thai khẩn cấp thôi.Nhưng giờ nó biến chứng hơn con phải làm sao đây?", "en": "A 20-year-old girl has felt heat, itching, and burning in her vagina over the past few days, which is very uncomfortable. She thought it was normal, but now, after nearly one week, she has self-examined and noticed roughness and small bumps near the inner labia, close to the vagina. Six months ago, she had unprotected intercourse. At the end of January, she had intercourse using a condom, but due to an accident, she had to take emergency contraceptive pills. Initially, she thought it was just an allergic reaction to the emergency contraceptive pills. However, now the condition has worsened, and she does not know what to do.", "zh": "20岁，几天前开始感觉阴道发烫、瘙痒和灼热，非常难受……我原本以为是正常现象，但到今天已经接近一周了，自己检查发现阴唇内侧出现颗粒状突起，且在阴道附近有疣状物。半年前有过不安全的性行为。上个月底有过使用避孕套的性行为，但由于意外，我不得不服用紧急避孕药。一开始我以为是紧急避孕药过敏，但现在情况变得更严重了，我该怎么办？"}, {"id": 969, "vi": "Thưa em có đặt vòng, cuối tháng rồi em vẫn có kinh nhưng không nhiều. Mấy ngày sau em có dấu hiệu buồn nôn, ngực căng và hay đau bụng, em nghi ngờ nên mua que về thử kết quả 2 vạch. Sau đó em đi khám thì bảo em có echo hỗn hợp nằm ngoài tử cung, nghi ngờ em có thai ngoài tử cung. Sau đó em đến bệnh viện siêu âm đầu dò âm đạo vẫn không thấy được gì, sau đó nghi ngờ là những khối phân rồi cho xét nghiệm máu thì kết quả không mang thai, giờ em vẫn còn hoang mang! Xin giải đáp thắc mắc giúp em với ạ.", "en": "Dear girl, I have an IUD, and at the end of the month I still have my period but it's not much. A few days later, I started experiencing nausea, breast fullness, and frequent abdominal pain. I suspect I should buy a test stick to check for two lines. After that, I went to see a doctor who told me there was a mixed echo outside the uterus, suspecting an ectopic pregnancy. Then I went to the hospital for transvaginal ultrasound, but nothing was visible. I then suspected it might be bowel masses and had a blood test, which showed negative for pregnancy. Now I'm still confused! I kindly request your clarification on this matter.", "zh": "您好，我放置了宫内节育器，上个月结束时仍有少量月经。几天后我出现恶心、乳房胀痛和腹痛的症状，怀疑自己可能怀孕，于是买了验孕棒测试，结果是两条线。之后我去医院检查，医生说我有位于子宫外的混合性包块，怀疑我患有宫外孕。后来我到医院做阴道超声检查，却未发现异常，怀疑是肠道占位，于是做了血检，结果显示未怀孕。现在我仍然很困惑！恳请帮我看一下这个问题。"}, {"id": 970, "vi": "Em có người bạn ạ, trước kia vài tháng có hay uống rượu, trong một tuần hay nửa tháng có uống đôi lần nhưng không nghiện ạ. Có bữa vì uống say quá nên thấy người mệt mỏi lại kèm thêm đói bụng, xây xẩm nên đã ngừng rượu bia, đồ cay nóng và có đi kiểm tra chức năng tuyến giáp và gan mật không ảnh hưởng gì, nhưng hiện tại mang theo trong mình một bệnh cứ đói hoài, đói bụng đến mức làm mất phương hướng ạ. Vậy cho em hỏi như vậy có tiền sử bệnh thần kinh gì đó không ạ?", "en": "I have a friend. A few months ago, she occasionally drank alcohol, once or twice per week or every half month, but she was not addicted. On one occasion, after drinking heavily, she felt tired and had a feeling of hunger and dizziness, so she stopped drinking alcohol, spicy foods, and hot foods, and had a thyroid and liver-bile function test, which showed no abnormalities. However, currently, she has a persistent feeling of hunger, with constant hunger in her stomach to the point of losing orientation. So I would like to ask, does this indicate any neurological disease history?", "zh": "我有个朋友，之前几个月经常喝酒，一星期或半个月喝一次，但没有上瘾。有一次喝醉了，感到疲倦，还伴有饥饿感和头晕，于是就停止了饮酒、辛辣食物，并检查了甲状腺和肝胆功能，结果都正常。但现在他一直感觉饥饿，肚子饿得厉害，甚至迷失方向。我想问一下，这种情况是否有神经系统方面的病史？"}, {"id": 971, "vi": "Em đang mang thai được 34 tuần và sang tuần thứ 35. Khi đi siêu âm bác sĩ có nói em bị canxi hóa độ 2,3 nên con hấp thu dinh dưỡng từ mẹ thấp. Vậy em có vấn đề gì nguy hiểm không ạ? Em nên ăn uống và bổ sung như thế nào để em bé khỏe mạnh và phát triển tốt?", "en": "I am 34 weeks pregnant and entering week 35. When I went for ultrasound, the doctor said I have stage 2.3 calcification, so the baby is absorbing nutrients from the mother poorly. So, is there any serious problem? What should I eat and supplement to ensure my baby is healthy and develops well?", "zh": "我目前怀孕34周，即将进入35周。做超声检查时，医生告诉我我有2.3级的钙化，因此胎儿从母体吸收营养较差。请问我会有什么危险吗？我应该如何饮食和补充营养，才能让宝宝健康且良好发育？"}, {"id": 972, "vi": "Gần đây con cảm thấy tròng trắng nhỏ lại, xin giúp con.", "en": "Recently, I have felt a small white spot. Please help me.", "zh": "最近孩子感觉白眼球变小了，麻烦帮忙。"}, {"id": 973, "vi": "Đau phần tủy răng của răng hàm đã bọc sứ được 2 năm có phải lấy tủy và hàn lại răng không?", "en": "Is it necessary to remove the pulp and restore the molar tooth that has had a ceramic crown for 2 years, experiencing pulp pain?", "zh": "镶瓷两年后，磨牙出现牙髓疼痛，是否需要进行根管治疗并重新牙体填充？"}, {"id": 974, "vi": "Bác sĩ cho em hỏi kết quả xét nghiệm nước tiểu của em, bệnh viện nghi ngờ nhiễm trùng đường tiểu nên cho em làm xét nghiệm cấy nước tiểu. Cho em hỏi nếu nhiễm trùng đường tiểu có ảnh hưởng gì đến thai nhi và gây ra bệnh gì cho thai nhi không ạ?", "en": "Doctor, I would like to ask about my urine test results. The hospital suspects a urinary tract infection and has recommended I undergo a urine culture test. I would like to know if a urinary tract infection can affect the fetus and cause any diseases in the fetus?", "zh": "医生请问一下我尿液检查的结果，医院怀疑我有尿路感染，所以让我做了尿液培养检查。我想问一下，如果患有尿路感染，会对胎儿产生什么影响，会不会导致胎儿患上什么疾病呢？"}, {"id": 975, "vi": "Tai đau sau mổ u tuyến nước bọt phải làm gì?", "en": "What should be done after surgery for pain in the right salivary gland?", "zh": "术后腮腺疼痛该怎么办？"}, {"id": 976, "vi": "Cho cháu xin hỏi ạ người cháu một ngày đi vệ sinh rất nhiều và đi vệ sinh vào ban đêm ạ ăn uống không được ngon và ngủ thì hay thỉnh giấc ạ ma song muốn ngủ lại nhưng không ngủ được ạ", "en": "May I ask about my child? My child has very frequent bowel movements and wakes up at night to use the toilet. The child has poor appetite and often wakes up during sleep, wanting to go back to sleep but unable to fall asleep.", "zh": "请问一下，我家孩子每天排便次数很多，晚上也经常排便，食欲不好，睡觉时容易醒来，想睡又睡不着。"}, {"id": 977, "vi": "Chuẩn đoán là tăng men gan, xét nghiệm máu trên SGOT 60 và SGPT 46, ngoài việc uống thuốc của bác sĩ kê đơn ra thì có những biện pháp nào để giảm bớt tăng men gan.", "en": "Diagnosis is elevated liver enzymes, blood test showing SGOT 60 and SGPT 46. Besides taking the medications prescribed by the doctor, what other measures can be taken to reduce elevated liver enzymes?", "zh": "诊断为肝酶升高，血清谷丙转氨酶（SGPT）为60，谷草转氨酶（SGOT）为46，除了遵医嘱服药外，还有哪些措施可以减轻肝酶升高？"}, {"id": 978, "vi": "Nội soi trực tràng có phát hiện bệnh lý đại tràng không", "en": "Can rectal endoscopy detect colorectal diseases?", "zh": "直肠镜检查能否发现结肠疾病？"}, {"id": 979, "vi": "Nam giới truyền dịch, kim tiêm dính máu đâm vào chai dịch có sao không?", "en": "Does a male patient have a problem if the needle with blood-stained syringe pierces the fluid bottle?", "zh": "男性输液时，针头刺破药液瓶有事吗？"}, {"id": 980, "vi": "Sau điều trị viêm tai giữa có hiện tượng đau đầu, vì sao?", "en": "After treatment of acute otitis media, there is a headache; why?", "zh": "治疗中耳炎后出现头痛，为什么？"}, {"id": 981, "vi": "Sau sinh mổ gần một tháng có cần uống canxi và vitamin không và uống trong bao lâu", "en": "After cesarean section surgery, about one month later, is it necessary to take calcium and vitamin, and for how long?", "zh": "产后近一个月手术后是否需要服用钙和维生素，需要服用多久"}, {"id": 982, "vi": "Trĩ nội độ 1 có cần điều trị không hay chỉ tăng cường ăn chất xơ là được.", "en": "Does stage 1 internal hemorrhoids need treatment or is it sufficient to just increase fiber intake only?", "zh": "IⅠ期内痔是否需要治疗，还是仅通过增加膳食纤维摄入即可？"}, {"id": 983, "vi": "Sau khi tiêm vắc xin Covid-19 mà tiếp xúc với F0 thì có bị coi là F1 và phải đi cách ly tập trung không?", "en": "After receiving the Covid-19 vaccine and coming into contact with a F0, will one be considered a F1 and required to undergo centralized quarantine?", "zh": "接种新冠疫苗后接触过感染者（F0），是否会被视为次密切接触者（F1）并需要集中隔离？"}, {"id": 984, "vi": "Kết quả Triple test: HC Down (Tr.21): 1: 755 Dị tật ống thần kinh: 0,6108 HC Edward (Tr.18): 1: 27143 AFP: 54,54 Beta hcg: 26230 uE3: 1,564", "en": "Triple test results: HC Down (Page 21): 1: 755 Neural tube defect: 0.6108 HC Edward (Page 18): 1: 27143 AFP: 54.54 Beta hcg: 26230 uE3: 1.564", "zh": "三联筛查结果：Down综合征风险（第21页）：1:755，神经管缺陷：0.6108，Edward综合征风险（第18页）：1:27143，AFP：54.54，β-hCG：26230 uE3：1.564"}, {"id": 985, "vi": "Con em mới bị sốt và viêm tai giữa dùng thuốc mấy hôm nay giờ đã khỏi. Cách bao nhiêu ngày thì tiêm được 5 trong 1 ạ", "en": "The child has just had fever and middle ear infection and took medicine for several days, and now is recovered. After how many days can the child receive the 5-in-1 vaccine?", "zh": "孩子刚发烧并患有中耳炎，吃了药几天后现在已经好了。多久后可以打五联疫苗啊？"}, {"id": 986, "vi": "Em bé hơn 1 tuổi cứ đi ngoài ngày 2-3 lần, phân lỏng hoặc sền sệt, thì có vấn đề gì không ạ?", "en": "Is there any problem if a child under 1 year old has 2-3 loose or pasty stools per day?", "zh": "1岁以内的宝宝每天大便2-3次，大便稀或黏稠，是否有什么问题？"}, {"id": 987, "vi": "Em kiểm tra hcg và có kết quả là 102.1 thì có nghĩa kết quả là 102,1 hay là 1021", "en": "I checked hcg and the result is 102.1, does this mean the result is 102.1 or 1021?", "zh": "检查HCG结果为102.1，那么结果是102.1还是1021？"}, {"id": 988, "vi": "Trường hợp nào mổ lấy nẹp vít xương đòn được chỉ định gây tê đám rối thần kinh cánh tay?", "en": "In which case is surgical removal of shoulder bone fixation screws indicated with brachial plexus nerve block?", "zh": "在哪些情况下需要进行臂丛神经阻滞的肩胛骨钢板取出手术？"}, {"id": 989, "vi": "Gây tê tủy sống khi sinh mổ có gây đau lưng không?", "en": "Does causing spinal anesthesia during cesarean section lead to back pain?", "zh": "剖腹产引起脊髓麻木是否会导致腰痛？"}, {"id": 990, "vi": "Bong võng mạc thoái hóa độ 4 có điều trị như bệnh Coats được không?", "en": "Is stage 4 macular degeneration treated like Coats disease?", "zh": "黄斑变性4级能否像Coats病一样治疗？"}, {"id": 991, "vi": "Nay em vừa nhận được kết quả xét nghiệm máu thấy lượng mỡ máu của em đang vượt ngưỡng. Hiện tại em mang thai tuần 30. Không biết có ảnh hưởng gì đến mẹ và bé không ạ? Có biện pháp nào giảm không ạ? Và cần ăn kiêng gì không ạ?", "en": "Today I just received my blood test results and I found my lipid levels are exceeding the normal range. I am currently at 30 weeks of pregnancy. I would like to know if this has any effect on mother and baby? Are there any measures to reduce it? And what should I eat?", "zh": "我最近刚拿到血检结果，发现我的血脂水平已经超标。目前我怀孕30周。不知道这对母亲和胎儿有没有影响？有没有什么方法可以降低？需要忌口什么吗？"}, {"id": 992, "vi": "Tiểu nhiều, nước tiểu đôi khi đục trắng là do đâu?", "en": "Frequent urination, sometimes cloudy urine, what is the cause?", "zh": "尿频，尿液有时呈白色浑浊是什么原因？"}, {"id": 993, "vi": "Cholesterol toàn phần 6.31 mmol/l, Triglycerid 1.09 mmol/l, LDL - C 4.49 mmol/l có nguy hiểm không?", "en": "Total cholesterol 6.31 mmol/l, Triglyceride 1.09 mmol/l, LDL-C 4.49 mmol/l, is dangerous?", "zh": "总胆固醇 6.31 mmol/l，甘油三酯 1.09 mmol/l，LDL-C 4.49 mmol/l 有危险吗？"}, {"id": 994, "vi": "Tôi năm nay 42 tuổi bị đau vùng thượng vị hay ợ hơi,đau chếch về bên trái nhiều hơn nhiều khi không nằm nghiêng sang trái được như vậy là tôi bị bệnh gì", "en": "I am 42 years old and have been experiencing upper abdominal pain and belching, with more left-sided pain, especially when not lying on my left side. What disease do I have?", "zh": "我今年42岁，经常感到上腹部疼痛和打嗝，左侧疼痛更明显，尤其是平躺时无法向左侧倾斜，这说明我得了什么病？"}, {"id": 995, "vi": "Đầu vú vừa mới xuất hiện 2-3 chấm đen giống mụn xé ra rỉ máu có phải bị ung thư vú không?", "en": "Have I developed breast cancer with the recent appearance of 2-3 black spots resembling ruptured pustules with bleeding?", "zh": "乳房刚刚出现2-3个类似破溃出血的黑点，是否意味着患有乳腺癌？"}, {"id": 996, "vi": "Nam giới có Gran# 1,9, Gran% 38,5, LYM% 53,3 có ảnh hưởng gì?", "en": "What impact does a male with Gran# 1.9, Gran% 38.5, LYM% 53.3 have?", "zh": "男性粒细胞计数为1.9，粒细胞百分比为38.5，淋巴细胞百分比为53.3，有何影响？"}, {"id": 997, "vi": "Mẹ em 58 tuổi, bị lang thận, có sao không?", "en": "My mother, 58 years old, has chronic kidney disease, is there any problem?", "zh": "我母亲58岁，患有慢性肾衰竭，会有什么问题吗？"}, {"id": 998, "vi": "Em xin tư vấn về các phương pháp điều trị sùi mào gà. Em có tìm hiểu về phương pháp ALA-PDT nhưng chưa hiểu rõ và không biết phương pháp này khác gì so với các phương pháp đốt điện thông thường?", "en": "I would like to consult about the treatment methods for genital warts. I have researched the ALA-PDT method, but I do not fully understand it and do not know how this method differs from conventional electrocautery?", "zh": "我想咨询关于尖锐湿疣的治疗方法。我了解到ALA-PDT这种方法，但还不太明白，它与常规的电灼治疗有什么不同？"}, {"id": 999, "vi": "Làm thế nào để dịch viêm tai giữa tự tiêu?", "en": "How to treat self-resolving middle ear inflammation?", "zh": "如何治疗自限性中耳炎？"}, {"id": 1000, "vi": "Em chuyển phôi được 4 ngày. Đến ngày 5 thì có dấu hiệu gì không ạ?", "en": "The embryo transfer was 4 days ago. By day 5, are there any signs?", "zh": "移植4天，到第5天会有哪些表现吗？"}, {"id": 1001, "vi": "Em bị hạch hai bên cổ ở dọc thượng đòn chùm. Đi khám bác sĩ bảo bị viêm hạch. Em có uống thuốc do bác sĩ kê đơn nhưng vẫn sờ thấy hạch hai bên cổ. Em bị 8 tháng rồi. Đây có phải là hạch ung thư không, hay em bị ung thư ở chỗ khác di căn vậy?", "en": "My neck lymph nodes on both sides along the upper border of the clavicle are enlarged. I visited a doctor who diagnosed me with lymph node inflammation. I have been taking the medication prescribed by the doctor, but I can still feel the enlarged lymph nodes on both sides of my neck. This has been going on for 8 months. Could this be cancerous lymph nodes, or could I have cancer elsewhere that has metastasized?", "zh": "我两侧颈部链状淋巴结肿大，医生检查后说是淋巴结炎。我按医生开的药吃了，但仍然能摸到两侧颈部的淋巴结，已经持续8个月了。这会不会是淋巴瘤？还是我其他地方有癌症转移了？"}, {"id": 1002, "vi": "Em 40 tuổi, đã làm hai lần IUI và hai lần IVF, đều đậu thai được 8-10 tuần thì mất tim thai. Em có đi làm xét nghiệm chỉ số của gen đông máu (chỉ số MTHFR677 và chỉ số MTHFR 1298 phát hiện đột biến dị hợp tử; PAI-1/Serpine phát hiện đột biến đồng hợp tử, được xét nghiệm sau khi thực hiện hai lần IUI và một lần IVF). Lần IVF chuyển cuối cùng đã tiêm thuốc chống đông máu từ khi bắt đầu chuyển phôi. Bây giờ em cần làm gì để tìm nguyên nhân và tiếp tục điều trị, thực hiện phương pháp nào là phù hợp?", "en": "I am 40 years old and have undergone two cycles of IUI and two cycles of IVF, all of which resulted in pregnancies that were lost at 8-10 weeks. I have undergone genetic coagulation factor testing (MTHFR677 and MTHFR 1298 mutation indices showing heterozygous mutations; PAI-1/Serpine mutation index showing homozygous mutation, tested after two IUI cycles and one IVF cycle). The final IVF cycle involved administration of anticoagulant medication from the beginning of embryo transfer. What should I do now to identify the cause and continue treatment, and which treatment method is most appropriate?", "zh": "我40岁，已进行过两次宫腔内人工授精（IUI）和两次体外受精（IVF），每次均成功着床8-10周后流产。我曾进行过基因凝血功能相关检查（检测MTHFR677和MTHFR1298位点的杂合突变；PAI-1/Serpin位点的纯合突变，分别在完成两次IUI和一次IVF后进行检测）。最后一次IVF移植过程中，从开始移植起即开始使用抗凝药物。现在我需要采取什么措施来查找流产原因，并继续治疗，哪种治疗方法更为合适？"}, {"id": 1003, "vi": "Làm thế nào để phòng tránh lây bệnh viêm gan B cho chồng và con?", "en": "How to prevent transmission of hepatitis B virus to husband and child?", "zh": "如何预防将乙型肝炎传染给丈夫和孩子？"}, {"id": 1004, "vi": "Hôm qua em đi khám bên bệnh viện thì được chẩn đoán là gai đôi đốt sống L5, hiện giờ em chưa thấy đau hay có vấn đề gì ạ, nhưng em đang rất lo lắng khi tìm hiểu về bệnh này, em mong được tư vấn phải chữa trị như thế nào ạ.", "en": "Yesterday I went to the hospital for a check-up and was diagnosed with L5 lumbar joint spurs. Currently, I haven't felt any pain or experienced any issues, but I am very worried after learning about this condition. I hope to get advice on how to treat it.", "zh": "昨天我去医院检查，被诊断为L5节段椎间盘突出，目前我还没有感到疼痛或出现其他问题，但我对这种病非常担心，希望得到关于应该如何治疗的建议。"}, {"id": 1005, "vi": "Em bị sỏi túi mật với kích thước 9x5mm, cho em hỏi là kích thước vậy có lớn không ạ? Có cần phải mổ không ạ? Nếu để vậy có nguy hiểm gì không, và khi bị sỏi em có cần kiêng những thực phẩm như sữa, trứng gà, dầu mỡ, chất béo không?", "en": "My mother has gallstones with a size of 9x5mm. I would like to know if this size is large? Do I need surgery? If left untreated, is there any danger? And when having gallstones, do I need to avoid foods such as milk, eggs, oil, and fats?", "zh": "我患有胆囊结石，大小为9x5mm，想咨询一下，这样的大小算大吗？需要手术吗？如果不处理会有危险吗？得了胆结石是否需要忌口，比如牛奶、鸡蛋、油炸食物、高脂肪食物等？"}, {"id": 1006, "vi": "Em đang bị hành kinh nguyệt vậy thì có chích ngừa ung thư cổ tử cung mũi thứ 2 được không ạ? Nếu chích trễ hơn lịch hẹn thì có ảnh hưởng tới tác dụng của thuốc không ạ?", "en": "My period is coming, so can I get the second dose of cervical cancer vaccine? If I get the second dose after the scheduled time, will it affect the drug's effectiveness?", "zh": "我正在经期，那么可以打宫颈癌疫苗第二针吗？如果延迟了预约时间，会对药物效果有影响吗？"}, {"id": 1007, "vi": "Em có dùng que thử HIV OraQuick sau khi có nguy cơ cách đây 5 tháng . Nó hiện 1 vạch ở C. Nhưng lúc dùng que thử cho vào nướu trên thì em quên nên phần lớn quét ở phần trong môi ( đối diện nướu ) nhiều hơn ( vẫn có quét nướu trên nhưng ít), còn nướu dưới em quét chuẩn. Làm như vậy có ảnh hưởng tới kết quả không.", "en": "I used the HIV OraQuick test five months ago after having potential exposure. It showed one line on the C side. However, when using the test, I forgot and mostly swabbed the inner mouth (opposite the gums) rather than the upper gums (although I still swabbed the upper gums a little), and I swabbed the lower gums properly. Will this affect the result?", "zh": "我五个月前有感染HIV的风险，之后使用了OraQuick HIV试纸，结果显示C区出现一条线。但当我将试纸放入上牙龈时，我忘记了，因此大部分擦拭的是内唇侧（对侧牙龈），上牙龈部分也有擦拭但较少，下牙龈则擦拭得正确。这样做会影响结果吗？"}, {"id": 1008, "vi": "Bé 2 tuổi bị gãy răng có uống thuốc giảm đau được không?", "en": "Can a 2-year-old child take pain relief medication after a tooth fracture?", "zh": "2岁幼儿牙齿折断是否可以服用止痛药？"}, {"id": 1009, "vi": "Ý nghĩa chỉ số Cyfra 21 – 1?", "en": "What is the significance of the Cyfra 21-1 index?", "zh": "Cyfra 21-1 指标的意义是什么？"}, {"id": 1010, "vi": "Tôi đang sử dụng thuốc tránh thai nhưng rất hay quên. Tôi muốn chuyển sang cây que tránh thai. Nên cấy vào thời điểm nào? Tác dụng phụ là gì ạ?", "en": "I am currently using oral contraceptive pills but often forget to take them. I would like to switch to a copper intrauterine device (IUD). When should it be inserted? What are the side effects?", "zh": "我正在使用避孕药，但经常忘记服用。我想改用宫内节育器。应该在什么时间植入？有哪些副作用？"}, {"id": 1011, "vi": "Trẻ không có phản ứng sau khi tiêm lao có phải tiêm lại không?", "en": "Does a child need to be revaccinated after no reaction following BCG injection?", "zh": "接种结核疫苗后儿童无反应是否需要重新接种？"}, {"id": 1012, "vi": "Vùng da ở nách xuất hiện đốm đỏ là triệu chứng của bệnh gì?", "en": "What disease is characterized by red spots in the axillary skin area?", "zh": "腋下皮肤出现红色斑点是哪种疾病的症状？"}, {"id": 1013, "vi": "Chỉ số của em triglycerid 5,0 và cholesterol 6,8 đã phải dùng thuốc chưa ạ?", "en": "Is medication already being used for your triglyceride level of 5.0 and cholesterol level of 6.8?", "zh": "我的甘油三酯为5.0，胆固醇为6.8，是否已经用药了？"}, {"id": 1014, "vi": "Tỷ lệ % bạch cầu trung tính giảm, bạch cầu lympho tăng, nồng độ và lượng huyết sắc tố, chỉ số creatinin giảm cảnh báo nguy cơ mắc bệnh gì?", "en": "What disease risk is signaled by a % decrease in neutrophils, an increase in lymphocytes, a decrease in hemoglobin concentration and volume, and a decrease in creatinine index?", "zh": "中性粒细胞比例下降，淋巴细胞比例升高，血红蛋白浓度和量、肌酐水平下降提示何种疾病风险？"}, {"id": 1015, "vi": "Em bị xuất tinh sớm, qua tìm hiểu em có thấy thuốc Dapoxetine (Priligy) điều trị xuất tinh sớm. Các bác sĩ có thể cho em biết tác dụng, cách sử dụng và liều lượng dùng của thuốc không ạ?", "en": "I have premature ejaculation. After researching, I found that Dapoxetine (Priligy) is used to treat premature ejaculation. Could the doctors please tell me the effects, usage method, and dosage of this medication?", "zh": "我有早射问题，通过查阅资料了解到达普oxetine（普瑞巴林）可用于治疗早射。医生能否告知该药的作用、用法以及剂量呢？"}, {"id": 1016, "vi": "Bé nhà em bị thiếu men G6PD. Bị thiếu men này bị dị ứng với một số loại thuốc và đậu. Vậy bác sĩ cho em hỏi dấu hiệu khi bị dị ứng là như thế nào?", "en": "My child has G6PD deficiency. People with this enzyme deficiency are allergic to certain medications and legumes. Therefore, I would like to ask the doctor what the signs are when an allergic reaction occurs?", "zh": "我孩子患有G6PD缺乏症。这种缺乏症会导致对某些药物和豆类产生过敏反应。因此我想请教一下，出现过敏时有哪些症状？"}, {"id": 1017, "vi": "Tiểu ra máu sau mổ sỏi thận nguyên nhân là gì?", "en": "What is the cause of hematuria after nephrolithotomy?", "zh": "肾结石手术后尿中带血的原因是什么？"}, {"id": 1018, "vi": "Run tay ở người trẻ có nguy hiểm không?", "en": "Is hand tremor in young people dangerous?", "zh": "年轻人出现手部抽搐是否有危险？"}, {"id": 1019, "vi": "Tăng vùng dị nhiễm sắc trên nhánh dài nhiễm sắc thể số 1 ở nữ có ảnh hưởng gì không, có chữa được không, để lâu về sau có bị sao không?", "en": "Does an increase in the number of structurally abnormal chromosomes on the long arm of chromosome 1 in females have any impact? Can it be treated? What might happen later if left untreated?", "zh": "1号染色体长臂上异染色区增加在女性中有什么影响？能否治疗？长期之后会有什么后果？"}, {"id": 1020, "vi": "Tinh dịch có phần đông giống như thạch là sao?", "en": "What does it mean if semen has a solid part resembling gel?", "zh": "精液中有类似凝胶的凝块是怎么回事？"}, {"id": 1021, "vi": "Mẹ tôi bị tăng huyết áp, nay mới được chẩn đoán là mắc đái tháo đường, hai bệnh này có mối liên quan tới nhau hay không thưa bác sĩ và mẹ tôi cần lưu ý gì để tránh biến chứng bệnh ạ?", "en": "My mother has hypertension, and she has just been diagnosed with diabetes. Doctor, are these two conditions related to each other? And what should my mother pay attention to in order to avoid complications of the diseases?", "zh": "我母亲患有高血压，最近才被诊断出患有糖尿病，这两种疾病之间是否有关联？医生，我母亲需要注意什么才能避免并发症呢？"}, {"id": 1022, "vi": "Hiện tại bà xã em đi siêu âm có kết quả thai đôi( chắc là cùng trứng vì trước khi qh có siêu âm 1 trứng rụng). Kết quả siêu âm tuần thứ 7. Nhưng hiện chỉ có 1 thai là có tim thai còn thai kia chưa có. Vậy xin bác sĩ cho em hỏi như vậy có bất thường không ? Và giờ cần làm gì ạ ?", "en": "Currently my wife has an ultrasound result showing a twin pregnancy (likely monozygotic since a previous ultrasound before intercourse showed one egg releasing). The ultrasound is from week 7. However, currently only one fetus has a detectable heartbeat, while the other does not. Therefore, I would like to ask the doctor whether this is abnormal, and what steps should be taken now?", "zh": "目前我女朋友做B超检查显示是双胞胎（很可能是同卵双胞胎，因为性生活前做过B超检查发现一个卵泡破裂）。目前是第7周的B超结果，但目前只有一个胎儿有心跳，另一个胎儿还没有心跳。请问医生这种情况是否异常？现在应该怎么办？"}, {"id": 1023, "vi": "Có thuốc kìm hãm sự phát triển của polyp túi mật không? Chỉ định phẫu thuật polyp túi mật khi nào?", "en": "Are there drugs that inhibit the development of gallbladder polyps? When is surgical indication for gallbladder polyps?", "zh": "有没有药物可以抑制胆囊息肉的生长？胆囊息肉手术的指征是什么？"}, {"id": 1024, "vi": "Em bị a-mi-đan, có nên đi cắt không?", "en": "My child has adenoid, should I have them operated?", "zh": "我有腺样体肥大，要不要手术？"}, {"id": 1025, "vi": "Con em lúc sinh có xét nghiệm máu gót chân - xét nghiệm sàng lọc rối loạn chuyển hoá bẩm sinh, kết quả có chỉ số Serine (Ser) là 528,15 trong khi ngưỡng cho phép từ 73->350. Bác sỹ có khuyến nghị đi đo định lượng acid amin tuy nhiên do con em mới sinh được 19 ngày nên cho em hỏi chỉ số này có ý nghĩa gì? Có cần thiết phải đi đo định lượng acid amin ngay không?", "en": "At birth, the baby underwent blood test from the heel for newborn screening of inborn metabolic disorders, with a serum serine (Ser) level of 528.15, while the acceptable range is 73–350. The doctor recommended further amino acid quantification testing; however, since the baby is only 19 days old, I would like to know what this value means. Is it necessary to proceed with amino acid quantification testing immediately?", "zh": "新生儿出生时进行了足跟血检测——用于筛查先天代谢障碍，结果显示丝氨酸（Ser）为528.15，而允许范围为73～350。医生建议进行氨基酸定量检测，但孩子出生仅19天，因此想咨询一下这个数值有什么意义？是否需要立即进行氨基酸定量检测？"}, {"id": 1026, "vi": "Bác sĩ cho em hỏi trước mang thai em có bị tiểu đường thai kì giờ em sinh cháu được hơn 2 tháng rồi thì bác sĩ cho em hỏi, em nên đi khám lại trong khoảng thời gian nào là tốt nhất và chế độ ăn uống sau sinh của em có cần phải kiêng như khi mang bầu không ạ?", "en": "Doctor, I would like to ask you: before pregnancy, I had gestational diabetes. Now my child has been born for more than 2 months. I would like to ask the doctor, when is the best time for me to have a follow-up visit, and does my postpartum diet need to be restricted like during pregnancy?", "zh": "医生请问，我怀孕前有妊娠糖尿病，现在孩子出生已经超过两个月了，医生请问，我最好在什么时候复查？产后饮食是否还需要像孕期那样忌口呢？"}, {"id": 1027, "vi": "Dừng thuốc chống động kinh có bị lại không?", "en": "Will seizures return after stopping anti-seizure medication?", "zh": "停抗癫痫药会复发吗？"}, {"id": 1028, "vi": "Tôi năm nay 35 tuổi, đã làm IVF một lần nhưng không đạt, kiểm tra CMV để xác định nguyên nhân thì chỉ số CMV IgG là 270. Sau ba tháng tập luyện thể thao để tăng sức đề kháng, xét nghiệm lại chỉ số CMV IgG tăng lên 297. Chỉ số này có ảnh hưởng đến sự phát triển của phôi không? Có cần xét nghiệm thêm IgM hay không? Tôi có thể tiếp tục chuẩn bị nội mạc để chuyển phôi lần nữa không?", "en": "I am 35 years old. I have undergone one IVF cycle but failed to achieve success. I performed a CMV test to determine the cause, and my CMV IgG level was 270. After three months of exercising to boost my immunity, the CMV IgG level increased to 297. Does this value affect embryo development? Do I need to undergo additional testing for CMV IgM? Can I continue preparing the endometrium for another embryo transfer?", "zh": "我今年35岁，曾进行过一次试管婴儿，但未成功。检查CMV以确定原因，CMV IgG滴度为270。经过三个月的体育锻炼以增强免疫力后，复查CMV IgG滴度上升至297。这个数值会影响胚胎发育吗？是否需要进一步检查IgM？我可以继续准备子宫内膜以进行第二次胚胎移植吗？"}, {"id": 1029, "vi": "Thi thoảng em đi tiểu bị đục đầu bãi như kiểu có cặn. Nước tiểu màu vàng. Có lúc màu vàng sẫm. Như vậy có là biểu hiện của bệnh không.", "en": "Occasionally I urinate and experience a cloudy head sensation, as if there is sediment. Urine is yellow. At times, it is dark yellow. Is this a sign of a disease?", "zh": "有时小便时会出现头部发沉的感觉，像是有沉淀物。尿液呈黄色，有时颜色较深。这种情况是疾病的表现吗？"}, {"id": 1030, "vi": "Các bác sĩ cho cháu hỏi cháu mới làm xét nghiệm máu. Cái protein của cháu cao có ảnh hưởng gì không ạ?", "en": "Doctors, I would like to ask you: my child just had a blood test. Does high protein level in my child have any impact?", "zh": "医生们请问一下，我孩子刚做了血检，孩子的蛋白偏高，这会有影响吗？"}, {"id": 1031, "vi": "Trẻ bị táo bón kéo dài là do đâu, có thể chuyển thành mãn tính không?", "en": "What is the cause of prolonged constipation in children, and can it become chronic?", "zh": "长期便秘是什么原因，会不会发展为慢性？"}, {"id": 1032, "vi": "Tại sao không nên bấm huyệt nhân trung với bệnh nhân đột quỵ?", "en": "Why should acupoint stimulation at the Renzhong point not be performed on stroke patients?", "zh": "为什么中风患者不应按压人中穴？"}, {"id": 1033, "vi": "Dùng nhiều thuốc bôi trị nấm da đầu gối không khỏi nên làm gì?", "en": "What should I do if using multiple topical antifungal medications for fungal infection of the scalp and knee has not worked?", "zh": "膝盖和头皮真菌感染使用多种外用药膏无效时该怎么办？"}, {"id": 1034, "vi": "Môi nổi mụn mủ vàng là dấu hiệu của bệnh lý gì", "en": "Yellow pus-filled bumps on the mouth are signs of what disease?", "zh": "口腔出现黄脓疱是何种疾病的征兆？"}, {"id": 1035, "vi": "Tôi có một số thắc mắc: 1. Nam giới sau khi tiêm vắc xin phòng Covid-19 của Astrazeneca mũi thứ nhất thì bao lâu có thể sinh con? 2. Nữ giới sau tiêm vắc xin phòng Covid-19 của Astrazeneca mũi thứ nhất trong khoảng từ 1-4 tháng sau tiêm, nếu có thai thì vắc xin trên có ảnh hưởng thế nào tới thai nhi và phụ nữ không? Giải pháp xử lý tốt nhất với trường hợp này?", "en": "I have several questions: 1. For men, after receiving the first dose of the Covid-19 vaccine from AstraZeneca, how long can they conceive? 2. For women, after receiving the first dose of the Covid-19 vaccine from AstraZeneca, if pregnancy occurs within 1-4 months after vaccination, what impact does this vaccine have on the fetus and the mother? What is the best management approach in this case?", "zh": "我有几个问题：1. 男性在接种阿斯利康新冠疫苗第一剂后，多久可以生育？2. 女性在接种阿斯利康新冠疫苗第一剂后1-4个月内怀孕，该疫苗对胎儿和母亲有何影响？对此情况的最佳处理方案是什么？"}, {"id": 1036, "vi": "Sau điều trị tràn dịch màng phổi bị dày dính màng phổi nên làm gì?", "en": "What should be done after treatment of pleural effusion with adhesion of the pleura?", "zh": "胸腔积液治疗后出现胸膜粘连应如何处理？"}, {"id": 1037, "vi": "Hội chứng thận hư có thể chữa dứt điểm không? Ăn gì hỗ trợ điều trị?", "en": "Can nephrotic syndrome be cured completely? What foods support treatment?", "zh": "肾病综合征能否根治？吃什么有助于治疗？"}, {"id": 1038, "vi": "Tôi bị ung thư tuyến giáp đã phẫu thuật và uống phóng xạ cách đây 3 năm. Tôi có tiền sử giảm tiểu cầu. Vậy tôi có nên tiêm vaccine không và nếu có thì loại nào phù hợp?", "en": "I have had thyroid cancer, underwent surgery and received radioactive iodine treatment three years ago. I have a history of thrombocytopenia. Should I receive a vaccine and, if so, which type is appropriate?", "zh": "我三年前接受了甲状腺癌手术并接受了放射性碘治疗，有血小板减少的病史。我是否应该接种疫苗？如果需要，应该接种哪种疫苗？"}, {"id": 1039, "vi": "Xét nghiệm ANA có ý nghĩa gì?", "en": "What does the ANA test mean?", "zh": "ANA检查有什么意义？"}, {"id": 1040, "vi": "Bị lạc nội mạc tử cung trong cơ có nguy hiểm không? Em làm IVF hai lần đều thất bại, liệu lạc nội mạc tử cung có phải là nguyên nhân khiến phôi không bám được vào tử cung không?", "en": "Is endometriosis in the pelvic cavity dangerous? I have undergone two rounds of IVF, both of which have failed. Could endometriosis be the reason why the embryo does not implant in the uterus?", "zh": "宫腔内膜异位症有危险吗？我做过两次试管婴儿都失败了，宫腔内膜异位症是否是导致胚胎无法着床的原因？"}, {"id": 1041, "vi": "Em bị bệnh xuất tinh sớm, dùng thuốc Estromen có cải thiện được không?", "en": "My son has early ejaculation; can Estromen be improved?", "zh": "我患有早射症，服用埃斯特梅恩能改善吗？"}, {"id": 1042, "vi": "Trẻ 4 tháng tuổi chỉ bú 500-700ml sữa một ngày liệu có đáng lo?", "en": "Is it concerning for a 4-month-old child to drink only 500-700ml of milk per day?", "zh": "4个月大的婴儿每天只喝500-700毫升奶是否需要担心？"}, {"id": 1043, "vi": "Bị cảm kéo dài, uống thuốc không đỡ là do đâu?", "en": "Chronic cold, not improved by medication, what is the cause?", "zh": "感冒持续不愈，服药无效的原因是什么？"}, {"id": 1044, "vi": "Chế độ dinh dưỡng hạn chế đạm nên ăn bao nhiêu gam thịt mỗi ngày?", "en": "What amount of meat in grams should be consumed daily on a protein-restricted diet?", "zh": "蛋白质摄入应每天吃多少克肉？"}, {"id": 1045, "vi": "Trẻ sốt cao, hắt hơi, chảy nước mũi sau tiêm phòng cúm có đáng lo?", "en": "Does a child with high fever, runny nose, and nasal discharge after influenza vaccination warrant concern?", "zh": "幼儿高烧、打喷嚏、流鼻涕，接种流感疫苗后需要担心吗？"}, {"id": 1046, "vi": "Vợ chồng e đang định làm xét nghiệm cho chồng, tinh dịch đồ và đứt gãy tinh trùng. Vậy trước khi đi làm xét nghiệm cần chuẩn bị gì k ạ?", "en": "The husband and wife are planning to have tests for the husband, semen analysis and sperm DNA fragmentation. So, what should be prepared before undergoing the test, sir?", "zh": "夫妻打算为丈夫做精液检查和精子断裂检查。那么在做检查前需要做哪些准备呢？"}, {"id": 1047, "vi": "Con em lúc 1 tháng đến bây giờ đã là 7 tháng rồi mà bé cứ hay buồn oẹ mà không oẹ được có tháng thì không thấy nhưng đa phần là mồm hay kiểu muốn oẹ ,bé vẫn hoạt động bình thường không có biểu hiện gì và giờ bé đang mọc răng ,vậy bé nhà em bị gì ạ và cho bé đi khám thì khám gì để biết", "en": "Since the baby was one month old until now, at seven months old, the baby keeps being fussy and unable to cry out; sometimes there are no episodes, but most of the time it is the mouth or wanting to cry. The baby is still active normally with no other signs. The baby is currently teething. So, what illness does my baby have, and what should I have the baby checked for to find out?", "zh": "宝宝从一个月到现在已经七个月了，一直经常想哭但哭不出来，有时一个月都看不到，大部分时间是嘴巴里想哭的样子。宝宝目前活动正常，没有其他异常表现，现在正在长牙。请问宝宝这是什么问题？去检查应该检查什么？"}, {"id": 1048, "vi": "Tôi phát hiện bị viêm gan siêu vi B từ năm 2005. Hiện tại, sức khỏe của tôi bình thường. Gần đây nhất, tôi đi khám và kiểm tra sức khỏe vào tháng 3/2019 tại Bạch Mai và tháng 9/2019 tại Bệnh viện Đa khoa Thăng Long. Các chỉ số như men gan đều ở mức bình thường, xét nghiệm kháng thể virus là dương tính. Tôi có những nguy cơ gì liên quan đến bệnh gan và cần phải làm gì để tránh các nguy cơ ấy?", "en": "I was diagnosed with chronic hepatitis B in 2005. Currently, my health is normal. Recently, I visited for check-up and examination in March 2019 at Bạch Mai Hospital and in September 2019 at Thăng Long General Hospital. Liver function tests show normal levels, and viral antibody tests are positive. What are my risks related to liver disease, and what should I do to avoid these risks?", "zh": "我于2005年发现患有乙型肝炎病毒性肝炎，目前身体状况正常。最近一次在2019年3月于白求恩医院、9月于升龙综合医院进行体检。肝功能指标均在正常范围内，病毒抗体检测结果为阳性。我与肝病相关的风险有哪些？我需要采取哪些措施来避免这些风险？"}, {"id": 1049, "vi": "Sau khi bị mèo cắn, hơi sợ gió khi nằm quạt có sao không?", "en": "After being bitten by a cat, is it normal to feel afraid of wind when lying near a fan?", "zh": "被猫咬后，躺风扇旁边怕风会有问题吗？"}, {"id": 1050, "vi": "Các biểu hiện gặp phải khi bị nhiễm HIV là gì?", "en": "What are the manifestations encountered when infected with HIV?", "zh": "感染HIV时会出现哪些症状？"}, {"id": 1051, "vi": "Vợ chồng em dạo gần đây có thả đúng ngày rụng trứng mà em vẫn không thấy có thai, chu kỳ của em rất đều, em có bị viêm lộ tuyến mức độ 2 và bác sĩ chỉ định dùng thuốc. Em muốn hỏi làm thế nào để em có thể có thai sớm nhất và trước khi có thai em nên uống và tiêm phòng những loại thuốc gì?", "en": "My husband and I have been having intercourse around the exact time of ovulation, yet I still haven't conceived. My cycle is very regular. I have cervical inflammation of degree 2, and the doctor has prescribed medication. I would like to ask how I can conceive as soon as possible, and what vaccines should I take and receive before trying to conceive?", "zh": "我最近和丈夫都按时在排卵日同房，但我还是没有怀孕，我的月经周期非常规律，我有轻度的宫颈糜烂，医生开了药。我想问一下，如何才能尽快怀孕？在怀孕前，我应该接种哪些疫苗？"}, {"id": 1052, "vi": "Mắt bị lên chắp lẹo có lây sang người khác không ạ?", "en": "Can eye inflammation caused by conjunctivitis be transmitted to other people?", "zh": "眼睛出现的皮疹会传染给其他人吗？"}, {"id": 1053, "vi": "Thường chói mắt khi nhìn màn hình điện thoại, máy tính có làm sao không?", "en": "Is it normal to experience glare when looking at a phone or computer screen?", "zh": "经常看手机、电脑屏幕会有什么问题吗？"}, {"id": 1054, "vi": "Em bị mụn nước mọc ở lưng khoảng 4 ngày. Vì không biết nên em tắm bây giờ mụn nước lây lan thành vùng. Đó có phải là bị bệnh thần kinh zola hay là bị kiến ba khoang không? Phương pháp điều trị là gì?", "en": "My skin has developed water blisters on the back about 4 days ago. Since I don't know what it is, I am now bathing, and the blisters are spreading to a larger area. Is this due to zoster nerve disease or due to a scorpion sting? What is the treatment method?", "zh": "我背部出现了水疱，大约4天了。因为不知道该怎么办，现在洗澡导致水疱扩散到了整个区域。这是带状疱疹还是虫咬引起的？治疗方法是什么？"}, {"id": 1055, "vi": "Điều trị ung thư buồng trứng tái phát thế nào hiệu quả?", "en": "How to effectively treat recurrent ovarian cancer?", "zh": "复发性卵巢癌如何有效治疗？"}, {"id": 1056, "vi": "Em có tiền sử 2 lần mang thai đều bị hội chứng antiphospholipid (đều phải tiêm thuốc). Em có thể tiêm vắc xin phòng COVID-19 được không ạ?", "en": "I have a history of two pregnancies both complicated by antiphospholipid syndrome (both requiring injection therapy). Can I receive the COVID-19 vaccine?", "zh": "我有两次妊娠均患有抗磷脂综合征（均需注射药物）的病史。我可以接种新冠疫苗吗？"}, {"id": 1057, "vi": "Nam giới có mắc bệnh ung thư vú không?", "en": "Can men get breast cancer?", "zh": "男性会得乳腺癌吗？"}, {"id": 1058, "vi": "Cách đây 3 tháng vợ em có tiêm ngừa 3 in 1 (sởi, quai bị, rubella). Sau khi tiêm 14 ngày vợ em bị viêm mang tai và sốt. Cho em hỏi biểu hiện viêm mang tai vậy có gây vô sinh ở nữ không? Em đang lo lắng tác dụng phụ của ngừa quai bị gây vô sinh ở nữ.", "en": "Three months ago, my wife received the 3-in-1 vaccine (measles, mumps, rubella). Fourteen days after vaccination, she developed otitis media and fever. I would like to know whether otitis media can cause infertility in women? I am worried about the risk of mumps vaccination causing infertility in women.", "zh": "三个月前我妻子接种了三联疫苗（麻疹、腮腺炎、风疹）。接种后14天，我妻子出现外耳道炎和发热。我想咨询一下，外耳道炎是否会导致女性不孕？我担心接种腮腺炎疫苗会引起女性不孕。"}, {"id": 1059, "vi": "Nước tiểu màu đỏ nhạt có nguy hiểm không?", "en": "Is pale red urine dangerous?", "zh": "尿液呈淡红色是否危险？"}, {"id": 1060, "vi": "Năm nay cháu 25 tuổi đã quan hệ tình dục nhưng kinh nguyệt của cháu không đều, 3 tháng nay cháu vẫn chưa thấy có kinh nguyệt, cháu đã từng bị sảy thai tự nhiên 2 lần, lần gần đây nhất là vào tháng 4 năm 2019. Như vậy là cháu bị sao vậy ạ và làm sao để cháu có kinh nguyệt trở lại ạ?", "en": "This year I am 25 years old and have had sexual intercourse, but my menstrual cycle has been irregular. For the past 3 months, I have not had any menstruation. I have previously experienced two spontaneous abortions, the most recent one being in April 2019. Thus, what is wrong with me and how can I restore my menstruation?", "zh": "今年我25岁，有性生活，但月经不规律，三个月来都没有来月经，曾自然流产两次，最近一次是2019年4月。这样是得了什么病啊？怎么才能让月经恢复呢？"}, {"id": 1061, "vi": "Tôi đang có thai 10 tuần, đi siêu âm bác sĩ nói có nhân xơ tử cung 15 mm, vậy có ảnh hưởng gi không?", "en": "I am 10 weeks pregnant, and the doctor said there is a uterine fibroid of 15 mm. Will this have any impact?", "zh": "我目前怀孕10周，做B超时医生说有子宫肌瘤，大小为15毫米，这会有影响吗？"}, {"id": 1062, "vi": "Vợ em đang bầu, em muốn làm xét nghiệm sàng lọc dị tật thì làm ở tuần thứ bao nhiêu ạ?", "en": "My wife is pregnant, I want to do the screening test for birth defects, at which week should I do it?", "zh": "我妻子正在怀孕，我想做畸形筛查，需要在怀孕几周做检查呢？"}, {"id": 1063, "vi": "Bác sĩ cho hỏi la sau mỗi lần điều trị hóa chất xong là mấy ngày sau tôi bị đau đầu ko thể ngủ yên được, các ngón tay bị tê quá. Và nữa là nhờ bác sĩ giúp về ăn uống dinh dưỡng như nào cho tốt với bệnh nhân bị ulimpho non hogkin tế bào T ngoại vi.", "en": "Doctor, I would like to ask: after each chemotherapy session, how many days later do I start experiencing headaches that make it impossible to sleep, and numbness in my fingers? Also, I would like to ask the doctor about proper nutrition for a patient with peripheral T-cell non-Hodgkin lymphoma.", "zh": "医生请问，每次化疗结束后，我几日会出现头痛，无法入睡，手指发麻的情况？另外，请医生帮忙指导一下，对于外周T细胞淋巴瘤患者，应该如何进行营养饮食？"}, {"id": 1064, "vi": "Bé bị bệnh nặng hay nhẹ nếu kết quả xét nghiệm G6PD là 0.3?", "en": "Is the child having a severe or mild illness if the G6PD test result is 0.3?", "zh": "如果G6PD检测结果为0.3，孩子病情是严重还是轻微？"}, {"id": 1065, "vi": "Rối loạn chuyển hóa bẩm sinh có di truyền trong gia đình không?", "en": "Is congenital metabolic disorder hereditary within the family?", "zh": "先天性代谢紊乱是否具有家族遗传性？"}, {"id": 1066, "vi": "Hôm qua theo lịch theo dõi chu kỳ kinh thì em đang tuần thai thứ 32 và có đi khám tổng quát. Em thấy kết quả siêu âm 4d thì ghi 31 tuần 2 ngày, cân nặng thì gần như không tăng so với mức siêu âm ngày 7/5. em có được đo huyết áp thì 2 lần 150/100, lần cuối cùng là 135/90, bàn tay hơi phù và đau khi ngủ dậy, chân phù và người bị ngứa. Sáng nay đo lại huyết áp, em không nhớ rõ chỉ số nhưng thấy bảo bình thường. em có xét nghiệm nước tiểu và chạy máy đo cơn co thì thấy bác sĩ nói bình thường. Xét nghiệm máu thì có 1 số chỗ tăng, giảm so với chuẩn. Tình trạng của em là như thế nào và có nguy hiểm đến thai không.", "en": "Yesterday according to the menstrual cycle monitoring schedule, I am at week 32 of pregnancy and had a general check-up. I noticed that the 4D ultrasound result shows 31 weeks and 2 days, and my weight has not increased significantly compared to the ultrasound done on May 7. I measured my blood pressure twice, showing 150/100, and the last reading was 135/90; my hands are slightly swollen and painful upon waking, my legs are swollen, and I have itching. This morning I measured my blood pressure again, but I do not remember the exact value, and I was told it is normal. I had a urine test and a contraction monitor test, and the doctor said both are normal. The blood test shows some values increased or decreased from the normal range. What is my current condition, and is there any risk to the pregnancy?", "zh": "昨天按照月经周期跟踪计划，我处于孕32周，并进行了常规检查。我看到4D超声结果记录为31周2天，与5月7日的超声相比体重几乎没有增加。我测过血压，两次均为150/100，最后一次为135/90，双手略肿，早晨醒来时有疼痛感，双脚水肿，身体有瘙痒感。今天早上再次测量血压，我不太记得具体数值，但感觉医生说正常。我做了尿液检查和宫缩监测，医生说结果正常。血检中有一些项目高于或低于正常范围。我的情况如何，对胎儿有危险吗？"}, {"id": 1067, "vi": "Em bé nhà em mới sinh, sau khi sinh được 5 ngày em có dùng dung dịch petadine để vệ sinh rốn cho bé (lúc này bé đã rụng rốn) trong vòng 12 ngày bang cách nho 1 -2 giọt vào rốn bé sau khi tam. Sau đó em được biết petadine không được dùng cho trẻ sơ sinh nên em đã không dùng cho bé nữa. Vậy em xin hỏi bác sĩ petadine thời gian dùng như vậy có ảnh hưởng đến bé không? Cần làm xét nghiệm gì không?", "en": "My baby was just born. After birth, on day 5, I used Pediadine solution to clean the navel (at that time the navel had fallen off). For the next 12 days, I applied 1-2 drops into the navel after each bath. Later, I learned that Pediadine is not recommended for use in newborns, so I stopped using it for my baby. Therefore, I would like to ask the doctor whether using Pediadine in this manner has affected my baby, and if any tests are needed.", "zh": "我女儿刚出生，出生后5天我用Petadine溶液清洁了宝宝的脐部（此时宝宝的脐带已经脱落），在接下来的12天里，每次换尿布后我会滴1-2滴Petadine到宝宝的脐部。后来我了解到Petadine不适用于新生儿，因此之后就停止使用了。我想咨询一下医生，这样使用Petadine会不会对宝宝有影响？需要做哪些检查？"}, {"id": 1068, "vi": "Tôi sắp đi công tác trong tháng này và đã tiêm một mũi vaccine AstraZeneca vào cuối tháng trước. Vậy khoảng cách giữa 2 mũi tiêm sớm nhất là sau bao nhiêu ngày để đạt hiệu quả của vaccine?", "en": "I am about to go on business this month and have received one dose of AstraZeneca vaccine at the end of last month. What is the earliest number of days between the two doses to achieve the vaccine's effectiveness?", "zh": "我即将本月出差，上个月末接种了一剂阿斯利康疫苗。那么两剂疫苗最早需要间隔多少天才能达到疫苗的最佳效果？"}, {"id": 1069, "vi": "Xét nghiệm FTA-ABS có ý nghĩa như thế nào? Các xét nghiệm RPR, TPHA có khả năng dương tính giả không ạ?", "en": "What is the significance of FTA-ABS testing? Can RPR, TPHA tests give false positive results?", "zh": "FTA-ABS 检查的意义是什么？RPR、TPHA 检查会出现假阳性吗？"}, {"id": 1070, "vi": "Nổi mảng da và ngứa ở vùng bẹn có phải nấm hoặc khuẩn 2 đầu không?", "en": "Are skin patches and itching in the inguinal region due to fungus or bacteria with two ends?", "zh": "阴部出现皮疹和瘙痒是否由真菌或两端菌引起？"}, {"id": 1071, "vi": "Xin hỏi, em bị viêm bao quy đầu. Bác sĩ kê thuốc uống Selazn và Pollezin thì có tác dụng và điều trị không?", "en": "May I ask, I have glans penis inflammation. Is it effective and treatable to take Selazn and Pollezin prescribed by the doctor?", "zh": "请问，我患有包皮炎。医生开的口服药是Selazn和Pollezin，这两种药有疗效和治疗作用吗？"}, {"id": 1072, "vi": "Em bị bệnh gout đã gần 10 năm uống nhiều loại thuốc (cả đông và tây y) nhưng không khỏi. Nhờ tư vấn giúp nên dùng loại thuốc nào để điều trị và thực đơn ăn uống thế nào cho hợp lý.", "en": "My child has had gout for nearly 10 years and has been taking many types of medication (both traditional and Western medicine), but it hasn't improved. With your advice, I would like to know which medication to use for treatment and what a reasonable diet should be.", "zh": "我患有痛风已近10年，服用过多种药物（包括中西药），但一直未痊愈。希望能得到建议，应该使用哪种药物治疗，以及如何合理安排饮食。"}, {"id": 1073, "vi": "Em đang mang bầu được 16 tuần hôm trước em có làm xét nghiệm Triple test ở tuần 15 (mã bệnh của em là 140819-1109017839). Kết quả em vẫn chưa hiểu lắm và em có nghe người quen nói làm xét nghiệm này ở tuần 18-20 mới chính xác nhất vậy em có phải làm xét nghiệm lại không? Mong giải thích giúp em với ạ.", "en": "I am currently 16 weeks pregnant. Previously, I did a Triple test at week 15 (my patient ID is 140819-1109017839). I still don't fully understand the results, and I heard from a friend that the Triple test is most accurate between weeks 18-20. Do I need to repeat the test? I hope for your explanation.", "zh": "我目前怀孕16周，之前在15周做过三联筛查（我的病例号是140819-1109017839）。但我对检查结果还不太理解，我听朋友说三联筛查在18-20周做才最准确，所以我是否需要重新做一次检查？希望得到您的解释。"}, {"id": 1074, "vi": "Mẹ tôi 50 tuổi, đã sạch kinh nửa năm nay. Đợt này thấy ra ít máu kinh nhưng rải rác trong tháng chứ không đều đặn. Mẹ tôi có khả năng bị ung thư không?", "en": "My mother is 50 years old and has been amenorrheic for half a year. This time, she has noticed occasional spotting but not regularly throughout the month. Is my mother at risk of cancer?", "zh": "我母亲50岁，过去半年没有月经。这次发现月经量很少，但每月零星出现，不规律。她有患癌症的可能吗？"}, {"id": 1075, "vi": "Phẫu thuật hẹp vách ngăn và thu gọn cánh mũi cùng 1 lần được không?", "en": "Can septal reduction and nasal wing correction be performed at the same time?", "zh": "鼻中隔成形术和鼻翼缩小术能否一次完成？"}, {"id": 1076, "vi": "Thường là 18 tuổi mới mọc răng khôn nhưng sao em 14 tuổi đã mọc 1 cái và đến nay 15 tuổi vừa mới mọc thêm 1 cái nữa tổng cộng là 2 cái, vậy là có vấn đề gì không", "en": "Usually, wisdom teeth start to emerge at 18 years old, but my sister had one emerge at 14 years old, and now at 15 years old, she has just grown another one, making a total of two. Is there any problem?", "zh": "通常18岁才长智齿，但妹妹14岁就已经长了1颗，到如今15岁又长了1颗，总共2颗，这种情况有问题吗？"}, {"id": 1077, "vi": "Em có làm xét nghiệm NST cho cả 2 vợ chồng, chồng em kết quả bình thường. Kết quả của em bị bất thường về NST, em muốn hỏi bác sĩ nếu như trường hợp của em thì có ảnh hưởng gì đến em bé sau này khi em mang thai không ạ?", "en": "I have done karyotyping testing for both my husband and myself, and my husband's results are normal. My results show abnormality in karyotype. I would like to ask the doctor whether, in my case, it will affect my future child when I am pregnant.", "zh": "我做了夫妻双方的染色体检查，丈夫结果正常，我的检查结果染色体异常。我想咨询一下医生，如果我的情况，将来我怀孕后对胎儿会有影响吗？"}, {"id": 1078, "vi": "Cổ họng khó chịu và có đờm là biểu hiện của bệnh gì?", "en": "Unpleasant throat and coughing up sputum are signs of what disease?", "zh": "喉咙不适和有痰是哪种疾病的症状？"}, {"id": 1079, "vi": "Bác sĩ cho em hỏi, trước khi sinh, các bác sĩ ở viện có lấy máu và nước tiểu để kiểm tra? Vậy bác sĩ cho em hỏi, các bác sĩ lấy mẫu như vậy để kiểm tra vấn đề gì ạ? Và sau sinh 3–4 hôm, em có nổi hạch ở nách gây đau! Liệu đó có phải đau do vết khâu nên nổi hạch như vậy không ạ?", "en": "Doctor, I would like to ask, before delivery, do the doctors at the hospital take blood and urine samples for examination? So, doctor, I would like to ask, what issues do the doctors check for by taking such samples? And after delivery, 3–4 days later, I have developed a swollen lymph node in my armpit causing pain! Could this pain be due to the surgical suture causing the lymph node to swell like this?", "zh": "医生请问一下，在分娩前，医院的医生会取血液和尿液进行检查吗？那么医生可以告诉一下，医生们取这些样本是为了检查什么问题吗？另外，生完孩子3到4天后，我腋下出现了肿块并伴有疼痛！这种情况是否是因为缝合伤口引起的腋下肿块疼痛呢？"}, {"id": 1080, "vi": "Mang thai 38 tuần, đau nhói bên phải gần ngực khi ngủ có phải dấu hiệu sắp sinh", "en": "At 38 weeks of pregnancy, sharp pain on the right side near the chest when sleeping, is this a sign of impending labor?", "zh": "怀孕38周，睡觉时右侧靠近胸部有刺痛感，是否是即将分娩的征兆？"}, {"id": 1081, "vi": "Tại sao cháu tôi đi khám da liễu bị chàm khi chi toa mua thuốc lại có thuốc sổ mửa viêm họng", "en": "Why does my child visit dermatology for atopic dermatitis and have a medication prescription that includes an antiemetic and throat inflammation drug?", "zh": "为什么我孩子去皮肤科就诊湿疹，开的药里却有治呕吐和咽喉炎的药物？"}, {"id": 1082, "vi": "Vì sao bệnh nhân tiểu đường type 2 cần phải làm kiểm tra HbA1C?", "en": "Why does a type 2 diabetes patient need to undergo HbA1C testing?", "zh": "为什么2型糖尿病患者需要进行HbA1C检查？"}, {"id": 1083, "vi": "Em có chu kỳ kinh là 28 ngày. Mấy hôm nay em thử que đều lên hai vạch và rõ dần. Chiều và tối hôm qua em có bị ra máu màu hồng nhạt, sáng nay em thấy ra chút máu nâu hơn nhưng không nhiều. Em không thấy có hiện tượng đau bụng. Siêu âm thì chưa thấy túi ối. Xét nghiệm thì có kết quả như sau. Bác sĩ tư vấn giúp em là liệu có bị thai ngoài tử cung hay dọa sảy không ạ?", "en": "I have a 28-day menstrual cycle. For several days I have tested with the pregnancy test and both lines have appeared and have become clearer. Last night, in the afternoon and evening, I had a little pinkish bleeding, and this morning I noticed a bit of brownish blood but not much. I do not feel any abdominal pain. The ultrasound did not show the amniotic sac. The lab test results are as follows. Doctor, could you please advise me whether I might have an ectopic pregnancy or a threatened miscarriage?", "zh": "我的月经周期是28天。最近几天用验孕棒检测都出现两条线，并且逐渐变清晰。昨天下午和晚上我有少量粉红色出血，今天早上又看到一点棕褐色的血，但量不多。我没有腹痛的感觉。超声检查未见羊水囊。化验结果如下。医生能帮我判断一下，我是不是宫外孕或者有流产的危险吗？"}, {"id": 1084, "vi": "Chiều cao của trẻ bị thấp hơn so với tuổi, cần làm gì?", "en": "The child's height is below the expected for age. What should be done?", "zh": "儿童身高低于同龄人，需要做什么？"}, {"id": 1085, "vi": "Trẻ 8 tuổi bị phì đại cơ thái dương hàm có ảnh hưởng gì không?", "en": "What impact does enlarged maxillary mandibular bone have in an 8-year-old child?", "zh": "8岁儿童患有下颌骨增生会有什么影响吗？"}, {"id": 1086, "vi": "Chụp X-quang phổi thấy kẽ phổi giãn, cần chăm sóc và điều trị như thế nào?", "en": "Chest X-ray shows widened interstitial spaces. What kind of care and treatment is needed?", "zh": "胸部X光显示肺纹理增宽，应如何护理和治疗？"}, {"id": 1087, "vi": "Em tên Hoàng, năm nay 20 tuổi, bị teo tinh hoàn hai bên. Xin tư vấn cách chữa trị hiệu quả cho em và việc chữa mất bao lâu mới khôi phục lại? Nếu chữa được có ảnh hưởng không khi em có dự định đi nghĩa vụ?", "en": "My name is Hoang, I am 20 years old and have bilateral testicular atrophy. I would like to consult on the most effective treatment for me and how long it will take to recover. If treatable, will it have any impact if I have military service plans?", "zh": "我叫黄，今年20岁，双侧睾丸萎缩。想咨询一下有效的治疗方法，以及治疗多久才能恢复？如果能治好，将来我服兵役会不会有影响？"}, {"id": 1088, "vi": "Vaccine COVID-19 AstraZeneca nên tiêm cho đối tượng nào? Giữa người cao tuổi và người trẻ tuổi, đối tượng nào tiêm vaccine sẽ an toàn hơn?", "en": "Which group should receive the COVID-19 AstraZeneca vaccine? Between the elderly and young people, which group is safer to administer the vaccine to?", "zh": "新冠疫苗阿斯利康应接种给哪些人群？在老年人和年轻人之间，哪个群体接种疫苗更安全？"}, {"id": 1089, "vi": "Dùng thuốc điều trị lao màng phổi 3 tháng có bầu gây ảnh hưởng gì không?", "en": "Does using anti-tuberculosis medication for 3 months during pregnancy cause any effects?", "zh": "使用3个月的肺结核胸膜病治疗药物是否会影响怀孕？"}, {"id": 1090, "vi": "Phương pháp điều trị tối ưu u mềm lây?", "en": "What is the optimal treatment approach for soft tissue sarcoma?", "zh": "软组织肿瘤的最佳治疗方法是什么？"}, {"id": 1091, "vi": "Tôi bị đau quanh vùng hông,cúi lên cúi xuống hay đứng lâu 1 chỗ là bị đau, không nhấc chân đi dc,phải vít tường 1 lúc mới nhấc chân đi dc.Tôi có đi chụp cộng hưởng từ cột sống thì không bị thoái hoá hay thoát vị,nhưng lại có 1 cái nang. Bác sĩ cho tôi hỏi cái nang đấy có nguy hiểm không .", "en": "I have pain around the hip area, pain when bending forward or backward, or standing in one place for a long time, making it difficult to lift my foot; I have to lean against the wall for a while before I can lift my foot. I have undergone MRI of the spine and it shows no degeneration or herniation, but there is a cyst. Doctor, I would like to ask whether this cyst is dangerous.", "zh": "我髋部周围疼痛，弯腰、俯身或长时间站立时会疼痛，无法抬起脚，必须靠墙站一会儿才能抬起脚。我做过脊柱MRI检查，没有发现退行性病变或椎间盘突出，但发现有一个囊肿。医生请问这个囊肿是否有危险？"}, {"id": 1092, "vi": "Xét nghiệm canxi cho trẻ sau tiêm vắc xin sởi có được không", "en": "Can calcium test be performed for children after measles vaccine injection?", "zh": "接种麻疹疫苗后是否需要给儿童做钙检测"}, {"id": 1093, "vi": "Em chào bác sĩ. Em chuyển phôi ngày 23/9, ngày 2/10 (9 scp) em xét nghiệm beta HCG tại Medlatec kết quả là 224. Hôm nay em xét nghiệm lại (15 scp) kết quả là 281. Bác sĩ cho em hỏi kết quả này có nghi ngờ khả năng thai bị thái hoá không ạ? Vì em không có biểu hiện đau bụng hay ra máu. Vài ngày trước em thấy có đau lâm râm nhưng chỉ 1 lúc thì hết.", "en": "Hello doctor. I transferred the embryo on September 23. On October 2 (9 SCP), I had a beta-HCG test at Medlatec and the result was 224. Today I tested again (15 SCP) and the result is 281. Doctor, I would like to ask if this result raises suspicion of possible miscarriage? Because I have no symptoms of abdominal pain or bleeding. A few days ago, I felt mild pain but it only lasted for a moment and then disappeared.", "zh": "您好，医生。我于9月23日进行胚胎移植，10月2日（9 SCP）在Medlatec做了β-hCG检测，结果为224。今天我再次检测（15 SCP），结果为281。医生请问这个结果是否提示胚胎可能发生了吸收？因为我没有腹痛或出血的症状。几天前我曾感到轻微的腹痛，但只持续了一瞬间就消失了。"}, {"id": 1094, "vi": "Có khả năng phục hồi thị lực khi mắt mù lòa không?", "en": "Is there a possibility of visual recovery in cases of complete blindness?", "zh": "失明的眼睛有恢复视力的可能吗？"}, {"id": 1095, "vi": "Điều trị viêm tai giữa như thế nào?", "en": "How to treat middle ear inflammation?", "zh": "中耳炎如何治疗？"}, {"id": 1096, "vi": "Em bị thoái hoá đốt sống lưng, bị đau không cúi xuống được, vận động khó khăn.. bác sỹ có phương pháp điều trị triệt để không", "en": "The child has lumbar spine degeneration, pain, unable to bend down, difficult movement... Doctor, is there a curative treatment method?", "zh": "我患有腰椎退行性病变，疼痛严重，无法弯腰，活动困难。医生有没有根治的方法？"}, {"id": 1097, "vi": "Glô-côm góc đóng tiến triển nặng dẫn đến mù lòa vĩnh viễn có cách nào điều trị không?", "en": "Does advanced closed-angle glaucoma progression leading to permanent blindness have any treatment options?", "zh": "闭角型青光眼进展性严重导致永久性失明是否有治疗方法？"}, {"id": 1098, "vi": "Đại tiện phân sống, khó tăng cân phải làm sao?", "en": "Stool is live, difficult to gain weight, what should I do?", "zh": "大便带血，体重增长困难怎么办？"}, {"id": 1099, "vi": "Thường xuyên căng thẳng, lo lắng, thở nông, đổ mồ hôi lòng bàn chân tay cần điều trị chuyên khoa nào?", "en": "Which specialty should treat frequent stress, anxiety, shallow breathing, and sweating of the palms and soles?", "zh": "经常紧张、焦虑、呼吸浅表、手脚出汗，需要就诊哪个专科？"}, {"id": 1100, "vi": "Nói ngọng do lưỡi đầy có thể phẫu thuật cắt bớt không?", "en": "Can the speech disorder due to a full tongue be treated by surgical reduction?", "zh": "舌根阻塞引起的言语不清是否可以手术切除？"}, {"id": 1101, "vi": "Tôi từ đầu năm 2020 đến giờ, mỗi lần đi vệ sinh lúc thì khó đi, lúc thì đi ra như là nước chảy, lúc thì đỡ hơn nhưng chưa đi được như bình thường hôm nào cả từ đầu năm 2020 đến giờ.", "en": "Since the beginning of 2020 until now, every time I go to the toilet, it is sometimes difficult to pass stool, sometimes it feels like water is flowing out, sometimes it improves but still not normal compared to any time since the beginning of 2020 until now.", "zh": "从2020年初至今，每次排便时有时困难，有时像水一样流出，有时有所缓解，但始终无法恢复到2020年初以来正常的排便状态。"}, {"id": 1102, "vi": "Vợ chồng tôi kết hôn được 5 năm, đã làm IVF 3 lần đều không nuôi được phôi ngày 5, chuyển phôi ngày 3 lần thứ 3 cũng không được. Chồng tôi đã xét nghiệm phân mảnh tinh trùng, kết quả 15,06%. Tôi bình thường. Bây giờ, tôi hoang mang không biết lý do tại sao và phương pháp hỗ trợ như thế nào nếu tiếp tục làm IVF?", "en": "My husband and I have been married for 5 years and have undergone IVF three times, all of which failed to develop embryos. On day 5, embryo transfer was performed three times, but all were unsuccessful. My husband has undergone sperm morphology analysis, with results showing 15.06%. I am otherwise normal. Currently, I am anxious and do not know the reason for this failure, and I am unsure about the appropriate support methods if we decide to continue with IVF?", "zh": "我夫妻结婚5年，已进行3次体外受精（IVF），每次均未能成功培育出胚胎，第3次移植日为第5天，也未能成功。我丈夫已进行精子形态学检查，结果为15.06%。我自身情况正常。现在我非常焦虑，不知为何无法成功，如果继续进行IVF，应采取何种辅助措施？"}, {"id": 1103, "vi": "Em bị nhu mô gan thu nhỏ nhẹ. Em đã xét nghiệm âm tính với viêm gan B và C. Bây giờ em có phải dùng thêm kháng sinh hay thuốc gì không ạ? Cho em xin đơn ạ.", "en": "My liver parenchyma is slightly reduced. I have tested negative for hepatitis B and C. Now, do I need to take additional antibiotics or any other medication? Please provide me with a prescription.", "zh": "我患有轻度肝实质缩小。我已检查出乙肝和丙肝均为阴性。现在我是否需要额外使用抗生素或其他药物？请给我开个处方。"}, {"id": 1104, "vi": "Đau nhức một bên mặt do mọc răng khôn phải làm sao?", "en": "What should I do for pain and discomfort on one side of the face due to wisdom tooth eruption?", "zh": "智齿一侧面部疼痛怎么办？"}, {"id": 1105, "vi": "Bị chuột rút tay chân làm xét nghiệm máu có ra bệnh hay không?", "en": "Will a blood test show any disease due to hand and foot cramps?", "zh": "手足抽筋是否会导致血液检查出病？"}, {"id": 1106, "vi": "Tai phát ra âm thanh có phải ù tai không?", "en": "Does the ear producing a sound mean tinnitus?", "zh": "耳部发出声音是否属于耳鸣？"}, {"id": 1107, "vi": "Viêm bàng quang ở nữ giới có chữa khỏi được không?", "en": "Can urinary tract infection in women be cured?", "zh": "女性膀胱炎能治愈吗？"}, {"id": 1108, "vi": "Chụp MRI não phát hiện tổn thương nhuyễn não rải rác ở thùy thái dương trái, thùy trán hai bên có sao không?", "en": "Does MRI of the brain show scattered soft brain lesions in the left temporal lobe and bilateral frontal lobes?", "zh": "脑部MRI检查发现左侧额叶和双侧额叶散在的软脑膜损伤吗？"}, {"id": 1109, "vi": "Đầu gối sưng nhẹ bình thường không đau nhưng ấn vào thì đau có phải dấu hiệu bệnh gút không?", "en": "Mild swelling of the knee, normally painless but painful on pressure, could be a sign of gout?", "zh": "膝盖轻微肿胀，通常不痛，但按压时会痛，这是否是痛风的征兆？"}, {"id": 1110, "vi": "Em năm nay 23 tuổi. Có bé 5 tuần thì ra máu và không đau bụng, đi khám thì thai được chẩn đoán là nằm lệch về bên phải buồng niêm mạc, cách lớp thanh mạc 6mm, chưa có phôi thai. Không có hình ảnh bóc tách túi thai. Vậy việc nằm lệch buồng niêm mạc tử cung có ảnh hưởng gì đến em bé không ạ? Có nguy cơ sảy thai không ạ? Và thai có thể di chuyển dần vào vị trí giữa ở giai đoạn này nữa không ạ?", "en": "I am 23 years old. At 5 weeks of pregnancy, I had vaginal bleeding without abdominal pain. Upon examination, the pregnancy was diagnosed as lying laterally to the right uterine cavity, 6 mm away from the peritoneal layer, with no embryo present. There was no image of trophoblastic detachment. Therefore, does lateral positioning of the uterine cavity affect the baby? Is there a risk of miscarriage? And can the fetus move gradually into a central position at this stage?", "zh": "我今年23岁。怀孕5周时出现阴道流血，但没有腹痛。去医院检查，医生诊断为胚胎位于子宫内膜偏右侧，距离腹膜层6毫米，尚未见到胚胎组织，也没有见到胎囊剥离的影像。请问，子宫内膜偏移是否会影响胎儿？有流产的风险吗？在这个阶段，胎儿能否逐渐移动到中间位置呢？"}, {"id": 1111, "vi": "Thâm đầu ngón tay sau bó bột có sao không?", "en": "Is there any problem with the fingertip's deep tissue after casting?", "zh": "夹板后手指变黑有没有问题？"}, {"id": 1112, "vi": "Đứt động mạch trên da của dương vật đi thẳng đến đoạn giữa bìu có ảnh hưởng đến sức khỏe không?", "en": "Does the rupture of the penile arterial supply going directly to the mid-scrotal region affect health?", "zh": "阴茎动脉损伤直接到达阴囊中部会对健康造成影响吗？"}, {"id": 1113, "vi": "Ba mình hiện tại bị tăng huyết áp dẫn đến bị liệt lưỡi không nói được. Hiện tại đang dùng thuốc cân bằng huyết áp. Cho hỏi có thể tuư vấn về vấn đề ăn uống hằng ngày và cách cân bằng huyết áp không ?", "en": "I currently have hypertension leading to tongue paralysis and inability to speak. I am currently taking medication to balance blood pressure. May I ask for advice regarding daily diet and methods to balance blood pressure?", "zh": "我目前患有高血压导致舌瘫无法说话。目前正在服用降压药。想咨询一下日常饮食以及如何平衡血压的问题？"}, {"id": 1114, "vi": "Triệu chứng tê một bên mặt là dấu hiệu bệnh gì?", "en": "What disease is indicated by the symptom of numbness on one side of the face?", "zh": "一侧面部麻木是哪种疾病的征兆？"}, {"id": 1115, "vi": "Xuất hiện mảng trắng trong miệng trẻ nhỏ là do đâu?", "en": "What is the cause of white patches in the mouth of young children?", "zh": "婴幼儿口腔出现白色斑块的原因是什么？"}, {"id": 1116, "vi": "Bác sĩ cho em hỏi, mẹ chồng em năm nay 45 tuổi, cách đây hơn 1 năm bị rong kinh cứ tầm nửa tháng xong lại hết 4-5 ngày rồi lại bị tiếp. Đi khám thì bác sĩ bảo là do rối loạn nội tiết tố các thứ và cho uống thuốc. Uống được tầm 2-3 tháng thì lại bị tái lại. Mà mẹ em không có biểu hiện đau bụng hay mệt mỏi gì cả. Lần này đi khám bác sĩ chỉ định cho nạo nội mạc tử cung. Như vậy là mẹ em bị làm sao ạ?", "en": "Doctor, I would like to ask you, my mother-in-law is 45 years old this year. About one year ago, she started having irregular periods, occurring roughly every half month, lasting for 4-5 days before stopping and then recurring again. When she went to the doctor, the doctor said it was due to hormonal disorders and prescribed medication. After taking the medication for about 2-3 months, the symptoms returned. However, my mother does not have any abdominal pain or fatigue. This time, the doctor has recommended a hysterectomy (endometrial ablation). So, what is the condition of my mother?", "zh": "医生请问一下，我婆婆今年45岁，一年前开始出现月经不规律，大约每半个月一次，持续4-5天后又停止，之后又复发。去医院检查，医生说是由于内分泌紊乱，开了药治疗。吃了大约2-3个月后又复发了。但婆婆没有任何腹痛或乏力的症状。这次去医院，医生建议做宫腔刮除术。请问婆婆这是什么情况？"}, {"id": 1117, "vi": "Em chuyển phôi IVF 21 ngày, siêu âm kết quả hình ảnh cho thấy có một trống âm đường kính 3,0 mm. Như thế có ý nghĩa như thế nào và kết quả đó có ổn không ạ? Sau chuyển phôi 18 ngày em xét nghiệm beta là 881,3. Em chuyển phôi ngày 2 ạ.", "en": "I transferred embryo via IVF on day 21, ultrasound imaging showed a fluid anechoic area with a diameter of 3.0 mm. What does this mean and is this result stable? Eighty-eight hundred and thirteen beta was detected on day 18 after embryo transfer. I transferred the embryo on day 2.", "zh": "体外受精移植21天，超声检查显示有一个3.0毫米的无回声区。这表示什么？这个结果是否正常？移植后18天，血清β-hCG值为881.3。移植是在第2天进行的。"}, {"id": 1118, "vi": "Kết quả xét nghiệm đề tie lệ tiểu cầu có kích thước lớn là 30,5 (tăng), Độ phân bố tiểu cầu(PDW) là 19,7(cao) thì là như thế nào, có bệnh gì không?", "en": "The result of the test requested for large platelet size is 30.5 (increased), and the platelet distribution width (PDW) is 19.7 (high); what does this mean, and is there any disease?", "zh": "血小板体积分布宽度（PDW）为19.7（增高），血小板大小分布宽度为30.5（增高），这种情况如何，是否有疾病？"}, {"id": 1119, "vi": "Mấy ngày gần đây em có triệu chứng hay bị ọe(không phải buồn nôn), ăn không ngon miệng, hay buồn tiểu, có khi vừa đi xong lại buồn tiếp (có lúc buồn nhưng không đi được có lúc lại tiểu dễ dàng, lượng nước tiểu mỗi lần vừa, nước tiểu lúc trong lúc vàng nhạt,không tiểu buốt hay rát, không tiểu đêm). vậy những triệu chứng này liên quan nhiều đến bệnh lý nào. Những triệu chứng này có liên quan đến thủ dâm nhiều không. Dạo gần đây em hay thủ dâm 1 lần/ngày.", "en": "Recently, I have been experiencing symptoms of frequent urination (not nausea), poor appetite, frequent urge to urinate, sometimes feeling the urge to urinate right after urinating (sometimes feeling the urge but unable to urinate, sometimes urinating easily, with a moderate amount of urine per episode, urine being clear at times and pale yellow at other times, no burning or pain during urination, no nocturia). Therefore, what diseases are these symptoms most closely related to? Are these symptoms related to frequent masturbation? Recently, I have been masturbating once a day.", "zh": "最近几天我经常有尿意（不是恶心），食欲不好，经常想小便，有时刚上完厕所又想上（有时想上却尿不出来，有时则容易排尿，每次尿量适中，尿液有时清亮有时略黄，没有排尿时疼痛或灼热感，也没有夜尿）。这些症状可能与哪些疾病有关？这些症状和频繁自慰有关吗？最近我每天自慰一次。"}, {"id": 1120, "vi": "Các chỉ số xét nghiệm nội tiết của em như vậy có phản ánh tình trạng bệnh đa nang buồng trứng và nhân xơ tử cung không ạ? (Em xét nghiệm vào giữa chu kỳ kinh.) Bác sĩ siêu âm kết luận theo dõi đa nang buồng trứng cả 2 bên và nhân xơ 8 mm. Nội mạc tử cung của em là 4,2 mm thì có phải mỏng và khó thụ thai không ạ?", "en": "Do these endocrine lab results reflect my polycystic ovary syndrome and uterine fibroids? (I had the test in the mid-cycle.) The ultrasound doctor concluded to monitor polycystic ovaries on both sides and fibroids measuring 8 mm. My endometrial thickness is 4.2 mm—does this mean my endometrium is thin and thus difficult to conceive?", "zh": "我的内分泌检查指标是否能反映多囊卵巢和子宫内膜纤维化的情况呢？（我是在月经中期进行检查的。）医生的超声检查结论是双侧多囊卵巢以及子宫内膜纤维化8毫米。我的子宫内膜厚度为4.2毫米，这是否属于偏薄，从而导致不孕呢？"}, {"id": 1121, "vi": "Dùng thuốc điều trị suy thận độ 3A thế nào cho hiệu quả?", "en": "How to use medication for effective treatment of stage 3A chronic kidney disease?", "zh": "如何治疗3A期肾功能不全药物治疗才能更有效？"}, {"id": 1122, "vi": "Em mới mang thai 13 tuần và em đi khám xét nhiêm độ mờ da gáy tỉ số bênh đao là 1:163 thì tỉ lệ bị đao như nào và độ mờ da gáy là 1,9mm.", "en": "I am 13 weeks pregnant and I had a screening test showing a nuchal fold thickness of 1.9 mm and a chromosomal abnormality risk ratio of 1:163. What is the risk of having Down syndrome in this case?", "zh": "我怀孕13周，检查出皮肤褶皱模糊，颈项透明层比值为1:163，那么患有唐氏综合症的风险是多少？皮肤褶皱模糊为1.9毫米。"}, {"id": 1123, "vi": "Bác sĩ cho em hỏi, em chụp CT bác sĩ kết luận giãn phế nang như mô thủy dưới phổi bên phải. Vậy bệnh này của em có chữa được không, sau này có biến chứng gì không? Thỉnh thoảng em thấy mệt, khó thở, thở nóng và nhanh.", "en": "Doctor, I would like to ask you, I had a CT scan and the doctor concluded that there is dilated alveolar spaces resembling pulmonary edema on the right lung. So, is this condition treatable for me, and what complications might occur later? Occasionally, I feel tired, short of breath, and have a hot, rapid breathing.", "zh": "医生请问一下，我做CT检查，医生诊断为右肺下叶肺大泡扩张，像肺组织下的液体。这种病能治好吗？以后会不会有并发症？我偶尔会感到疲倦、呼吸困难、呼吸急促和发热。"}, {"id": 1124, "vi": "Cho em hỏi em đi xét nghiệm hiv ở viện pateur sau 5 tháng 1 tuần quan hệ ko an toàn âm tính thì em đã an toàn 100% không bị chưa ạ. Do em mấy nay hay ngứa người và nổi đỏ như hình nên em sợ", "en": "May I ask, after 5 months since unsafe sexual contact, I got a negative HIV test result at Paulet Hospital, does that mean I am 100% safe from HIV infection? Because lately I have been experiencing itching all over my body and redness as shown in the picture, so I am worried.", "zh": "我想咨询一下，我在帕特医院进行HIV检测，距离不安全性行为5个月后，结果为阴性，那么我是否已经100%安全，没有被感染呢？因为最近我经常感到身体瘙痒，并且出现红肿，像图片所示，所以我很担心。"}, {"id": 1125, "vi": "Cho em hỏi nếu trong thời gian đặt thuốc chữa viêm âm đạo nhưng lỡ quan hệ tình dục (có sử dụng bao cao su) thì có ảnh hưởng gì không  ?", "en": "May I ask, if during the period of taking medication for vaginal inflammation, I have sexual intercourse (using condoms), will it have any effect?", "zh": "我想问一下，如果在使用阴道炎药物期间不小心发生了性行为（使用了安全套），会不会有影响？"}, {"id": 1126, "vi": "Có thể điều trị bệnh thoái hóa cơ tủy do mất đoạn gen không?", "en": "Can degenerative spinal cord disease caused by gene deletion be treated?", "zh": "可以治疗由基因缺失引起的脊髓退行性疾病吗？"}, {"id": 1127, "vi": "Đã tiêm phòng mũi đầu vắc-xin 6 trong 1, mũi tiếp theo có thể tiêm vắc-xin 5 trong 1 không?", "en": "After receiving the first dose of the 6-in-1 vaccine, can the next dose be given as the 5-in-1 vaccine?", "zh": "第一针接种了六联疫苗，第二针可以接种五联疫苗吗？"}, {"id": 1128, "vi": "Phương pháp nào điều trị mờ mắt do máu tụ trong não chèn dây thần kinh? Có ảnh hưởng và để lại di chứng về mắt sau này không?", "en": "What method treats vision loss due to cerebral thrombosis compressing the optic nerve? Does it have any impact and leave any long-term eye complications?", "zh": "哪种方法治疗脑内血栓导致的视神经压迫性视力下降？是否会影响并留下后遗症？"}, {"id": 1129, "vi": "Trong gan xuất hiện ổ dịch có nguy hiểm không?", "en": "Is a purulent focus in the liver dangerous?", "zh": "肝脏出现脓肿危险吗？"}, {"id": 1130, "vi": "Bác sĩ cho em hỏi chân của em nó nổi mụn có dịch cứ khỏi đơt này lại mọc mụn đợt khác lên là sao.", "en": "Doctor, I would like to ask: my leg has pus-filled bumps that keep coming back; every time one goes away, new bumps appear again. Why is that happening?", "zh": "医生请问，我的脚上反复出现有脓液的痘痘，每次好了又会出现新的，这是怎么回事？"}, {"id": 1131, "vi": "Sau khi phẫu thuật tuyến giáp có thể sinh con được không?", "en": "Can a person conceive after thyroid surgery?", "zh": "甲状腺手术后可以生育吗？"}, {"id": 1132, "vi": "Chụp phim não xuất hiện màu đen bất thường", "en": "Brain MRI shows abnormal black appearance", "zh": "脑部X光片出现异常黑色"}, {"id": 1133, "vi": "Cận thị tuổi 31 nên mổ hay đeo kính?", "en": "Should a 31-year-old with nearsightedness have surgery or wear glasses?", "zh": "31岁高度近视应该手术还是戴眼镜？"}, {"id": 1134, "vi": "Em vừa hút thai lưu hôm qua. Sau khi hút em có đau bụng và ra máu, 2 tiếng sau khi hút em đi vệ sinh thì có ra cục máu đông, sau đó đau quặn bụng 1 lát. Sau khi hút xong thì 3 tiếng đầu e ra nhiều máu, sau ra ít dần và hết ngày thì không ra máu nữa ạ. Nằm nghỉ thì em không đau bụng, đi lại thì hơi đau. Em không bị mệt, không bị sốt, ăn uống bình thường. Vậy em chỉ ra máu 1 ngày như vậy thì có sao không?", "en": "I had a suction abortion yesterday. After the procedure, I experienced abdominal pain and bleeding. Two hours after the procedure, when I went to the toilet, I passed a blood clot, followed by a brief episode of cramping. In the first three hours after the procedure, I had a lot of bleeding, then the amount gradually decreased and stopped by the end of the day. When lying down, I do not feel abdominal pain, but there is slight pain when moving around. I am not feeling tired, no fever, and my eating and drinking are normal. So, is it normal to only have bleeding for one day like this?", "zh": "我昨天刚做了清宫手术。清宫后我出现腹痛和出血，两小时后我上厕所时发现有血块排出，之后腹痛一阵。清宫后头三小时出血较多，之后逐渐减少，到第二天就没有出血了。平躺休息时没有腹痛，活动时会有点痛。我没有感到疲倦，没有发烧，饮食正常。请问我只出血一天这样会不会有问题？"}, {"id": 1135, "vi": "Chồng em nhóm máu O+, em nhóm máu AB+. vậy có khả năng em bị hội chứng antiphosphipid không? Do trước kia em bị thai lưu 1 lần lúc 7 tuần, và 1 lần thai sinh hóa sau IUI, lần này may mắn có tự nhiên, nhưng em muốn phòng tránh những khả năng gây ra sảy thai. Vậy em cần làm những xét nghiệm gì?", "en": "My husband has blood group O+, and I have blood group AB+. Therefore, is it possible for me to have antiphospholipid syndrome? Previously, I had one miscarriage at 7 weeks, and one biochemical pregnancy after IUI; this time, I had a natural pregnancy, but I want to prevent any potential causes of miscarriage. So, what tests do I need to undergo?", "zh": "我丈夫血型O+，我血型AB+，我有患上抗磷脂综合征的可能吗？之前我有过一次7周的流产，以及一次体外受精后胚胎停育，这次是自然怀孕，但我想预防再次发生流产。因此我需要做哪些检查？"}, {"id": 1136, "vi": "Em có xét nghiệm beta ngày 7 sau chuyển phôi được 3.87. Hôm nay ngày 12 sau chuyển phôi xét nghiệm beta được 22.71 liệu có còn hy vọng không?", "en": "My beta test on day 7 after embryo transfer was 3.87. Today, on day 12 after embryo transfer, my beta test result is 22.71. Is there still hope?", "zh": "我移植后第7天测到β值为3.87，今天移植后第12天测到β值为22.71，还有希望吗？"}, {"id": 1137, "vi": "Bác sĩ cho em hỏi có cách nào phòng chống đột quỵ không ạ?", "en": "Doctor, I would like to ask if there is any way to prevent stroke?", "zh": "医生请问有没有预防中风的方法？"}, {"id": 1138, "vi": "Kết quả xét nghiệm giun đũa chó POS 0.71 có nặng không?", "en": "Is the result of the dog roundworm test POS 0.71 severe?", "zh": "蛔虫犬检测结果为0.71，是否严重？"}, {"id": 1139, "vi": "Chảy mủ dương vật điều trị như thế nào?", "en": "How to treat discharge from the penis?", "zh": "如何治疗尿道流脓？"}, {"id": 1140, "vi": "Trẻ bị sún răng và ố vàng cần phải làm gì?", "en": "What should be done for a child with enamel hypoplasia and tooth discoloration?", "zh": "孩子有牙齿发育不全和牙齿发黄怎么办？"}, {"id": 1141, "vi": "Mẹ em bị tiểu đường nhiều năm nay nhưng hay thức khuya. Không biết liệu thức khuya như vậy thì có ảnh hưởng tới đường máu của mẹ em không ạ? Mẹ em bị tiểu đường tuýp II ạ.", "en": "My mother has had diabetes for many years but often stays up late. I want to know if staying up late like this affects my mother's blood sugar levels. My mother has type II diabetes.", "zh": "我母亲多年来患有2型糖尿病，但经常熬夜。不知道这种情况是否会影响母亲的血糖水平？母亲是2型糖尿病。"}, {"id": 1142, "vi": "Xét nghiệm hiv khẳng định ở ngày thứ 26 sau nguy cơ thì yên tâm đc bao nhiêu %.", "en": "How much can one feel reassured after a confirmatory HIV test on day 26 after exposure?", "zh": "在暴露风险后第26天进行HIV确认检测，可以放心的百分比是多少%？"}, {"id": 1143, "vi": "Nữ giới viêm chân răng, sưng lợi điều trị như thế nào?", "en": "How to treat female patients with periodontal inflammation and swollen gums?", "zh": "女性牙周炎、牙龈肿胀如何治疗？"}, {"id": 1144, "vi": "Cha mẹ cần chăm sóc và theo dõi sức khỏe cho trẻ bị dậy thì sớm như thế nào?", "en": "How should parents care for and monitor the health of a child with early puberty?", "zh": "父母应该如何照顾和监测早熟儿童的健康？"}, {"id": 1145, "vi": "Điều trị trĩ nội độ 2 3 búi trĩ ở vị trí 3 giờ 5 giờ 11 giờ nên xử lí như nào?", "en": "How to manage internal hemorrhoids stage 2-3 with 3-5-11 o'clock piles?", "zh": "II、III度内痔，3点、5点、11点位置的痔核应如何处理？"}, {"id": 1146, "vi": "Bé nhà mình được 7 tháng, mấy hôm vừa rồi mình có bị cúm và bé đã bị lây cúm từ mình. Bé có hơi sốt khoảng 37.1 độ, nhiều nước mũi, mệt mỏi, nhưng bú và ăn vẫn bình thường. Cho mình hỏi có phương pháp tự nhiên nào để chữa khỏi cho bé không ạ, tại bé còn nhỏ nên mình chưa muốn cho bé sử dụng kháng sinh.", "en": "Our child is 7 months old. A few days ago, I got the flu and the baby caught it from me. The baby has a slight fever around 37.1 degrees, runny nose, fatigue, but still feeds and eats normally. I would like to ask if there are any natural methods to treat the baby, as the baby is so young and I haven't wanted to give the baby antibiotics.", "zh": "我家宝宝7个月大，前几天我得了流感，宝宝也从我这里感染了。宝宝有点发烧，约37.1度，流清鼻涕，感到疲倦，但吃奶和进食都正常。想咨询一下，有没有天然的治疗方法可以给宝宝治疗，因为宝宝还小，我还不想给宝宝用抗生素。"}, {"id": 1147, "vi": "Sưng đỏ đuôi mắt tự phát do nguyên nhân gì?", "en": "What is the cause of spontaneous red and swollen eyelid?", "zh": "眼尾自发性红肿的原因是什么？"}, {"id": 1148, "vi": "Uống nhiều nước lạnh là nguyên nhân mọc hạch cổ?", "en": "Does drinking a lot of cold water cause neck lymph node enlargement?", "zh": "经常喝冷水会引发颈部淋巴结肿大吗？"}, {"id": 1149, "vi": "Em bị u nang buồng trứng, đã mổ nội soi và lạc nội mạc tử cung không bóc tách được. Vợ chồng em để tự nhiên một năm rồi nhưng chưa có thai. Trường hợp của em nên làm IUI hay IVF? Trước khi làm có phải can thiệp gì không?", "en": "I have ovarian cysts, have undergone laparoscopic surgery and could not completely resect endometriosis. My couple has been trying naturally for one year without conceiving. Should my case undergo IUI or IVF? Are there any interventions required before proceeding?", "zh": "我患有卵巢囊肿，已做腹腔镜手术，但子宫内膜异位症未能完全切除。夫妻自然同居一年未孕。我的情况应该选择宫腔内人工授精（IUI）还是体外受精（IVF）？在进行治疗前是否需要进行其他干预？"}, {"id": 1150, "vi": "Phòng ngừa bệnh mạch vành như thế nào?", "en": "How to prevent coronary artery disease?", "zh": "如何预防冠心病？"}, {"id": 1151, "vi": "Tôi năm nay 66 tuổi, đang điều trị bệnh Basedow hàng ngày bằng cách uống thuốc, có bệnh dị ứng thời tiết hắt hơi, sổ mũi. Xin hỏi tôi có nên tiêm chủng vaccine COVID-19 không? Nếu tiêm thì loại thuốc nào thích hợp?", "en": "I am 66 years old and am taking medication daily for Graves' disease, and I have a weather-related allergy with sneezing and runny nose. I would like to ask whether I should receive the COVID-19 vaccine? If so, which medication is appropriate?", "zh": "我今年66岁，目前正在每日服用药物治疗格雷夫斯病，同时有季节性过敏症状，如打喷嚏、流鼻涕。我想咨询一下，我是否应该接种新冠疫苗？如果接种，应该选择哪种疫苗？"}, {"id": 1152, "vi": "Sau sinh 1 năm, vết tiêm gây tê còn đau nhức có nguy hiểm không?", "en": "One year after delivery, is it dangerous if the injection site for anesthesia still causes pain and discomfort?", "zh": "产后一年，注射麻醉的疤痕仍有疼痛和不适，是否有危险？"}, {"id": 1153, "vi": "Bị mất hơi, tiết nhiều nước bọt khi cố đẩy hơi để thổi sáo, do đâu?", "en": "Loss of breath, excessive salivation when trying to blow air into a flute, what is the cause?", "zh": "用力吹笛时出现气息消失、唾液分泌增多，是由于什么原因？"}, {"id": 1154, "vi": "Em bầu 20 tuần mà bị ra khí hư nhiều loãng như nước mà không có mùi. Vậy có ảnh hưởng gì không", "en": "I am 20 weeks pregnant and have a lot of loose vaginal discharge that looks like water and has no odor. Is this a problem?", "zh": "我怀孕20周，白带增多、稀薄如水，没有异味。这种情况会有影响吗？"}, {"id": 1155, "vi": "Trẻ bại não nên điều trị thế nào?", "en": "How should a child with cerebral palsy be treated?", "zh": "脑性瘫痪儿童应该如何治疗？"}, {"id": 1156, "vi": "Hôi miệng lâu năm là do đâu và cách điều trị như thế nào?", "en": "What is the cause of long-term bad breath and how is it treated?", "zh": "长期口臭的原因是什么，该如何治疗？"}, {"id": 1157, "vi": "Nam giới có khả năng bị cường giáp không? Gần đây tôi thấy bị tim đập nhanh không rõ nguyên nhân, hồi hộp.", "en": "Can men get hyperthyroidism? Recently I have been experiencing a rapid heartbeat without a clear cause, palpitations.", "zh": "男性会有甲状腺功能亢进的可能吗？最近我感觉心跳加快，且原因不明。"}, {"id": 1158, "vi": "Em muốn hỏi bệnh hoang tưởng bị hại nhẹ. Có phương pháp trị liệu tâm lý hay lời khuyên nào cho bệnh này không ạ?", "en": "I would like to ask about mild delusional disorder. Is there any psychological treatment or advice available for this condition?", "zh": "我想问一下被害性妄想障碍是否有心理治疗方法或建议吗？"}, {"id": 1159, "vi": "Em hay thường bị đi tiểu nhiều và khó đi tiểu và cứ cảm giác có hòn sỏi chăn đường nước tiếu ở dưới háng không biết em bị mắc bệnh gì và có ảnh hưởng gì không?", "en": "I often have frequent urination and difficulty urinating, and I always feel like there is a stone blocking the urinary tract below the hip. I don't know what disease I have and whether it has any impact.", "zh": "我经常出现尿频、排尿困难，还总感觉在髋部下方有结石堵塞尿路，我不知道我得了什么病，会不会有影响？"}, {"id": 1160, "vi": "Tôi muốn hỏi sau khi tiêm vắc xin bao lâu thì mới qua ngưỡng an toàn?", "en": "I would like to ask, how long after vaccination do I need to wait before I pass the safety threshold?", "zh": "我想问一下接种疫苗后多久才能达到安全阈值？"}, {"id": 1161, "vi": "Có thể tiêm phòng thủy đậu cho phụ nữ đang mang thai không?", "en": "Can varicella vaccination be given to pregnant women?", "zh": "孕妇可以接种水痘疫苗吗？"}, {"id": 1162, "vi": "Em 31 tuổi: cho em hỏi em thường bị sốt cao vào giữa trưa và chiều mà không biết nguyên nhân. Bác sĩ tư vấn dùm em được không ạ?", "en": "I am 31 years old: I would like to ask, I often have high fever in the midday and afternoon, but I don't know the cause. Can the doctor advise me?", "zh": "我31岁：请问医生，我经常在中午和下午出现高烧，但不知道原因。医生能给我一点建议吗？"}, {"id": 1163, "vi": "Bệnh tiểu đường không chữa trị thì có sao không?", "en": "What happens if diabetes is not treated?", "zh": "如果不治疗糖尿病会怎样？"}, {"id": 1164, "vi": "Em bị sảy thai thì bao lâu có thể có con tiếp ạ? Và cần kiêng và bổ sung gì ạ?", "en": "How long after a spontaneous abortion can I try to conceive again? And what should I avoid and supplement?", "zh": "宝宝流产之后多久可以再次怀孕呢？需要忌口和补充什么吗？"}, {"id": 1165, "vi": "Phát hiện bị viêm amidan khi đang điều trị bệnh lao thì phải làm sao?", "en": "What should I do if adenoiditis is detected while being treated for tuberculosis?", "zh": "在治疗肺结核期间发现患有扁桃体炎该怎么办？"}, {"id": 1166, "vi": "Sốt li bì không rõ nguyên nhân khi điều trị lao phổi có sao không?", "en": "Is it dangerous to have undetermined fever when treating pulmonary tuberculosis?", "zh": "肺结核治疗期间不明原因的弛张热会有问题吗？"}, {"id": 1167, "vi": "Sau khi bị chó dại cắn, làm thế nào để biết mình có bị bệnh dại hay không?", "en": "After being bitten by a rabid dog, how can I know if I have rabies or not?", "zh": "被疯狗咬伤后，如何知道是否感染了狂犬病？"}, {"id": 1168, "vi": "Rụng tóc nhiều bất thường nguyên nhân là gì?", "en": "What is the cause of abnormal excessive hair loss?", "zh": "什么原因导致异常脱发？"}, {"id": 1169, "vi": "Trước chuyển phôi 3 tháng em có làm xét nghiệm Anti đều âm tính. Vậy bây giờ em vừa chuyển phôi xong, đến nay đã có Beta hcg. Mong bác sĩ tư vấn cho em có cần phải làm lại xét nghiệm Anti lần nữa không.", "en": "Three months before embryo transfer, I had performed the Anti test, which came back negative. Now, I have just completed embryo transfer, and I have detected beta hCG. I hope the doctor can advise whether I need to repeat the Anti test again.", "zh": "在移植前3个月我做过抗磷脂抗体检测，结果为阴性。现在我刚刚完成移植，至今已检测出β-hCG。请问医生，我现在是否需要再次进行抗磷脂抗体检测？"}, {"id": 1170, "vi": "Tôi được chuẩn đoán hen phế quản 2 tháng nay. Trước đây tôi rất thích chạy bộ và đi bơi lội, nhưng dạo gần đây chạy bộ được gần 1 tiếng là tôi thấy khó thở. Vậy tôi có nên tiếp tục chạy bộ hàng ngày không? Tôi còn có thể tập môn thể thao nào khác không?", "en": "I have been diagnosed with asthma for the past two months. Previously, I used to enjoy running and swimming, but recently, after running for nearly one hour, I start to feel short of breath. Should I continue running every day? Are there any other sports I can still do?", "zh": "我最近两个月被诊断为支气管哮喘。以前我非常喜欢跑步和游泳，但最近跑大约一小时就会感到呼吸困难。我是否应该继续每天跑步？我还可不可以尝试其他运动项目？"}, {"id": 1171, "vi": "Thời gian làm xét nghiệm nhuộm hóa mô miễn dịch để chẩn đoán xác định CK7 bao lâu?", "en": "How long does it take to perform the immunohistochemical staining test to definitively diagnose CK7?", "zh": "确定CK7免疫组化染色的组织病理学检测需要多长时间？"}, {"id": 1172, "vi": "Tôi bị tiểu đường 10 năm nay, hiện đang dùng thuốc tiêm và uống, tôi không có bệnh gì khác. Tôi xin hỏi tôi có được tiêm vắc xin Covid-19 không?", "en": "I have had diabetes for 10 years, and currently I am taking both injected and oral medications. I do not have any other diseases. I would like to ask whether I can receive the COVID-19 vaccine?", "zh": "我患有糖尿病已有10年，目前在使用注射和口服药物，没有其他疾病。我想咨询一下，我可以接种新冠疫苗吗？"}, {"id": 1173, "vi": "Trẻ 17 tháng tuổi mọc răng được 8 chiếc liệu có bình thường?", "en": "Is it normal for a 17-month-old child to have 8 teeth erupted?", "zh": "17个月大的孩子长出8颗牙是否正常？"}, {"id": 1174, "vi": "Sáng con ngủ dậy có cảm thấy hốc mắt của con đau nhức, còn mắt thì không đau, không biết con bị gì và có thể làm gì tại nhà để giảm đau không", "en": "In the morning, the child feels pain in the eye socket, but not in the eye. I don't know what the child has and what can be done at home to relieve the pain.", "zh": "孩子早上醒来时感觉眼窝疼痛，眼睛本身没有疼痛，不知道孩子得了什么病，家里可以采取什么方法来缓解疼痛？"}, {"id": 1175, "vi": "Em từng bị bệnh giang mai, vì điều trị không đến nơi đến chốn nên dẫn tới biến chứng tắc ống dẫn tinh bên trái. Nay em kết hôn đã lâu mà chưa có con, đi khám thì một bên ống dẫn tinh bị tắc, một bên bình thường nhưng bác sĩ kết luận khả năng có con không cao. Vậy có biện pháp nào giúp em tăng cơ hội có con không, liệu em có thực hiện thụ tinh nhân tạo được không?", "en": "I used to have syphilis; due to inadequate treatment, I developed a complication of left vasectomy blockage. Now, after a long time of marriage, I have not been able to conceive. Upon examination, one vas deferens is blocked while the other is normal, but the doctor concluded that my chances of conceiving are low. Are there any measures that can help increase my chances of conceiving? Can I undergo in vitro fertilization?", "zh": "我曾患过梅毒，由于治疗不彻底，导致左侧输精管堵塞。现在我结婚多年但一直未生育，检查发现一侧输精管堵塞，另一侧正常，但医生认为生育几率较低。请问是否有办法可以提高我的生育机会？我是否可以进行体外受精？"}, {"id": 1176, "vi": "Bác sĩ cho em hỏi, nếu mẹ uống thuốc say xe Dimenhydrinate 50mg thì có thể cho em bé 3 tháng bú mẹ được không? Nếu không được thì liệu có loại thuốc chống say xe nào phù hợp cho người đang cho con bú không ạ?", "en": "Doctor, I would like to ask, if mother takes motion sickness medication Dimenhydrinate 50mg, can she still breastfeed her 3-month-old baby? If not, is there any motion sickness medication that is suitable for someone who is breastfeeding?", "zh": "医生请问，如果妈妈服用晕车药dimenhydrinate 50mg，是否可以哺乳三个月的婴儿？如果不可以，那么有没有适合哺乳期妈妈的抗晕车药物呢？"}, {"id": 1177, "vi": "Đau háng chân trái khi ép ngang ép dọc và chạy nước rút", "en": "Left hip pain when lateral and axial compression and sprinting", "zh": "左髋部在横向和纵向压迫及冲刺跑时疼痛"}, {"id": 1178, "vi": "Có kết quả xét nghiệm HBsAb 13 UI/mL phải tiêm bao nhiêu mũi vắc-xin viêm gan B?", "en": "What is the number of hepatitis B vaccine doses to be administered if the HBsAb test result is 13 UI/mL?", "zh": "乙肝表面抗体检测结果为13 UI/mL，需要接种几剂乙肝疫苗？"}, {"id": 1179, "vi": "Rối loạn cảm xúc lưỡng cực có điều trị dứt điểm được không?", "en": "Can bipolar affective disorder be cured?", "zh": "双相情感障碍能否根治？"}, {"id": 1180, "vi": "Con tôi ê buốt răng hàm do mở tuỷ có nên đi nhổ không?", "en": "My child has molars pain due to root canal treatment - should I have them extracted?", "zh": "我孩子因开髓导致牙痛，是否需要拔牙？"}, {"id": 1181, "vi": "Nguyên nhân gây đau hốc mắt thường xuyên.", "en": "Causes of frequent pain in the orbital region.", "zh": "引起眼窝持续疼痛的常见原因。"}, {"id": 1182, "vi": "Thiếu máu điều trị như thế nào?", "en": "How is anemia treated?", "zh": "贫血如何治疗？"}, {"id": 1183, "vi": "Bé nhà mình 14 tuổi, 2 hôm nay xuất hiện nổi 2 nốt mụn có mủ trắng ở hai bên vòm miệng hàm trên. Xin tư vấn từ bác sĩ là thuốc gì có thể điều trị và có nguy hiểm không?", "en": "Our child is 14 years old and has appeared two white pus-filled bumps on both sides of the upper palate over the past two days. I would like to consult the doctor regarding which medication can treat this and whether it is dangerous?", "zh": "我家孩子14岁，这两天在上颌两侧口腔顶出现了两个化脓的白色脓疱。请医生咨询一下，用什么药可以治疗，有没有危险？"}, {"id": 1184, "vi": "Em hay bị nghẹn ở cổ và lúc nào cũng như có gì đè lên cổ khiến em thấy rất khó thở ạ .Không biết em bị làm sao ạ", "en": "I often experience choking in the neck and always feel as if something is pressing on the neck, making me feel very short of breath. I don't know what's wrong with me.", "zh": "我经常感到喉咙堵塞，总觉得有什么东西压在脖子上，导致我呼吸非常困难。不知道我是什么情况。"}, {"id": 1185, "vi": "Rụng tóc sau khi làm tóc có sao không?", "en": "Is it harmful to lose hair after hairdressing?", "zh": "做头发后会掉头发吗？"}, {"id": 1186, "vi": "Nên làm gì khi bị bệnh tổ đỉa hơn 1 tuần?", "en": "What should I do when having fungal infection longer than 1 week?", "zh": "患疱疹超过一周该怎么办？"}, {"id": 1187, "vi": "Hậu môn của em bình thường hay chảy dịch có mùi hôi khó chịu, gây rát thành hậu môn. Tình trạng này có giảm khi em rửa hậu môn sau mỗi lần vệ sinh thay vì chỉ chùi bằng giấy. Mấy hôm nay sau khi rửa, em thấm khô lại bằng giấy thì thấy vết dịch màu tím, lượng dịch tím ít.", "en": "My anus normally discharges a smelly, unpleasant fluid, causing irritation of the anal skin. This condition improves when I wash my anus after each bowel movement instead of just wiping with paper. Recently, after washing, when I dry it with paper, I have noticed a purple-colored discharge, with a small amount of purple fluid.", "zh": "我妹妹肛门平时会流出有难闻气味的分泌物，导致肛门瘙痒。这种情况在每次如厕后用清水清洗肛门，而不是只用纸巾擦拭时会有所改善。最近几次在清洗后，用纸巾擦拭时发现有紫色分泌物，量较少。"}, {"id": 1188, "vi": "Trong miệng bé có nhiều đốm trắng xong bà nội của bé láy mật ong về rơ lưỡi cho bé..Giờ bé buổi tối khó ngủ và bú như kiểu miệng bé bị đau và mỗi lần bú là khóc và khó bú nữa..Bé có bị làm sao không có cần đi khám không.Tình trạng của bé biểu hiện đã 2 ngày rồi.", "en": "There are many white spots in the baby's mouth, and the baby's grandmother mixed honey and applied it to the baby's tongue. Now, the baby has difficulty sleeping and breastfeeding, as if the baby's mouth is painful, and each time feeding, the baby cries and becomes harder to feed. Is the baby okay without needing to see a doctor? The baby's condition has been present for two days.", "zh": "孩子嘴里有很多白色斑点，奶奶用蜂蜜涂在孩子的舌头上。现在孩子晚上睡觉困难，吃奶时像嘴巴疼一样，每次吃奶都哭闹，而且吃奶更困难了。孩子会不会有问题，需要去看医生吗？这种情况已经持续两天了。"}, {"id": 1189, "vi": "Tôi năm nay 28 tuổi. Ngón tay áp út của tôi sáng ngủ dậy bổng dưng đau ở khớp chổ lòng bàn tay. Co lại thì vẫn bình thường. Nhưng duỗi thẳng thì bị đau nên không thể duỗi thẳng được. Tôi không làm gì bị va đập hay chấn thương cả. Thế nên nhờ bác sĩ tư vấn giúp đấy là triệu chứng của bệnh gì ?", "en": "I am 28 years old. My thumb joint at the base of the hand suddenly hurts when I wake up in the morning. It is normal when I bend it, but when I straighten it, I experience pain and cannot fully straighten it. I have not done anything that would cause injury or trauma. Therefore, I am seeking advice from the doctor—what disease could this symptom indicate?", "zh": "我今年28岁。我早上醒来时，手指掌侧的关节突然出现疼痛，手指弯曲时正常，但伸直时会疼痛，因此无法伸直。我没有受到任何撞击或外伤。所以想咨询一下医生，这是什么疾病的症状？"}, {"id": 1190, "vi": "Tôi muốn xét nghiệm CA 19-9 để biết theo dõi xem mình có bị bệnh này không. Khi đọc về CA 19-9 thấy có ghi chú người có kháng nguyên nhóm máu lewis a-b sẽ không tổng hợp được CA 19-9. Ghi chú này có liên quan gì đến nhóm máu không (tôi có nhóm máu AB+).", "en": "I would like to undergo CA 19-9 testing to monitor whether I have this condition. When reading about CA 19-9, I noticed a note stating that individuals with Lewis a-b blood group antigens will not be able to synthesize CA 19-9. Is this note related to blood group (I have blood group AB+)?", "zh": "我想检查CA 19-9以了解自己是否患有此病。阅读关于CA 19-9的内容时，发现有注释指出，具有Lewis A-B血型抗原的人不会合成CA 19-9。这条注释与血型有什么关系（我的血型是AB+）？"}, {"id": 1191, "vi": "Hiện nay tình hình dịch bệnh như thế nào? Chúng ta có thể nghĩ rằng đã đạt đỉnh trong đợt bùng phát dịch Covid-19 lần thứ 4 này chưa và nếu chưa thì diễn biến sẽ đi đến đâu ạ?", "en": "What is the current situation of the disease? Can we consider that we have reached the peak during the fourth wave of the COVID-19 outbreak, and if not, where will the course go?", "zh": "目前疫情形势如何？我们是否可以认为本轮新冠第四波疫情已经到达顶峰？如果还没有到达顶峰，后续发展会如何？"}, {"id": 1192, "vi": "Kết quả khám tiểu đường 300 mg/dL có nguy hiểm không?", "en": "Is a diabetes examination result of 300 mg/dL dangerous?", "zh": "空腹血糖检测结果为300 mg/dL是否危险？"}, {"id": 1193, "vi": "Tại sao bị sâu răng không được tẩy trắng răng?", "en": "Why do cavities not qualify for teeth whitening?", "zh": "为什么蛀牙患者不能进行牙齿美白？"}, {"id": 1194, "vi": "Mất thị lực do biến chứng đái tháo đường có khả năng cứu chữa không?", "en": "Can diabetic complications causing vision loss be cured?", "zh": "2型糖尿病并发症导致的视力丧失能否治愈？"}, {"id": 1195, "vi": "Khi nào uống thuốc tránh thai hàng ngày viên đầu tiên?", "en": "When to take the first daily oral contraceptive pill?", "zh": "什么时候服用每日一片口服避孕药的第一片？"}, {"id": 1196, "vi": "Lớp 9 chưa dậy thì có vấn đề gì không?", "en": "Does a 9th-grade student not experiencing puberty have any problems?", "zh": "九年级还没发育正常有问题吗？"}, {"id": 1197, "vi": "Cho tôi hỏi dạo này tôi bị tình trạng xuất tinh sớm, tình trạng này đã kéo dài được gần 1 năm. Lúc trước tôi quan hệ được tầm 2 đến 3 phút nhưng thời gian gần đây chỉ được dưới 1 phút. Xin được tư vấn.", "en": "May I ask, recently I have been experiencing premature ejaculation, a condition that has lasted for nearly 1 year. Previously, I could last about 2 to 3 minutes during intercourse, but recently I can only last less than 1 minute. I would like to seek advice.", "zh": "请问，最近我出现早泄症状，这种情况已经持续将近一年了。以前性生活持续约2到3分钟，但最近仅持续不到1分钟。想咨询一下。"}, {"id": 1198, "vi": "Cháu bị côn trùng đốt ở chân được 9 ngày rồi. Ngày mai bác sĩ bảo khâu lại cho đẹp. Bên trong hố sâu đã hình thành các tổ chức hạt đầy dần lên nhưng vẫn chảy dịch phải thấm băng. Đang điều trị Zinnat 500 mg, ngày 2 viên. Quanh miệng vết thương hơi cứng. Như vậy có bình thường không, liệu khâu có ổn không vì cháu thấy điều trị lâu khỏi quá?", "en": "I was bitten by an insect on my foot 9 days ago. Tomorrow the doctor will suture it to make it look better. Inside the deep wound, granulation tissue has gradually formed but still drains fluid and needs to be absorbed with gauze. I am currently being treated with Zinnat 500 mg, two tablets per day. There is slight hardness around the edges of the wound. Is this normal? Will the suturing be stable since I feel the healing process is taking too long?", "zh": "我脚上被昆虫叮咬已经9天了。医生说明天要缝合以美观。伤口深处已经形成了颗粒状组织，但仍然有渗液，需要用纱布吸收。目前正在服用锌酸（Zinnat）500毫克，每天2粒。伤口周围稍有硬结。这种情况正常吗？缝合后会不会稳定？因为我感觉治疗时间太长了，恢复得比较慢。"}, {"id": 1199, "vi": "Viêm lợi trùm có điều trị dứt điểm được không?", "en": "Can gingivitis be cured completely?", "zh": "根尖周炎能根治吗？"}, {"id": 1200, "vi": "Phát hiện bệnh lý nguy hiểm cần phải làm gì?", "en": "What should be done when a serious disease is detected?", "zh": "发现危险疾病需要做什么？"}]}'''

In [ ]:
if FAILED is None:
    with stage('variants'):
        # One row per text to encode: the full query (always), its sub-queries (USE_QUERY_SPLIT) and its EN / ZH
        # translations (USE_TRANSLATION).
        TRANS_TEXT = [{} for _ in QUERY_TEXT]   # qi -> {'en': text, 'zh': text}
        if USE_TRANSLATION:
            assert TRANSLATIONS_JSON, 'USE_TRANSLATION: rebuild the notebook with --translations query_translations.json'
            by_id = {int(item['id']): item for item in json.loads(TRANSLATIONS_JSON)['items']}
            assert all(qid in by_id for qid in QUERY_IDS), 'translations missing for some query ids'
            assert all(by_id[qid]['vi'] == text for qid, text in zip(QUERY_IDS, QUERY_TEXT)), 'translations of other query texts'
            TRANS_TEXT = [{lang: by_id[qid][lang] for lang in ('en', 'zh') if by_id[qid].get(lang)} for qid in QUERY_IDS]
        VARIANTS = []
        for qi, text in enumerate(QUERY_TEXT):
            VARIANTS.append(dict(qi=qi, kind='full', text=text))
            if USE_QUERY_SPLIT:
                for sub in sub_queries(text, SPLIT_MIN_WORDS, MIN_SUB_WORDS, MAX_SUBQ):
                    VARIANTS.append(dict(qi=qi, kind='sub', text=sub))
            for lang, translated in TRANS_TEXT[qi].items():
                VARIANTS.append(dict(qi=qi, kind='trans', lang=lang, text=translated))
        TASK = INSTRUCT_CUSTOM if USE_CUSTOM_INSTRUCT else INSTRUCT_DEFAULT
        (OUT / 'query_variants.json').write_text(json.dumps(dict(task=TASK, variants=VARIANTS), ensure_ascii=False) + '\n',
                                                 encoding='utf-8')
        n_sub = sum(v['kind'] == 'sub' for v in VARIANTS)
        print(f'{len(VARIANTS)} texts to encode: {len(QUERY_TEXT)} full, {n_sub} sub-queries '
              f'({len({v["qi"] for v in VARIANTS if v["kind"] == "sub"})} queries split)')

## 3. Encode queries with the active models (USE_CUSTOM_INSTRUCT picks the instruction)

In [ ]:
if FAILED is None:
    with stage('encode'):
        started = time.perf_counter()
        QUERY_VECTORS = {}
        for model_key in ACTIVE_MODEL_WEIGHTS:
            spec = EMBEDDING_SPECS[model_key]
            metadata = dict(model=spec['model'], model_key=model_key, task=TASK, variants=VARIANTS,
                            query_ids=QUERY_IDS, model_revision=MODEL_REVISIONS[model_key],
                            query_format='Instruct: {task}\nQuery:{text}', max_length=1024)
            if ENCODE_HERE:
                from transformers import AutoModel, AutoTokenizer
                import torch.nn.functional as F
                os.environ['HF_HUB_OFFLINE'] = '1'
                tok = AutoTokenizer.from_pretrained(str(MODEL_PATHS[model_key]), local_files_only=True)
                tok.padding_side = 'left'
                model = AutoModel.from_pretrained(str(MODEL_PATHS[model_key]), torch_dtype=torch.bfloat16,
                                                 attn_implementation='sdpa', local_files_only=True).cuda().eval()
                assert model.config.hidden_size == spec['dim']
                note(f'{model_key}: query encoder loaded')

                @torch.inference_mode()
                def encode(texts, batch=16):
                    tokenized = tok(texts, truncation=True, max_length=1024)['input_ids']
                    order = np.argsort([-len(row) for row in tokenized], kind='stable')
                    vectors = np.empty((len(texts), spec['dim']), dtype=np.float32)
                    for start in range(0, len(order), batch):
                        index = order[start:start + batch]
                        enc = tok.pad({'input_ids': [tokenized[i] for i in index]}, return_tensors='pt').to('cuda')
                        hidden = model(input_ids=enc['input_ids'], attention_mask=enc['attention_mask'], use_cache=False).last_hidden_state
                        mask = enc['attention_mask']
                        if mask[:, -1].sum() == mask.shape[0]:
                            pooled = hidden[:, -1]
                        else:
                            lengths = mask.sum(dim=1) - 1
                            pooled = hidden[torch.arange(hidden.shape[0], device=hidden.device), lengths]
                        vectors[index] = F.normalize(pooled.float(), p=2, dim=1).cpu().numpy()
                    return vectors

                vectors = encode([f'Instruct: {TASK}\nQuery:{v["text"]}' for v in VARIANTS])
                del model, tok
                torch.cuda.empty_cache()
            else:
                cache_hits = sorted(glob.glob(f'/kaggle/input/**/query_variants_{model_key}.json', recursive=True))
                legacy = False
                if not cache_hits and model_key == 'qwen4b':
                    cache_path = one_match('/kaggle/input/**/query_variants.json', 'legacy 4B query variants')
                    vector_path = cache_path.with_name('query_variant_vectors.npy')
                    legacy = True
                else:
                    assert len(cache_hits) == 1, f'{model_key}: attach exactly one matching query variant cache, found {cache_hits}'
                    cache_path = Path(cache_hits[0])
                    vector_path = cache_path.with_name(f'query_variant_vectors_{model_key}.npy')
                saved = json.loads(cache_path.read_text(encoding='utf-8'))
                vectors = np.load(vector_path, allow_pickle=False)
                validate_query_cache(saved, vectors, model_key, VARIANTS, TASK, QUERY_IDS, legacy=legacy,
                                     revision=MODEL_REVISIONS[model_key])
            validate_query_cache(metadata, vectors, model_key, VARIANTS, TASK, QUERY_IDS,
                                 revision=MODEL_REVISIONS[model_key])
            QUERY_VECTORS[model_key] = vectors
            np.save(OUT / f'query_variant_vectors_{model_key}.npy', vectors)
            (OUT / f'query_variants_{model_key}.json').write_text(json.dumps(metadata, ensure_ascii=False), encoding='utf-8')
            if list(ACTIVE_MODEL_WEIGHTS) == ['qwen4b']:
                np.save(OUT / 'query_variant_vectors.npy', vectors)  # legacy output name
            note(f'{model_key}: {len(vectors):,} query variants ready, dim={spec["dim"]}')
        TIMINGS['encode_s'] = round(time.perf_counter() - started, 1)

## 4. Configure sequential vector loading (one model at a time)

In [ ]:
if FAILED is None:
    with stage('load_vectors'):
        # Load/search/release one embedding model at a time, so fusion does not require all three matrices resident.
        READ_ROWS = 32768
        ALL_IDS = None
        TIMINGS['load_vectors_s'] = 0.0


        def load_vector_parts(model_key):
            global ALL_IDS
            started = time.perf_counter()
            paths = MODEL_VECTOR_PATHS[model_key]
            dim = EMBEDDING_SPECS[model_key]['dim']
            parts, ids = [], []
            read_ids = ALL_IDS is None
            for key in sorted(paths):
                t0 = time.perf_counter()
                array = np.load(paths[key], mmap_mode='r', allow_pickle=False)
                assert array.dtype == np.float16 and array.shape[1] == dim
                gpu = torch.empty(array.shape, dtype=torch.float16, device='cuda')
                for start in range(0, array.shape[0], READ_ROWS):
                    gpu[start:start + READ_ROWS] = torch.from_numpy(np.array(array[start:start + READ_ROWS])).cuda()
                del array
                if read_ids:
                    ids_path = Path(paths[key]).with_name(f'{key}-ids.parquet')
                    ids.append(pq.read_table(ids_path, columns=['chunk_id'])['chunk_id'].to_numpy())
                parts.append(gpu)
                note(f'{model_key}/{key}: {gpu.shape[0]:,} rows on GPU in {time.perf_counter() - t0:.0f}s')
            offsets = np.cumsum([0] + [p.shape[0] for p in parts])
            if read_ids:
                ALL_IDS = np.concatenate(ids)
            assert len(ALL_IDS) == offsets[-1]
            TIMINGS['load_vectors_s'] += round(time.perf_counter() - started, 1)
            return parts, offsets

## 5. Exact dense search: full query, sub-queries, PRF (USE_PRF)

In [ ]:
if FAILED is None:
    with stage('search'):
        torch.backends.cuda.matmul.allow_tf32 = SEARCH_TF32
        ROW_BLOCK, QUERY_BATCH = 262144, 1024


        @torch.inference_mode()
        def search(query_vectors, k):
            k = min(k, len(ALL_IDS))
            assert k > 0
            q = torch.from_numpy(np.ascontiguousarray(query_vectors, dtype=np.float32)).cuda()
            best_s = torch.full((len(q), k), -float('inf'), device='cuda')
            best_i = torch.full((len(q), k), -1, dtype=torch.int64, device='cuda')
            for part, offset in zip(PARTS, OFFSETS[:-1]):
                for row in range(0, part.shape[0], ROW_BLOCK):
                    block = part[row:row + ROW_BLOCK].float()
                    for qs in range(0, len(q), QUERY_BATCH):
                        scores = q[qs:qs + QUERY_BATCH] @ block.T
                        top_s, top_i = torch.topk(scores, min(k, scores.shape[1]), dim=1)
                        cat_s = torch.cat([best_s[qs:qs + QUERY_BATCH], top_s], dim=1)
                        cat_i = torch.cat([best_i[qs:qs + QUERY_BATCH], top_i + int(offset) + row], dim=1)
                        keep_s, order = torch.topk(cat_s, k, dim=1)
                        best_s[qs:qs + QUERY_BATCH] = keep_s
                        best_i[qs:qs + QUERY_BATCH] = torch.gather(cat_i, 1, order)
                    del block
            result_s, result_i = best_s.cpu().numpy(), best_i.cpu().numpy()
            assert (result_i >= 0).all() and np.isfinite(result_s).all()
            return result_s, result_i


        def gather_vectors(rows):
            flat = np.asarray(rows).ravel()
            part_of = np.searchsorted(OFFSETS, flat, side='right') - 1
            out = torch.empty((len(flat), DIM), dtype=torch.float32, device='cuda')
            for p in np.unique(part_of):
                mask = np.flatnonzero(part_of == p)
                local = torch.from_numpy(flat[mask] - OFFSETS[p]).cuda()
                out[torch.from_numpy(mask).cuda()] = PARTS[p].index_select(0, local).float()
            return out.view(*np.shape(rows), DIM)


        started = time.perf_counter()
        kinds = np.array([v['kind'] for v in VARIANTS])
        owner = np.array([v['qi'] for v in VARIANTS])
        FULL = np.flatnonzero(kinds == 'full')
        assert (owner[FULL] == np.arange(len(QUERY_TEXT))).all()
        MODEL_RESULTS = {}
        for model_key in ACTIVE_MODEL_WEIGHTS:
            t0 = time.perf_counter()
            DIM = EMBEDDING_SPECS[model_key]['dim']
            PARTS, OFFSETS = load_vector_parts(model_key)
            variant_vectors = QUERY_VECTORS[model_key]
            d_scores, d_rows = search(variant_vectors, DENSE_K)
            subs = [[] for _ in QUERY_TEXT]
            for i in np.flatnonzero(kinds == 'sub'):
                subs[owner[i]].append(ALL_IDS[d_rows[i, :SUB_K]])
            trans = [[] for _ in QUERY_TEXT]   # EN / ZH translations of the query (USE_TRANSLATION)
            for i in np.flatnonzero(kinds == 'trans'):
                trans[owner[i]].append(ALL_IDS[d_rows[i, :TRANS_K]])
            prf_ids = None
            if USE_PRF:
                with torch.inference_mode():
                    feedback = gather_vectors(d_rows[FULL, :PRF_K]).mean(dim=1)
                    q = torch.from_numpy(variant_vectors[FULL]).cuda()
                    prf = torch.nn.functional.normalize((1 - PRF_ALPHA) * q + PRF_ALPHA * feedback, p=2, dim=1).cpu().numpy()
                _, prf_rows = search(prf, PRF_KEEP)
                prf_ids = ALL_IDS[prf_rows]
                del feedback, q, prf, prf_rows
            MODEL_RESULTS[model_key] = dict(full_ids=ALL_IDS[d_rows[FULL]], full_scores=d_scores[FULL],
                                           sub_lists=subs, trans_lists=trans, prf_ids=prf_ids)
            del PARTS, variant_vectors, d_scores, d_rows
            torch.cuda.empty_cache()
            note(f'{model_key}: full/sub/PRF retrieval complete in {time.perf_counter() - t0:.0f}s; GPU vectors released')

        if len(ACTIVE_MODEL_WEIGHTS) == 1:
            result = MODEL_RESULTS[next(iter(ACTIVE_MODEL_WEIGHTS))]
            DENSE_FULL_IDS, DENSE_FULL_SCORES = result['full_ids'], result['full_scores']
        else:
            # Reference dense ranking for overlap/export; final hybrid fusion adds every model's lists once.
            dense_fused = [rrf([(list(MODEL_RESULTS[key]['full_ids'][qi]), weight)
                                for key, weight in ACTIVE_MODEL_WEIGHTS.items()], RRF_K)[:DENSE_K]
                           for qi in range(len(QUERY_TEXT))]
            DENSE_FULL_IDS = np.array([[cid for cid, _ in row] for row in dense_fused], dtype=object)
            DENSE_FULL_SCORES = np.array([[score for _, score in row] for row in dense_fused], dtype=np.float32)
        TIMINGS['search_s'] = round(time.perf_counter() - started - TIMINGS['load_vectors_s'], 1)

## 6. BM25 candidates (USE_BM25)

In [ ]:
if FAILED is None:
    with stage('bm25'):
        BM25_IDS = [[] for _ in QUERY_TEXT]
        if BM25_ACTIVE:
            with np.load(BM25_PATH, allow_pickle=True) as z:  # each z[key] access re-reads the array: read once
                bm25_qids, bm25_scores, bm25_ids = z['query_ids'], z['scores'], z['chunk_ids']
            row_of = {int(q): i for i, q in enumerate(bm25_qids)}
            for qi, qid in enumerate(QUERY_IDS):
                r = row_of[qid]
                keep = bm25_scores[r, :BM25_K] > 0  # no match = score 0
                BM25_IDS[qi] = bm25_ids[r, :BM25_K][keep]
            known = set(ALL_IDS.tolist())
            unknown = sum(cid not in known for ids in BM25_IDS for cid in ids)
            assert unknown == 0, f'{unknown} BM25 chunk ids are not in the vector parts (different chunk corpus?)'
            print('BM25 candidates per query: median', int(np.median([len(x) for x in BM25_IDS])))

## 7. Weighted RRF pool

In [ ]:
if FAILED is None:
    with stage('fuse'):
        # One weighted RRF stage across all model full/sub/PRF lists; BM25 appears once.
        ONLY_DENSE = len(ACTIVE_MODEL_WEIGHTS) == 1 and not (USE_QUERY_SPLIT or USE_PRF or BM25_ACTIVE)
        FUSED, POOLS = [], []
        for qi in range(len(QUERY_TEXT)):
            lists = model_candidate_lists(MODEL_RESULTS, ACTIVE_MODEL_WEIGHTS, qi, W_SUB, W_PRF,
                                          W_TRANS if USE_TRANSLATION else 0.0)
            if len(BM25_IDS[qi]):
                lists.append((list(BM25_IDS[qi]), W_BM25))
            fused = rrf(lists, RRF_K)
            FUSED.append(fused)
            POOLS.append(fused[:POOL_MAX] if USE_RERANK else fused)
        pool_sizes = [len(p) for p in POOLS]
        print('fused candidates per query: median', int(np.median([len(f) for f in FUSED])),
              '| pool: median', int(np.median(pool_sizes)), 'total', sum(pool_sizes))

## 8. Chunk text and document groups

In [ ]:
if FAILED is None:
    with stage('records'):
        # Chunk records for every fused candidate (documents); text only for the chunks that need it.
        started = time.perf_counter()
        need_text = set()
        for qi, pool in enumerate(POOLS):
            for cid, _ in (pool if USE_RERANK else pool[:N_CHUNKS]):
                need_text.add(cid)
            need_text.update(DENSE_FULL_IDS[qi, :N_CHUNKS].tolist())  # for the overlap check against plain dense
        wanted = {cid for fused in FUSED for cid, _ in fused}
        wanted_arr, text_arr = pa.array(sorted(wanted)), pa.array(sorted(need_text))
        RECORDS, LANG_OF = {}, {}
        for path in CHUNK_FILES:
            lang = Path(path).parent.name   # chunks/<lang>/part-*.parquet
            table = pq.read_table(path, columns=['chunk_id', 'text_sha256', 'rep_doc_id', 'order', 'chunk_text'])
            table = table.filter(pc.is_in(table['chunk_id'], value_set=wanted_arr))
            with_text = pc.is_in(table['chunk_id'], value_set=text_arr).to_pylist()
            for cid, sha, doc, order, text, keep in zip(table['chunk_id'].to_pylist(), table['text_sha256'].to_pylist(),
                                                        table['rep_doc_id'].to_pylist(), table['order'].to_pylist(),
                                                        table['chunk_text'].to_pylist(), with_text):
                RECORDS[cid] = (sha, int(doc), int(order), text if keep else None)
                LANG_OF[cid] = lang
            del table
        missing = wanted - set(RECORDS)
        assert not missing, f'{len(missing)} candidate chunks have no row in the chunk corpus, e.g. {sorted(missing)[:3]}'

        groups = pd.read_parquet(HASH_PATH, columns=['text_sha256', 'doc_ids', 'fanout_safe'])
        GROUP = {sha: [int(d) for d in doc_ids] for sha, doc_ids, safe in
                 zip(groups['text_sha256'], groups['doc_ids'], groups['fanout_safe']) if bool(safe)}
        del groups


        def docs_of(cid):
            sha, doc, _, _ = RECORDS[cid]
            return GROUP.get(sha) or [doc]


        TIMINGS['records_s'] = round(time.perf_counter() - started, 1)
        print(f'{len(RECORDS):,} candidate chunks, {len(need_text):,} with text, {len(GROUP):,} fan-out groups')

## 9. Rerank the pool (USE_RERANK)

In [ ]:
if FAILED is None:
    with stage('rerank'):
        # Final chunk scores. Reranker: P(relevant) of (full query, chunk). Without it: fused RRF score, or the
        # cosine when the full-query dense list is the only list (then DOC_AGG='max' is the old ranking exactly).
        started = time.perf_counter()
        FINAL = []
        if USE_RERANK:
            from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer

            pairs = [(QUERY_TEXT[qi], RECORDS[cid][3]) for qi, pool in enumerate(POOLS) for cid, _ in pool]
            n_base, same_lang_of = len(pairs), []   # extra pair -> index of its (VI query, chunk) pair
            if USE_TRANSLATION and RERANK_SAME_LANG:
                position = 0
                for qi, pool in enumerate(POOLS):
                    for cid, _ in pool:
                        translated = TRANS_TEXT[qi].get(LANG_OF[cid])
                        if translated:
                            pairs.append((translated, RECORDS[cid][3]))
                            same_lang_of.append(position)
                        position += 1
            TIMINGS['same_lang_pairs'] = len(same_lang_of)
            scores = np.empty(len(pairs), dtype=np.float32)
            order = np.argsort([len(d) for _, d in pairs], kind='stable')  # similar lengths per batch: less padding
            if RERANKER.startswith('Qwen3-Reranker'):
                tok = AutoTokenizer.from_pretrained(str(RERANK_PATH), padding_side='left')
                model = AutoModelForCausalLM.from_pretrained(str(RERANK_PATH), torch_dtype=torch.bfloat16).cuda().eval()
                yes, no = tok.convert_tokens_to_ids('yes'), tok.convert_tokens_to_ids('no')
                instruct = 'Given a Vietnamese medical question, judge whether the document contains the answer to the question'
                batch = 16

                def score_batch(index):
                    enc = qwen_reranker_inputs(tok, [pairs[i][0] for i in index], [pairs[i][1] for i in index],
                                               instruct, 1024).to('cuda')
                    logits = model(**enc).logits[:, -1, :]
                    two = torch.stack([logits[:, no], logits[:, yes]], dim=1).float()
                    return torch.softmax(two, dim=1)[:, 1]
            else:
                tok = AutoTokenizer.from_pretrained(str(RERANK_PATH))
                model = AutoModelForSequenceClassification.from_pretrained(str(RERANK_PATH), torch_dtype=torch.bfloat16).cuda().eval()
                batch = 32

                def score_batch(index):
                    enc = tok([pairs[i][0] for i in index], [pairs[i][1] for i in index], padding=True, truncation=True,
                              max_length=1024, return_tensors='pt').to('cuda')
                    return torch.sigmoid(model(**enc).logits.view(-1).float())

            note(f'{RERANKER} loaded; {len(pairs):,} pairs')
            t0, next_report = time.perf_counter(), 0.05
            with torch.inference_mode():
                for start in range(0, len(order), batch):
                    index = order[start:start + batch]
                    scores[index] = score_batch(index).cpu().numpy()
                    done = (start + len(index)) / len(order)
                    if done >= next_report:
                        rate = (start + len(index)) / (time.perf_counter() - t0)
                        note(f'rerank {done:.0%} | {rate:.0f} pairs/s | ETA {(len(order) - start) / rate / 60:.0f} min')
                        next_report += 0.05
            del model, tok
            torch.cuda.empty_cache()
            TIMINGS['rerank_pairs_per_s'] = round(len(pairs) / (time.perf_counter() - t0), 1)
            if same_lang_of:   # EN/ZH chunk: the better of the VI query and the query in the chunk's language
                TIMINGS['same_lang_raised'] = int((scores[n_base:] > scores[same_lang_of]).sum())
                np.maximum.at(scores, np.array(same_lang_of), scores[n_base:])
            scores = scores[:n_base]
            position = 0
            for pool in POOLS:
                part = scores[position:position + len(pool)]
                position += len(pool)
                ranked = sorted(zip([cid for cid, _ in pool], part.tolist()), key=lambda item: -item[1])
                FINAL.append(ranked)
        elif ONLY_DENSE:
            FINAL = [list(zip(DENSE_FULL_IDS[qi].tolist(), DENSE_FULL_SCORES[qi].tolist())) for qi in range(len(QUERY_TEXT))]
        else:
            FINAL = [list(pool) for pool in POOLS]
        TIMINGS['rerank_s'] = round(time.perf_counter() - started, 1)

## 10. Documents (DOC_AGG), chunks, ZIP under the size limit, manifest

In [ ]:
if FAILED is None:
    with stage('submit'):
        started = time.perf_counter()
        corpus_ids = set(map(int, pd.read_parquet(CORPUS_PATH, columns=['id'])['id']))


        def build(n_chunks):
            submission = []
            for qi, qid in enumerate(QUERY_IDS):
                scored_ids = {cid for cid, _ in FINAL[qi]}
                fill = [cid for cid, _ in FUSED[qi] if cid not in scored_ids]
                docs, chunks = assemble(FINAL[qi], fill, docs_of, N_DOCS, n_chunks, DOC_AGG)
                submission.append(dict(id=qid, relevant_docs=docs, relevant_chunks=[
                    dict(doc_id=doc, chunk_text=RECORDS[cid][3], chunk_order=RECORDS[cid][2]) for doc, cid in chunks]))
            for entry in submission:
                listed = set(entry['relevant_docs'])
                assert entry['relevant_chunks'] and listed <= corpus_ids and len(listed) == len(entry['relevant_docs'])
                assert all(c['doc_id'] in listed and c['chunk_text'] and c['chunk_text'].strip() for c in entry['relevant_chunks'])
            return submission


        def write(submission):
            json_path, zip_path = OUT / 'medxir-hybrid-submission.json', OUT / 'medxir-hybrid-submission.zip'
            json_path.write_text(json.dumps(submission, ensure_ascii=False, separators=(',', ':')), encoding='utf-8')
            with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=9) as archive:
                archive.write(json_path, arcname=json_path.name)
            with zipfile.ZipFile(zip_path) as archive:
                assert archive.namelist() == [json_path.name]
            json_path.unlink()
            return zip_path


        # How far the new lists moved from plain dense (same sets = same score; PROGRESS rule before submitting).
        dense_docs = [doc_ranking(list(zip(DENSE_FULL_IDS[qi].tolist(), DENSE_FULL_SCORES[qi].tolist())), docs_of, 'max')[:N_DOCS]
                      for qi in range(len(QUERY_TEXT))]
        n_chunks = N_CHUNKS
        while True:
            submission = build(n_chunks)
            if QUERY_LIMIT:
                print('QUERY_LIMIT set: submission not written')
                zip_path = None
                break
            zip_path = write(submission)
            zip_mb = zip_path.stat().st_size / 1e6
            note(f'{n_chunks} chunks/query -> ZIP {zip_mb:.1f} MB')
            if zip_mb <= MAX_ZIP_MB:
                break
            n_chunks -= 10

        chunk_overlap = np.mean([len({cid for cid, _ in FINAL[qi][:n_chunks]} & set(DENSE_FULL_IDS[qi, :n_chunks].tolist())) / n_chunks
                                 for qi in range(len(QUERY_TEXT))])
        doc_overlap = np.mean([len(set(e['relevant_docs'][:N_DOCS]) & set(dense_docs[qi])) / max(len(dense_docs[qi]), 1)
                               for qi, e in enumerate(submission)])

        pool_rows = []
        for qi, qid in enumerate(QUERY_IDS):
            dense_rank = {cid: r for r, cid in enumerate(DENSE_FULL_IDS[qi].tolist())}
            bm25_rank = {cid: r for r, cid in enumerate(BM25_IDS[qi])}
            fused = dict(FUSED[qi])
            for rank, (cid, score) in enumerate(FINAL[qi]):
                pool_rows.append((qid, cid, rank, float(score), fused.get(cid, 0.0), dense_rank.get(cid, -1), bm25_rank.get(cid, -1)))
        pd.DataFrame(pool_rows, columns=['query_id', 'chunk_id', 'final_rank', 'final_score', 'rrf', 'dense_rank', 'bm25_rank']) \
          .to_parquet(OUT / 'pool.parquet', index=False, compression='zstd')
        TIMINGS['submit_s'] = round(time.perf_counter() - started, 1)

        manifest = dict(
            config=CONFIG, task_instruction=TASK, gpu=GPU_NAME, queries=len(QUERY_IDS),
            inputs=INPUT_PROVENANCE, bm25=BM25_PROVENANCE,
            sub_queries=int(sum(v['kind'] == 'sub' for v in VARIANTS)), chunks_per_query=n_chunks, docs_per_query=N_DOCS,
            zip=str(zip_path) if zip_path else None, zip_mb=round(zip_path.stat().st_size / 1e6, 1) if zip_path else None,
            pool_median=int(np.median([len(p) for p in POOLS])),
            chunk_overlap_with_dense=round(float(chunk_overlap), 4), doc_overlap_with_dense=round(float(doc_overlap), 4),
            timings=TIMINGS, elapsed_s=round(time.perf_counter() - T0, 1))
        (OUT / 'hybrid_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
        print(json.dumps(manifest, ensure_ascii=False, indent=2))

## 11. Extra ZIPs: neighbour spans (EXPAND_NB) and bge document rerank (DOC_RERANK)

In [ ]:
# Copied from scripts/expand_chunks.py by scripts/build_hybrid_notebook.py. Edit the script and rebuild.
# Run in its own namespace EXPAND: its VARIANTS and one_match would shadow this notebook's.
import types

EXPAND_SOURCE = r'''"""Rebuild the chunk lists of the hybrid submission with neighbouring chunks merged into longer spans.

A reference passage counts as found when a submitted text of the same document covers >= 40% of it (token
LCS). Chunk recall (0.18) is far below document recall (0.47): the right documents are found, the passages
are not. Here the top-ranked chunks bring their neighbours (order - 1, order + 1) along, and contiguous
chunks of one text are merged into one span (the 50-word overlaps are dropped once). The byte budget per
query stays that of the hybrid submission, so lower-ranked chunks make room for the neighbours.

The document lists are kept as submitted; documents of new spans are appended (as in assemble()).
Optional per variant: n_docs extends the document list with the next pool documents (best chunk first);
quota reserves that share of the chunk bytes for ZH/EN texts; doc_quota swaps the lowest VI documents of the
list for the best ZH/EN pool documents (both lost on the leaderboard, 29/10); doc_rr re-orders the
candidate documents by weighted RRF of their submitted rank and document-reranker scores (doc_rank.parquet,
written by the doc-rerank notebook, kaggle_doc_rerank.ipynb), so the reranker decides which make the cut.
conf_gamma moves chunk bytes between queries, total unchanged: a query's confidence is its number of pool
chunks with rerank score >= conf_t; confident queries lose budget (their lowest-ranked chunks go) and the
budget goes to queries with few strong chunks, which reach deeper into their pool.
Runs on a Kaggle CPU session (Internet OFF is fine). Inputs:
  medxir-chunk-corpus      chunks/*/part-*.parquet, hash_groups.parquet
  medxir-hybrid-retrieval  pool.parquet, medxir-hybrid-submission.zip   (notebook output or uploaded dataset)
  doc_rank.parquet         only for doc_rr variants (in OUT_DIR or an input)
Output: one ZIP per variant in /kaggle/working (or OUT_DIR), plus expand_manifest.json.
"""
import glob
import json
import math
import os
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

# nb: number of top chunks whose neighbours are added (0 = only merge, no new neighbours);
#     'whole' = probe: full texts of the top documents instead of ranked chunks.
# n_docs: document list length (0 = as submitted). quota / doc_quota: min ZH/EN share of chunk bytes / documents.
# doc_rr: {score column of doc_rank.parquet: RRF weight}; w_base: RRF weight of the submitted document rank;
# keep_top: that many top candidates kept as submitted, the rest of the list re-ordered.
# conf_gamma: budget weight (confidence / median confidence) ** -gamma, clipped to CONF_CLIP, renormalised.
# min_chars: spans shorter than this are not submitted. A span matches when it covers 40% of a reference
# passage, so a 5-character ZH entry ('食管炎') can only match an equally short passage; such spans were 11% of
# the entries of the 0.2464 ZIP (< 200 chars) for 0.35% of its text.
# min_score: ranked chunks with a rerank score below it are not taken (nor their neighbours); the bytes a query
# leaves unused go to the queries that used their whole budget, by the same confidence weights. bge scores are
# high: at 0.2 the cut removes ~46 chunks in 16% of the queries (the unsure ones) and none elsewhere.
# zip: 'bzip2' writes the ZIP with bzip2 instead of deflate: the 0.2477 JSON is 86.5 MB deflated, 56.5 MB with
# bzip2 (LZMA 54.4 MB, 10x slower), so ~1.5x more text fits under the 100 MB limit. budget_scale replaces
# BUDGET_SCALE; the size loop shrinks it until the ZIP fits. Under the limit chunk recall grows with the bytes
# (each gold document holds ~1 reference passage, so more bytes = more documents reached).
# cover (first, last): after the nb top chunks (with neighbours), the best pool chunk of every listed document
# at ranks first..last that has none yet, then the rest of the pool. 30/10 (Qwen-8B, 0.2509): ranks 100-200 of
# the document list got chunk text for 10% of the documents, ranks 50-100 for 48%, while ~15% of them are
# relevant and a relevant document holds ~1 reference passage.
# win: every entry is a window of the chunk +- win neighbours of the same text (the entry shape of the 0.2755
# ensemble ZIP: no tiny spans, no 5+ chunk merges); a centre inside a chosen window is skipped. Centres in the
# order of w_bge * bge + (1 - w_bge) * rrf, both min-max per query (0.2755 used bge 0.25 over its fused list).
# cover_top: first the best window of each of the top cover_top listed documents, then one window per text in
# score order, then second windows. A relevant document holds ~1 reference passage, so a second window of one
# text rarely adds recall, while ~23% of the top-20 documents of the 0.2755 ZIP had no window.
# doc_conf (t, lo, hi): per query, the documents with doc-reranker score >= t, at least lo and at most hi. A
# document pays off in F2 when P(relevant) > ~F2 / 5 (~0.065 at Doc F2 0.32); bge scores scaled to the 0.161 Doc P
# of the 0.2539 ZIP (mean bge of its top 200 = 0.514) put that at t = 0.208 (~189 documents on average).
# bm25w (needs bm25_words.py outputs attached): doc_rr column 'bm25w' = the word-BM25 rank of the best
# BM25W_TEXTS texts (their documents), fused by RRF with bge; w_bm25w adds the word-BM25 chunk score (best
# BM25W_CHUNKS chunks, also those outside the pool) to the window order. 30/10: 48% of the word-BM25 top 40
# documents are in none of our 206-document lists.
VARIANTS = {
    'nb0': dict(nb=0), 'nb10': dict(nb=10), 'nb30': dict(nb=30), 'nb60': dict(nb=60), 'wholedoc': dict(nb='whole'),
    'nb60-d300': dict(nb=60, n_docs=300),
    'nb60-q20': dict(nb=60, quota=0.2),
    'nb60-q20-dq20': dict(nb=60, quota=0.2, doc_quota=0.2),
    'nb60-drr-bge': dict(nb=60, doc_rr={'bge-reranker-v2-m3': 1.0}, w_base=1.0),
    'nb60-drr-bge-only': dict(nb=60, doc_rr={'bge-reranker-v2-m3': 1.0}, w_base=0.0),
    # Qwen3-Reranker-4B scores only the zone around the cut (ranks 120-360): the top 120 stay as submitted.
    'nb60-drr-qwen4b': dict(nb=60, doc_rr={'Qwen3-Reranker-4B': 1.0}, w_base=1.0, keep_top=120),
    'nb60-drr-qwen4b-only': dict(nb=60, doc_rr={'Qwen3-Reranker-4B': 1.0}, w_base=0.0, keep_top=120),
    'nb60-drr-both': dict(nb=60, doc_rr={'bge-reranker-v2-m3': 1.0, 'Qwen3-Reranker-4B': 1.0}, w_base=1.0,
                          keep_top=120),
    # Bge-only document list, chunk bytes moved between queries. 30/10 on v2 (bge-only 0.2464): glow (to unsure
    # queries) 0.2352, ghigh (to confident queries) 0.2477; macro F2 rewards the queries that already score.
    'nb60-drr-bge-only-glow': dict(nb=60, doc_rr={'bge-reranker-v2-m3': 1.0}, w_base=0.0, conf_gamma=0.5, conf_t=0.5),
    'nb60-drr-bge-only-ghigh': dict(nb=60, doc_rr={'bge-reranker-v2-m3': 1.0}, w_base=0.0, conf_gamma=-0.5, conf_t=0.5),
    'nb60-drr-bge-only-ghigh1': dict(nb=60, doc_rr={'bge-reranker-v2-m3': 1.0}, w_base=0.0, conf_gamma=-1.0,
                                     conf_t=0.5),
    # Precision: no tiny spans; 'nball' gives every ranked chunk its neighbours (no single chunks filling the
    # budget), the entry shape of the x1-c70 submission (Chunk P 0.208 vs 0.155).
    'nb60-drr-bge-only-ghigh1-min200': dict(nb=60, doc_rr={'bge-reranker-v2-m3': 1.0}, w_base=0.0, conf_gamma=-1.0,
                                            conf_t=0.5, min_chars=200),
    'nb60-drr-bge-only-ghigh-min200-s20': dict(nb=60, doc_rr={'bge-reranker-v2-m3': 1.0}, w_base=0.0,
                                               conf_gamma=-0.5, conf_t=0.5, min_chars=200, min_score=0.2),
    'nb60-drr-bge-only-ghigh1-min200-s20': dict(nb=60, doc_rr={'bge-reranker-v2-m3': 1.0}, w_base=0.0,
                                                conf_gamma=-1.0, conf_t=0.5, min_chars=200, min_score=0.2),
    'nb60-drr-bge-only-ghigh-bz': dict(nb=60, doc_rr={'bge-reranker-v2-m3': 1.0}, w_base=0.0, conf_gamma=-0.5,
                                       conf_t=0.5, zip='bzip2', budget_scale=1.6),
    'nb60-drr-bge-only-ghigh-cover': dict(nb=60, doc_rr={'bge-reranker-v2-m3': 1.0}, w_base=0.0, conf_gamma=-0.5,
                                          conf_t=0.5, cover=(20, 200)),
    'nb30-drr-bge-only-ghigh-cover': dict(nb=30, doc_rr={'bge-reranker-v2-m3': 1.0}, w_base=0.0, conf_gamma=-0.5,
                                          conf_t=0.5, cover=(20, 200)),
    'nb60-drr-bge-only-ghigh-min200': dict(nb=60, doc_rr={'bge-reranker-v2-m3': 1.0}, w_base=0.0, conf_gamma=-0.5,
                                           conf_t=0.5, min_chars=200),
    'nball-drr-bge-only-ghigh-min200': dict(nb=10 ** 6, doc_rr={'bge-reranker-v2-m3': 1.0}, w_base=0.0,
                                            conf_gamma=-0.5, conf_t=0.5, min_chars=200),
    'win1-cov50-drr-bge-only': dict(nb=0, win=1, w_bge=0.25, cover_top=50, doc_rr={'bge-reranker-v2-m3': 1.0},
                                    w_base=0.0, budget_scale=1.15),
    'win1-cov50-drr-bge-conf': dict(nb=0, win=1, w_bge=0.25, cover_top=50, doc_rr={'bge-reranker-v2-m3': 1.0},
                                    w_base=0.0, budget_scale=1.15, doc_conf=(0.208, 80, 300)),
    # Ensemble pool of the 0.2916 ZIP (POOL_FILE=pool-bgefull.parquet: final_score = bge on all 500 pool chunks,
    # rrf = ensemble score), windows in its order (ensemble 0.75, bge 0.25), BZIP2 with the budget loop filling
    # ~99 MB. cov0: one window per text before second windows; cov50: the top 50 listed documents first.
    'ensbge-win1-cov50-bz': dict(nb=0, win=1, w_bge=0.25, cover_top=50, doc_rr={'bge-reranker-v2-m3': 1.0},
                                 w_base=0.0, zip='bzip2', budget_scale=2.0),
    'ensbge-win1-cov0-bz': dict(nb=0, win=1, w_bge=0.25, cover_top=0, doc_rr={'bge-reranker-v2-m3': 1.0},
                                w_base=0.0, zip='bzip2', budget_scale=2.0),
    'win1-cov50-bm25w-drr': dict(nb=0, win=1, w_bge=0.25, w_bm25w=0.25, cover_top=50,
                                 doc_rr={'bge-reranker-v2-m3': 1.0, 'bm25w': 1.0}, w_base=0.0, budget_scale=1.15),
}
ONLY = ['nb60-d300']   # variants to build this run; [] = all
BUDGET_SCALE = 1.08   # 87.5 MB hybrid ZIP * 1.08 stays below the 100 MB limit
MAX_ZIP_MB = 99.0
CONF_CLIP = (0.5, 2.0)
BM25W_CHUNKS, BM25W_TEXTS = 200, 100   # word-BM25 candidates read per query
RRF_K = 60
INPUT = os.environ.get('INPUT_ROOT', '/kaggle/input')
OUT = Path(os.environ.get('OUT_DIR', '/kaggle/working'))


def one_match(pattern):
    hits = sorted(p for p in glob.glob(f'{INPUT}/**/{pattern}', recursive=True))
    if len(hits) != 1:
        for root in sorted(glob.glob(f'{INPUT}/*/*/*')) or sorted(glob.glob(f'{INPUT}/*')):
            print('input:', root, sorted(os.listdir(root))[:8] if os.path.isdir(root) else '')
    assert len(hits) == 1, f'{pattern}: expected one match, found {hits[:10]}'
    return Path(hits[0])


def load_submission():
    # An uploaded dataset is unzipped by Kaggle: accept the JSON too.
    hits = sorted(glob.glob(f'{INPUT}/**/medxir-hybrid-submission.json', recursive=True))
    if hits:
        return json.loads(Path(hits[0]).read_text(encoding='utf-8'))
    with zipfile.ZipFile(one_match('medxir-hybrid-submission.zip')) as archive:
        return json.loads(archive.read(archive.namelist()[0]))


def merge_run(rows):
    """One text out of chunks of one document text with consecutive orders (rows sorted by order)."""
    text, end = rows[0]['text'], rows[0]['end']
    for row in rows[1:]:
        text += row['text'][max(0, end - row['start']):]
        end = row['end']
    return text


def build_query(ranked, n_neighbour, budget, rows, by_pos, docs_of, quota=0.0, scores=None, min_score=0.0):
    """Selected chunk ids for one query: ranked chunks (+ neighbours of the top ones) until the budget is used.

    quota > 0: a first pass caps VI chunks at (1 - quota) of the budget so ZH/EN chunks further down the pool
    get the rest; a second pass fills what ZH/EN could not use.
    """
    chosen, used, vi_used = [], 0, 0
    for vi_cap in ([budget * (1 - quota)] if quota else []) + [budget]:
        for rank, cid in enumerate(ranked):
            if min_score and scores[rank] < min_score:
                break   # pool is in score order
            group = [cid]
            if rank < n_neighbour:
                sha, order = rows[cid]['sha'], rows[cid]['order']
                group += [by_pos[(sha, o)] for o in (order - 1, order + 1) if (sha, o) in by_pos]
            for c in group:
                if c in chosen:
                    continue
                cost = len(rows[c]['text'].encode('utf-8')) * len(docs_of(c))
                vi = rows[c].get('lang', 'vi') == 'vi'
                if used + cost > budget or (vi and vi_used + cost > vi_cap):
                    continue  # a long or wide-fanout chunk does not fit; smaller ones further down may
                chosen.append(c)
                used += cost
                vi_used += cost if vi else 0
    return chosen


def doc_list(docs, ranked, docs_of, lang_of, n_docs=0, doc_quota=0.0):
    """Submitted documents with the ZH/EN quota applied, then extended to n_docs with pool documents."""
    docs = list(docs)
    pool_docs = list(dict.fromkeys(d for cid in ranked for d in docs_of(cid)))
    if doc_quota:
        listed = set(docs)
        extra = [d for d in pool_docs if d not in listed and lang_of.get(d, 'vi') != 'vi']
        need = math.ceil(doc_quota * len(docs)) - sum(lang_of.get(d, 'vi') != 'vi' for d in docs)
        vi_slots = [i for i, d in enumerate(docs) if lang_of.get(d, 'vi') == 'vi']
        slots = vi_slots[len(vi_slots) - min(max(need, 0), len(extra), len(vi_slots)):]
        for i, doc in zip(slots, extra):   # lowest VI slots, best ZH/EN document first
            docs[i] = doc
    if n_docs > len(docs):
        listed = set(docs)
        docs += [d for d in pool_docs if d not in listed][:n_docs - len(docs)]
    return docs


def best_chunks(ranked, docs_of):
    """{doc: its pool chunks, best first}. Documents filled from beyond the reranked pool have none."""
    best = {}
    for cid in ranked:
        for doc in docs_of(cid):
            best.setdefault(doc, []).append(cid)
    return best


def doc_text(best, rows, by_text, lead_chars=400, n_chunks=1):
    """Text a document reranker sees: the start of the document text (title / lead) + its best pool chunks.

    best: the document's pool chunks, best first; the n_chunks best are shown in text order.
    """
    shown = sorted(best[:n_chunks], key=lambda c: rows[c]['order'])
    parts = [rows[c]['text'] for c in shown]
    lead = rows[by_text[rows[shown[0]]['sha']][0]]
    if lead['order'] != rows[shown[0]]['order']:
        parts.insert(0, lead['text'][:lead_chars])
    return '\n...\n'.join(parts)


def fuse_docs(candidates, scores, weights, w_base=1.0, k=RRF_K, offset=0):
    """Candidates (submitted order) re-ordered by weighted RRF of their position and reranker scores.

    scores: {column: {doc: score}}; a document without a score in a column gets no term from it.
    offset: submitted rank of the first candidate (when the documents above it are kept fixed).
    """
    fused = {doc: w_base / (k + offset + rank) for rank, doc in enumerate(candidates)}
    for column, weight in weights.items():
        scored = sorted((d for d in candidates if d in scores[column]), key=lambda d: -scores[column][d])
        for rank, doc in enumerate(scored):
            fused[doc] += weight / (k + rank)
    return sorted(candidates, key=lambda d: -fused[d])


def spans(chosen, rows):
    """Merge chosen chunks of the same text with consecutive orders. Returns [(first chunk id, text)] in rank order."""
    by_sha = {}
    for cid in chosen:
        by_sha.setdefault(rows[cid]['sha'], []).append(cid)
    first_rank = {cid: i for i, cid in enumerate(chosen)}
    out = []
    for cids in by_sha.values():
        cids = sorted(cids, key=lambda c: rows[c]['order'])
        run = [cids[0]]
        for c in cids[1:]:
            if rows[c]['order'] == rows[run[-1]]['order'] + 1:
                run.append(c)
            else:
                out.append(run)
                run = [c]
        out.append(run)
    out.sort(key=lambda run: min(first_rank[c] for c in run))
    return [(run[0], merge_run([rows[c] for c in run])) for run in out]


def build_whole_docs(ranked, docs, budget, rows, by_text, docs_of):
    """Probe: every chunk of the submitted documents, best document first, until the budget is used.

    Tells whether chunk recall is lost inside the right documents (recall jumps) or by missing documents.
    """
    text_of = {}
    for cid in ranked:
        for doc in docs_of(cid):
            text_of.setdefault(doc, rows[cid]['sha'])
    chosen, used = [], 0
    for doc in docs:
        if doc not in text_of:
            continue
        for c in by_text[text_of[doc]]:
            cost = len(rows[c]['text'].encode('utf-8')) * len(docs_of(c))
            if used + cost > budget:
                return chosen
            if c not in chosen:
                chosen.append(c)
                used += cost
    return chosen


def load_inputs(pool_path=None, base=None, chunk_files=None):
    """Pool ranking, base submission, every chunk of the candidate texts and the document groups.

    The hybrid notebook passes its own pool.parquet, in-memory submission and validated chunk parts;
    otherwise they are found under INPUT.
    """
    pool_path = pool_path or one_match(os.environ.get('POOL_FILE', 'pool.parquet'))
    columns = ['query_id', 'chunk_id', 'final_rank', 'final_score']
    pool = pd.read_parquet(pool_path, columns=columns + (['rrf'] if 'rrf' in pq.read_schema(pool_path).names else []))
    pool = pool.sort_values(['query_id', 'final_rank'])
    ranked = {qid: g['chunk_id'].tolist() for qid, g in pool.groupby('query_id', sort=False)}
    score = {qid: g['final_score'].tolist() for qid, g in pool.groupby('query_id', sort=False)}
    rrf = {qid: g['rrf'].tolist() for qid, g in pool.groupby('query_id', sort=False)} if 'rrf' in pool else None
    base = load_submission() if base is None else base
    needed_sha_prefix = {cid.rsplit('-', 1)[0] for ids in ranked.values() for cid in ids}
    bm25w = load_bm25w()
    if bm25w:
        needed_sha_prefix |= {cid.rsplit('-', 1)[0] for ids, _ in bm25w['chunks'].values() for cid in ids}
        needed_sha_prefix |= {sha[:16] for shas, _ in bm25w['texts'].values() for sha in shas}

    # Every chunk of the candidate texts: the neighbours are rows of the same text_sha256.
    rows, by_pos = {}, {}
    chunk_files = chunk_files or sorted(glob.glob(f'{INPUT}/**/chunks/*/part-*.parquet', recursive=True))
    assert chunk_files, 'no chunk parts: attach medxir-chunk-corpus'
    columns = ['chunk_id', 'text_sha256', 'rep_doc_id', 'order', 'start', 'end', 'chunk_text']
    for path in chunk_files:
        table = pq.read_table(path, columns=columns)
        prefix = pc.utf8_slice_codeunits(table['text_sha256'], 0, 16)
        table = table.filter(pc.is_in(prefix, value_set=pa.array(sorted(needed_sha_prefix))))
        lang = Path(path).parent.name   # chunks/<lang>/part-*.parquet
        for cid, sha, doc, order, start, end, text in zip(*(table[c].to_pylist() for c in columns)):
            rows[cid] = dict(sha=sha, doc=int(doc), order=int(order), start=int(start), end=int(end), text=text,
                             lang=lang)
            by_pos[(sha, int(order))] = cid
        print(Path(path).parent.name, Path(path).name, f'{len(rows):,} rows kept', flush=True)
    missing = {cid for ids in ranked.values() for cid in ids} - set(rows)
    assert not missing, f'{len(missing)} pool chunks not in the chunk corpus (chunk id != sha prefix?), e.g. {sorted(missing)[:3]}'

    groups = pd.read_parquet(one_match('hash_groups.parquet'), columns=['text_sha256', 'doc_ids', 'fanout_safe'])
    group = {sha: [int(d) for d in ids] for sha, ids, safe in
             zip(groups['text_sha256'], groups['doc_ids'], groups['fanout_safe']) if bool(safe)}
    del groups

    def docs_of(cid):
        return group.get(rows[cid]['sha']) or [rows[cid]['doc']]

    by_text = {}
    for cid in sorted(rows, key=lambda c: rows[c]['order']):
        by_text.setdefault(rows[cid]['sha'], []).append(cid)

    def text_docs(sha):
        return group.get(sha) or ([rows[by_text[sha][0]]['doc']] if sha in by_text else [])

    if bm25w:   # word-BM25 chunks whose rows were not found (other corpus parts) are dropped
        for q, (ids, scores) in bm25w['chunks'].items():
            keep = [(c, s) for c, s in zip(ids, scores) if c in rows]
            bm25w['chunks'][q] = ([c for c, _ in keep], [s for _, s in keep])
    return dict(ranked=ranked, score=score, rrf=rrf, bm25w=bm25w, text_docs=text_docs,
                base=base, rows=rows, by_pos=by_pos, by_text=by_text, docs_of=docs_of,
                lang_of={doc: row['lang'] for cid, row in rows.items() for doc in docs_of(cid)})


def load_bm25w(n_chunks=None, n_texts=None):
    """Word-BM25 candidates of bm25_words.py, {'chunks': {qid: (ids, scores)}, 'texts': {qid: (shas, scores)}}, or
    None when bm25w-chunks-k500.npz is not attached. Only positive scores, the BM25W_CHUNKS / BM25W_TEXTS best."""
    if not glob.glob(f'{INPUT}/**/bm25w-chunks-k500.npz', recursive=True):
        return None
    out = {}
    for key, name, column, n in (('chunks', 'bm25w-chunks-k500.npz', 'chunk_ids', n_chunks or BM25W_CHUNKS),
                                 ('texts', 'bm25w-texts-k500.npz', 'text_sha256', n_texts or BM25W_TEXTS)):
        with np.load(one_match(name), allow_pickle=True) as z:
            out[key] = {int(q): ([str(x) for x, s in zip(ids[:n], sc[:n]) if s > 0], [float(s) for s in sc[:n] if s > 0])
                        for q, ids, sc in zip(z['query_ids'], z[column], z['scores'])}
    return out


def load_doc_scores():
    """{query_id: {column: {doc: score}}} from doc_rank.parquet (OUT_DIR first, then the inputs)."""
    path = OUT / 'doc_rank.parquet'
    frame = pd.read_parquet(path if path.exists() else one_match('doc_rank.parquet'))
    columns = [c for c in frame.columns if c not in ('query_id', 'doc_id', 'base_rank')]
    out = {}
    for qid, g in frame.groupby('query_id'):
        out[int(qid)] = {c: {int(d): float(s) for d, s in zip(g['doc_id'], g[c]) if s == s} for c in columns}
    return out


def confidence_budget(budget, score, gamma, threshold):
    """Per-query budgets re-weighted by (confidence / median) ** -gamma, the total kept."""
    conf = {qid: sum(s >= threshold for s in score.get(qid, [])) + 1 for qid in budget}
    median = sorted(conf.values())[len(conf) // 2]
    weight = {qid: min(max((c / median) ** -gamma, CONF_CLIP[0]), CONF_CLIP[1]) for qid, c in conf.items()}
    norm = sum(budget.values()) / sum(budget[q] * weight[q] for q in budget)
    return {qid: budget[qid] * weight[qid] * norm for qid in budget}


def cover_order(ranked, n_head, docs, first, last, docs_of):
    """Pool order with the best chunk of each listed document at ranks first..last moved right after the top n_head
    chunks, unless a top chunk already belongs to it."""
    head = ranked[:n_head]
    seen = {d for c in head for d in docs_of(c)}
    best = {}
    for cid in ranked[n_head:]:
        for doc in docs_of(cid):
            best.setdefault(doc, cid)
    promoted = list(dict.fromkeys(best[d] for d in docs[first:last] if d in best and d not in seen))
    moved = set(promoted)
    return head + promoted + [c for c in ranked[n_head:] if c not in moved]


def blended_order(ranked, columns, weights):
    """Chunks sorted by sum(weight * column), each column min-max scaled within the query."""
    def scaled(values):
        lo, hi = min(values), max(values)
        return [(v - lo) / (hi - lo) if hi > lo else 0.0 for v in values]
    blend = [sum(w * v for w, v in zip(weights, row)) for row in zip(*(scaled(c) for c in columns))]
    return [ranked[i] for i in sorted(range(len(ranked)), key=lambda i: -blend[i])]


def with_bm25w(ranked, bge, rrf, bm25w):
    """Pool chunks + the word-BM25 chunks outside the pool, with (bge, rrf, bm25w) columns; absent scores are the
    column minimum (0 for bge and BM25, the pool minimum for rrf)."""
    ids, scores = bm25w
    by_id = dict(zip(ids, scores))
    extra = [c for c in ids if c not in set(ranked)]
    out = list(ranked) + extra
    rrf_floor = min(rrf) if rrf else 0.0
    return (out, list(bge) + [0.0] * len(extra), list(rrf) + [rrf_floor] * len(extra),
            [by_id.get(c, 0.0) for c in out])


def window_ids(cid, context, rows, by_pos):
    """Chunk ids of the window order - context .. order + context of the chunk's text, text order."""
    sha, order = rows[cid]['sha'], rows[cid]['order']
    return [by_pos[(sha, o)] for o in range(order - context, order + context + 1) if (sha, o) in by_pos]


def build_windows(order, docs, n_cover, context, budget, rows, by_pos, docs_of):
    """[(centre, window chunk ids)] for one query until the budget is used.

    Passes: the best chunk of each of the top n_cover documents, then the chunks in order with one window per
    text, then the chunks in order again (second windows). A centre inside a chosen window is skipped.
    """
    best = {}
    for cid in order:
        for doc in docs_of(cid):
            best.setdefault(doc, cid)
    head = list(dict.fromkeys(best[d] for d in docs[:n_cover] if d in best))
    chosen, covered, used = [], {}, 0
    for one_per_text, candidates in ((True, head), (True, order), (False, order)):
        for cid in candidates:
            sha, o = rows[cid]['sha'], rows[cid]['order']
            if o in covered.get(sha, ()) or (one_per_text and sha in covered):
                continue
            ids = window_ids(cid, context, rows, by_pos)
            cost = len(merge_run([rows[c] for c in ids]).encode('utf-8')) * len(docs_of(cid))
            if used + cost > budget:
                continue   # a long or wide-fanout window does not fit; smaller ones further down may
            chosen.append((cid, ids))
            used += cost
            covered.setdefault(sha, set()).update(range(o - context, o + context + 1))
    return chosen


def conf_doc_count(q_scores, column, threshold, lo, hi):
    """Documents with reranker score >= threshold, clipped to [lo, hi]."""
    return min(max(sum(s >= threshold for s in q_scores.get(column, {}).values()), lo), hi)


def chosen_bytes(chosen, rows, docs_of):
    return sum(len(rows[c]['text'].encode('utf-8')) * len(docs_of(c)) for c in chosen)


def score_cut_budget(budget, ctx, n_neighbour, min_score):
    """Budgets after the score cut: what cut queries leave unused goes to the queries the cut did not stop,
    in proportion to their budgets."""
    used, full = {}, []
    for qid, b in budget.items():
        args = (ctx['ranked'][qid], n_neighbour, b, ctx['rows'], ctx['by_pos'], ctx['docs_of'])
        cut = chosen_bytes(build_query(*args, scores=ctx['score'][qid], min_score=min_score), ctx['rows'], ctx['docs_of'])
        if cut < chosen_bytes(build_query(*args), ctx['rows'], ctx['docs_of']):
            used[qid] = cut
        else:
            full.append(qid)
    freed = sum(budget[q] - used[q] for q in used)
    grow = 1 + freed / max(sum(budget[q] for q in full), 1)
    print(f'score cut {min_score}: {len(used)} queries cut, {freed / 1e6:.1f} MB moved to {len(full)} queries', flush=True)
    return {qid: used[qid] if qid in used else budget[qid] * grow for qid in budget}


def build_variants(ctx, only=None, doc_scores=None):
    ranked, base, rows, by_pos, by_text, docs_of, lang_of = (
        ctx[k] for k in ('ranked', 'base', 'rows', 'by_pos', 'by_text', 'docs_of', 'lang_of'))
    base_bytes = {e['id']: sum(len(c['chunk_text'].encode('utf-8')) for c in e['relevant_chunks'])
                  for e in base}
    only = ONLY if only is None else only
    manifest = {'budget_scale': BUDGET_SCALE, 'variants': {}}
    for name, spec in VARIANTS.items():
        if only and name not in only:
            continue
        if spec.get('win') is not None and ctx.get('rrf') is None:
            print(name, 'skipped: pool.parquet has no rrf column', flush=True)
            continue
        uses_bm25w = 'bm25w' in spec.get('doc_rr', {}) or spec.get('w_bm25w')
        if uses_bm25w and not ctx.get('bm25w'):
            print(name, 'skipped: bm25w-chunks-k500.npz not attached', flush=True)
            continue
        if spec.get('doc_rr'):
            doc_scores = doc_scores or load_doc_scores()
            some = next(iter(doc_scores.values()))
            absent = [c for c in spec['doc_rr'] if c not in some and c != 'bm25w']
            if absent:
                print(name, 'skipped: no doc_rank column', absent, flush=True)
                continue
        n_neighbour = spec['nb']
        budget = {qid: b * spec.get('budget_scale', BUDGET_SCALE) for qid, b in base_bytes.items()}
        q_budget = (confidence_budget(budget, ctx['score'], spec['conf_gamma'], spec.get('conf_t', 0.5))
                    if spec.get('conf_gamma') else budget)
        if spec.get('min_score'):
            q_budget = score_cut_budget(q_budget, ctx, n_neighbour, spec['min_score'])
        scale = 1.0
        while True:
            submission, n_spans, n_chunks, n_docs, other_docs, other_bytes, all_bytes, moved = [], 0, 0, 0, 0, 0, 0, 0
            dropped = 0
            for entry in base:
                qid = entry['id']
                if spec.get('doc_rr'):
                    q_scores = doc_scores.get(qid, {})
                    if 'bm25w' in spec['doc_rr']:
                        q_scores = dict(q_scores, bm25w={})
                        for sha, s in zip(*ctx['bm25w']['texts'].get(qid, ([], []))):
                            for doc in ctx['text_docs'](sha):
                                q_scores['bm25w'].setdefault(doc, s)
                    candidates = list(dict.fromkeys(
                        doc_list(entry['relevant_docs'], ranked[qid], docs_of, lang_of, n_docs=10 ** 6)
                        + [d for c in spec['doc_rr'] for d in q_scores.get(c, {})]))
                    keep = spec.get('n_docs') or len(entry['relevant_docs'])
                    if spec.get('doc_conf'):
                        keep = conf_doc_count(q_scores, next(iter(spec['doc_rr'])), *spec['doc_conf'])
                    fixed = candidates[:spec.get('keep_top', 0)]
                    docs = fixed + fuse_docs(candidates[len(fixed):], q_scores, spec['doc_rr'], spec.get('w_base', 1.0),
                                             offset=len(fixed))[:keep - len(fixed)]
                    moved += len(set(docs) - set(entry['relevant_docs']))
                else:
                    docs = doc_list(entry['relevant_docs'], ranked[qid], docs_of, lang_of,
                                    spec.get('n_docs', 0), spec.get('doc_quota', 0.0))
                windows = None
                if spec.get('win') is not None:
                    if spec.get('w_bm25w'):
                        ids, bge, rrf, bm = with_bm25w(ranked[qid], ctx['score'][qid], ctx['rrf'][qid],
                                                       ctx['bm25w']['chunks'].get(qid, ([], [])))
                        weights = [spec['w_bge'], 1 - spec['w_bge'] - spec['w_bm25w'], spec['w_bm25w']]
                        order = blended_order(ids, [bge, rrf, bm], weights)
                    else:
                        order = blended_order(ranked[qid], [ctx['score'][qid], ctx['rrf'][qid]],
                                              [spec['w_bge'], 1 - spec['w_bge']])
                    windows = build_windows(order, docs, spec.get('cover_top', 0), spec['win'], q_budget[qid] * scale,
                                            rows, by_pos, docs_of)
                    chosen = list(dict.fromkeys(c for _, ids in windows for c in ids))
                elif n_neighbour == 'whole':
                    chosen = build_whole_docs(ranked[qid], entry['relevant_docs'], q_budget[qid] * scale,
                                              rows, by_text, docs_of)
                elif spec.get('cover'):
                    order = cover_order(ranked[qid], n_neighbour, docs, *spec['cover'], docs_of)
                    chosen = build_query(order, n_neighbour, q_budget[qid] * scale, rows, by_pos, docs_of)
                else:
                    chosen = build_query(ranked[qid], n_neighbour, q_budget[qid] * scale, rows, by_pos, docs_of,
                                         spec.get('quota', 0.0), ctx['score'][qid], spec.get('min_score', 0.0))
                for c in chosen:
                    size = len(rows[c]['text'].encode('utf-8'))
                    all_bytes += size
                    other_bytes += size if rows[c]['lang'] != 'vi' else 0
                listed = set(docs)
                chunks = []
                entries = ([(cid, merge_run([rows[c] for c in ids])) for cid, ids in windows] if windows is not None
                           else spans(chosen, rows))
                for cid, text in entries:
                    if len(text) < spec.get('min_chars', 0):
                        dropped += 1
                        continue
                    for doc in docs_of(cid):
                        chunks.append(dict(doc_id=doc, chunk_text=text, chunk_order=rows[cid]['order']))
                        if doc not in listed:
                            listed.add(doc)
                            docs.append(doc)
                n_spans += len(chunks)
                n_docs += len(docs)
                other_docs += sum(lang_of.get(d, 'vi') != 'vi' for d in docs) / max(len(docs), 1)
                n_chunks += len(chosen)
                submission.append(dict(id=qid, relevant_docs=docs, relevant_chunks=chunks))
            zip_path = OUT / f'medxir-expand-{name}-submission.zip'
            json_name = zip_path.name.replace('.zip', '.json')
            method = zipfile.ZIP_BZIP2 if spec.get('zip') == 'bzip2' else zipfile.ZIP_DEFLATED
            with zipfile.ZipFile(zip_path, 'w', compression=method, compresslevel=9) as archive:
                archive.writestr(json_name, json.dumps(submission, ensure_ascii=False, separators=(',', ':')))
            mb = zip_path.stat().st_size / 1e6
            if mb <= MAX_ZIP_MB:
                break
            scale *= 0.95 * MAX_ZIP_MB / mb
        stats = dict(spec, zip_mb=round(mb, 1), budget_scale_applied=round(scale, 3),
                     chunks_per_query=round(n_chunks / len(base), 1), entries_per_query=round(n_spans / len(base), 1),
                     docs_per_query=round(n_docs / len(base), 1), short_dropped_per_query=round(dropped / len(base), 1),
                     zh_en_doc_share=round(other_docs / len(base), 3),
                     zh_en_chunk_byte_share=round(other_bytes / max(all_bytes, 1), 3))
        if spec.get('doc_rr'):
            stats['new_docs_per_query'] = round(moved / len(base), 1)   # documents not in the submitted list
        manifest['variants'][name] = stats
        print(name, stats, flush=True)
    (OUT / 'expand_manifest.json').write_text(json.dumps(manifest, indent=2))
    return manifest


def main():
    build_variants(load_inputs())
'''
EXPAND = types.ModuleType('expand_chunks')
exec(compile(EXPAND_SOURCE, 'expand_chunks.py', 'exec'), EXPAND.__dict__)

In [ ]:
if FAILED is None:
    with stage('expand_inputs'):
        # Every chunk of the pool texts (neighbours need start/end/order), read again from the validated parts.
        EXPAND_ON = not QUERY_LIMIT and (EXPAND_NB > 0 or DOC_RERANK != 'off')
        if not EXPAND_ON:
            print('no extra ZIPs (EXPAND_NB = 0 and DOC_RERANK off, or QUERY_LIMIT set)')
        else:
            started = time.perf_counter()
            EXPAND.OUT = OUT
            CTX = EXPAND.load_inputs(pool_path=OUT / 'pool.parquet', base=submission, chunk_files=CHUNK_FILES)
            TIMINGS['expand_inputs_s'] = round(time.perf_counter() - started, 1)
            note(f'expand inputs: {len(CTX["rows"]):,} chunk rows')

In [ ]:
if FAILED is None:
    with stage('doc_rerank'):
        # Candidate documents per query: the submitted N_DOCS, then the other pool documents (best chunk first), up to
        # DOC_RERANK_DEPTH. bge scores (query, document lead + best pool chunk); fan-out documents share a text: scored once.
        DOC_SCORES = None
        if EXPAND_ON and DOC_RERANK != 'off':
            from transformers import AutoModelForSequenceClassification, AutoTokenizer

            started = time.perf_counter()
            keys, cand_rows, text_lang = {}, [], {}   # (qi, text) -> documents sharing it / language of the text
            for qi, entry in enumerate(submission):
                ranked = CTX['ranked'][entry['id']]
                best = EXPAND.best_chunks(ranked, CTX['docs_of'])
                cands = EXPAND.doc_list(entry['relevant_docs'], ranked, CTX['docs_of'], CTX['lang_of'], n_docs=10 ** 6)
                for rank, doc in enumerate(cands[:DOC_RERANK_DEPTH]):
                    cand_rows.append((entry['id'], doc, rank))
                    if doc in best:   # documents filled from beyond the pool have no chunk text: they keep their rank
                        text = EXPAND.doc_text(best[doc], CTX['rows'], CTX['by_text'], DOC_LEAD_CHARS)
                        keys.setdefault((qi, text), []).append(doc)
                        text_lang[(qi, text)] = CTX['rows'][best[doc][0]]['lang']
            pairs = list(keys)
            print('example document text:', pairs[0][1][:600].replace('\n', ' | '))
            # (query text, document text) to score: VI query for every document, then (RERANK_SAME_LANG) the query
            # translated to the language of each EN/ZH document; a document keeps the better score.
            scored = [(QUERY_TEXT[qi], text) for qi, text in pairs]
            same_lang_of = []
            if USE_TRANSLATION and RERANK_SAME_LANG:
                for i, (qi, text) in enumerate(pairs):
                    translated = TRANS_TEXT[qi].get(text_lang[(qi, text)])
                    if translated:
                        scored.append((translated, text))
                        same_lang_of.append(i)
            TIMINGS['doc_same_lang_pairs'] = len(same_lang_of)

            tok = AutoTokenizer.from_pretrained(str(DOC_RERANK_PATH))
            model = AutoModelForSequenceClassification.from_pretrained(str(DOC_RERANK_PATH), torch_dtype=torch.bfloat16).cuda().eval()
            order = np.argsort([len(t) for _, t in scored], kind='stable')   # similar lengths per batch: less padding
            values = np.empty(len(scored), dtype=np.float32)
            note(f'document rerank: {len(cand_rows):,} candidates, {len(pairs):,} texts, {len(scored):,} pairs')
            t0, next_report = time.perf_counter(), 0.05
            with torch.inference_mode():
                for start in range(0, len(order), 64):
                    index = order[start:start + 64]
                    enc = tok([scored[i][0] for i in index], [scored[i][1] for i in index], padding=True,
                              truncation=True, max_length=512, return_tensors='pt').to('cuda')
                    values[index] = torch.sigmoid(model(**enc).logits.view(-1).float()).cpu().numpy()
                    done = (start + len(index)) / len(order)
                    if done >= next_report:
                        rate = (start + len(index)) / (time.perf_counter() - t0)
                        note(f'document rerank {done:.0%} | {rate:.0f} pairs/s | ETA {(len(order) - start) / rate / 60:.0f} min')
                        next_report += 0.05
            del model, tok
            torch.cuda.empty_cache()
            if same_lang_of:
                np.maximum.at(values, np.array(same_lang_of), values[len(pairs):])
            values = values[:len(pairs)]

            score = {(QUERY_IDS[qi], doc): float(v) for (qi, text), v in zip(pairs, values) for doc in keys[(qi, text)]}
            DOC_SCORES = {qid: {'bge-reranker-v2-m3': {}} for qid in QUERY_IDS}
            for (qid, doc), v in score.items():
                DOC_SCORES[qid]['bge-reranker-v2-m3'][doc] = v
            pd.DataFrame([(qid, doc, rank, score.get((qid, doc), np.nan)) for qid, doc, rank in cand_rows],
                         columns=['query_id', 'doc_id', 'base_rank', 'bge-reranker-v2-m3']) \
              .to_parquet(OUT / 'doc_rank.parquet', index=False, compression='zstd')
            TIMINGS['doc_rerank_s'] = round(time.perf_counter() - started, 1)

In [ ]:
if FAILED is None:
    with stage('expand'):
        # One ZIP per document ordering, all with the nb chunk spans; documents re-ordered only with DOC_RERANK.
        if EXPAND_ON:
            orderings = {'off': [None], 'fusion': [('drr-bge', 1.0)], 'only': [('drr-bge-only', 0.0)],
                         'both': [('drr-bge', 1.0), ('drr-bge-only', 0.0)]}[DOC_RERANK]
            names = []
            for ordering in orderings:
                name, spec = f'nb{EXPAND_NB}-d{N_DOCS}', dict(nb=EXPAND_NB)
                if ordering:
                    name += '-' + ordering[0]
                    spec.update(doc_rr={'bge-reranker-v2-m3': 1.0}, w_base=ordering[1])
                EXPAND.VARIANTS[name] = spec
                names.append(name)
            expand_manifest = EXPAND.build_variants(CTX, only=names, doc_scores=DOC_SCORES)
            expand_manifest.update(n_docs=N_DOCS, doc_rerank=DOC_RERANK, doc_rerank_depth=DOC_RERANK_DEPTH,
                                   doc_lead_chars=DOC_LEAD_CHARS, timings=TIMINGS, elapsed_s=round(time.perf_counter() - T0, 1))
            (OUT / 'expand_manifest.json').write_text(json.dumps(expand_manifest, indent=2) + '\n')
            for name in names:
                note(f'medxir-expand-{name}-submission.zip: {expand_manifest["variants"][name]}')

## 12. Report

In [ ]:
note('finished; status ' + ('error' if FAILED else 'ok'))
if FAILED:
    (OUT / 'stage_status.json').write_text(json.dumps(
        dict(status='error', failed_stage=FAILED, error_file='retrieval_error.txt'), indent=2) + '\n')